# Original code order, outputs removed

All 153 non-empty original code cells are present in original order. The two empty original cells (1 and 2) were omitted. This is the authoritative source-order record for comparing the split notebooks. The original uploaded .ipynb file remains unchanged outside this repository.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import sys
import os
import json
import platform
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata

import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

folders = [
    "data/metadata",
    "data/processed",
    "features/acoustic",
    "features/linguistic",
    "models",
    "results/tables",
    "results/figures",
    "reports",
    "checkpoints",
    "paper",
]

for folder in folders:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

packages = [
    "numpy", "pandas", "scipy", "scikit-learn",
    "librosa", "soundfile", "torch",
    "transformers", "jiwer", "parselmouth"
]

versions = {}

for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "NOT INSTALLED"

info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": [
        torch.cuda.get_device_name(i)
        for i in range(torch.cuda.device_count())
    ] if torch.cuda.is_available() else [],
    "packages": versions,
}

version_file = PROJECT / "reports/environment_versions.txt"
version_file.write_text(
    json.dumps(info, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress_file = PROJECT / "checkpoints/research_progress.json"

if not progress_file.exists():
    progress = {
        "project": "ASR Fluency Robustness",
        "current_step": 1,
        "completed_steps": [0],
        "dataset": "Speechocean762",
        "total_audio_files": None,
        "processed_audio_files": 0,
        "failed_audio_files": 0,
        "random_seed": 42,
        "next_task": "Verify Kaggle dataset access",
        "updated_at": datetime.now(timezone.utc).isoformat()
    }
    progress_file.write_text(
        json.dumps(progress, indent=2),
        encoding="utf-8"
    )

print("=== ENVIRONMENT ===")
print(json.dumps(info, indent=2))

print("\n=== INPUT DATASETS ===")
input_dir = Path("/kaggle/input")

if input_dir.exists():
    print([p.name for p in input_dir.iterdir()])
else:
    print("Kaggle input directory not found")

print("\n=== PROJECT ===")
print(PROJECT)
print("Environment and checkpoint files created.")


In [ ]:

import urllib.request
import json

url = (
    "https://huggingface.co/api/datasets/"
    "mispeech/speechocean762"
)

try:
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "research-dataset-check/1.0"}
    )

    with urllib.request.urlopen(request, timeout=30) as response:
        data = json.load(response)

    print("=== DATASET CONNECTION ===")
    print("Status: SUCCESS")
    print("Dataset:", data.get("id"))
    print("Private:", data.get("private"))
    print("Revision:", data.get("sha"))

    info = data.get("cardData") or {}
    print("License:", info.get("license"))

    print("\n=== DATASET FILES ===")
    for item in data.get("siblings", []):
        filename = item.get("rfilename", "")
        if filename.endswith((".parquet", ".json", ".md")):
            print(filename)

except Exception as e:
    print("Status: FAILED")
    print("Error:", type(e).__name__, str(e))


In [ ]:

import io
import urllib.request
import pyarrow.parquet as pq

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

url = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

print("=== PARQUET SCHEMA CHECK ===")

try:
    request = urllib.request.Request(
        url,
        headers={"Range": "bytes=0-65535"}
    )

    with urllib.request.urlopen(request, timeout=60) as response:
        data = response.read()
        status = response.status

    print("HTTP status:", status)
    print("Bytes downloaded:", len(data))

    if status == 206:
        print("Partial download supported.")
        print("First bytes:", data[:4])
        print(
            "Full Parquet schema requires the footer. "
            "We will use a different metadata method."
        )
    else:
        print("Server returned full file.")
        pf = pq.ParquetFile(io.BytesIO(data))

        print("Total rows:", pf.metadata.num_rows)
        print("Row groups:", pf.metadata.num_row_groups)
        print("\n=== COLUMNS ===")
        print(pf.schema_arrow)

except Exception as e:
    print("Error:", type(e).__name__, str(e))


In [ ]:

import pyarrow.parquet as pq
import pyarrow.fs as pafs

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

url = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

print("=== REMOTE PARQUET METADATA ===")

try:
    filesystem, path = pafs.FileSystem.from_uri(url)

    parquet_file = pq.ParquetFile(
        path,
        filesystem=filesystem
    )

    print("Total rows:", parquet_file.metadata.num_rows)
    print("Row groups:", parquet_file.metadata.num_row_groups)

    print("\n=== COLUMNS ===")
    print(parquet_file.schema_arrow)

except Exception as e:
    print("FAILED:", type(e).__name__)
    print(str(e))


In [ ]:

import urllib.request
import struct
import pyarrow.parquet as pq
import pyarrow as pa

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

def read_range(start, end):
    req = urllib.request.Request(
        URL,
        headers={
            "Range": f"bytes={start}-{end}",
            "Accept-Encoding": "identity"
        }
    )

    with urllib.request.urlopen(req, timeout=60) as response:
        if response.status != 206:
            raise RuntimeError(
                f"Expected HTTP 206, got {response.status}"
            )
        return response.read()

print("=== PARQUET FOOTER CHECK ===")

# 1. Get the last 8 bytes and total file size
req = urllib.request.Request(
    URL,
    headers={
        "Range": "bytes=-8",
        "Accept-Encoding": "identity"
    }
)

with urllib.request.urlopen(req, timeout=60) as response:
    if response.status != 206:
        raise RuntimeError("HTTP Range not supported")

    footer = response.read()
    content_range = response.headers["Content-Range"]

file_size = int(content_range.split("/")[-1])

print("File size:", file_size)
print("Footer bytes:", footer)

if len(footer) != 8 or footer[-4:] != b"PAR1":
    raise ValueError("Invalid Parquet footer")

metadata_length = struct.unpack("<I", footer[:4])[0]

print("Metadata length:", metadata_length)

# 2. Fetch metadata bytes
metadata_start = file_size - 8 - metadata_length

metadata_bytes = read_range(
    metadata_start,
    file_size - 9
)

# 3. Construct a minimal Parquet file
minimal_parquet = (
    b"PAR1"
    + metadata_bytes
    + footer
)

# Metadata can contain row-group offsets referring
# to the original file. Read only schema/metadata.
try:
    reader = pq.ParquetFile(
        pa.BufferReader(minimal_parquet)
    )

    print("\n=== DATASET METADATA ===")
    print("Total rows:", reader.metadata.num_rows)
    print("Row groups:", reader.metadata.num_row_groups)

    print("\n=== COLUMNS ===")
    print(reader.schema_arrow)

except Exception as e:
    print("\nMetadata extraction requires adjustment:")
    print(type(e).__name__, str(e))


In [ ]:

from pathlib import Path
import requests
import time
import os
import json

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

TARGET = RAW / "speechocean762_train.parquet"
PART = RAW / "speechocean762_train.parquet.part"

EXPECTED_SIZE = 332843486

if TARGET.exists() and TARGET.stat().st_size == EXPECTED_SIZE:
    print("Train file already downloaded.")
else:
    for attempt in range(5):
        existing = PART.stat().st_size if PART.exists() else 0

        headers = {}
        if existing:
            headers["Range"] = f"bytes={existing}-"

        try:
            with requests.get(
                URL,
                headers=headers,
                stream=True,
                timeout=(30, 120)
            ) as response:
                response.raise_for_status()

                if existing and response.status == 200:
                    # Server ignored the Range header
                    existing = 0

                mode = "ab" if existing and response.status == 206 else "wb"

                with open(PART, mode) as f:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)

            if PART.stat().st_size == EXPECTED_SIZE:
                os.replace(PART, TARGET)
                print("Download completed.")
                break

            print("Incomplete file. Retrying...")

        except Exception as e:
            print(f"Attempt {attempt + 1}: {e}")
            time.sleep(3)

    else:
        raise RuntimeError("Download incomplete after retries.")

print("=== TRAIN DATASET ===")
print("File:", TARGET)
print("Size:", TARGET.stat().st_size)

# Verify Parquet integrity and row count
import pyarrow.parquet as pq

pf = pq.ParquetFile(TARGET)

print("Rows:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)
print("Columns:", pf.schema_arrow.names)

# Update checkpoint
progress_file = PROJECT / "checkpoints" / "research_progress.json"

progress = json.loads(progress_file.read_text(encoding="utf-8"))
progress["train_downloaded"] = True
progress["train_rows"] = pf.metadata.num_rows
progress["dataset_revision"] = REVISION
progress["next_task"] = "Inspect train metadata and speaker IDs"

progress_file.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nCheckpoint updated.")


In [ ]:

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = (
    PROJECT / "data/raw/speechocean762_train.parquet"
)

# Read metadata only, without audio bytes
columns = [
    "speaker",
    "gender",
    "age",
    "fluency",
    "accuracy",
    "completeness",
    "prosodic",
    "total",
    "text"
]

df = pq.read_table(
    TRAIN,
    columns=columns
).to_pandas()

print("=== BASIC INFORMATION ===")
print("Total rows:", len(df))
print("Unique speakers:", df["speaker"].nunique())
print("Duplicate rows:", df.duplicated().sum())

print("\n=== FLUENCY SCORES ===")
print(df["fluency"].describe())

print("\nScore distribution:")
print(df["fluency"].value_counts().sort_index())

print("\n=== SPEAKER DISTRIBUTION ===")
print(df.groupby("speaker").size().describe())

print("\n=== GENDER ===")
print(df["gender"].value_counts(dropna=False))

print("\n=== AGE ===")
print(df["age"].describe())

print("\n=== MISSING VALUES ===")
print(df.isna().sum())

print("\n=== SAMPLE RECORDS ===")
print(
    df[
        ["speaker", "fluency", "text"]
    ].head(5).to_string(index=False)
)

# Save compact initial metadata
output = PROJECT / "data/metadata/train_metadata_initial.csv"

df.to_csv(
    output,
    index=False,
    encoding="utf-8"
)

print("\nMetadata saved:", output)


In [ ]:

import io
from pathlib import Path
import pyarrow.parquet as pq
import soundfile as sf
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"

# Read only first row group and audio column
pf = pq.ParquetFile(TRAIN)

table = pf.read_row_group(
    0,
    columns=["audio"]
)

audio_values = table.column("audio").to_pylist()

print("=== AUDIO INSPECTION ===")
print("Row group audio count:", len(audio_values))

records = []

for i, audio in enumerate(audio_values[:5], start=1):

    audio_bytes = audio.get("bytes")
    audio_path = audio.get("path")

    if not audio_bytes:
        print(f"Audio {i}: missing bytes")
        continue

    with sf.SoundFile(io.BytesIO(audio_bytes)) as f:
        sample_rate = f.samplerate
        channels = f.channels
        frames = f.frames
        duration = frames / sample_rate
        audio_format = f.format
        subtype = f.subtype

    records.append({
        "row": i,
        "path": audio_path,
        "sample_rate": sample_rate,
        "channels": channels,
        "duration_sec": round(duration, 3),
        "format": audio_format,
        "subtype": subtype
    })

result = pd.DataFrame(records)

print(result.to_string(index=False))

output = PROJECT / "data/metadata/audio_inspection_5.csv"
result.to_csv(output, index=False)

print("\nSaved:", output)


In [ ]:

import io
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"

OUT = PROJECT / "data/metadata/train_audio_technical_metadata.csv"
ERRORS = PROJECT / "data/metadata/train_audio_errors.csv"
PROGRESS = PROJECT / "checkpoints/research_progress.json"

pf = pq.ParquetFile(TRAIN)

records = []
errors = []
global_row = 0

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(group_id, columns=["audio"])
    audios = table.column("audio").to_pylist()

    for audio in audios:
        global_row += 1

        try:
            audio_bytes = audio.get("bytes")
            audio_path = audio.get("path")

            if not audio_bytes:
                raise ValueError("Missing audio bytes")

            with sf.SoundFile(io.BytesIO(audio_bytes)) as f:
                records.append({
                    "row": global_row,
                    "audio_id": Path(audio_path).stem,
                    "audio_path": audio_path,
                    "sample_rate": f.samplerate,
                    "channels": f.channels,
                    "duration_sec": round(f.frames / f.samplerate, 4),
                    "format": f.format,
                    "subtype": f.subtype
                })

        except Exception as e:
            errors.append({
                "row": global_row,
                "error": str(e)
            })

    print(
        f"Row group {group_id + 1}/"
        f"{pf.metadata.num_row_groups} completed."
    )

    # Checkpoint after each row group
    pd.DataFrame(records).to_csv(OUT, index=False)
    pd.DataFrame(errors, columns=["row", "error"]).to_csv(
        ERRORS, index=False
    )

df_audio = pd.DataFrame(records)

print("\n=== FULL AUDIO VALIDATION ===")
print("Total examined:", global_row)
print("Successful:", len(records))
print("Failed:", len(errors))

print("\nSample rate distribution:")
print(df_audio["sample_rate"].value_counts())

print("\nChannel distribution:")
print(df_audio["channels"].value_counts())

print("\nFormat distribution:")
print(df_audio["format"].value_counts())

print("\nDuration statistics:")
print(df_audio["duration_sec"].describe())

print("\nUnique audio IDs:", df_audio["audio_id"].nunique())
print("Duplicate audio IDs:", df_audio["audio_id"].duplicated().sum())

progress = json.loads(PROGRESS.read_text(encoding="utf-8"))

progress["audio_technical_validation"] = {
    "examined": global_row,
    "successful": len(records),
    "failed": len(errors),
    "metadata_file": str(OUT),
    "error_file": str(ERRORS)
}
progress["next_task"] = "Validate metadata joins and train/test strategy"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

PROGRESS.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nFiles saved:")
print(OUT)
print(ERRORS)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

META = PROJECT / "data/metadata/train_metadata_initial.csv"
AUDIO = PROJECT / "data/metadata/train_audio_technical_metadata.csv"
OUTPUT = PROJECT / "data/metadata/dataset_metadata_train.csv"
PROGRESS = PROJECT / "checkpoints/research_progress.json"

df_meta = pd.read_csv(
    META,
    dtype={"speaker": "string"}
)

df_audio = pd.read_csv(
    AUDIO,
    dtype={"audio_id": "string", "audio_path": "string"}
)

# Original Parquet row order is preserved in both CSVs.
df_meta.insert(0, "row", range(1, len(df_meta) + 1))

assert len(df_meta) == 2500
assert len(df_audio) == 2500
assert df_meta["row"].is_unique
assert df_audio["row"].is_unique
assert df_audio["audio_id"].is_unique
assert df_audio["row"].between(1, len(df_meta)).all()

df = df_meta.merge(
    df_audio,
    on="row",
    how="inner",
    validate="one_to_one"
)

assert len(df) == 2500
assert df["fluency"].notna().all()
assert df["speaker"].notna().all()
assert df["audio_id"].notna().all()

df.insert(1, "split", "train")

# Check the audio ID against the speaker ID
# before relying on this relationship.
df["audio_speaker_prefix"] = df["audio_id"].str[:4]

prefix_matches = (
    df["audio_speaker_prefix"] == df["speaker"]
)

print("=== METADATA JOIN VALIDATION ===")
print("Total rows:", len(df))
print("Unique audio IDs:", df["audio_id"].nunique())
print("Unique speakers:", df["speaker"].nunique())
print("Missing fluency:", df["fluency"].isna().sum())
print("Speaker-prefix matches:", prefix_matches.sum())
print("Speaker-prefix mismatches:", (~prefix_matches).sum())

print("\n=== SAMPLE ===")
print(
    df[
        [
            "row", "audio_id", "speaker",
            "fluency", "duration_sec", "text"
        ]
    ].head(5).to_string(index=False)
)

# Keep speaker IDs as strings to preserve leading zeros.
df.to_csv(OUTPUT, index=False, encoding="utf-8")

progress = json.loads(PROGRESS.read_text(encoding="utf-8"))
progress["train_metadata_file"] = str(OUTPUT)
progress["train_metadata_rows"] = len(df)
progress["next_task"] = "Validate train speaker identities and test split"
PROGRESS.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nSaved:", OUTPUT)
print("Checkpoint updated.")


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
FILE = PROJECT / "data/metadata/dataset_metadata_train.csv"

df = pd.read_csv(
    FILE,
    dtype={"speaker": "string", "audio_id": "string"}
)

print("=== AUDIO ID STRUCTURE ===")
print("Audio ID lengths:")
print(df["audio_id"].str.len().value_counts().sort_index())

print("\nFirst 15 examples:")
print(
    df[["audio_id", "speaker"]]
    .head(15)
    .to_string(index=False)
)

print("\n=== POSITION MATCH ANALYSIS ===")

for start in range(6):
    extracted = df["audio_id"].str.slice(start, start + 4)
    matches = (extracted == df["speaker"]).sum()

    print(
        f"Positions {start}:{start+4} "
        f"=> matches: {matches}/2500"
    )

print("\n=== SPEAKER EXAMPLES ===")
sample_speakers = df["speaker"].drop_duplicates().head(5)

for speaker in sample_speakers:
    values = (
        df.loc[df["speaker"] == speaker, "audio_id"]
        .head(5)
        .tolist()
    )
    print(f"Speaker {speaker}: {values}")

print("\n=== BASIC VALIDATION ===")
print("Unique audio IDs:", df["audio_id"].nunique())
print("Unique speakers:", df["speaker"].nunique())
print("Duplicate audio IDs:", df["audio_id"].duplicated().sum())


In [ ]:

from pathlib import Path

print("=== KAGGLE FILE RECOVERY CHECK ===")

locations = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/kaggle/working/acoustic_linguistic_research"),
]

for folder in locations:
    print(f"\nDirectory: {folder}")

    if not folder.exists():
        print("NOT FOUND")
        continue

    files = list(folder.iterdir())

    if not files:
        print("EMPTY")
    else:
        for item in files[:30]:
            if item.is_file():
                print(
                    "FILE:",
                    item.name,
                    "| Size:",
                    round(item.stat().st_size / 1024 / 1024, 2),
                    "MB"
                )
            else:
                print("DIR:", item.name)

print("\n=== IMPORTANT FILES ===")

project = Path(
    "/kaggle/working/acoustic_linguistic_research"
)

important_files = [
    "data/raw/speechocean762_train.parquet",
    "data/metadata/train_metadata_initial.csv",
    "data/metadata/train_audio_technical_metadata.csv",
    "data/metadata/dataset_metadata_train.csv",
    "checkpoints/research_progress.json",
]

for filename in important_files:
    path = project / filename
    print(
        "FOUND" if path.exists() else "MISSING",
        filename
    )


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import io
import json
import os
import zipfile

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data/raw"
META = PROJECT / "data/metadata"
CHECK = PROJECT / "checkpoints"

for folder in [RAW, META, CHECK]:
    folder.mkdir(parents=True, exist_ok=True)

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"
EXPECTED_SIZE = 332843486

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

TARGET = RAW / "speechocean762_train.parquet"
PART = RAW / "speechocean762_train.parquet.part"

# Download with resume support
if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:
    for attempt in range(5):
        existing = PART.stat().st_size if PART.exists() else 0
        headers = {"Range": f"bytes={existing}-"} if existing else {}

        try:
            with requests.get(
                URL, headers=headers, stream=True,
                timeout=(30, 120)
            ) as response:
                response.raise_for_status()

                resume = existing > 0 and response.status == 206
                mode = "ab" if resume else "wb"

                with open(PART, mode) as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            if PART.stat().st_size == EXPECTED_SIZE:
                os.replace(PART, TARGET)
                break

        except Exception as e:
            print("Retry:", attempt + 1, str(e))

    if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:
        raise RuntimeError("Train dataset download failed")

print("Train Parquet restored.")

pf = pq.ParquetFile(TARGET)
assert pf.metadata.num_rows == 2500

# Read non-audio metadata
columns = [
    "speaker", "gender", "age", "fluency",
    "accuracy", "completeness", "prosodic",
    "total", "text"
]

df_meta = pq.read_table(
    TARGET, columns=columns
).to_pandas()

df_meta.insert(0, "row", range(1, len(df_meta) + 1))
df_meta["speaker"] = df_meta["speaker"].astype("string")

# Restore audio metadata
records = []
row_number = 0

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        row_number += 1
        audio_bytes = audio["bytes"]
        audio_path = audio["path"]

        if not audio_bytes:
            raise ValueError(f"Missing audio at row {row_number}")

        with sf.SoundFile(io.BytesIO(audio_bytes)) as f:
            records.append({
                "row": row_number,
                "audio_id": Path(audio_path).stem,
                "audio_path": audio_path,
                "sample_rate": f.samplerate,
                "channels": f.channels,
                "duration_sec": round(f.frames / f.samplerate, 4),
                "format": f.format,
                "subtype": f.subtype
            })

    print(f"Audio metadata: group {group_id + 1}/{pf.metadata.num_row_groups}")

df_audio = pd.DataFrame(records)

df = df_meta.merge(
    df_audio, on="row",
    how="inner", validate="one_to_one"
)
df.insert(1, "split", "train")

assert len(df) == 2500
assert df["audio_id"].is_unique
assert df["speaker"].nunique() == 125
assert df["fluency"].notna().all()
assert (df["sample_rate"] == 16000).all()
assert (df["channels"] == 1).all()

df_meta.to_csv(META / "train_metadata_initial.csv", index=False)
df_audio.to_csv(META / "train_audio_technical_metadata.csv", index=False)
df.to_csv(META / "dataset_metadata_train.csv", index=False)

progress = {
    "project": "ASR Fluency Robustness",
    "current_step": 1,
    "completed_steps": [0],
    "dataset": "Speechocean762",
    "dataset_revision": REVISION,
    "train_rows": len(df),
    "train_speakers": int(df["speaker"].nunique()),
    "audio_validation_successful": len(df_audio),
    "audio_validation_failed": 0,
    "random_seed": 42,
    "next_task": "Investigate audio ID and speaker mapping",
    "updated_at": datetime.now(timezone.utc).isoformat()
}

(CHECK / "research_progress.json").write_text(
    json.dumps(progress, indent=2),
    encoding="utf-8"
)

# Backup metadata and checkpoint (not audio)
ZIP = Path("/kaggle/working/asr_fluency_step1_backup.zip")

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [META, CHECK]:
        for file in folder.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(PROJECT))

print("\n=== RECOVERY COMPLETE ===")
print("Train rows:", len(df))
print("Speakers:", df["speaker"].nunique())
print("Unique audio IDs:", df["audio_id"].nunique())
print("Metadata restored:", (META / "dataset_metadata_train.csv").exists())
print("Checkpoint restored:", (CHECK / "research_progress.json").exists())
print("Backup ZIP:", ZIP)
print("Backup size (MB):", round(ZIP.stat().st_size / 1048576, 2))


In [ ]:

import pandas as pd
from pathlib import Path
import json
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
FILE = PROJECT / "data/metadata/dataset_metadata_train.csv"

df = pd.read_csv(
    FILE,
    dtype={"speaker": "string", "audio_id": "string"}
)

print("=== AUDIO ID STRUCTURE ===")
print("Audio ID lengths:")
print(df["audio_id"].str.len().value_counts().sort_index())

print("\nFirst 15 examples:")
print(
    df[["audio_id", "speaker"]]
    .head(15)
    .to_string(index=False)
)

print("\n=== POSITION MATCH ANALYSIS ===")

for start in range(6):
    extracted = df["audio_id"].str.slice(start, start + 4)
    matches = (extracted == df["speaker"]).sum()

    print(
        f"Positions {start}:{start + 4} "
        f"=> matches: {matches}/{len(df)}"
    )

print("\n=== SPEAKER EXAMPLES ===")

for speaker in df["speaker"].drop_duplicates().head(5):
    audio_ids = df.loc[
        df["speaker"] == speaker, "audio_id"
    ].head(5).tolist()

    print(f"Speaker {speaker}: {audio_ids}")

print("\n=== VALIDATION ===")
print("Rows:", len(df))
print("Unique audio IDs:", df["audio_id"].nunique())
print("Unique speakers:", df["speaker"].nunique())
print("Duplicate audio IDs:", df["audio_id"].duplicated().sum())

# Save the actual inspection results
report = PROJECT / "reports/audio_speaker_mapping_check.txt"
report.parent.mkdir(parents=True, exist_ok=True)

lines = [
    "STEP 1 - Audio Speaker Mapping Check",
    f"Rows: {len(df)}",
    f"Unique audio IDs: {df['audio_id'].nunique()}",
    f"Unique speakers: {df['speaker'].nunique()}",
    "",
    "Position matches:"
]

for start in range(6):
    matches = (
        df["audio_id"].str.slice(start, start + 4)
        == df["speaker"]
    ).sum()
    lines.append(f"{start}:{start + 4} = {matches}")

report.write_text("\n".join(lines), encoding="utf-8")

progress_path = PROJECT / "checkpoints/research_progress.json"
progress = json.loads(progress_path.read_text(encoding="utf-8"))
progress["speaker_mapping_check"] = "completed"
progress["next_task"] = "Verify official test speaker independence"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

progress_path.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nReport saved:", report)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
import requests
import pyarrow.parquet as pq
import pandas as pd
import json
import os
import time

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/test-00000-of-00001.parquet"
)

TARGET = RAW / "speechocean762_test.parquet"
PART = RAW / "speechocean762_test.parquet.part"

# Get expected file size using a 1-byte Range request
response = requests.get(
    URL,
    headers={"Range": "bytes=0-0"},
    timeout=60
)
response.raise_for_status()

if response.status_code != 206:
    raise RuntimeError("HTTP Range response required")

EXPECTED_SIZE = int(
    response.headers["Content-Range"].split("/")[-1]
)

print("Expected test file size:", EXPECTED_SIZE)

# Download with retry and resume
if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:

    for attempt in range(5):
        existing = PART.stat().st_size if PART.exists() else 0

        headers = (
            {"Range": f"bytes={existing}-"}
            if existing else {}
        )

        try:
            with requests.get(
                URL,
                headers=headers,
                stream=True,
                timeout=(30, 120)
            ) as r:
                r.raise_for_status()

                resume = existing > 0 and r.status_code == 206
                mode = "ab" if resume else "wb"

                with open(PART, mode) as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            if PART.stat().st_size == EXPECTED_SIZE:
                os.replace(PART, TARGET)
                print("Test file downloaded.")
                break

            print("Incomplete download, retrying...")

        except Exception as e:
            print("Attempt:", attempt + 1, str(e))
            time.sleep(3)

    if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:
        raise RuntimeError("Test download failed")

# Read speaker column only
test_speakers = pq.read_table(
    TARGET,
    columns=["speaker"]
).to_pandas()["speaker"].astype("string")

train_meta = pd.read_csv(
    PROJECT / "data/metadata/dataset_metadata_train.csv",
    dtype={"speaker": "string"}
)

train_ids = set(train_meta["speaker"].dropna())
test_ids = set(test_speakers.dropna())

overlap = train_ids.intersection(test_ids)

print("\n=== SPEAKER INDEPENDENCE ===")
print("Train speakers:", len(train_ids))
print("Test speakers:", len(test_ids))
print("Shared speakers:", len(overlap))
print("Test rows:", len(test_speakers))
print("Missing test speaker IDs:", test_speakers.isna().sum())

if overlap:
    print("WARNING - overlapping speakers:", sorted(overlap))
else:
    print("PASS - official splits are speaker-disjoint")

assert len(test_speakers) == 2500
assert test_speakers.notna().all()
assert len(overlap) == 0, "Speaker leakage detected!"

# Save verification report
REPORT = PROJECT / "reports/speaker_independence_report.json"
REPORT.parent.mkdir(parents=True, exist_ok=True)

report = {
    "dataset": "Speechocean762",
    "revision": REVISION,
    "train_speakers": len(train_ids),
    "test_speakers": len(test_ids),
    "shared_speakers": sorted(overlap),
    "test_rows": len(test_speakers),
    "speaker_independent": len(overlap) == 0
}

REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

# Update checkpoint
PROGRESS = PROJECT / "checkpoints/research_progress.json"
progress = json.loads(PROGRESS.read_text(encoding="utf-8"))
progress["test_downloaded"] = True
progress["speaker_independence_verified"] = len(overlap) == 0
progress["next_task"] = "Finalize STEP 1 dataset metadata"
PROGRESS.write_text(
    json.dumps(progress, indent=2),
    encoding="utf-8"
)

print("\nReport saved:", REPORT)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import pyarrow.parquet as pq
import importlib.metadata as metadata
import platform
import json
import zipfile
import hashlib
import sys

from IPython.display import display, FileLink

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
TEST = PROJECT / "data/raw/speechocean762_test.parquet"
META = PROJECT / "data/metadata/dataset_metadata_train.csv"
REPORT_DIR = PROJECT / "reports"
CHECK_DIR = PROJECT / "checkpoints"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHECK_DIR.mkdir(parents=True, exist_ok=True)

# Verify files before finalizing STEP 1
assert TRAIN.exists() and TEST.exists() and META.exists()

train_pf = pq.ParquetFile(TRAIN)
test_pf = pq.ParquetFile(TEST)

assert train_pf.metadata.num_rows == 2500
assert test_pf.metadata.num_rows == 2500

df = pd.read_csv(
    META,
    dtype={"speaker": "string", "audio_id": "string"}
)

assert len(df) == 2500
assert df["audio_id"].is_unique
assert df["speaker"].nunique() == 125
assert df["fluency"].notna().all()

test_speakers = set(
    pq.read_table(TEST, columns=["speaker"])
    .to_pandas()["speaker"].astype(str)
)
train_speakers = set(df["speaker"].astype(str))

assert len(train_speakers & test_speakers) == 0

# Record installed package versions
packages = [
    "numpy", "pandas", "pyarrow", "scipy",
    "scikit-learn", "librosa", "soundfile",
    "torch", "transformers", "requests",
    "jiwer", "praat-parselmouth"
]

versions = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {}
}

for package in packages:
    try:
        versions["packages"][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions["packages"][package] = "NOT INSTALLED"

(REPORT_DIR / "environment_versions.txt").write_text(
    json.dumps(versions, indent=2),
    encoding="utf-8"
)

report_text = f"""# STEP 1 — Environment and Dataset Verification

Date (UTC): {datetime.now(timezone.utc).isoformat()}

Dataset: Speechocean762
Dataset revision: 06385584fad212b26134c656fdd3ccf9f093f33e

## Verified results

- Train utterances: 2500
- Test utterances: 2500
- Train speakers: 125
- Test speakers: {len(test_speakers)}
- Train/test shared speakers: 0
- Train unique audio IDs: 2500
- Train missing fluency values: 0
- Train audio technical validation: 2500/2500 successful
- Train audio sample rate: 16000 Hz
- Train audio channels: mono
- Train audio format: WAV
- Train audio mean duration: 4.141644 seconds
- Train fluency mean: 7.7284
- Train fluency SD: 1.555249
- Speaker mapping: audio_id[1:5] matches speaker for all train rows

## Research safeguards

- Official test labels have not been inspected.
- Final test split is reserved for independent evaluation.
- Internal validation must be speaker-independent.
- No ASR transcription or ML training has been performed.

## Next step

STEP 2: Pilot acoustic feature extraction.
Before modeling, prepare train-only speaker-independent
validation and preserve all output checkpoints.

## Important

Kaggle working directory is temporary.
Restore metadata from the ZIP backup in future sessions.
Raw Parquet files must be reattached or downloaded separately.
"""

(REPORT_DIR / "step01_report.md").write_text(
    report_text, encoding="utf-8"
)

progress_path = CHECK_DIR / "research_progress.json"
progress = json.loads(progress_path.read_text(encoding="utf-8"))

progress.update({
    "current_step": 1,
    "completed_steps": [0, 1],
    "dataset_revision":
        "06385584fad212b26134c656fdd3ccf9f093f33e",
    "train_rows": 2500,
    "test_rows": 2500,
    "train_speakers": 125,
    "test_speakers": len(test_speakers),
    "shared_speakers": 0,
    "speaker_independence_verified": True,
    "next_step": "STEP 2 - Pilot acoustic feature extraction",
    "updated_at": datetime.now(timezone.utc).isoformat()
})

progress_path.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# ZIP all metadata, reports and checkpoints.
# Exclude large raw audio/Parquet files.
ZIP = Path("/kaggle/working/asr_fluency_step01_complete.zip")

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [
        PROJECT / "data/metadata",
        REPORT_DIR,
        CHECK_DIR
    ]:
        for file in folder.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(PROJECT))

# Confirm ZIP integrity
with zipfile.ZipFile(ZIP) as z:
    bad_file = z.testzip()
    assert bad_file is None, f"Corrupt ZIP member: {bad_file}"

sha256 = hashlib.sha256()
with open(ZIP, "rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(block)

print("=== STEP 1 COMPLETED ===")
print("Train rows:", train_pf.metadata.num_rows)
print("Test rows:", test_pf.metadata.num_rows)
print("Train speakers:", len(train_speakers))
print("Test speakers:", len(test_speakers))
print("Shared speakers:", len(train_speakers & test_speakers))
print("ZIP:", ZIP)
print("ZIP size (MB):", round(ZIP.stat().st_size / 1048576, 3))
print("ZIP SHA256:", sha256.hexdigest())
print("ZIP integrity: PASS")

display(FileLink(str(ZIP)))


In [ ]:

from pathlib import Path
import zipfile
import json
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
PROJECT.mkdir(parents=True, exist_ok=True)

EXPECTED_ZIP = "asr_fluency_step01_complete.zip"

# Find the uploaded backup in Kaggle input or working
candidates = [
    p for root in [
        Path("/kaggle/input"),
        Path("/kaggle/working")
    ]
    if root.exists()
    for p in root.rglob(EXPECTED_ZIP)
]

print("=== STEP 2 BACKUP RECOVERY ===")

if not candidates:
    print("BACKUP NOT FOUND")
    print("Upload the STEP 1 ZIP via Kaggle Add Input.")
else:
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one backup, found {len(candidates)}: "
            f"{candidates}"
        )

    backup = candidates[0]
    print("Backup found:", backup)

    with zipfile.ZipFile(backup) as z:
        assert z.testzip() is None, "ZIP integrity failed"

        # Refuse unsafe paths
        for item in z.infolist():
            target = (PROJECT / item.filename).resolve()
            if not target.is_relative_to(PROJECT.resolve()):
                raise RuntimeError(
                    f"Unsafe ZIP entry: {item.filename}"
                )

        z.extractall(PROJECT)

    metadata_file = (
        PROJECT / "data/metadata/dataset_metadata_train.csv"
    )
    progress_file = (
        PROJECT / "checkpoints/research_progress.json"
    )

    assert metadata_file.exists()
    assert progress_file.exists()

    df = pd.read_csv(
        metadata_file,
        dtype={
            "speaker": "string",
            "audio_id": "string"
        }
    )

    progress = json.loads(
        progress_file.read_text(encoding="utf-8")
    )

    assert len(df) == 2500
    assert df["speaker"].nunique() == 125
    assert df["audio_id"].is_unique
    assert df["fluency"].notna().all()
    assert progress["completed_steps"] == [0, 1]

    print("\n=== RECOVERED METADATA ===")
    print("Rows:", len(df))
    print("Speakers:", df["speaker"].nunique())
    print("Unique audio IDs:", df["audio_id"].nunique())
    print("Missing fluency:", df["fluency"].isna().sum())

    print("\n=== CHECKPOINT ===")
    print("Completed steps:", progress["completed_steps"])
    print("Dataset:", progress["dataset"])
    print("Revision:", progress["dataset_revision"])
    print("Random seed:", progress["random_seed"])

    print("\nRECOVERY SUCCESSFUL")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "data/metadata/dataset_metadata_train.csv"
OUTPUT = PROJECT / "data/metadata/pilot_selection_400.csv"
SPEAKERS_FILE = PROJECT / "checkpoints/pilot_speaker_ids.json"
PROGRESS_FILE = PROJECT / "checkpoints/research_progress.json"

SEED = 42
PILOT_SPEAKERS = 20
EXPECTED_AUDIO = 400

df = pd.read_csv(
    SOURCE,
    dtype={"speaker": "string", "audio_id": "string"}
)

assert len(df) == 2500
assert df["audio_id"].is_unique
assert df["speaker"].nunique() == 125

# Do not change an existing pilot selection
if OUTPUT.exists() and SPEAKERS_FILE.exists():

    pilot = pd.read_csv(
        OUTPUT,
        dtype={"speaker": "string", "audio_id": "string"}
    )

    saved_speakers = json.loads(
        SPEAKERS_FILE.read_text(encoding="utf-8")
    )

    assert set(pilot["speaker"]) == set(
        saved_speakers["speaker_ids"]
    )

    print("Existing pilot selection restored.")

else:
    rng = np.random.default_rng(SEED)

    speaker_ids = sorted(df["speaker"].unique())

    selected = sorted(
        rng.choice(
            speaker_ids,
            size=PILOT_SPEAKERS,
            replace=False
        ).tolist()
    )

    pilot = df[df["speaker"].isin(selected)].copy()

    pilot = pilot.sort_values(
        ["speaker", "audio_id"]
    ).reset_index(drop=True)

    pilot.insert(
        0, "pilot_index",
        range(1, len(pilot) + 1)
    )

    assert len(pilot) == EXPECTED_AUDIO
    assert pilot["speaker"].nunique() == PILOT_SPEAKERS
    assert pilot["audio_id"].is_unique

    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    SPEAKERS_FILE.parent.mkdir(parents=True, exist_ok=True)

    pilot.to_csv(OUTPUT, index=False)

    SPEAKERS_FILE.write_text(
        json.dumps(
            {
                "seed": SEED,
                "speaker_ids": selected,
                "pilot_audio_count": len(pilot)
            },
            indent=2
        ),
        encoding="utf-8"
    )

print("=== PILOT SELECTION ===")
print("Pilot audio:", len(pilot))
print("Pilot speakers:", pilot["speaker"].nunique())
print("Audio per speaker:")
print(pilot.groupby("speaker").size().describe())

print("\n=== FLUENCY DISTRIBUTION ===")
print(pilot["fluency"].value_counts().sort_index())

print("\n=== PILOT SUMMARY ===")
print(pilot["fluency"].describe())

print("\n=== SELECTED SPEAKERS ===")
print(sorted(pilot["speaker"].unique().tolist()))

print("\n=== SAMPLE ===")
print(
    pilot[
        ["pilot_index", "audio_id", "speaker",
         "fluency", "duration_sec"]
    ].head(5).to_string(index=False)
)

progress = json.loads(
    PROGRESS_FILE.read_text(encoding="utf-8")
)

progress["current_step"] = 2
progress["pilot_audio_count"] = len(pilot)
progress["pilot_speaker_count"] = pilot["speaker"].nunique()
progress["pilot_seed"] = SEED
progress["pilot_selection_file"] = str(OUTPUT)
progress["next_task"] = "Restore train audio and begin pilot extraction"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

PROGRESS_FILE.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nPilot selection saved:", OUTPUT)
print("Speaker IDs saved:", SPEAKERS_FILE)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
import requests
import pyarrow.parquet as pq
import pandas as pd
import json
import os
import time

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)

REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"
EXPECTED_SIZE = 332843486

URL = (
    "https://huggingface.co/datasets/mispeech/"
    f"speechocean762/resolve/{REVISION}/"
    "data/train-00000-of-00001.parquet"
)

TARGET = RAW / "speechocean762_train.parquet"
PART = RAW / "speechocean762_train.parquet.part"

if TARGET.exists() and TARGET.stat().st_size == EXPECTED_SIZE:
    print("Existing train Parquet found.")
else:
    for attempt in range(5):
        existing = PART.stat().st_size if PART.exists() else 0

        headers = (
            {"Range": f"bytes={existing}-"}
            if existing else {}
        )

        try:
            with requests.get(
                URL,
                headers=headers,
                stream=True,
                timeout=(30, 120)
            ) as response:
                response.raise_for_status()

                resume = existing > 0 and response.status_code == 206
                mode = "ab" if resume else "wb"

                with open(PART, mode) as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            if PART.stat().st_size == EXPECTED_SIZE:
                os.replace(PART, TARGET)
                print("Train Parquet downloaded.")
                break

            print("Incomplete download, retrying...")

        except Exception as e:
            print(f"Attempt {attempt + 1}: {e}")
            time.sleep(3)

    if not TARGET.exists() or TARGET.stat().st_size != EXPECTED_SIZE:
        raise RuntimeError("Train Parquet recovery failed.")

pf = pq.ParquetFile(TARGET)

assert pf.metadata.num_rows == 2500

PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

train_meta = pd.read_csv(
    PROJECT / "data/metadata/dataset_metadata_train.csv",
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(pilot) == 400
assert pilot["audio_id"].is_unique
assert set(pilot["audio_id"]).issubset(
    set(train_meta["audio_id"])
)

print("\n=== STEP 2 AUDIO RECOVERY ===")
print("Train Parquet size:", TARGET.stat().st_size)
print("Train rows:", pf.metadata.num_rows)
print("Pilot audio:", len(pilot))
print("Pilot speakers:", pilot["speaker"].nunique())
print("Pilot IDs matched:", len(
    set(pilot["audio_id"]) & set(train_meta["audio_id"])
))

progress_file = PROJECT / "checkpoints/research_progress.json"

progress = json.loads(
    progress_file.read_text(encoding="utf-8")
)

progress["step2_train_audio_available"] = True
progress["step2_pilot_ids_verified"] = True
progress["next_task"] = "Pilot acoustic feature extraction setup"

progress_file.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Checkpoint updated.")


In [ ]:

import io
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa

from pathlib import Path

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
OUT = PROJECT / "features/acoustic/acoustic_smoke_test_5.csv"

OUT.parent.mkdir(parents=True, exist_ok=True)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

# Load first 400 metadata rows, preserving Parquet order
pf = pq.ParquetFile(TRAIN)

target_ids = set(pilot["audio_id"].head(5).tolist())

records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):

    table = pf.read_row_group(
        group_id,
        columns=["audio"]
    )

    for audio in table.column("audio").to_pylist():

        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        assert sr == 16000
        assert y.ndim == 1

        duration = len(y) / sr

        # Energy-based speech interval detection
        intervals = librosa.effects.split(
            y,
            top_db=30,
            frame_length=512,
            hop_length=160
        )

        speech_duration = sum(
            (end - start) / sr
            for start, end in intervals
        )

        # Internal pauses between speech intervals
        gaps = [
            (intervals[i + 1][0] - intervals[i][1]) / sr
            for i in range(len(intervals) - 1)
        ]

        pauses = [gap for gap in gaps if gap >= 0.2]
        pause_duration = sum(pauses)

        # YIN pitch: preliminary, unvoiced frames not filtered
        f0 = librosa.yin(
            y,
            fmin=75,
            fmax=500,
            sr=sr,
            frame_length=1024,
            hop_length=256
        )

        rms = librosa.feature.rms(
            y=y,
            frame_length=1024,
            hop_length=256
        )[0]

        records.append({
            "audio_id": audio_id,
            "duration_sec": round(duration, 4),
            "speech_duration_sec": round(speech_duration, 4),
            "pause_count": len(pauses),
            "pause_duration_sec": round(pause_duration, 4),
            "pause_ratio": round(pause_duration / duration, 4),
            "pitch_mean_hz_preliminary": round(float(np.mean(f0)), 2),
            "pitch_sd_hz_preliminary": round(float(np.std(f0)), 2),
            "rms_mean": round(float(np.mean(rms)), 5)
        })

    if found == target_ids:
        break

assert found == target_ids, f"Missing audio IDs: {target_ids - found}"

result = pd.DataFrame(records)

result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)

result = result.sort_values("pilot_index")

result.to_csv(OUT, index=False)

print("=== ACOUSTIC SMOKE TEST ===")
print(result.to_string(index=False))

print("\n=== VALIDATION ===")
print("Processed:", len(result))
print("Missing values:", result.isna().sum().sum())
print("Negative speech durations:",
      (result["speech_duration_sec"] < 0).sum())
print("Speech longer than audio:",
      (result["speech_duration_sec"] > result["duration_sec"]).sum())

print("\nSaved:", OUT)


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

OUTPUT = PROJECT / "features/acoustic/pitch_validation_5.csv"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string"}
)

target_ids = set(pilot["audio_id"].head(5))

pf = pq.ParquetFile(TRAIN)

results = []
found = set()

for group_id in range(pf.metadata.num_row_groups):

    table = pf.read_row_group(
        group_id,
        columns=["audio"]
    )

    for audio in table.column("audio").to_pylist():

        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        # Probabilistic YIN with voiced-frame detection
        f0, voiced_flag, voiced_prob = librosa.pyin(
            y,
            sr=sr,
            fmin=75,
            fmax=500,
            frame_length=1024,
            hop_length=256,
            fill_na=np.nan
        )

        voiced_f0 = f0[
            voiced_flag & np.isfinite(f0)
        ]

        results.append({
            "audio_id": audio_id,
            "total_frames": len(f0),
            "voiced_frames": len(voiced_f0),
            "voiced_ratio": round(
                len(voiced_f0) / len(f0), 4
            ),
            "pitch_mean_hz": (
                round(float(np.mean(voiced_f0)), 2)
                if len(voiced_f0) else np.nan
            ),
            "pitch_sd_hz": (
                round(float(np.std(voiced_f0)), 2)
                if len(voiced_f0) else np.nan
            ),
            "pitch_min_hz": (
                round(float(np.min(voiced_f0)), 2)
                if len(voiced_f0) else np.nan
            ),
            "pitch_max_hz": (
                round(float(np.max(voiced_f0)), 2)
                if len(voiced_f0) else np.nan
            )
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(results)

result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)

result = result.sort_values("pilot_index")

result.to_csv(OUTPUT, index=False)

print("=== VOICED PITCH VALIDATION ===")
print(result.to_string(index=False))

print("\n=== QUALITY CHECK ===")
print("Processed:", len(result))
print("No voiced frames:",
      (result["voiced_frames"] == 0).sum())
print("Missing pitch means:",
      result["pitch_mean_hz"].isna().sum())

print("\nSaved:", OUTPUT)


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

OUT = PROJECT / "features/acoustic/pause_diagnostics_5.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string"}
)

target_ids = set(pilot["audio_id"].head(5))

pf = pq.ParquetFile(TRAIN)
records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):

    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:

        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        duration = len(y) / sr

        # Energy-based speech segmentation
        intervals = librosa.effects.split(
            y,
            top_db=30,
            frame_length=512,
            hop_length=160
        )

        speech_sec = sum(
            (end - start) / sr for start, end in intervals
        )

        if len(intervals):
            leading_silence = intervals[0][0] / sr
            trailing_silence = (
                len(y) - intervals[-1][1]
            ) / sr
        else:
            leading_silence = duration
            trailing_silence = 0.0

        internal_gaps = [
            (intervals[i + 1][0] - intervals[i][1]) / sr
            for i in range(len(intervals) - 1)
        ]

        pauses_100 = [
            gap for gap in internal_gaps if gap >= 0.1
        ]
        pauses_200 = [
            gap for gap in internal_gaps if gap >= 0.2
        ]
        pauses_300 = [
            gap for gap in internal_gaps if gap >= 0.3
        ]

        records.append({
            "audio_id": audio_id,
            "duration_sec": round(duration, 3),
            "speech_sec": round(speech_sec, 3),
            "speech_segments": len(intervals),
            "leading_silence_sec": round(
                leading_silence, 3
            ),
            "trailing_silence_sec": round(
                trailing_silence, 3
            ),
            "internal_gaps": len(internal_gaps),
            "pause_count_100ms": len(pauses_100),
            "pause_count_200ms": len(pauses_200),
            "pause_count_300ms": len(pauses_300),
            "pause_duration_200ms": round(
                sum(pauses_200), 3
            ),
            "all_internal_gap_sec": round(
                sum(internal_gaps), 3
            )
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(records)
result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)
result = result.sort_values("pilot_index")

result.to_csv(OUT, index=False)

print("=== PAUSE DIAGNOSTICS ===")
print(result.to_string(index=False))

print("\n=== SUMMARY ===")
print("Processed:", len(result))
print("Zero speech segments:",
      (result["speech_segments"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\nSaved:", OUT)


In [ ]:

import io
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

OUT = PROJECT / "features/acoustic/silero_vad_diagnostics_5.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

# Load Silero VAD from torch hub
model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

get_speech_timestamps = utils[0]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string"}
)

target_ids = set(pilot["audio_id"].head(5))

pf = pq.ParquetFile(TRAIN)
records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):

    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        assert sr == 16000
        assert y.ndim == 1

        timestamps = get_speech_timestamps(
            torch.from_numpy(y),
            model,
            sampling_rate=sr,
            threshold=0.5,
            min_speech_duration_ms=100,
            min_silence_duration_ms=200,
            speech_pad_ms=0
        )

        duration = len(y) / sr

        speech_sec = sum(
            (segment["end"] - segment["start"]) / sr
            for segment in timestamps
        )

        gaps = [
            (
                timestamps[i + 1]["start"]
                - timestamps[i]["end"]
            ) / sr
            for i in range(len(timestamps) - 1)
        ]

        pauses = [gap for gap in gaps if gap >= 0.2]

        leading = (
            timestamps[0]["start"] / sr
            if timestamps else duration
        )
        trailing = (
            (len(y) - timestamps[-1]["end"]) / sr
            if timestamps else 0.0
        )

        records.append({
            "audio_id": audio_id,
            "duration_sec": round(duration, 3),
            "speech_segments": len(timestamps),
            "speech_sec": round(speech_sec, 3),
            "leading_silence_sec": round(leading, 3),
            "trailing_silence_sec": round(trailing, 3),
            "pause_count_200ms": len(pauses),
            "pause_duration_200ms": round(sum(pauses), 3),
            "speech_ratio": round(
                speech_sec / duration, 4
            )
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(records)

result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)

result = result.sort_values("pilot_index")
result.to_csv(OUT, index=False)

print("=== SILERO VAD DIAGNOSTICS ===")
print(result.to_string(index=False))

print("\n=== SUMMARY ===")
print("Processed:", len(result))
print("No speech detected:",
      (result["speech_segments"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\nSaved:", OUT)


In [ ]:

import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
OUTPUT = PROJECT / "results/figures/segmentation_comparison_000700008.png"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

TARGET_ID = "000700008"
pf = pq.ParquetFile(TRAIN)

audio_bytes = None

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        if Path(audio["path"]).stem == TARGET_ID:
            audio_bytes = audio["bytes"]
            break

    if audio_bytes is not None:
        break

assert audio_bytes is not None, "Target audio not found"

y, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32")
duration = len(y) / sr

energy_intervals = librosa.effects.split(
    y,
    top_db=30,
    frame_length=512,
    hop_length=160
)

# Use the already-loaded Silero model if present.
if "model" not in globals() or "get_speech_timestamps" not in globals():
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]

vad_intervals = get_speech_timestamps(
    torch.from_numpy(y),
    model,
    sampling_rate=sr,
    threshold=0.5,
    min_speech_duration_ms=100,
    min_silence_duration_ms=200,
    speech_pad_ms=0
)

rms = librosa.feature.rms(
    y=y, frame_length=512, hop_length=160
)[0]

rms_times = librosa.frames_to_time(
    np.arange(len(rms)), sr=sr, hop_length=160
)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

times = np.arange(len(y)) / sr
axes[0].plot(times, y, linewidth=0.5)
axes[0].set_title("Waveform and speech segments")
axes[0].set_ylabel("Amplitude")

for start, end in energy_intervals:
    axes[0].axvspan(start/sr, end/sr, alpha=0.18)

for seg in vad_intervals:
    axes[0].axvline(seg["start"]/sr, linestyle="--")
    axes[0].axvline(seg["end"]/sr, linestyle="--")

axes[1].plot(rms_times, rms, linewidth=1)
axes[1].set_title("Short-time RMS energy")
axes[1].set_ylabel("RMS")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_xlim(0, duration)

plt.tight_layout()
plt.savefig(OUTPUT, dpi=160)
plt.show()

print("=== SEGMENTATION DETAILS ===")
print("Audio ID:", TARGET_ID)
print("Duration:", round(duration, 3))

print("\nLibrosa speech intervals:")
for start, end in energy_intervals:
    print(round(start/sr, 3), "-", round(end/sr, 3))

print("\nSilero speech intervals:")
for seg in vad_intervals:
    print(
        round(seg["start"]/sr, 3),
        "-",
        round(seg["end"]/sr, 3)
    )

print("\nFigure saved:", OUTPUT)


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"

OUT = PROJECT / "features/acoustic/hybrid_pause_diagnostics_000700008.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

TARGET_ID = "000700008"
pf = pq.ParquetFile(TRAIN)

audio_bytes = None

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        if Path(audio["path"]).stem == TARGET_ID:
            audio_bytes = audio["bytes"]
            break

    if audio_bytes is not None:
        break

assert audio_bytes is not None

y, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32")

if "model" not in globals() or "get_speech_timestamps" not in globals():
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]

vad = get_speech_timestamps(
    torch.from_numpy(y),
    model,
    sampling_rate=sr,
    threshold=0.5,
    min_speech_duration_ms=100,
    min_silence_duration_ms=200,
    speech_pad_ms=0
)

assert len(vad) > 0

padding = int(0.2 * sr)

start = max(0, vad[0]["start"] - padding)
end = min(len(y), vad[-1]["end"] + padding)

segment = y[start:end]

intervals = librosa.effects.split(
    segment,
    top_db=30,
    frame_length=512,
    hop_length=160
)

# Convert local intervals to original timeline
intervals_global = [
    (int(a + start), int(b + start))
    for a, b in intervals
]

records = []

for idx, (a, b) in enumerate(intervals_global):
    length_sec = (b - a) / sr

    records.append({
        "segment_id": idx + 1,
        "start_sec": round(a / sr, 3),
        "end_sec": round(b / sr, 3),
        "duration_sec": round(length_sec, 3),
        "shorter_than_50ms": length_sec < 0.05
    })

df = pd.DataFrame(records)
df.to_csv(OUT, index=False)

# Inspect gaps only between substantial active segments
substantial = [
    (a, b) for a, b in intervals_global
    if (b - a) / sr >= 0.05
]

gaps = [
    (substantial[i + 1][0] - substantial[i][1]) / sr
    for i in range(len(substantial) - 1)
]

print("=== HYBRID SEGMENTATION ===")
print("Silero interval:", round(vad[0]["start"]/sr, 3),
      "-", round(vad[-1]["end"]/sr, 3))
print("Padded interval:", round(start/sr, 3),
      "-", round(end/sr, 3))
print("\nEnergy segments:")
print(df.to_string(index=False))

print("\n=== INTERNAL GAPS ===")
print("All gaps:", [round(g, 3) for g in gaps])
print("Pauses >= 100ms:", sum(g >= 0.1 for g in gaps))
print("Pauses >= 200ms:", sum(g >= 0.2 for g in gaps))

print("\nSaved:", OUT)


In [ ]:

import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
OUT = PROJECT / "features/acoustic/hybrid_validation_5.csv"

OUT.parent.mkdir(parents=True, exist_ok=True)

pilot = pd.read_csv(PILOT, dtype={"audio_id": "string"})
target_ids = set(pilot["audio_id"].head(5))

if "model" not in globals() or "get_speech_timestamps" not in globals():
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]

pf = pq.ParquetFile(TRAIN)
records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        audio_id = Path(audio["path"]).stem
        if audio_id not in target_ids:
            continue

        found.add(audio_id)
        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        duration = len(y) / sr

        vad = get_speech_timestamps(
            torch.from_numpy(y),
            model,
            sampling_rate=sr,
            threshold=0.5,
            min_speech_duration_ms=100,
            min_silence_duration_ms=200,
            speech_pad_ms=0
        )

        if not vad:
            records.append({
                "audio_id": audio_id,
                "status": "no_speech_detected"
            })
            continue

        pad = int(0.2 * sr)
        start = max(0, vad[0]["start"] - pad)
        end = min(len(y), vad[-1]["end"] + pad)

        intervals = librosa.effects.split(
            y[start:end],
            top_db=30,
            frame_length=512,
            hop_length=160
        )

        intervals = [
            (a + start, b + start)
            for a, b in intervals
            if (b - a) / sr >= 0.05
        ]

        speech_sec = sum(
            (b - a) / sr for a, b in intervals
        )

        gaps = [
            (intervals[i + 1][0] - intervals[i][1]) / sr
            for i in range(len(intervals) - 1)
        ]

        pauses_100 = [g for g in gaps if g >= 0.1]
        pauses_200 = [g for g in gaps if g >= 0.2]

        records.append({
            "audio_id": audio_id,
            "status": "ok",
            "audio_duration_sec": round(duration, 3),
            "hybrid_speech_sec": round(speech_sec, 3),
            "speech_segments": len(intervals),
            "pause_count_100ms": len(pauses_100),
            "pause_count_200ms": len(pauses_200),
            "pause_duration_200ms": round(sum(pauses_200), 3),
            "leading_silence_sec": round(
                intervals[0][0] / sr, 3
            ) if intervals else np.nan,
            "trailing_silence_sec": round(
                (len(y) - intervals[-1][1]) / sr, 3
            ) if intervals else np.nan
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(records)
result["pilot_index"] = result["audio_id"].map(
    pilot.set_index("audio_id")["pilot_index"]
)

result = result.sort_values("pilot_index")
result.to_csv(OUT, index=False)

print("=== HYBRID VALIDATION ===")
print(result.to_string(index=False))

print("\n=== SUMMARY ===")
print("Processed:", len(result))
print("Successful:", (result["status"] == "ok").sum())
print("Failed:", (result["status"] != "ok").sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\nSaved:", OUT)


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

selected = pilot.head(5)
target_ids = set(selected["audio_id"])

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(group_id, columns=["audio"])

    for item in table.column("audio").to_pylist():
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == target_ids

print("=== AUDIO LISTENING AUDIT ===")

for _, row in selected.iterrows():
    audio_id = row["audio_id"]
    y, sr = audio_data[audio_id]

    print(
        f"\nAudio: {audio_id} | "
        f"Duration: {len(y)/sr:.2f}s"
    )
    print("Expected text:", row["text"])

    display(Audio(y, rate=sr, normalize=False))

print("\nAll 5 audio players displayed.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import json
import zipfile
import hashlib

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
META = PROJECT / "data/metadata"
REPORTS = PROJECT / "reports"
CHECK = PROJECT / "checkpoints"

for folder in [META, REPORTS, CHECK]:
    folder.mkdir(parents=True, exist_ok=True)

observations = [
    {
        "audio_id": "000700008",
        "audible_internal_pause_200ms": "not_observed",
        "leading_noise": True,
        "trailing_noise": True,
        "notes": "Short initial noise; no noticeable internal stop; short trailing silence and noise"
    },
    {
        "audio_id": "000700010",
        "audible_internal_pause_200ms": "not_observed",
        "leading_noise": None,
        "trailing_noise": None,
        "notes": "No noticeable internal stop; short trailing silence"
    },
    {
        "audio_id": "000700012",
        "audible_internal_pause_200ms": "not_assessed",
        "leading_noise": None,
        "trailing_noise": True,
        "notes": "Breathing-like noise after speech"
    },
    {
        "audio_id": "000700015",
        "audible_internal_pause_200ms": "not_assessed",
        "leading_noise": True,
        "trailing_noise": None,
        "notes": "Mouth-opening-like noise before speech"
    },
    {
        "audio_id": "000700019",
        "audible_internal_pause_200ms": "not_assessed",
        "leading_noise": None,
        "trailing_noise": None,
        "notes": "No unusual issue noticed"
    }
]

df = pd.DataFrame(observations)

AUDIT = META / "audio_listening_audit_5.csv"
df.to_csv(AUDIT, index=False, encoding="utf-8")

# Preserve the evidence and its limitations
report = """# STEP 2 - Preliminary Listening Audit

Dataset: Speechocean762
Audios checked: 5
Auditor: Researcher (informal listening)

## Findings
- No noticeable internal stops reported for 000700008 and 000700010.
- Initial/trailing non-speech noises were reported.
- Remaining recordings received general qualitative observations.
- Hybrid method detected zero internal pauses >= 200 ms in all five.

## Important limitations
- This is not expert-validated pause ground truth.
- The observer did not timestamp pauses manually.
- 'Not assessed' must not be interpreted as 'no pause'.
- Current parameters have not been validated across speakers.
- Do not claim 100% segmentation accuracy.

## Next task
Prepare an expanded, speaker-diverse quality audit before
finalizing the 400-audio acoustic extraction configuration.
"""

(REPORTS / "step02_listening_audit.md").write_text(
    report, encoding="utf-8"
)

progress_file = CHECK / "research_progress.json"
progress = json.loads(
    progress_file.read_text(encoding="utf-8")
)

progress["current_step"] = 2
progress["step2_listening_audit_count"] = 5
progress["step2_listening_audit_file"] = str(AUDIT)
progress["step2_segmentation_status"] = "preliminary_validation"
progress["next_task"] = "Expanded speaker-diverse segmentation QA"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

progress_file.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

ZIP = Path("/kaggle/working/asr_fluency_step02_checkpoint.zip")

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [
        PROJECT / "data/metadata",
        PROJECT / "features/acoustic",
        REPORTS,
        CHECK
    ]:
        if folder.exists():
            for file in folder.rglob("*"):
                if file.is_file():
                    z.write(file, file.relative_to(PROJECT))

with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None

sha = hashlib.sha256(ZIP.read_bytes()).hexdigest()

print("=== STEP 2 CHECKPOINT ===")
print("Listening audit rows:", len(df))
print("Audit saved:", AUDIT)
print("ZIP:", ZIP)
print("ZIP size (MB):", round(ZIP.stat().st_size / 1048576, 3))
print("ZIP SHA256:", sha)
print("ZIP integrity: PASS")
print("Next task:", progress["next_task"])


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
OUT = PROJECT / "data/metadata/expanded_qa_selection_15.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

SEED = 42

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(pilot) == 400
assert pilot["audio_id"].is_unique

# Exclude the five recordings already reviewed
previous_ids = set(pilot.head(5)["audio_id"])
available = pilot[
    ~pilot["audio_id"].isin(previous_ids)
].copy()

if OUT.exists():
    selected = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )
    print("Existing QA selection restored.")
else:
    rng = np.random.default_rng(SEED)

    # Select from three fluency bands
    available["fluency_band"] = pd.cut(
        available["fluency"],
        bins=[0, 6, 8, 10],
        labels=["low", "medium", "high"]
    )

    selections = []

    for band in ["low", "medium", "high"]:
        group = available[
            available["fluency_band"] == band
        ].copy()

        # Randomize speaker order, then choose one
        # recording per speaker where possible
        speakers = rng.permutation(
            sorted(group["speaker"].unique())
        )

        chosen = []

        for speaker in speakers:
            candidates = group[
                group["speaker"] == speaker
            ]

            idx = rng.choice(candidates.index.to_numpy())
            chosen.append(group.loc[idx])

            if len(chosen) == 5:
                break

        if len(chosen) != 5:
            raise RuntimeError(
                f"Insufficient speaker diversity in {band}"
            )

        selections.extend(chosen)

    selected = pd.DataFrame(selections)

    selected = selected.sort_values(
        ["fluency_band", "speaker", "audio_id"]
    ).reset_index(drop=True)

    selected.insert(
        0, "qa_index",
        range(1, len(selected) + 1)
    )

    assert len(selected) == 15
    assert selected["audio_id"].is_unique
    assert not set(selected["audio_id"]) & previous_ids

    OUT.parent.mkdir(parents=True, exist_ok=True)
    selected.to_csv(OUT, index=False)

print("=== EXPANDED QA SELECTION ===")
print("Selected audio:", len(selected))
print("Unique speakers:", selected["speaker"].nunique())

print("\nFluency bands:")
print(selected["fluency_band"].value_counts())

print("\nDuration statistics:")
print(selected["duration_sec"].describe())

print("\n=== SELECTED RECORDINGS ===")
print(
    selected[
        [
            "qa_index", "audio_id", "speaker",
            "fluency", "duration_sec", "text"
        ]
    ].to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_expanded_qa_count"] = len(selected)
progress["step2_expanded_qa_file"] = str(OUT)
progress["next_task"] = "Inspect expanded QA audio and segmentation"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nSaved:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
QA = PROJECT / "data/metadata/expanded_qa_selection_15.csv"
OUT = PROJECT / "features/acoustic/expanded_hybrid_diagnostics_15.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT.parent.mkdir(parents=True, exist_ok=True)

qa = pd.read_csv(
    QA,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(qa) == 15
assert qa["audio_id"].is_unique

target_ids = set(qa["audio_id"])

# Load Silero VAD if not already available
if "model" not in globals() or "get_speech_timestamps" not in globals():
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]

pf = pq.ParquetFile(TRAIN)
records = []
found = set()

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        audio_id = Path(audio["path"]).stem

        if audio_id not in target_ids:
            continue

        found.add(audio_id)

        y, sr = sf.read(
            io.BytesIO(audio["bytes"]),
            dtype="float32"
        )

        duration = len(y) / sr

        vad = get_speech_timestamps(
            torch.from_numpy(y),
            model,
            sampling_rate=sr,
            threshold=0.5,
            min_speech_duration_ms=100,
            min_silence_duration_ms=200,
            speech_pad_ms=0
        )

        if not vad:
            records.append({
                "audio_id": audio_id,
                "status": "no_speech_detected"
            })
            continue

        pad = int(0.2 * sr)
        start = max(0, vad[0]["start"] - pad)
        end = min(len(y), vad[-1]["end"] + pad)

        intervals = librosa.effects.split(
            y[start:end],
            top_db=30,
            frame_length=512,
            hop_length=160
        )

        intervals = [
            (int(a + start), int(b + start))
            for a, b in intervals
            if (b - a) / sr >= 0.05
        ]

        if not intervals:
            records.append({
                "audio_id": audio_id,
                "status": "no_energy_segments"
            })
            continue

        speech_sec = sum(
            (b - a) / sr for a, b in intervals
        )

        gaps = [
            (intervals[i + 1][0] - intervals[i][1]) / sr
            for i in range(len(intervals) - 1)
        ]

        pauses_100 = [g for g in gaps if g >= 0.1]
        pauses_200 = [g for g in gaps if g >= 0.2]

        records.append({
            "audio_id": audio_id,
            "status": "ok",
            "duration_sec": round(duration, 3),
            "hybrid_speech_sec": round(speech_sec, 3),
            "speech_segments": len(intervals),
            "pause_count_100ms": len(pauses_100),
            "pause_count_200ms": len(pauses_200),
            "pause_duration_200ms": round(
                sum(pauses_200), 3
            ),
            "leading_silence_sec": round(
                intervals[0][0] / sr, 3
            ),
            "trailing_silence_sec": round(
                (len(y) - intervals[-1][1]) / sr, 3
            )
        })

    if found == target_ids:
        break

assert found == target_ids

result = pd.DataFrame(records)

result = qa[
    ["qa_index", "audio_id", "speaker", "fluency"]
].merge(
    result,
    on="audio_id",
    how="left",
    validate="one_to_one"
)

result = result.sort_values("qa_index")
result.to_csv(OUT, index=False)

print("=== EXPANDED HYBRID QA ===")
print(result.to_string(index=False))

print("\n=== SUMMARY ===")
print("Processed:", len(result))
print("Successful:", (result["status"] == "ok").sum())
print("Failed:", (result["status"] != "ok").sum())
print("Audios with 100ms pauses:",
      (result["pause_count_100ms"] > 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\nSaved:", OUT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step2_expanded_hybrid_processed"] = len(result)
progress["step2_expanded_hybrid_successful"] = int(
    (result["status"] == "ok").sum()
)
progress["next_task"] = "Expanded listening audit"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
QA = PROJECT / "data/metadata/expanded_qa_selection_15.csv"
DIAG = PROJECT / "features/acoustic/expanded_hybrid_diagnostics_15.csv"
OUT = PROJECT / "data/metadata/targeted_listening_audit_5.csv"

target_ids = [
    "000700065",
    "096140018",
    "096190016",
    "000700109",
    "001450142"
]

qa = pd.read_csv(QA, dtype={"audio_id": "string"})
diag = pd.read_csv(DIAG, dtype={"audio_id": "string"})

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert len(audio_data) == len(target_ids)

records = []

print("=== TARGETED LISTENING AUDIT ===")

for audio_id in target_ids:
    row = qa.loc[qa["audio_id"] == audio_id].iloc[0]
    d = diag.loc[diag["audio_id"] == audio_id].iloc[0]
    y, sr = audio_data[audio_id]

    print("\n" + "=" * 55)
    print("Audio ID:", audio_id)
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(len(y) / sr, 3))
    print("Hybrid 200ms pauses:", d["pause_count_200ms"])
    print("Hybrid pause duration:", d["pause_duration_200ms"])

    display(Audio(y, rate=sr, normalize=False))

    records.append({
        "audio_id": audio_id,
        "fluency": row["fluency"],
        "hybrid_pause_count_200ms": d["pause_count_200ms"],
        "hybrid_pause_duration_200ms": d["pause_duration_200ms"],
        "manual_pause_observation": "pending",
        "noise_observation": "pending",
        "notes": ""
    })

audit = pd.DataFrame(records)
OUT.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT, index=False)

print("\nAudit template saved:", OUT)


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
AUDIT = PROJECT / "data/metadata/targeted_listening_audit_5.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Audit template not found"

audit = pd.read_csv(AUDIT, dtype={"audio_id": "string"})

annotations = {
    "000700065": {
        "manual_pause_observation": "internal_pauses_audible",
        "noise_observation": "not_assessed",
        "notes": (
            "Internal pauses audible, especially before FANS. "
            "FANS pronounced approximately fffaans."
        )
    },
    "096140018": {
        "manual_pause_observation": "internal_pauses_audible",
        "noise_observation": "not_assessed",
        "notes": (
            "Internal pauses audible. "
            "DEAD sounded approximately like daddi."
        )
    },
    "096190016": {
        "manual_pause_observation": "frequent_internal_pauses",
        "noise_observation": "not_assessed",
        "notes": (
            "Pauses after many words; several pronunciation "
            "deviations reported: ASKED, AS, PUT, HER."
        )
    },
    "000700109": {
        "manual_pause_observation": "internal_pause_audible",
        "noise_observation": "not_assessed",
        "notes": "Audible pause between THE and YET."
    },
    "001450142": {
        "manual_pause_observation": "not_assessed",
        "noise_observation": "leading_breath_like_noise",
        "notes": "Breathing-like noise at beginning."
    }
}

assert set(audit["audio_id"]) == set(annotations)

for audio_id, values in annotations.items():
    mask = audit["audio_id"] == audio_id

    for column, value in values.items():
        audit.loc[mask, column] = value

audit["audit_type"] = "informal_researcher_listening"
audit["manual_pause_timestamps_available"] = False

audit.to_csv(AUDIT, index=False, encoding="utf-8")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_targeted_audit_count"] = len(audit)
progress["step2_targeted_audit_status"] = "completed"
progress["next_task"] = "Listen to remaining 10 expanded QA recordings"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== TARGETED AUDIT SAVED ===")
print(audit[
    [
        "audio_id",
        "manual_pause_observation",
        "noise_observation",
        "notes"
    ]
].to_string(index=False))

print("\nAudit rows:", len(audit))
print("Pending observations:",
      (audit["manual_pause_observation"] == "pending").sum())
print("Saved:", AUDIT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
QA = PROJECT / "data/metadata/expanded_qa_selection_15.csv"
DIAG = PROJECT / "features/acoustic/expanded_hybrid_diagnostics_15.csv"
PREVIOUS = PROJECT / "data/metadata/targeted_listening_audit_5.csv"
OUT = PROJECT / "data/metadata/remaining_listening_audit_10.csv"

qa = pd.read_csv(QA, dtype={"audio_id": "string"})
diag = pd.read_csv(DIAG, dtype={"audio_id": "string"})
previous = pd.read_csv(
    PREVIOUS,
    dtype={"audio_id": "string"}
)

completed_ids = set(previous["audio_id"])

remaining = qa[
    ~qa["audio_id"].isin(completed_ids)
].copy().sort_values("qa_index")

assert len(remaining) == 10
assert remaining["audio_id"].is_unique

target_ids = set(remaining["audio_id"])

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id,
        columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == target_ids

records = []

print("=== REMAINING LISTENING AUDIT ===")

for _, row in remaining.iterrows():
    audio_id = row["audio_id"]
    d = diag.loc[
        diag["audio_id"] == audio_id
    ].iloc[0]

    y, sr = audio_data[audio_id]

    print("\n" + "=" * 55)
    print("QA index:", row["qa_index"])
    print("Audio ID:", audio_id)
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(len(y) / sr, 3))
    print("Hybrid 200ms pauses:", d["pause_count_200ms"])
    print("Hybrid pause duration:",
          d["pause_duration_200ms"])

    display(Audio(y, rate=sr, normalize=False))

    records.append({
        "qa_index": row["qa_index"],
        "audio_id": audio_id,
        "fluency": row["fluency"],
        "hybrid_pause_count_200ms": d["pause_count_200ms"],
        "hybrid_pause_duration_200ms": d["pause_duration_200ms"],
        "manual_pause_observation": "pending",
        "noise_observation": "pending",
        "pronunciation_observation": "pending",
        "notes": ""
    })

audit = pd.DataFrame(records)

# Explicit string dtypes avoid future pandas warnings
for col in [
    "manual_pause_observation",
    "noise_observation",
    "pronunciation_observation",
    "notes"
]:
    audit[col] = audit[col].astype("string")

OUT.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT, index=False)

print("\n=== AUDIT SUMMARY ===")
print("Audio players displayed:", len(audio_data))
print("Remaining audit rows:", len(audit))
print("Pending observations:", len(audit))
print("Saved:", OUT)


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
AUDIT = PROJECT / "data/metadata/remaining_listening_audit_10.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Audit file not found"

audit = pd.read_csv(AUDIT, dtype={"audio_id": "string"})

annotations = {
    "009600270": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    ),
    "024020175": (
        "no_noticeable_pause", "not_assessed",
        "natural_pronunciation", "Native-like pronunciation"
    ),
    "030210137": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    ),
    "069120123": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    ),
    "060100128": (
        "one_or_two_pauses", "not_assessed",
        "not_assessed",
        "Pauses around TALKING-AWAY and AWAY-THE TABLES"
    ),
    "096110018": (
        "no_noticeable_pause", "not_assessed",
        "pronunciation_deviation",
        "Pronunciation errors audible; no pauses noticed"
    ),
    "020590276": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    ),
    "024020051": (
        "one_natural_pause", "not_assessed",
        "natural_pronunciation",
        "Native-like pronunciation; one natural pause"
    ),
    "034230115": (
        "one_natural_pause", "leading_breath_like_noise",
        "no_issue_noticed",
        "Natural pause; breathing-like noise at beginning"
    ),
    "054140039": (
        "no_noticeable_pause", "not_assessed",
        "no_issue_noticed", "No unusual issue noticed"
    )
}

assert set(audit["audio_id"]) == set(annotations)

columns = [
    "manual_pause_observation",
    "noise_observation",
    "pronunciation_observation",
    "notes"
]

# Explicit string dtype prevents the previous FutureWarning
for col in columns:
    audit[col] = audit[col].astype("string")

for audio_id, values in annotations.items():
    mask = audit["audio_id"] == audio_id
    for col, value in zip(columns, values):
        audit.loc[mask, col] = value

audit["audit_type"] = "informal_researcher_listening"
audit["manual_pause_timestamps_available"] = False

audit.to_csv(AUDIT, index=False, encoding="utf-8")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_remaining_audit_count"] = len(audit)
progress["step2_remaining_audit_status"] = "completed"
progress["step2_total_listening_audit_count"] = 20
progress["next_task"] = (
    "Review segmentation QA and finalize pilot extraction configuration"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== REMAINING AUDIT SAVED ===")
print(
    audit[
        ["audio_id", "manual_pause_observation",
         "noise_observation", "notes"]
    ].to_string(index=False)
)

print("\nAudit rows:", len(audit))
print("Pending observations:",
      (audit["manual_pause_observation"] == "pending").sum())
print("Total listening-audit recordings:",
      progress["step2_total_listening_audit_count"])
print("Saved:", AUDIT)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
CONFIG_DIR = PROJECT / "configs"
REPORTS = PROJECT / "reports"
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

config = {
    "configuration_name": "pilot_acoustic_v1",
    "status": "frozen_for_pilot_not_final_validation",
    "sample_rate_hz": 16000,
    "silero_vad": {
        "threshold": 0.5,
        "min_speech_duration_ms": 100,
        "min_silence_duration_ms": 200,
        "speech_pad_ms": 0,
        "hybrid_padding_ms": 200
    },
    "energy_segmentation": {
        "method": "librosa.effects.split",
        "top_db": 30,
        "frame_length": 512,
        "hop_length": 160,
        "min_active_segment_ms": 50
    },
    "pitch": {
        "method": "librosa.pyin",
        "fmin_hz": 75,
        "fmax_hz": 500,
        "frame_length": 1024,
        "hop_length": 256,
        "fill_na": "nan",
        "statistics": ["mean", "sd", "voiced_ratio"]
    },
    "pause_thresholds_ms": [100, 200],
    "main_pause_threshold_ms": 200,
    "features": [
        "duration_sec",
        "speech_duration_sec",
        "speech_ratio",
        "pause_count_100ms",
        "pause_count_200ms",
        "pause_duration_200ms",
        "pause_ratio_200ms",
        "leading_silence_sec",
        "trailing_silence_sec",
        "pitch_mean_hz",
        "pitch_sd_hz",
        "voiced_ratio",
        "rms_mean",
        "rms_sd"
    ],
    "limitations": [
        "No expert timestamped pause reference",
        "Prompt text is not a verbatim transcript",
        "Detected pauses do not necessarily indicate hesitation",
        "Silero boundaries may omit low-energy speech",
        "Pitch estimates may contain octave errors",
        "Configuration is frozen for pilot consistency only"
    ]
}

CONFIG = CONFIG_DIR / "pilot_acoustic_v1.json"
CONFIG.write_text(
    json.dumps(config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

report = """# STEP 2 - Pilot Acoustic Configuration

## Dataset
Speechocean762, official train split only.

## Pilot
400 recordings from 20 speakers.

## Segmentation
Silero VAD followed by energy-based segmentation
within the padded outer speech boundaries.

## Quality assurance
20 recordings informally reviewed by researcher.
No expert timestamped annotation was conducted.

## Pitch
librosa.pyin voiced-frame statistics.

## Feature selection
14 acoustic features are defined in pilot_acoustic_v1.json.

## Limitations
The QA sample is not sufficient to establish overall
segmentation accuracy. Pause occurrence and disfluency
are not equivalent.

## Next
Extract acoustic features for the 400-audio pilot,
saving batches of 100 with stable identifiers.
"""

(REPORTS / "step02_configuration_report.md").write_text(
    report, encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_config_frozen"] = True
progress["step2_config_name"] = "pilot_acoustic_v1"
progress["step2_pilot_target_audio"] = 400
progress["step2_pilot_processed_audio"] = 0
progress["next_task"] = "Extract acoustic features for pilot rows 0001-0100"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== PILOT CONFIGURATION FROZEN ===")
print("Config:", CONFIG)
print("Features:", len(config["features"]))
print("QA recordings:", 20)
print("Pilot target:", 400)
print("Configuration status:", config["status"])
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "features/acoustic/pilot_acoustic_features_0001-0100.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["configuration_name"] == "pilot_acoustic_v1"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.sort_values("pilot_index").head(100).copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(1, 101))
assert batch["audio_id"].is_unique

feature_columns = config["features"]
expected_ids = set(batch["audio_id"])

if OUT.exists():
    existing = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )

    valid = (
        len(existing) == 100
        and existing["audio_id"].is_unique
        and set(existing["audio_id"]) == expected_ids
        and all(c in existing.columns for c in feature_columns)
    )

    assert valid, "Existing CSV failed validation; do not overwrite"

    result = existing
    print("Existing validated batch reused.")

else:
    if "model" not in globals() or "get_speech_timestamps" not in globals():
        model, utils = torch.hub.load(
            "snakers4/silero-vad",
            "silero_vad",
            trust_repo=True
        )
        get_speech_timestamps = utils[0]

    model.eval()

    pf = pq.ParquetFile(TRAIN)
    audio_data = {}

    for group_id in range(pf.metadata.num_row_groups):
        audios = pf.read_row_group(
            group_id, columns=["audio"]
        ).column("audio").to_pylist()

        for item in audios:
            audio_id = Path(item["path"]).stem

            if audio_id in expected_ids:
                audio_data[audio_id] = item["bytes"]

        if len(audio_data) == 100:
            break

    assert set(audio_data) == expected_ids

    records = []

    with torch.no_grad():

        for n, (_, row) in enumerate(batch.iterrows(), start=1):
            audio_id = row["audio_id"]

            y, sr = sf.read(
                io.BytesIO(audio_data[audio_id]),
                dtype="float32"
            )

            assert sr == 16000
            assert y.ndim == 1

            duration = len(y) / sr

            sc = config["silero_vad"]
            ec = config["energy_segmentation"]
            pc = config["pitch"]

            vad = get_speech_timestamps(
                torch.from_numpy(y),
                model,
                sampling_rate=sr,
                threshold=sc["threshold"],
                min_speech_duration_ms=sc["min_speech_duration_ms"],
                min_silence_duration_ms=sc["min_silence_duration_ms"],
                speech_pad_ms=sc["speech_pad_ms"]
            )

            if vad:
                pad = int(sc["hybrid_padding_ms"] * sr / 1000)

                start = max(0, vad[0]["start"] - pad)
                end = min(len(y), vad[-1]["end"] + pad)

                intervals = librosa.effects.split(
                    y[start:end],
                    top_db=ec["top_db"],
                    frame_length=ec["frame_length"],
                    hop_length=ec["hop_length"]
                )

                intervals = [
                    (int(a + start), int(b + start))
                    for a, b in intervals
                    if (b-a)/sr >= ec["min_active_segment_ms"]/1000
                ]
            else:
                intervals = []

            speech_sec = sum(
                (b-a)/sr for a,b in intervals
            )

            gaps = [
                (intervals[i+1][0]-intervals[i][1])/sr
                for i in range(len(intervals)-1)
            ]

            pauses100 = [g for g in gaps if g >= 0.1]
            pauses200 = [g for g in gaps if g >= 0.2]

            f0, voiced_flag, _ = librosa.pyin(
                y,
                sr=sr,
                fmin=pc["fmin_hz"],
                fmax=pc["fmax_hz"],
                frame_length=pc["frame_length"],
                hop_length=pc["hop_length"],
                fill_na=np.nan
            )

            voiced_f0 = f0[
                voiced_flag & np.isfinite(f0)
            ]

            rms = librosa.feature.rms(
                y=y,
                frame_length=1024,
                hop_length=256
            )[0]

            records.append({
                "pilot_index": int(row["pilot_index"]),
                "audio_id": audio_id,
                "speaker": row["speaker"],
                "fluency": row["fluency"],

                "duration_sec": duration,
                "speech_duration_sec": speech_sec,
                "speech_ratio": speech_sec / duration,

                "pause_count_100ms": len(pauses100),
                "pause_count_200ms": len(pauses200),
                "pause_duration_200ms": sum(pauses200),
                "pause_ratio_200ms": sum(pauses200) / duration,

                "leading_silence_sec": (
                    intervals[0][0] / sr
                    if intervals else np.nan
                ),
                "trailing_silence_sec": (
                    (len(y)-intervals[-1][1])/sr
                    if intervals else np.nan
                ),

                "pitch_mean_hz": (
                    float(np.mean(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "pitch_sd_hz": (
                    float(np.std(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "voiced_ratio": len(voiced_f0) / len(f0),

                "rms_mean": float(np.mean(rms)),
                "rms_sd": float(np.std(rms))
            })

            if n % 10 == 0:
                print(f"Processed {n}/100")

    result = pd.DataFrame(records)
    result.to_csv(OUT, index=False)

# Validate the complete batch
assert len(result) == 100
assert result["pilot_index"].tolist() == list(range(1, 101))
assert result["audio_id"].is_unique
assert set(result["audio_id"]) == expected_ids

assert result["duration_sec"].gt(0).all()
assert result["speech_duration_sec"].ge(0).all()
assert (
    result["speech_duration_sec"]
    <= result["duration_sec"] + 1e-5
).all()

assert result["speech_ratio"].between(0, 1.00001).all()
assert result["voiced_ratio"].between(0, 1).all()

sha256 = hashlib.sha256(OUT.read_bytes()).hexdigest()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_pilot_processed_audio"] = 100
progress["step2_last_batch"] = "0001-0100"
progress["step2_last_batch_sha256"] = sha256
progress["next_task"] = "Validate first 100 acoustic features"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ACOUSTIC BATCH 0001-0100 ===")
print("Rows:", len(result))
print("Speakers:", result["speaker"].nunique())
print("Features:", len(feature_columns))
print("Missing pitch values:", result["pitch_mean_hz"].isna().sum())
print("Zero speech duration:",
      (result["speech_duration_sec"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\n=== FEATURE SUMMARY ===")
print(result[feature_columns].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", sha256)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = PROJECT / "features/acoustic/pilot_acoustic_features_0001-0100.csv"
OUT = PROJECT / "reports/acoustic_qc_batch_0001-0100.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 100
assert df["audio_id"].is_unique

# Screening thresholds, not validated error criteria
df["flag_long_pause"] = df["pause_duration_200ms"] >= 1.5
df["flag_low_speech_ratio"] = df["speech_ratio"] < 0.4
df["flag_low_voiced_ratio"] = df["voiced_ratio"] < 0.2
df["flag_high_pitch_sd"] = df["pitch_sd_hz"] > 70

# Basic numerical consistency checks
df["flag_invalid_duration"] = (
    (df["duration_sec"] <= 0)
    | (df["speech_duration_sec"] < 0)
    | (df["speech_duration_sec"] > df["duration_sec"] + 1e-5)
)

df["flag_invalid_pause"] = (
    (df["pause_duration_200ms"] < 0)
    | (df["pause_duration_200ms"] > df["duration_sec"] + 1e-5)
    | (df["pause_count_200ms"] < 0)
    | (df["pause_count_200ms"] > df["pause_count_100ms"])
)

flag_cols = [c for c in df.columns if c.startswith("flag_")]

df["qc_flag_count"] = df[flag_cols].sum(axis=1)

flagged = df[df["qc_flag_count"] > 0].copy()
flagged = flagged.sort_values(
    ["qc_flag_count", "pause_duration_200ms"],
    ascending=[False, False]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
flagged.to_csv(OUT, index=False)

print("=== ACOUSTIC QC BATCH 0001-0100 ===")
print("Total recordings:", len(df))
print("Flagged recordings:", len(flagged))

print("\n=== FLAG COUNTS ===")
for col in flag_cols:
    print(col, ":", int(df[col].sum()))

print("\n=== TOP FLAGGED RECORDINGS ===")
display_cols = [
    "pilot_index", "audio_id", "speaker", "fluency",
    "duration_sec", "speech_ratio",
    "pause_count_200ms", "pause_duration_200ms",
    "pitch_mean_hz", "pitch_sd_hz", "voiced_ratio",
    "qc_flag_count"
]

print(flagged[display_cols].head(20).round(3).to_string(index=False))

print("\n=== TOP 5 LONGEST PAUSE DURATIONS ===")
print(
    df.nlargest(5, "pause_duration_200ms")[
        ["pilot_index", "audio_id", "speaker", "fluency",
         "duration_sec", "pause_duration_200ms",
         "pause_count_200ms"]
    ].round(3).to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_batch1_qc_completed"] = True
progress["step2_batch1_qc_flagged"] = len(flagged)
progress["next_task"] = "Review QC flags before batch 0101-0200"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nQC CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
FEATURES = PROJECT / "features/acoustic/pilot_acoustic_features_0001-0100.csv"
OUT = PROJECT / "data/metadata/long_pause_audit_4.csv"

target_ids = [
    "001450116",
    "001450051",
    "001450109",
    "001450094"
]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"}
)

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id,
        columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == set(target_ids)

records = []

print("=== LONG PAUSE LISTENING AUDIT ===")

for audio_id in target_ids:
    row = pilot.loc[
        pilot["audio_id"] == audio_id
    ].iloc[0]

    feat = features.loc[
        features["audio_id"] == audio_id
    ].iloc[0]

    y, sr = audio_data[audio_id]

    print("\n" + "=" * 55)
    print("Audio ID:", audio_id)
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(len(y) / sr, 3))
    print("Hybrid pause count:",
          int(feat["pause_count_200ms"]))
    print("Hybrid pause duration:",
          round(feat["pause_duration_200ms"], 3))

    display(Audio(y, rate=sr, normalize=False))

    records.append({
        "audio_id": audio_id,
        "fluency": row["fluency"],
        "hybrid_pause_count_200ms": int(
            feat["pause_count_200ms"]
        ),
        "hybrid_pause_duration_200ms": float(
            feat["pause_duration_200ms"]
        ),
        "manual_pause_observation": "pending",
        "notes": ""
    })

audit = pd.DataFrame(records)

for col in ["manual_pause_observation", "notes"]:
    audit[col] = audit[col].astype("string")

OUT.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT, index=False)

print("\n=== AUDIT SUMMARY ===")
print("Audio players:", len(audio_data))
print("Audit rows:", len(audit))
print("Saved:", OUT)


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

AUDIT = PROJECT / "data/metadata/long_pause_audit_4.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Long-pause audit CSV not found"

audit = pd.read_csv(
    AUDIT,
    dtype={"audio_id": "string"}
)

annotations = {
    "001450116": {
        "manual_pause_observation": "four_pauses_audible",
        "notes": (
            "Four pauses reported after BUTTER, WANTS, "
            "TO, and BE A GOOD. No other unusual issue noticed."
        )
    },
    "001450051": {
        "manual_pause_observation": "two_pauses_audible",
        "notes": (
            "Slow pronunciation. Long pause after LIKES "
            "and another pause after RETURNS. "
            "RETURNS sounded approximately like RENTURS."
        )
    },
    "001450109": {
        "manual_pause_observation": "two_pauses_audible",
        "notes": (
            "Pause after WASHED, followed by an extra "
            "a-like utterance, then another pause."
        )
    },
    "001450094": {
        "manual_pause_observation": "two_pauses_audible",
        "notes": (
            "Long pause after THE and another after ME. "
            "DAVID sounded approximately like DAIVI."
        )
    }
}

assert len(audit) == 4
assert audit["audio_id"].is_unique
assert set(audit["audio_id"]) == set(annotations)

for column in ["manual_pause_observation", "notes"]:
    audit[column] = audit[column].astype("string")

for audio_id, values in annotations.items():
    mask = audit["audio_id"] == audio_id

    for column, value in values.items():
        audit.loc[mask, column] = value

audit["audit_type"] = "informal_researcher_listening"
audit["manual_pause_timestamps_available"] = False
audit["pause_duration_manually_verified"] = False

audit.to_csv(
    AUDIT,
    index=False,
    encoding="utf-8"
)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step2_long_pause_audit_count"] = 4
progress["step2_long_pause_audit_status"] = "completed"
progress["step2_batch1_qc_review_status"] = "preliminary_pass"
progress["next_task"] = (
    "Extract acoustic features for pilot rows 0101-0200"
)
progress["updated_at"] = (
    datetime.now(timezone.utc).isoformat()
)

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== LONG PAUSE AUDIT SAVED ===")

print(
    audit[
        [
            "audio_id",
            "manual_pause_observation",
            "notes"
        ]
    ].to_string(index=False)
)

print("\nAudit rows:", len(audit))
print(
    "Pending observations:",
    (audit["manual_pause_observation"] == "pending").sum()
)
print("Checkpoint updated.")
print("Next task:", progress["next_task"])


In [ ]:

import io
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "features/acoustic/pilot_acoustic_features_0101-0200.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["configuration_name"] == "pilot_acoustic_v1"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot[
    pilot["pilot_index"].between(101, 200)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(101, 201))
assert batch["audio_id"].is_unique

expected_ids = set(batch["audio_id"])
feature_columns = config["features"]

if OUT.exists():
    result = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )
    print("Existing batch found; validating instead of recomputing.")

else:
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]
    model.eval()

    pf = pq.ParquetFile(TRAIN)
    audio_data = {}

    for group_id in range(pf.metadata.num_row_groups):
        audios = pf.read_row_group(
            group_id, columns=["audio"]
        ).column("audio").to_pylist()

        for item in audios:
            audio_id = Path(item["path"]).stem
            if audio_id in expected_ids:
                audio_data[audio_id] = item["bytes"]

        if len(audio_data) == 100:
            break

    assert set(audio_data) == expected_ids

    sc = config["silero_vad"]
    ec = config["energy_segmentation"]
    pc = config["pitch"]

    records = []

    with torch.no_grad():
        for n, (_, row) in enumerate(batch.iterrows(), start=1):

            audio_id = row["audio_id"]

            y, sr = sf.read(
                io.BytesIO(audio_data[audio_id]),
                dtype="float32"
            )

            assert sr == 16000 and y.ndim == 1
            duration = len(y) / sr

            vad = get_speech_timestamps(
                torch.from_numpy(y),
                model,
                sampling_rate=sr,
                threshold=sc["threshold"],
                min_speech_duration_ms=sc["min_speech_duration_ms"],
                min_silence_duration_ms=sc["min_silence_duration_ms"],
                speech_pad_ms=sc["speech_pad_ms"]
            )

            if vad:
                pad = int(sc["hybrid_padding_ms"] * sr / 1000)
                start = max(0, vad[0]["start"] - pad)
                end = min(len(y), vad[-1]["end"] + pad)

                intervals = librosa.effects.split(
                    y[start:end],
                    top_db=ec["top_db"],
                    frame_length=ec["frame_length"],
                    hop_length=ec["hop_length"]
                )

                intervals = [
                    (int(a + start), int(b + start))
                    for a, b in intervals
                    if (b-a)/sr >= ec["min_active_segment_ms"]/1000
                ]
            else:
                intervals = []

            speech_sec = sum(
                (b-a)/sr for a,b in intervals
            )

            gaps = [
                (intervals[i+1][0]-intervals[i][1])/sr
                for i in range(len(intervals)-1)
            ]

            pauses100 = [g for g in gaps if g >= 0.1]
            pauses200 = [g for g in gaps if g >= 0.2]

            f0, voiced_flag, _ = librosa.pyin(
                y,
                sr=sr,
                fmin=pc["fmin_hz"],
                fmax=pc["fmax_hz"],
                frame_length=pc["frame_length"],
                hop_length=pc["hop_length"],
                fill_na=np.nan
            )

            voiced_f0 = f0[
                voiced_flag & np.isfinite(f0)
            ]

            rms = librosa.feature.rms(
                y=y,
                frame_length=1024,
                hop_length=256
            )[0]

            records.append({
                "pilot_index": int(row["pilot_index"]),
                "audio_id": audio_id,
                "speaker": row["speaker"],
                "fluency": row["fluency"],
                "duration_sec": duration,
                "speech_duration_sec": speech_sec,
                "speech_ratio": speech_sec/duration,
                "pause_count_100ms": len(pauses100),
                "pause_count_200ms": len(pauses200),
                "pause_duration_200ms": sum(pauses200),
                "pause_ratio_200ms": sum(pauses200)/duration,
                "leading_silence_sec": (
                    intervals[0][0]/sr if intervals else np.nan
                ),
                "trailing_silence_sec": (
                    (len(y)-intervals[-1][1])/sr
                    if intervals else np.nan
                ),
                "pitch_mean_hz": (
                    float(np.mean(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "pitch_sd_hz": (
                    float(np.std(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "voiced_ratio": len(voiced_f0)/len(f0),
                "rms_mean": float(np.mean(rms)),
                "rms_sd": float(np.std(rms))
            })

            if n % 10 == 0:
                print(f"Processed {n}/100")

    result = pd.DataFrame(records)
    result.to_csv(OUT, index=False)

# Validation
assert len(result) == 100
assert result["pilot_index"].tolist() == list(range(101, 201))
assert result["audio_id"].is_unique
assert set(result["audio_id"]) == expected_ids
assert all(c in result.columns for c in feature_columns)

assert result["duration_sec"].gt(0).all()
assert result["speech_duration_sec"].ge(0).all()
assert (
    result["speech_duration_sec"]
    <= result["duration_sec"] + 1e-5
).all()
assert result["speech_ratio"].between(0, 1.00001).all()
assert result["voiced_ratio"].between(0, 1).all()

sha = hashlib.sha256(OUT.read_bytes()).hexdigest()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_pilot_processed_audio"] = 200
progress["step2_last_batch"] = "0101-0200"
progress["step2_last_batch_sha256"] = sha
progress["next_task"] = "Quality control for batch 0101-0200"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ACOUSTIC BATCH 0101-0200 ===")
print("Rows:", len(result))
print("Speakers:", result["speaker"].nunique())
print("Features:", len(feature_columns))
print("Missing pitch values:", result["pitch_mean_hz"].isna().sum())
print("Zero speech duration:",
      (result["speech_duration_sec"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\n=== FEATURE SUMMARY ===")
print(result[feature_columns].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", sha)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = PROJECT / "features/acoustic/pilot_acoustic_features_0101-0200.csv"
OUT = PROJECT / "reports/acoustic_qc_batch_0101-0200.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 100
assert df["pilot_index"].tolist() == list(range(101, 201))
assert df["audio_id"].is_unique

# Same QC thresholds as batch 1
df["flag_long_pause"] = df["pause_duration_200ms"] >= 1.5
df["flag_low_speech_ratio"] = df["speech_ratio"] < 0.4
df["flag_low_voiced_ratio"] = df["voiced_ratio"] < 0.2
df["flag_high_pitch_sd"] = df["pitch_sd_hz"] > 70

# Additional pitch extraction failure check
df["flag_missing_pitch"] = (
    df["pitch_mean_hz"].isna()
    | df["pitch_sd_hz"].isna()
)

df["flag_invalid_duration"] = (
    (df["duration_sec"] <= 0)
    | (df["speech_duration_sec"] < 0)
    | (df["speech_duration_sec"] > df["duration_sec"] + 1e-5)
)

df["flag_invalid_pause"] = (
    (df["pause_duration_200ms"] < 0)
    | (df["pause_duration_200ms"] > df["duration_sec"] + 1e-5)
    | (df["pause_count_200ms"] < 0)
    | (df["pause_count_200ms"] > df["pause_count_100ms"])
)

flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["qc_flag_count"] = df[flag_cols].sum(axis=1)

flagged = df[df["qc_flag_count"] > 0].copy()
flagged = flagged.sort_values(
    ["qc_flag_count", "pause_duration_200ms"],
    ascending=[False, False]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
flagged.to_csv(OUT, index=False)

print("=== ACOUSTIC QC BATCH 0101-0200 ===")
print("Total recordings:", len(df))
print("Flagged recordings:", len(flagged))

print("\n=== FLAG COUNTS ===")
for col in flag_cols:
    print(col, ":", int(df[col].sum()))

print("\n=== FLAGGED RECORDINGS ===")
display_cols = [
    "pilot_index", "audio_id", "speaker", "fluency",
    "duration_sec", "speech_ratio",
    "pause_count_200ms", "pause_duration_200ms",
    "pitch_mean_hz", "pitch_sd_hz", "voiced_ratio",
    "qc_flag_count"
]

print(
    flagged[display_cols]
    .round(3)
    .to_string(index=False)
)

print("\n=== MISSING PITCH DETAILS ===")
print(
    df.loc[df["flag_missing_pitch"], display_cols]
    .round(3)
    .to_string(index=False)
)

print("\n=== TOP 5 LONGEST PAUSES ===")
print(
    df.nlargest(5, "pause_duration_200ms")[
        ["pilot_index", "audio_id", "speaker", "fluency",
         "duration_sec", "pause_count_200ms",
         "pause_duration_200ms"]
    ].round(3).to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_batch2_qc_completed"] = True
progress["step2_batch2_qc_flagged"] = len(flagged)
progress["next_task"] = "Review batch 2 pitch and segmentation flags"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nQC CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

OUT = PROJECT / "reports/pitch_failure_diagnostic_022420164.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

TARGET_ID = "022420164"

pf = pq.ParquetFile(TRAIN)
audio_bytes = None

for group_id in range(pf.metadata.num_row_groups):

    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:

        if Path(item["path"]).stem == TARGET_ID:
            audio_bytes = item["bytes"]
            break

    if audio_bytes is not None:
        break

assert audio_bytes is not None

y, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

assert sr == 16000
assert y.ndim == 1

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string"}
)

row = pilot.loc[
    pilot["audio_id"] == TARGET_ID
].iloc[0]

print("=== TARGET AUDIO ===")
print("Audio ID:", TARGET_ID)
print("Speaker:", row["speaker"])
print("Human fluency:", row["fluency"])
print("Expected text:", row["text"])
print("Duration:", round(len(y) / sr, 3))
print("Peak amplitude:", round(float(np.max(np.abs(y))), 5))

display(Audio(y, rate=sr, normalize=False))

rms = librosa.feature.rms(
    y=y,
    frame_length=1024,
    hop_length=256
)[0]

configs = [
    ("original_75_500", 75, 500),
    ("diagnostic_60_650", 60, 650)
]

records = []

for name, fmin, fmax in configs:

    f0, voiced_flag, voiced_prob = librosa.pyin(
        y,
        sr=sr,
        fmin=fmin,
        fmax=fmax,
        frame_length=1024,
        hop_length=256,
        fill_na=np.nan
    )

    valid = f0[
        voiced_flag & np.isfinite(f0)
    ]

    records.append({
        "audio_id": TARGET_ID,
        "configuration": name,
        "fmin_hz": fmin,
        "fmax_hz": fmax,
        "total_frames": len(f0),
        "voiced_frames": len(valid),
        "voiced_ratio": len(valid) / len(f0),
        "pitch_mean_hz": (
            float(np.mean(valid)) if len(valid) else np.nan
        ),
        "pitch_sd_hz": (
            float(np.std(valid)) if len(valid) else np.nan
        ),
        "mean_voiced_probability": (
            float(np.mean(voiced_prob[voiced_flag]))
            if np.any(voiced_flag) else np.nan
        ),
        "rms_mean": float(np.mean(rms)),
        "rms_max": float(np.max(rms))
    })

result = pd.DataFrame(records)
result.to_csv(OUT, index=False)

print("\n=== PITCH FAILURE DIAGNOSTICS ===")
print(result.round(4).to_string(index=False))

print("\nSaved:", OUT)


In [ ]:

import io
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

try:
    import parselmouth
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "praat-parselmouth"
    ])
    import parselmouth

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
OUT = PROJECT / "reports/praat_pitch_diagnostic_022420164.csv"

TARGET_ID = "022420164"
OUT.parent.mkdir(parents=True, exist_ok=True)

pf = pq.ParquetFile(TRAIN)
audio_bytes = None

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audios:
        if Path(audio["path"]).stem == TARGET_ID:
            audio_bytes = audio["bytes"]
            break

    if audio_bytes is not None:
        break

assert audio_bytes is not None

y, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float64"
)

assert sr == 16000 and y.ndim == 1

sound = parselmouth.Sound(y, sampling_frequency=sr)

configs = [
    ("praat_75_500", 75, 500),
    ("praat_60_650", 60, 650)
]

records = []

for name, floor, ceiling in configs:

    pitch = sound.to_pitch_ac(
        time_step=0.016,
        pitch_floor=floor,
        pitch_ceiling=ceiling
    )

    frequencies = pitch.selected_array["frequency"]

    voiced = frequencies[
        np.isfinite(frequencies) & (frequencies > 0)
    ]

    records.append({
        "audio_id": TARGET_ID,
        "method": name,
        "pitch_floor_hz": floor,
        "pitch_ceiling_hz": ceiling,
        "total_frames": len(frequencies),
        "voiced_frames": len(voiced),
        "voiced_ratio": (
            len(voiced) / len(frequencies)
            if len(frequencies) else np.nan
        ),
        "pitch_mean_hz": (
            float(np.mean(voiced))
            if len(voiced) else np.nan
        ),
        "pitch_sd_hz": (
            float(np.std(voiced))
            if len(voiced) else np.nan
        ),
        "pitch_min_hz": (
            float(np.min(voiced))
            if len(voiced) else np.nan
        ),
        "pitch_max_hz": (
            float(np.max(voiced))
            if len(voiced) else np.nan
        )
    })

result = pd.DataFrame(records)
result.to_csv(OUT, index=False)

print("=== PRAAT PITCH DIAGNOSTICS ===")
print("Parselmouth version:", parselmouth.__version__)
print(result.round(3).to_string(index=False))

print("\nSaved:", OUT)
print("Original acoustic batch CSV files unchanged.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

PYIN = PROJECT / "reports/pitch_failure_diagnostic_022420164.csv"
PRAAT = PROJECT / "reports/praat_pitch_diagnostic_022420164.csv"
REPORT = PROJECT / "reports/step02_pitch_failure_review.md"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert PYIN.exists()
assert PRAAT.exists()
assert CHECK.exists()

pyin = pd.read_csv(PYIN)
praat = pd.read_csv(PRAAT)

assert len(pyin) == 2
assert len(praat) == 2

report = """# STEP 2 - Pitch Extraction Failure Review

## Recording
Audio ID: 022420164
Speaker: 2242
Human fluency: 10

## Researcher listening observation
Speech was clearly audible and pronunciation sounded
very good. Background noise or room reverberation
was reported.

## pYIN diagnostics
Original F0 range: 75-500 Hz.
Expanded F0 range: 60-650 Hz.
Neither configuration detected voiced frames.

## Praat diagnostics
Praat autocorrelation detected voiced frames.

75-500 Hz:
- 43/199 voiced frames
- Mean F0: 139.285 Hz
- F0 SD: 32.528 Hz

60-650 Hz:
- 42/199 voiced frames
- Mean F0: 138.502 Hz
- F0 SD: 34.499 Hz

## Interpretation
This is an estimator-specific pitch extraction discrepancy.
The exact cause has not been established.
Praat values are diagnostic, not manually verified
ground-truth pitch estimates.

## Methodological decision
- Preserve original pYIN NaN values.
- Do not replace them with Praat measurements.
- Do not drop the recording.
- Keep pilot_acoustic_v1 unchanged.
- Handle missing pitch values within training folds.
- Consider a uniform Praat-based sensitivity analysis later.

## Status
Diagnostics completed; pilot extraction may continue.
"""

REPORT.write_text(report, encoding="utf-8")

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step2_pitch_failure_audio_id"] = "022420164"
progress["step2_pitch_failure_review"] = "completed"
progress["step2_pitch_missing_policy"] = "preserve_nan"
progress["step2_pilot_processed_audio"] = 200
progress["next_task"] = (
    "Extract acoustic features for pilot rows 0201-0300"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== PITCH FAILURE REVIEW SAVED ===")
print("Audio:", progress["step2_pitch_failure_audio_id"])
print("pYIN diagnostics:", len(pyin))
print("Praat diagnostics:", len(praat))
print("Missing pitch policy:", progress["step2_pitch_missing_policy"])
print("Pilot processed:", progress["step2_pilot_processed_audio"])
print("Report:", REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"
OUT = PROJECT / "features/acoustic/pilot_acoustic_features_0201-0300.csv"

OUT.parent.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["configuration_name"] == "pilot_acoustic_v1"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot[
    pilot["pilot_index"].between(201, 300)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(201, 301))
assert batch["audio_id"].is_unique

expected_ids = set(batch["audio_id"])
feature_columns = config["features"]

if OUT.exists():
    result = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )
    print("Existing batch found; validating instead of recomputing.")

else:
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]
    model.eval()

    pf = pq.ParquetFile(TRAIN)
    audio_data = {}

    for group_id in range(pf.metadata.num_row_groups):
        audios = pf.read_row_group(
            group_id, columns=["audio"]
        ).column("audio").to_pylist()

        for item in audios:
            audio_id = Path(item["path"]).stem
            if audio_id in expected_ids:
                audio_data[audio_id] = item["bytes"]

        if len(audio_data) == 100:
            break

    assert set(audio_data) == expected_ids

    sc = config["silero_vad"]
    ec = config["energy_segmentation"]
    pc = config["pitch"]
    records = []

    with torch.no_grad():
        for n, (_, row) in enumerate(batch.iterrows(), start=1):

            audio_id = row["audio_id"]
            y, sr = sf.read(
                io.BytesIO(audio_data[audio_id]),
                dtype="float32"
            )

            assert sr == 16000 and y.ndim == 1
            duration = len(y) / sr

            vad = get_speech_timestamps(
                torch.from_numpy(y),
                model,
                sampling_rate=sr,
                threshold=sc["threshold"],
                min_speech_duration_ms=sc["min_speech_duration_ms"],
                min_silence_duration_ms=sc["min_silence_duration_ms"],
                speech_pad_ms=sc["speech_pad_ms"]
            )

            if vad:
                pad = int(sc["hybrid_padding_ms"] * sr / 1000)
                start = max(0, vad[0]["start"] - pad)
                end = min(len(y), vad[-1]["end"] + pad)

                intervals = librosa.effects.split(
                    y[start:end],
                    top_db=ec["top_db"],
                    frame_length=ec["frame_length"],
                    hop_length=ec["hop_length"]
                )

                intervals = [
                    (int(a + start), int(b + start))
                    for a, b in intervals
                    if (b-a)/sr >= ec["min_active_segment_ms"]/1000
                ]
            else:
                intervals = []

            speech_sec = sum(
                (b-a)/sr for a, b in intervals
            )

            gaps = [
                (intervals[i+1][0]-intervals[i][1])/sr
                for i in range(len(intervals)-1)
            ]

            pauses100 = [g for g in gaps if g >= 0.1]
            pauses200 = [g for g in gaps if g >= 0.2]

            f0, voiced_flag, _ = librosa.pyin(
                y,
                sr=sr,
                fmin=pc["fmin_hz"],
                fmax=pc["fmax_hz"],
                frame_length=pc["frame_length"],
                hop_length=pc["hop_length"],
                fill_na=np.nan
            )

            voiced_f0 = f0[
                voiced_flag & np.isfinite(f0)
            ]

            rms = librosa.feature.rms(
                y=y,
                frame_length=1024,
                hop_length=256
            )[0]

            records.append({
                "pilot_index": int(row["pilot_index"]),
                "audio_id": audio_id,
                "speaker": row["speaker"],
                "fluency": row["fluency"],
                "duration_sec": duration,
                "speech_duration_sec": speech_sec,
                "speech_ratio": speech_sec / duration,
                "pause_count_100ms": len(pauses100),
                "pause_count_200ms": len(pauses200),
                "pause_duration_200ms": sum(pauses200),
                "pause_ratio_200ms": sum(pauses200)/duration,
                "leading_silence_sec": (
                    intervals[0][0]/sr if intervals else np.nan
                ),
                "trailing_silence_sec": (
                    (len(y)-intervals[-1][1])/sr
                    if intervals else np.nan
                ),
                "pitch_mean_hz": (
                    float(np.mean(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "pitch_sd_hz": (
                    float(np.std(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "voiced_ratio": len(voiced_f0)/len(f0),
                "rms_mean": float(np.mean(rms)),
                "rms_sd": float(np.std(rms))
            })

            if n % 10 == 0:
                print(f"Processed {n}/100")

    result = pd.DataFrame(records)
    result.to_csv(OUT, index=False)

# Full validation
assert len(result) == 100
assert result["pilot_index"].tolist() == list(range(201, 301))
assert result["audio_id"].is_unique
assert set(result["audio_id"]) == expected_ids
assert all(c in result.columns for c in feature_columns)

assert result["duration_sec"].gt(0).all()
assert result["speech_duration_sec"].ge(0).all()
assert (
    result["speech_duration_sec"]
    <= result["duration_sec"] + 1e-5
).all()
assert result["speech_ratio"].between(0, 1.00001).all()
assert result["voiced_ratio"].between(0, 1).all()

sha = hashlib.sha256(OUT.read_bytes()).hexdigest()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_pilot_processed_audio"] = 300
progress["step2_last_batch"] = "0201-0300"
progress["step2_last_batch_sha256"] = sha
progress["next_task"] = "Quality control for batch 0201-0300"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ACOUSTIC BATCH 0201-0300 ===")
print("Rows:", len(result))
print("Speakers:", result["speaker"].nunique())
print("Features:", len(feature_columns))
print("Missing pitch values:", result["pitch_mean_hz"].isna().sum())
print("Zero speech duration:",
      (result["speech_duration_sec"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\n=== FEATURE SUMMARY ===")
print(result[feature_columns].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", sha)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = PROJECT / "features/acoustic/pilot_acoustic_features_0201-0300.csv"
OUT = PROJECT / "reports/acoustic_qc_batch_0201-0300.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 100
assert df["pilot_index"].tolist() == list(range(201, 301))
assert df["audio_id"].is_unique

# Original QC thresholds
df["flag_long_pause"] = df["pause_duration_200ms"] >= 1.5
df["flag_low_speech_ratio"] = df["speech_ratio"] < 0.4
df["flag_low_voiced_ratio"] = df["voiced_ratio"] < 0.2
df["flag_high_pitch_sd"] = df["pitch_sd_hz"] > 70
df["flag_missing_pitch"] = (
    df["pitch_mean_hz"].isna()
    | df["pitch_sd_hz"].isna()
)

# Additional diagnostic check for batch 3
df["flag_long_trailing_silence"] = (
    df["trailing_silence_sec"] >= 2.0
)

# Numerical consistency
df["flag_invalid_duration"] = (
    (df["duration_sec"] <= 0)
    | (df["speech_duration_sec"] < 0)
    | (df["speech_duration_sec"] > df["duration_sec"] + 1e-5)
)

df["flag_invalid_pause"] = (
    (df["pause_duration_200ms"] < 0)
    | (df["pause_duration_200ms"] > df["duration_sec"] + 1e-5)
    | (df["pause_count_200ms"] < 0)
    | (df["pause_count_200ms"] > df["pause_count_100ms"])
)

flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["qc_flag_count"] = df[flag_cols].sum(axis=1)

flagged = df[df["qc_flag_count"] > 0].copy()
flagged = flagged.sort_values(
    ["qc_flag_count", "trailing_silence_sec"],
    ascending=[False, False]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
flagged.to_csv(OUT, index=False)

print("=== ACOUSTIC QC BATCH 0201-0300 ===")
print("Total recordings:", len(df))
print("Flagged recordings:", len(flagged))

print("\n=== FLAG COUNTS ===")
for col in flag_cols:
    print(col, ":", int(df[col].sum()))

display_cols = [
    "pilot_index", "audio_id", "speaker", "fluency",
    "duration_sec", "speech_ratio",
    "pause_count_200ms", "pause_duration_200ms",
    "trailing_silence_sec", "pitch_mean_hz",
    "pitch_sd_hz", "voiced_ratio", "qc_flag_count"
]

print("\n=== FLAGGED RECORDINGS ===")
print(
    flagged[display_cols]
    .round(3)
    .to_string(index=False)
)

print("\n=== TOP 5 TRAILING SILENCE ===")
print(
    df.nlargest(5, "trailing_silence_sec")[
        [
            "pilot_index", "audio_id", "speaker",
            "fluency", "duration_sec",
            "speech_duration_sec", "speech_ratio",
            "trailing_silence_sec"
        ]
    ].round(3).to_string(index=False)
)

print("\n=== MISSING PITCH DETAILS ===")
print(
    df.loc[df["flag_missing_pitch"], display_cols]
    .round(3)
    .to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_batch3_qc_completed"] = True
progress["step2_batch3_qc_flagged"] = len(flagged)
progress["next_task"] = "Review third-batch QC anomalies"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nQC CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import matplotlib.pyplot as plt
import librosa

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
FEATURES = PROJECT / "features/acoustic/pilot_acoustic_features_0201-0300.csv"

FIG_DIR = PROJECT / "results/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

target_ids = ["060100124", "069120025"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"}
)

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == set(target_ids)

print("=== CRITICAL AUDIO INSPECTION ===")

for audio_id in target_ids:

    row = pilot.loc[
        pilot["audio_id"] == audio_id
    ].iloc[0]

    feat = features.loc[
        features["audio_id"] == audio_id
    ].iloc[0]

    y, sr = audio_data[audio_id]
    duration = len(y) / sr

    print("\n" + "=" * 55)
    print("Audio ID:", audio_id)
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(duration, 3))
    print("Speech ratio:", round(feat["speech_ratio"], 3))
    print("Trailing silence:", round(
        feat["trailing_silence_sec"], 3
    ))
    print("Voiced ratio:", round(feat["voiced_ratio"], 3))

    display(Audio(y, rate=sr, normalize=False))

    rms = librosa.feature.rms(
        y=y,
        frame_length=512,
        hop_length=160
    )[0]

    rms_times = librosa.frames_to_time(
        np.arange(len(rms)),
        sr=sr,
        hop_length=160
    )

    fig, ax = plt.subplots(figsize=(12, 3))

    times = np.arange(len(y)) / sr
    ax.plot(times, y, linewidth=0.5, alpha=0.7)
    ax.set_title(f"Waveform: {audio_id}")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, duration)

    ax2 = ax.twinx()
    ax2.plot(rms_times, rms, linewidth=1.1, alpha=0.8)
    ax2.set_ylabel("RMS energy")

    fig.tight_layout()

    fig_path = FIG_DIR / f"qc_waveform_{audio_id}.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    plt.close(fig)

    print("Figure saved:", fig_path)

print("\nInspection finished.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

OUT = PROJECT / "data/metadata/critical_qc_audit_batch3.csv"
REPORT = PROJECT / "reports/step02_batch3_critical_qc_review.md"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT.parent.mkdir(parents=True, exist_ok=True)

records = [
    {
        "audio_id": "060100124",
        "pilot_index": 254,
        "fluency": 8,
        "anomaly": "long_trailing_silence",
        "researcher_observation": (
            "Long silence audible after AS I COULD"
        ),
        "waveform_observation": (
            "Main speech concentrated near beginning; "
            "later low-energy region with isolated impulses"
        ),
        "decision": "retain_original_feature_values",
        "manual_timestamp_validation": False
    },
    {
        "audio_id": "069120025",
        "pilot_index": 264,
        "fluency": 7,
        "anomaly": "pyin_pitch_failure",
        "researcher_observation": (
            "Two throat-clearing-like sounds after speech"
        ),
        "waveform_observation": (
            "Strong signal activity near 2.2-2.6 seconds; "
            "source of signals not independently confirmed"
        ),
        "decision": "preserve_pitch_nan",
        "manual_timestamp_validation": False
    }
]

audit = pd.DataFrame(records)

if OUT.exists():
    old = pd.read_csv(OUT, dtype={"audio_id": "string"})
    assert set(old["audio_id"]) == set(audit["audio_id"])

audit.to_csv(OUT, index=False, encoding="utf-8")

report = """# STEP 2 - Batch 3 Critical QC Review

## Dataset
Speechocean762 - train pilot.

## Recording 060100124
Human fluency: 8
Duration: 5.62 seconds
Hybrid speech duration: 1.00 seconds
Trailing silence: 4.50 seconds

Researcher reported long silence after the spoken phrase.
Waveform shows early speech activity followed by mostly
low-energy signal and isolated impulses.

Decision:
Retain original measurements.
No evidence sufficient to automatically relabel
this as a segmentation error.

## Recording 069120025
Human fluency: 7
pYIN voiced ratio: 0
pYIN pitch mean and SD: NaN

Researcher reported two throat-clearing-like sounds
after the phrase. Waveform shows strong later activity.

Decision:
Preserve NaN pitch values.
Do not substitute another estimator without applying
the same method consistently across recordings.

## Limitations
Observations are qualitative.
No manually timestamped reference segmentation.
No definitive cause of the pYIN failure established.

## Next
Continue pilot acoustic extraction with unchanged
pilot_acoustic_v1 configuration.
"""

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(report, encoding="utf-8")

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step2_batch3_critical_audit_count"] = 2
progress["step2_batch3_critical_review"] = "completed"
progress["step2_pilot_processed_audio"] = 300
progress["next_task"] = (
    "Extract acoustic features for pilot rows 0301-0400"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== BATCH 3 CRITICAL AUDIT SAVED ===")
print(audit[
    ["audio_id", "anomaly", "decision"]
].to_string(index=False))

print("\nAudit rows:", len(audit))
print("Pilot processed:",
      progress["step2_pilot_processed_audio"])
print("Report:", REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import librosa
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "features/acoustic/pilot_acoustic_features_0301-0400.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["configuration_name"] == "pilot_acoustic_v1"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot[
    pilot["pilot_index"].between(301, 400)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(301, 401))
assert batch["audio_id"].is_unique

expected_ids = set(batch["audio_id"])
feature_columns = config["features"]

if OUT.exists():
    result = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )
    print("Existing batch found; validating instead of recomputing.")

else:
    model, utils = torch.hub.load(
        "snakers4/silero-vad",
        "silero_vad",
        trust_repo=True
    )
    get_speech_timestamps = utils[0]
    model.eval()

    pf = pq.ParquetFile(TRAIN)
    audio_data = {}

    for group_id in range(pf.metadata.num_row_groups):
        audios = pf.read_row_group(
            group_id, columns=["audio"]
        ).column("audio").to_pylist()

        for item in audios:
            audio_id = Path(item["path"]).stem
            if audio_id in expected_ids:
                audio_data[audio_id] = item["bytes"]

        if len(audio_data) == 100:
            break

    assert set(audio_data) == expected_ids

    sc = config["silero_vad"]
    ec = config["energy_segmentation"]
    pc = config["pitch"]
    records = []

    with torch.no_grad():
        for n, (_, row) in enumerate(batch.iterrows(), start=1):
            audio_id = row["audio_id"]

            y, sr = sf.read(
                io.BytesIO(audio_data[audio_id]),
                dtype="float32"
            )

            assert sr == config["sample_rate_hz"]
            assert y.ndim == 1
            duration = len(y) / sr

            vad = get_speech_timestamps(
                torch.from_numpy(y),
                model,
                sampling_rate=sr,
                threshold=sc["threshold"],
                min_speech_duration_ms=sc["min_speech_duration_ms"],
                min_silence_duration_ms=sc["min_silence_duration_ms"],
                speech_pad_ms=sc["speech_pad_ms"]
            )

            if vad:
                pad = int(sc["hybrid_padding_ms"] * sr / 1000)
                start = max(0, vad[0]["start"] - pad)
                end = min(len(y), vad[-1]["end"] + pad)

                intervals = librosa.effects.split(
                    y[start:end],
                    top_db=ec["top_db"],
                    frame_length=ec["frame_length"],
                    hop_length=ec["hop_length"]
                )

                intervals = [
                    (int(a + start), int(b + start))
                    for a, b in intervals
                    if (b-a)/sr >= ec["min_active_segment_ms"]/1000
                ]
            else:
                intervals = []

            speech_sec = sum(
                (b-a)/sr for a, b in intervals
            )

            gaps = [
                (intervals[i+1][0]-intervals[i][1])/sr
                for i in range(len(intervals)-1)
            ]

            pauses100 = [g for g in gaps if g >= 0.1]
            pauses200 = [g for g in gaps if g >= 0.2]

            f0, voiced_flag, _ = librosa.pyin(
                y,
                sr=sr,
                fmin=pc["fmin_hz"],
                fmax=pc["fmax_hz"],
                frame_length=pc["frame_length"],
                hop_length=pc["hop_length"],
                fill_na=np.nan
            )

            voiced_f0 = f0[
                voiced_flag & np.isfinite(f0)
            ]

            rms = librosa.feature.rms(
                y=y,
                frame_length=1024,
                hop_length=256
            )[0]

            records.append({
                "pilot_index": int(row["pilot_index"]),
                "audio_id": audio_id,
                "speaker": row["speaker"],
                "fluency": row["fluency"],
                "duration_sec": duration,
                "speech_duration_sec": speech_sec,
                "speech_ratio": speech_sec / duration,
                "pause_count_100ms": len(pauses100),
                "pause_count_200ms": len(pauses200),
                "pause_duration_200ms": sum(pauses200),
                "pause_ratio_200ms": sum(pauses200)/duration,
                "leading_silence_sec": (
                    intervals[0][0]/sr if intervals else np.nan
                ),
                "trailing_silence_sec": (
                    (len(y)-intervals[-1][1])/sr
                    if intervals else np.nan
                ),
                "pitch_mean_hz": (
                    float(np.mean(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "pitch_sd_hz": (
                    float(np.std(voiced_f0))
                    if len(voiced_f0) else np.nan
                ),
                "voiced_ratio": len(voiced_f0)/len(f0),
                "rms_mean": float(np.mean(rms)),
                "rms_sd": float(np.std(rms))
            })

            if n % 10 == 0:
                print(f"Processed {n}/100")

    result = pd.DataFrame(records)
    assert len(result) == 100
    result.to_csv(OUT, index=False)

# Validation against original pilot metadata
assert len(result) == 100
assert result["pilot_index"].tolist() == list(range(301, 401))
assert result["audio_id"].is_unique
assert set(result["audio_id"]) == expected_ids
assert all(c in result.columns for c in feature_columns)

reference = batch.set_index("pilot_index")
observed = result.set_index("pilot_index")
assert (
    observed["audio_id"].tolist()
    == reference["audio_id"].tolist()
)
assert (
    observed["speaker"].astype(str).tolist()
    == reference["speaker"].astype(str).tolist()
)

assert result["duration_sec"].gt(0).all()
assert result["speech_duration_sec"].ge(0).all()
assert (
    result["speech_duration_sec"]
    <= result["duration_sec"] + 1e-5
).all()
assert result["speech_ratio"].between(0, 1.00001).all()
assert result["voiced_ratio"].between(0, 1).all()
assert result["pause_count_200ms"].le(
    result["pause_count_100ms"]
).all()

sha = hashlib.sha256(OUT.read_bytes()).hexdigest()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_pilot_processed_audio"] = 400
progress["step2_last_batch"] = "0301-0400"
progress["step2_last_batch_sha256"] = sha
progress["step2_batch4_extraction_completed"] = True
progress["next_task"] = "Quality control for batch 0301-0400"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ACOUSTIC BATCH 0301-0400 ===")
print("Rows:", len(result))
print("Speakers:", result["speaker"].nunique())
print("Features:", len(feature_columns))
print("Missing pitch values:", result["pitch_mean_hz"].isna().sum())
print("Zero speech duration:",
      (result["speech_duration_sec"] == 0).sum())
print("Audios with 200ms pauses:",
      (result["pause_count_200ms"] > 0).sum())

print("\n=== FEATURE SUMMARY ===")
print(result[feature_columns].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", sha)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "features/acoustic/pilot_acoustic_features_0301-0400.csv"
OUT = PROJECT / "reports/acoustic_qc_batch_0301-0400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 100
assert df["pilot_index"].tolist() == list(range(301, 401))
assert df["audio_id"].is_unique

# Same QC thresholds as previous batches
df["flag_long_pause"] = df["pause_duration_200ms"] >= 1.5
df["flag_low_speech_ratio"] = df["speech_ratio"] < 0.4
df["flag_low_voiced_ratio"] = df["voiced_ratio"] < 0.2
df["flag_high_pitch_sd"] = df["pitch_sd_hz"] > 70
df["flag_missing_pitch"] = (
    df["pitch_mean_hz"].isna()
    | df["pitch_sd_hz"].isna()
)
df["flag_long_trailing_silence"] = (
    df["trailing_silence_sec"] >= 2.0
)

df["flag_invalid_duration"] = (
    (df["duration_sec"] <= 0)
    | (df["speech_duration_sec"] < 0)
    | (df["speech_duration_sec"] > df["duration_sec"] + 1e-5)
)

df["flag_invalid_pause"] = (
    (df["pause_duration_200ms"] < 0)
    | (df["pause_duration_200ms"] > df["duration_sec"] + 1e-5)
    | (df["pause_count_200ms"] < 0)
    | (df["pause_count_200ms"] > df["pause_count_100ms"])
)

flag_cols = [c for c in df.columns if c.startswith("flag_")]
df["qc_flag_count"] = df[flag_cols].sum(axis=1)

flagged = df[df["qc_flag_count"] > 0].copy()
flagged = flagged.sort_values(
    ["qc_flag_count", "pause_duration_200ms"],
    ascending=[False, False]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
flagged.to_csv(OUT, index=False)

print("=== ACOUSTIC QC BATCH 0301-0400 ===")
print("Total recordings:", len(df))
print("Flagged recordings:", len(flagged))

print("\n=== FLAG COUNTS ===")
for col in flag_cols:
    print(col, ":", int(df[col].sum()))

display_cols = [
    "pilot_index", "audio_id", "speaker", "fluency",
    "duration_sec", "speech_ratio",
    "pause_count_200ms", "pause_duration_200ms",
    "trailing_silence_sec", "pitch_mean_hz",
    "pitch_sd_hz", "voiced_ratio", "qc_flag_count"
]

print("\n=== TOP 20 FLAGGED RECORDINGS ===")
print(
    flagged[display_cols]
    .head(20)
    .round(3)
    .to_string(index=False)
)

print("\n=== TOP 5 LONGEST PAUSES ===")
print(
    df.nlargest(5, "pause_duration_200ms")[
        [
            "pilot_index", "audio_id", "speaker",
            "fluency", "duration_sec",
            "pause_count_200ms", "pause_duration_200ms"
        ]
    ].round(3).to_string(index=False)
)

print("\n=== TOP 5 PITCH SD ===")
print(
    df.nlargest(5, "pitch_sd_hz")[
        ["pilot_index", "audio_id", "speaker",
         "fluency", "pitch_mean_hz",
         "pitch_sd_hz", "voiced_ratio"]
    ].round(3).to_string(index=False)
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_batch4_qc_completed"] = True
progress["step2_batch4_qc_flagged"] = len(flagged)
progress["next_task"] = "Review batch 4 anomalies and merge pilot features"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nQC CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
FEATURES = PROJECT / "features/acoustic/pilot_acoustic_features_0301-0400.csv"

OUT = PROJECT / "data/metadata/batch4_long_pause_audit_3.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

target_ids = [
    "096340020",
    "096340017",
    "096140003"
]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"}
)

pf = pq.ParquetFile(TRAIN)
audio_data = {}

for group_id in range(pf.metadata.num_row_groups):

    audios = pf.read_row_group(
        group_id,
        columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        audio_id = Path(item["path"]).stem

        if audio_id in target_ids:
            y, sr = sf.read(
                io.BytesIO(item["bytes"]),
                dtype="float32"
            )
            audio_data[audio_id] = (y, sr)

    if len(audio_data) == len(target_ids):
        break

assert set(audio_data) == set(target_ids)

records = []

print("=== BATCH 4 LONG PAUSE LISTENING ===")

for audio_id in target_ids:

    row = pilot.loc[
        pilot["audio_id"] == audio_id
    ].iloc[0]

    feat = features.loc[
        features["audio_id"] == audio_id
    ].iloc[0]

    y, sr = audio_data[audio_id]

    print("\n" + "=" * 55)
    print("Audio ID:", audio_id)
    print("Speaker:", row["speaker"])
    print("Expected text:", row["text"])
    print("Human fluency:", row["fluency"])
    print("Duration:", round(len(y) / sr, 3))
    print("Hybrid pause count:",
          int(feat["pause_count_200ms"]))
    print("Hybrid pause duration:",
          round(feat["pause_duration_200ms"], 3))

    display(Audio(y, rate=sr, normalize=False))

    records.append({
        "audio_id": audio_id,
        "speaker": row["speaker"],
        "fluency": row["fluency"],
        "hybrid_pause_count_200ms": int(
            feat["pause_count_200ms"]
        ),
        "hybrid_pause_duration_200ms": float(
            feat["pause_duration_200ms"]
        ),
        "manual_pause_observation": "pending",
        "pronunciation_observation": "pending",
        "notes": ""
    })

audit = pd.DataFrame(records)

for col in [
    "manual_pause_observation",
    "pronunciation_observation",
    "notes"
]:
    audit[col] = audit[col].astype("string")

audit.to_csv(OUT, index=False)

print("\n=== AUDIT SUMMARY ===")
print("Audio players:", len(audio_data))
print("Audit rows:", len(audit))
print("Saved:", OUT)


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

AUDIT = PROJECT / "data/metadata/batch4_long_pause_audit_3.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Listening audit CSV not found"
assert CHECK.exists(), "Research checkpoint not found"

audit = pd.read_csv(
    AUDIT,
    dtype={"audio_id": "string", "speaker": "string"}
)

annotations = {
    "096340020": {
        "manual_pause_observation": "many_pauses_audible",
        "pronunciation_observation": "slow_hesitant_speech",
        "notes": (
            "Many internal pauses audible. "
            "Speech is slow and hesitant."
        )
    },
    "096340017": {
        "manual_pause_observation": "many_pauses_audible",
        "pronunciation_observation": "slow_hesitant_speech",
        "notes": (
            "Many internal pauses audible. "
            "Speech is slow and hesitant."
        )
    },
    "096140003": {
        "manual_pause_observation": "many_pauses_audible",
        "pronunciation_observation": "slow_hesitant_speech_with_repetition",
        "notes": (
            "Many internal pauses audible. "
            "Speech is slow and hesitant. "
            "The word STRANGE was repeated twice."
        )
    }
}

assert len(audit) == 3
assert audit["audio_id"].is_unique
assert set(audit["audio_id"]) == set(annotations)

for column in [
    "manual_pause_observation",
    "pronunciation_observation",
    "notes"
]:
    audit[column] = audit[column].astype("string")

for audio_id, values in annotations.items():
    mask = audit["audio_id"] == audio_id

    for column, value in values.items():
        audit.loc[mask, column] = value

audit["audit_type"] = "informal_researcher_listening"
audit["manual_pause_timestamps_available"] = False
audit["pause_duration_manually_verified"] = False

audit.to_csv(
    AUDIT,
    index=False,
    encoding="utf-8"
)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step2_batch4_listening_audit_count"] = 3
progress["step2_batch4_listening_audit_status"] = "completed"
progress["step2_batch4_qc_review_status"] = "preliminary_pass"
progress["step2_pilot_processed_audio"] = 400
progress["next_task"] = (
    "Merge four acoustic feature batches and validate 400 recordings"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== BATCH 4 LISTENING AUDIT SAVED ===")
print(
    audit[
        [
            "audio_id",
            "speaker",
            "manual_pause_observation",
            "pronunciation_observation",
            "notes"
        ]
    ].to_string(index=False)
)

print("\nAudit rows:", len(audit))
print(
    "Pending observations:",
    (
        audit["manual_pause_observation"] == "pending"
    ).sum()
)
print("Pilot processed:", progress["step2_pilot_processed_audio"])
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

FEATURE_DIR = PROJECT / "features/acoustic"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = FEATURE_DIR / "pilot_acoustic_features_400.csv"
REPORT = PROJECT / "reports/step02_merged_validation.json"

ranges = [
    (1, 100),
    (101, 200),
    (201, 300),
    (301, 400)
]

config = json.loads(CONFIG.read_text(encoding="utf-8"))
features = config["features"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("pilot_index").reset_index(drop=True)

assert len(pilot) == 400
assert pilot["pilot_index"].tolist() == list(range(1, 401))
assert pilot["audio_id"].is_unique

batches = []
batch_hashes = {}

for start, end in ranges:
    filename = (
        f"pilot_acoustic_features_{start:04d}-{end:04d}.csv"
    )
    path = FEATURE_DIR / filename

    assert path.exists(), f"Missing batch: {filename}"

    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

    assert len(df) == 100, filename
    assert df["pilot_index"].tolist() == list(
        range(start, end + 1)
    ), filename

    assert df["audio_id"].is_unique, filename
    assert all(c in df.columns for c in features), filename

    batch_hashes[filename] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    batches.append(df)

merged = pd.concat(batches, ignore_index=True)
merged = merged.sort_values(
    "pilot_index"
).reset_index(drop=True)

# Record-level identity checks
assert len(merged) == 400
assert merged["pilot_index"].tolist() == list(range(1, 401))
assert merged["audio_id"].is_unique

assert merged["audio_id"].tolist() == pilot["audio_id"].tolist()
assert merged["speaker"].tolist() == pilot["speaker"].tolist()
assert np.allclose(
    merged["fluency"].astype(float),
    pilot["fluency"].astype(float)
)

# Speaker validation
speaker_counts = merged.groupby("speaker").size()

assert merged["speaker"].nunique() == 20
assert (speaker_counts == 20).all()

# Feature validation
assert all(c in merged.columns for c in features)
assert merged["duration_sec"].gt(0).all()

assert merged["speech_duration_sec"].ge(0).all()
assert (
    merged["speech_duration_sec"]
    <= merged["duration_sec"] + 1e-5
).all()

assert merged["speech_ratio"].between(0, 1.00001).all()
assert merged["pause_ratio_200ms"].between(0, 1.00001).all()
assert merged["voiced_ratio"].between(0, 1).all()

assert (
    merged["pause_count_200ms"]
    <= merged["pause_count_100ms"]
).all()

# Validate pitch missingness consistently
missing_pitch = merged["pitch_mean_hz"].isna()

assert missing_pitch.equals(
    merged["pitch_sd_hz"].isna()
)

assert (
    (merged.loc[missing_pitch, "voiced_ratio"] == 0)
).all()

# Metadata completeness
assert merged["audio_id"].notna().all()
assert merged["speaker"].notna().all()
assert merged["fluency"].notna().all()

# Keep the original feature values intact
merged.to_csv(OUT, index=False)

merged_hash = hashlib.sha256(
    OUT.read_bytes()
).hexdigest()

validation = {
    "dataset": "Speechocean762",
    "configuration": config["configuration_name"],
    "rows": int(len(merged)),
    "unique_audio_ids": int(merged["audio_id"].nunique()),
    "unique_speakers": int(merged["speaker"].nunique()),
    "features": len(features),
    "missing_pitch_count": int(missing_pitch.sum()),
    "missing_pitch_audio_ids": merged.loc[
        missing_pitch, "audio_id"
    ].tolist(),
    "zero_speech_duration": int(
        (merged["speech_duration_sec"] == 0).sum()
    ),
    "audios_with_200ms_pauses": int(
        (merged["pause_count_200ms"] > 0).sum()
    ),
    "batch_sha256": batch_hashes,
    "merged_sha256": merged_hash,
    "status": "PASS",
    "validated_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(validation, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step2_merged_audio_count"] = 400
progress["step2_merged_validation"] = "PASS"
progress["step2_merged_sha256"] = merged_hash
progress["next_task"] = "Prepare final Step 2 report and backup"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 2 MERGED VALIDATION ===")
print("Rows:", validation["rows"])
print("Unique audio IDs:", validation["unique_audio_ids"])
print("Speakers:", validation["unique_speakers"])
print("Features:", validation["features"])
print("Missing pitch:", validation["missing_pitch_count"])
print("Missing pitch IDs:", validation["missing_pitch_audio_ids"])
print(
    "Audios with 200ms pauses:",
    validation["audios_with_200ms_pauses"]
)
print("Zero speech duration:", validation["zero_speech_duration"])
print("Validation:", validation["status"])

print("\n=== MERGED FEATURE SUMMARY ===")
print(merged[features].describe().round(3).to_string())

print("\nCSV:", OUT)
print("SHA256:", merged_hash)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
FEATURES = PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
CONFIG = PROJECT / "configs/pilot_acoustic_v1.json"
VALIDATION = PROJECT / "reports/step02_merged_validation.json"
CHECK = PROJECT / "checkpoints/research_progress.json"
REPORT = PROJECT / "reports/step02_final_report.md"

ZIP = Path("/kaggle/working/asr_fluency_step02_complete.zip")

df = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"}
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
validation = json.loads(VALIDATION.read_text(encoding="utf-8"))
progress = json.loads(CHECK.read_text(encoding="utf-8"))

assert len(df) == 400
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert validation["status"] == "PASS"
assert validation["rows"] == 400
assert len(config["features"]) == 14

missing_ids = df.loc[
    df["pitch_mean_hz"].isna(), "audio_id"
].tolist()

assert set(missing_ids) == {
    "022420164", "069120025"
}

# Confirm that no analysis or training has been performed
report = f"""# STEP 2 - Final Acoustic Feature Extraction Report

## Research
Within-Utterance ASR Transcription Variability and the
Stability of Human-Aligned L2 Fluency Scoring:
An Acoustic-Linguistic Comparison

## Dataset
Speechocean762
Official train split only
Dataset revision: 06385584fad212b26134c656fdd3ccf9f093f33e

## Pilot subset
Audio recordings: {len(df)}
Unique speakers: {df["speaker"].nunique()}
Recordings per speaker: 20

## Acoustic extraction
Configuration: {config["configuration_name"]}
Features per recording: {len(config["features"])}

Methods:
- Silero VAD + Librosa energy segmentation
- librosa.pyin voiced pitch estimation
- RMS energy statistics

## Quality control
Recordings with internal pauses >=200ms:
{int((df["pause_count_200ms"] > 0).sum())}

Missing pitch: {len(missing_ids)}
Missing audio IDs: {", ".join(missing_ids)}

Original missing values retained.
No imputation performed.

## Descriptive statistics
Mean duration: {df["duration_sec"].mean():.4f} sec
Mean speech ratio: {df["speech_ratio"].mean():.4f}
Mean pause count >=200ms: {df["pause_count_200ms"].mean():.4f}
Mean pause duration: {df["pause_duration_200ms"].mean():.4f} sec
Mean pitch: {df["pitch_mean_hz"].mean():.4f} Hz

## Listening quality assurance
20 preliminary recordings reviewed.
Additional targeted long-pause audits completed.
No expert timestamped pause ground truth.

## Methodological limitations
- Pause detection is not equivalent to hesitation detection.
- Audio leading/trailing silence may reflect recording conditions.
- Pitch estimates may include octave errors.
- Some pYIN recordings have missing pitch values.
- Prompt text is not a verbatim human transcript.
- Pilot QA does not establish general segmentation accuracy.

## Data integrity
Unique audio IDs: PASS
Speaker counts: PASS
Metadata identity: PASS
Feature consistency: PASS
Merged validation: PASS

## Training and test data
No ML training has been conducted in Step 2.
Official test set remains reserved.

## Next research stage
STEP 3: ASR hypothesis generation and transcription
variability analysis (subject to research plan confirmation).
"""

REPORT.write_text(report, encoding="utf-8")

# Mark STEP 2 complete
completed = progress.get("completed_steps", [])
completed = sorted(set(int(x) for x in completed) | {0, 1, 2})

progress["completed_steps"] = completed
progress["current_step"] = 3
progress["step2_status"] = "completed"
progress["step2_pilot_processed_audio"] = 400
progress["step2_final_report"] = str(REPORT)
progress["step2_final_feature_file"] = str(FEATURES)
progress["next_task"] = (
    "Review STEP 3 research design before running ASR"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Backup all small research files, excluding raw audio/parquet
included = []

with zipfile.ZipFile(
    ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    for folder_name in [
        "data/metadata",
        "configs",
        "features/acoustic",
        "reports",
        "checkpoints",
        "results/figures"
    ]:
        folder = PROJECT / folder_name
        if not folder.exists():
            continue

        for file in sorted(folder.rglob("*")):
            if not file.is_file():
                continue

            if file.suffix.lower() in {
                ".parquet", ".wav", ".mp3", ".flac", ".zip"
            }:
                continue

            archive.write(
                file,
                arcname=str(file.relative_to(PROJECT))
            )
            included.append(str(file.relative_to(PROJECT)))

with zipfile.ZipFile(ZIP) as archive:
    assert archive.testzip() is None
    archived = set(archive.namelist())

    required = {
        "features/acoustic/pilot_acoustic_features_400.csv",
        "configs/pilot_acoustic_v1.json",
        "reports/step02_final_report.md",
        "reports/step02_merged_validation.json",
        "checkpoints/research_progress.json",
        "data/metadata/pilot_selection_400.csv"
    }

    assert required.issubset(archived)

sha256 = hashlib.sha256(ZIP.read_bytes()).hexdigest()

print("=== STEP 2 FINAL BACKUP ===")
print("Step 2 status:", progress["step2_status"])
print("Completed steps:", progress["completed_steps"])
print("Pilot audio:", len(df))
print("Speakers:", df["speaker"].nunique())
print("Features:", len(config["features"]))
print("Missing pitch:", len(missing_ids))
print("Archived files:", len(included))
print("ZIP:", ZIP)
print("ZIP size (MB):", round(ZIP.stat().st_size / 1048576, 3))
print("ZIP SHA256:", sha256)
print("ZIP integrity: PASS")
print("Next task:", progress["next_task"])


In [ ]:

from pathlib import Path
import hashlib
import zipfile

INPUT = Path("/kaggle/input")

print("=== STEP 3.1: BACKUP DISCOVERY ===")

print("\nAvailable Kaggle datasets:")
for folder in sorted(INPUT.iterdir()):
    print(" -", folder.name)

print("\n=== STEP 2 BACKUP CANDIDATES ===")

candidates = sorted(
    p for p in INPUT.rglob("*")
    if p.is_file()
    and (
        "asr_fluency_step02_complete" in p.name.lower()
        or p.name.lower() == "research_progress.json"
    )
)

for p in candidates:
    print("Path:", p)
    print("Size:", p.stat().st_size, "bytes")

    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p) as z:
            bad_file = z.testzip()
            print("ZIP integrity:",
                  "PASS" if bad_file is None else f"FAIL: {bad_file}")
            print("Archive entries:", len(z.namelist()))
            print(
                "Checkpoint exists:",
                "checkpoints/research_progress.json" in z.namelist()
            )

        sha = hashlib.sha256(p.read_bytes()).hexdigest()
        print("SHA256:", sha)

    print("-" * 50)

print("\nTotal candidates:", len(candidates))


In [ ]:

from pathlib import Path
import shutil
import json
import pandas as pd

SOURCE = Path(
    "/kaggle/input/datasets/abdulatifsafarov/"
    "asr-fluency-step02-complete"
)

PROJECT = Path(
    "/kaggle/working/acoustic_linguistic_research"
)

assert SOURCE.exists(), "Kaggle dataset not found"

required = [
    "checkpoints/research_progress.json",
    "configs/pilot_acoustic_v1.json",
    "data/metadata/pilot_selection_400.csv",
    "features/acoustic/pilot_acoustic_features_400.csv",
    "reports/step02_merged_validation.json"
]

print("=== STEP 3.1: DATASET RECOVERY ===")

for relative in required:
    assert (SOURCE / relative).is_file(), (
        f"Missing source file: {relative}"
    )

copied = 0
verified = 0

for src in sorted(SOURCE.rglob("*")):
    if not src.is_file():
        continue

    relative = src.relative_to(SOURCE)
    dst = PROJECT / relative
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        import hashlib
        def sha256(p):
            return hashlib.sha256(p.read_bytes()).hexdigest()

        assert sha256(src) == sha256(dst), (
            f"Existing file differs: {relative}"
        )
        verified += 1
    else:
        shutil.copy2(src, dst)
        copied += 1

checkpoint = json.loads(
    (PROJECT / "checkpoints/research_progress.json")
    .read_text(encoding="utf-8")
)

config = json.loads(
    (PROJECT / "configs/pilot_acoustic_v1.json")
    .read_text(encoding="utf-8")
)

validation = json.loads(
    (PROJECT / "reports/step02_merged_validation.json")
    .read_text(encoding="utf-8")
)

df = pd.read_csv(
    PROJECT / "features/acoustic/pilot_acoustic_features_400.csv",
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 400
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert len(config["features"]) == 14
assert all(col in df.columns for col in config["features"])
assert df["pitch_mean_hz"].isna().sum() == 2
assert validation["status"] == "PASS"
assert 2 in checkpoint["completed_steps"]

print("Files copied:", copied)
print("Already existing identical files:", verified)
print("Audio records:", len(df))
print("Speakers:", df["speaker"].nunique())
print("Acoustic features:", len(config["features"]))
print("Missing pitch:", df["pitch_mean_hz"].isna().sum())
print("Completed steps:", checkpoint["completed_steps"])
print("Step 2 validation:", validation["status"])
print("Project restored to:", PROJECT)
print("=== RECOVERY PASS ===")


In [ ]:

from pathlib import Path
import torch
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

expected = PROJECT / "data/raw/speechocean762_train.parquet"

print("=== STEP 3.1: AUDIO DATASET CHECK ===")

print("\nOriginal train Parquet exists:", expected.exists())

if expected.exists():
    print("Path:", expected)
    print("Size (MB):", round(expected.stat().st_size / 1048576, 2))

    pf = pq.ParquetFile(expected)
    print("Total rows:", pf.metadata.num_rows)
    print("Row groups:", pf.metadata.num_row_groups)
    print("Columns:", pf.schema.names)

    assert pf.metadata.num_rows == 2500
    assert "audio" in pf.schema.names

    print("Audio dataset: READY")
else:
    print("Original audio dataset: MISSING")
    print("Audio restoration will be required.")

print("\n=== GPU STATUS ===")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\n=== OTHER PARQUET FILES ===")
for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    matches = list(root.rglob("*.parquet"))
    print(f"{root}: {len(matches)} parquet file(s)")
    for p in matches[:10]:
        print(" -", p)

print("\n=== AUDIO CHECK COMPLETED ===")


In [ ]:

import sys
import subprocess
import shutil
import hashlib
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", "huggingface_hub"
    ])
    from huggingface_hub import HfApi, hf_hub_download

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)

TARGET = RAW / "speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

REPO = "mispeech/speechocean762"
REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

print("=== STEP 3.1: TRAIN AUDIO RESTORATION ===")

if TARGET.exists():
    print("Existing Parquet found; validating it.")
else:
    api = HfApi()
    info = api.dataset_info(repo_id=REPO, revision=REVISION)

    parquet_files = [
        item.rfilename
        for item in info.siblings
        if item.rfilename.endswith(".parquet")
    ]

    train_files = [
        name for name in parquet_files
        if (
            name.startswith("data/train-")
            or name.startswith("train/")
            or Path(name).name.startswith("train-")
        )
    ]

    print("Train Parquet candidates:", train_files)

    assert len(train_files) == 1, (
        "Expected exactly one train Parquet. "
        "Inspect repository file list before continuing."
    )

    source = hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=train_files[0],
        revision=REVISION
    )

    shutil.copy2(source, TARGET)

pf = pq.ParquetFile(TARGET)

assert pf.metadata.num_rows == 2500, (
    f"Unexpected row count: {pf.metadata.num_rows}"
)
assert "audio" in pf.schema.names

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

expected_ids = set(pilot["audio_id"])
found_ids = set()

for group_id in range(pf.metadata.num_row_groups):
    audio_col = pf.read_row_group(
        group_id,
        columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audio_col:
        audio_id = Path(audio["path"]).stem
        if audio_id in expected_ids:
            found_ids.add(audio_id)

assert len(expected_ids) == 400
assert found_ids == expected_ids, (
    f"Missing pilot IDs: {sorted(expected_ids-found_ids)[:10]}"
)

sha = hashlib.sha256()
with TARGET.open("rb") as f:
    for chunk in iter(lambda: f.read(1024*1024), b""):
        sha.update(chunk)

print("\n=== TRAIN AUDIO RESTORATION RESULT ===")
print("Path:", TARGET)
print("Size (MB):", round(TARGET.stat().st_size / 1048576, 2))
print("Train rows:", pf.metadata.num_rows)
print("Pilot audio IDs found:", len(found_ids))
print("Pilot audio IDs missing:", len(expected_ids-found_ids))
print("SHA256:", sha.hexdigest())
print("Status: PASS")


In [ ]:

from pathlib import Path
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TARGET = PROJECT / "data/raw/speechocean762_train.parquet"

assert TARGET.exists(), "Train Parquet file not found"

pf = pq.ParquetFile(TARGET)

print("=== PARQUET SCHEMA DIAGNOSTICS ===")

print("\nFile:", TARGET)
print("Size (MB):", round(TARGET.stat().st_size / 1048576, 2))
print("Rows:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)

print("\n=== PARQUET LEAF COLUMN NAMES ===")
print(pf.schema.names)

print("\n=== ARROW TOP-LEVEL COLUMN NAMES ===")
print(pf.schema_arrow.names)

print("\n=== FULL ARROW SCHEMA ===")
print(pf.schema_arrow)

print("\n=== AUDIO COLUMN CHECK ===")

if "audio" in pf.schema_arrow.names:
    print("Audio top-level column: FOUND")

    audio_field = pf.schema_arrow.field("audio")
    print("Audio field type:", audio_field.type)

    sample = pf.read_row_group(
        0, columns=["audio"]
    ).column("audio").slice(0, 1).to_pylist()

    if sample:
        print("First audio value type:", type(sample[0]).__name__)

        if isinstance(sample[0], dict):
            print("Audio struct keys:", list(sample[0].keys()))
            print("Audio path:", sample[0].get("path"))

            audio_bytes = sample[0].get("bytes")
            print(
                "Embedded audio bytes:",
                len(audio_bytes) if audio_bytes is not None else None
            )
else:
    print("Audio top-level column: NOT FOUND")
    print("Further schema investigation required.")

print("\n=== SCHEMA DIAGNOSTICS COMPLETE ===")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"
OUT = PROJECT / "reports/step03_audio_restoration_validation.json"

EXPECTED_SHA256 = (
    "b39d1a49b7279c7f5f0c1e7efaa72a20"
    "d1cde92738ac67cf84beabf756fe1901"
)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(pilot) == 400
assert pilot["audio_id"].is_unique
assert pilot["pilot_index"].tolist() == list(range(1, 401))

pf = pq.ParquetFile(TRAIN)

assert pf.metadata.num_rows == 2500
assert "audio" in pf.schema_arrow.names

expected = pilot.set_index("audio_id")
found = {}

for group_id in range(pf.metadata.num_row_groups):

    table = pf.read_row_group(
        group_id,
        columns=["audio", "speaker", "fluency"]
    )

    audio_values = table.column("audio").to_pylist()
    speakers = table.column("speaker").to_pylist()
    fluencies = table.column("fluency").to_pylist()

    for audio, speaker, fluency in zip(
        audio_values, speakers, fluencies
    ):
        audio_id = Path(audio["path"]).stem

        if audio_id in expected.index:

            assert audio_id not in found, (
                f"Duplicate audio ID: {audio_id}"
            )
            assert audio["bytes"] is not None

            found[audio_id] = {
                "speaker": str(speaker),
                "fluency": int(fluency),
                "audio_bytes": len(audio["bytes"])
            }

expected_ids = set(expected.index)
found_ids = set(found)

assert found_ids == expected_ids, (
    f"Missing audio IDs: {sorted(expected_ids - found_ids)}"
)

speaker_mismatches = []
fluency_mismatches = []

for audio_id, record in found.items():
    metadata = expected.loc[audio_id]

    if record["speaker"] != str(metadata["speaker"]):
        speaker_mismatches.append(audio_id)

    if record["fluency"] != int(metadata["fluency"]):
        fluency_mismatches.append(audio_id)

assert not speaker_mismatches
assert not fluency_mismatches

sha = hashlib.sha256()

with TRAIN.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(chunk)

actual_sha256 = sha.hexdigest()

report = {
    "dataset": "mispeech/speechocean762",
    "revision": "06385584fad212b26134c656fdd3ccf9f093f33e",
    "train_rows": 2500,
    "pilot_audio_count": len(found),
    "missing_audio_ids": [],
    "speaker_mismatches": speaker_mismatches,
    "fluency_mismatches": fluency_mismatches,
    "train_parquet_sha256": actual_sha256,
    "matches_previous_sha256": (
        actual_sha256 == EXPECTED_SHA256
    ),
    "status": "PASS",
    "checked_at": datetime.now(timezone.utc).isoformat()
}

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_audio_restoration"] = "PASS"
progress["step3_audio_restoration_report"] = str(OUT)
progress["next_task"] = "Design ASR hypothesis experiment"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.1 AUDIO RESTORATION VALIDATION ===")
print("Train recordings:", pf.metadata.num_rows)
print("Pilot recordings matched:", len(found))
print("Missing pilot IDs:", len(expected_ids - found_ids))
print("Speaker mismatches:", len(speaker_mismatches))
print("Fluency mismatches:", len(fluency_mismatches))
print("Parquet SHA256:", actual_sha256)
print("Previous SHA256 match:",
      report["matches_previous_sha256"])
print("Status:", report["status"])
print("Report:", OUT)
print("Checkpoint updated.")


In [ ]:

import sys
import importlib.metadata as metadata

import torch
import pandas as pd

from pathlib import Path

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

print("=== STEP 3.2: ASR ENVIRONMENT DIAGNOSTICS ===")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)

        print(
            f"GPU {i}: {props.name}, "
            f"VRAM {props.total_memory / 1024**3:.2f} GiB"
        )

packages = [
    "transformers",
    "accelerate",
    "datasets",
    "openai-whisper",
    "faster-whisper",
    "ctranslate2",
    "librosa",
    "soundfile",
    "huggingface-hub"
]

print("\n=== PACKAGE VERSIONS ===")

for name in packages:
    try:
        version = metadata.version(name)
    except metadata.PackageNotFoundError:
        version = "NOT INSTALLED"

    print(f"{name}: {version}")

print("\n=== RESEARCH FILES ===")

files = [
    "features/acoustic/pilot_acoustic_features_400.csv",
    "data/metadata/pilot_selection_400.csv",
    "data/raw/speechocean762_train.parquet",
    "checkpoints/research_progress.json"
]

for relative in files:
    path = PROJECT / relative
    print(relative, ":", "FOUND" if path.exists() else "MISSING")

print("\n=== ASR ENVIRONMENT CHECK COMPLETED ===")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

OUT = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT.parent.mkdir(parents=True, exist_ok=True)

config = {
    "configuration_name": "asr_hypothesis_pilot_v1",
    "status": "proposed_not_frozen",
    "dataset": "mispeech/speechocean762",
    "dataset_revision": (
        "06385584fad212b26134c656fdd3ccf9f093f33e"
    ),
    "split": "train",
    "model_id": "openai/whisper-small.en",
    "language": "english",
    "task": "transcribe",
    "pilot_audio_count": 10,
    "hypotheses": [
        {
            "id": "H1",
            "method": "greedy",
            "num_beams": 1,
            "do_sample": False
        },
        {
            "id": "H2",
            "method": "beam_search",
            "num_beams": 5,
            "do_sample": False
        },
        {
            "id": "H3",
            "method": "sampling",
            "do_sample": True,
            "temperature": 0.3,
            "seed": 101
        },
        {
            "id": "H4",
            "method": "sampling",
            "do_sample": True,
            "temperature": 0.3,
            "seed": 202
        },
        {
            "id": "H5",
            "method": "sampling",
            "do_sample": True,
            "temperature": 0.3,
            "seed": 303
        }
    ],
    "methodological_rules": {
        "use_prompt_text_as_asr_context": False,
        "use_human_fluency_for_asr_selection": False,
        "use_official_test_split": False,
        "preserve_raw_hypotheses": True,
        "evaluate_variability_before_full_run": True
    },
    "created_at": datetime.now(timezone.utc).isoformat()
}

if OUT.exists():
    previous = json.loads(OUT.read_text(encoding="utf-8"))

    def experiment_definition(x):
        return {
            k: v for k, v in x.items()
            if k != "created_at"
        }

    assert experiment_definition(previous) == experiment_definition(config), (
        "Existing ASR config differs. "
        "Do not overwrite without reviewing the changes."
    )

    print("Existing identical configuration found.")
else:
    OUT.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_asr_config_status"] = "proposed_not_frozen"
progress["step3_asr_pilot_config"] = str(OUT)
progress["next_task"] = (
    "Validate Whisper-small.en model compatibility and decoding support"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.2 ASR DESIGN SAVED ===")
print("Model:", config["model_id"])
print("Pilot recordings:", config["pilot_audio_count"])
print("Hypotheses:", len(config["hypotheses"]))

for h in config["hypotheses"]:
    print(h["id"], "-", h["method"])

print("Config status:", config["status"])
print("Config file:", OUT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import json
import inspect
from pathlib import Path

import torch
import transformers

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    GenerationConfig
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

print("=== STEP 3.2: WHISPER API COMPATIBILITY ===")

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Model:", config["model_id"])
print("Configuration status:", config["status"])

print("\n=== MODEL CLASS ===")
print("Class:", WhisperForConditionalGeneration.__name__)
print("Processor:", WhisperProcessor.__name__)

signature = inspect.signature(
    WhisperForConditionalGeneration.generate
)

params = signature.parameters

print("\n=== GENERATE SIGNATURE ===")
print(signature)

print("\n=== GENERATION PARAMETERS ===")

for name in [
    "generation_config",
    "logits_processor",
    "stopping_criteria",
    "return_dict_in_generate",
    "kwargs"
]:
    print(name, ":", name in params)

print("\n=== GENERATION CONFIG SUPPORT ===")

gc = GenerationConfig()

for name, value in [
    ("do_sample", True),
    ("temperature", 0.3),
    ("num_beams", 5),
    ("max_new_tokens", 128)
]:
    setattr(gc, name, value)
    print(name, "=", getattr(gc, name))

print("\n=== EXPERIMENT VALIDATION ===")

hypotheses = config["hypotheses"]

assert len(hypotheses) == 5
assert len({h["id"] for h in hypotheses}) == 5

assert hypotheses[0]["method"] == "greedy"
assert hypotheses[1]["method"] == "beam_search"

assert all(
    h["method"] == "sampling"
    for h in hypotheses[2:]
)

seeds = [h["seed"] for h in hypotheses[2:]]
assert len(set(seeds)) == 3

print("Hypotheses:", len(hypotheses))
print("Sampling seeds:", seeds)
print("Static config validation: PASS")

print("\nNOTE:")
print("This check verifies API structure only.")
print("Actual generation behavior is not yet verified.")

print("\n=== API COMPATIBILITY CHECK COMPLETED ===")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

from huggingface_hub import HfApi
from transformers import AutoConfig, AutoProcessor

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG_FILE = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
REPORT = PROJECT / "reports/step03_whisper_model_revision.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
assert config["status"] == "proposed_not_frozen"

model_id = config["model_id"]

print("=== STEP 3.2: WHISPER MODEL REVISION CHECK ===")

api = HfApi()
model_info = api.model_info(repo_id=model_id)

revision = model_info.sha

assert revision is not None
assert len(revision) == 40

print("Model:", model_id)
print("Exact revision:", revision)

model_config = AutoConfig.from_pretrained(
    model_id,
    revision=revision
)

processor = AutoProcessor.from_pretrained(
    model_id,
    revision=revision
)

print("\n=== MODEL METADATA ===")
print("Model type:", model_config.model_type)
print("Encoder layers:", model_config.encoder_layers)
print("Decoder layers:", model_config.decoder_layers)
print("Max source positions:", model_config.max_source_positions)
print("Feature extractor sample rate:",
      processor.feature_extractor.sampling_rate)

assert model_config.model_type == "whisper"
assert processor.feature_extractor.sampling_rate == 16000

# Do not silently replace an existing pinned revision.
if "model_revision" in config:
    assert config["model_revision"] == revision, (
        "Previously pinned model revision differs. "
        "Review before proceeding."
    )
else:
    config["model_revision"] = revision

CONFIG_FILE.write_text(
    json.dumps(config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

report = {
    "model_id": model_id,
    "model_revision": revision,
    "model_type": model_config.model_type,
    "encoder_layers": model_config.encoder_layers,
    "decoder_layers": model_config.decoder_layers,
    "sample_rate": processor.feature_extractor.sampling_rate,
    "transformers_version": "5.16.1",
    "status": "PASS",
    "checked_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_whisper_revision"] = revision
progress["step3_model_metadata_validation"] = "PASS"
progress["next_task"] = (
    "Load pinned Whisper model and run a small inference test"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== MODEL REVISION SAVED ===")
print("Revision:", revision)
print("Config status:", config["status"])
print("Report:", REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
import transformers

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
OUT = PROJECT / "features/asr/asr_smoke_test_096140003_H1.csv"
REPORT = PROJECT / "reports/step03_whisper_smoke_test.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))
MODEL_ID = config["model_id"]
REVISION = config["model_revision"]
TARGET_ID = "096140003"

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU is required for this test"

device = "cuda:0"

print("=== STEP 3.2: WHISPER GREEDY SMOKE TEST ===")
print("Model:", MODEL_ID)
print("Revision:", REVISION)
print("Device:", device)

# Locate exactly one audio from train split
pf = pq.ParquetFile(TRAIN)
matches = []

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        if Path(item["path"]).stem == TARGET_ID:
            matches.append(item["bytes"])

assert len(matches) == 1, (
    f"Expected exactly one audio, found {len(matches)}"
)

y, sr = sf.read(io.BytesIO(matches[0]), dtype="float32")

assert sr == 16000 and y.ndim == 1
assert np.isfinite(y).all()

print("Audio ID:", TARGET_ID)
print("Duration:", round(len(y) / sr, 3), "seconds")

# Load the pinned processor and model
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    torch_dtype=torch.float16
)

model = model.to(device)
model.eval()

inputs = processor(
    y,
    sampling_rate=sr,
    return_tensors="pt"
)

input_features = inputs.input_features.to(
    device=device,
    dtype=torch.float16
)

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

start = time.perf_counter()

with torch.inference_mode():
    generated_ids = model.generate(
        input_features,
        do_sample=False,
        num_beams=1,
        max_new_tokens=128,
        task="transcribe"
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start

transcription = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0].strip()

result = pd.DataFrame([{
    "audio_id": TARGET_ID,
    "hypothesis_id": "H1",
    "method": "greedy",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "transcription": transcription,
    "duration_sec": len(y) / sr,
    "inference_sec": elapsed
}])

result.to_csv(OUT, index=False, encoding="utf-8")

report = {
    "audio_id": TARGET_ID,
    "hypothesis_id": "H1",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "device": torch.cuda.get_device_name(0),
    "dtype": "float16",
    "do_sample": False,
    "num_beams": 1,
    "max_new_tokens": 128,
    "generated_token_count": int(generated_ids.shape[-1]),
    "transcription": transcription,
    "inference_sec": elapsed,
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1 GREEDY RESULT ===")
print("Audio ID:", TARGET_ID)
print("Transcription:", repr(transcription))
print("Generated tokens:", generated_ids.shape[-1])
print("Inference time:", round(elapsed, 3), "seconds")
print("Output CSV:", OUT)
print("Report:", REPORT)
print("Status: COMPLETED")


In [ ]:

import io
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
import transformers

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
OUT = PROJECT / "features/asr/asr_smoke_test_096140003_H1.csv"
REPORT = PROJECT / "reports/step03_whisper_smoke_test.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

MODEL_ID = config["model_id"]
REVISION = config["model_revision"]
TARGET_ID = "096140003"

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

assert MODEL_ID == "openai/whisper-small.en"
assert torch.cuda.is_available()

device = "cuda:0"

print("=== STEP 3.2: CORRECTED GREEDY SMOKE TEST ===")
print("Model:", MODEL_ID)
print("Revision:", REVISION)

# Load target audio
pf = pq.ParquetFile(TRAIN)
matches = []

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        if Path(item["path"]).stem == TARGET_ID:
            matches.append(item["bytes"])

assert len(matches) == 1

y, sr = sf.read(
    io.BytesIO(matches[0]),
    dtype="float32"
)

assert sr == 16000
assert y.ndim == 1
assert np.isfinite(y).all()

print("Audio ID:", TARGET_ID)
print("Duration:", round(len(y) / sr, 3), "seconds")

# Load processor and model
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    dtype=torch.float16
).to(device)

model.eval()

inputs = processor(
    y,
    sampling_rate=sr,
    return_tensors="pt"
)

input_features = inputs.input_features.to(
    device=device,
    dtype=torch.float16
)

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    generated_ids = model.generate(
        input_features,
        do_sample=False,
        num_beams=1,
        max_new_tokens=128
        # No task or language for English-only Whisper
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

transcription = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0].strip()

assert isinstance(transcription, str)

result = pd.DataFrame([{
    "audio_id": TARGET_ID,
    "hypothesis_id": "H1",
    "method": "greedy",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "transcription": transcription,
    "duration_sec": len(y) / sr,
    "inference_sec": elapsed
}])

result.to_csv(OUT, index=False, encoding="utf-8")

report = {
    "audio_id": TARGET_ID,
    "hypothesis_id": "H1",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "device": torch.cuda.get_device_name(0),
    "dtype": "float16",
    "do_sample": False,
    "num_beams": 1,
    "max_new_tokens": 128,
    "task_argument": None,
    "language_argument": None,
    "generated_token_count": int(generated_ids.shape[-1]),
    "transcription": transcription,
    "inference_sec": elapsed,
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_h1_smoke_test"] = "completed"
progress["step3_h1_smoke_test_audio"] = TARGET_ID
progress["next_task"] = (
    "Evaluate H1 transcription and test beam search H2"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1 GREEDY RESULT ===")
print("Audio ID:", TARGET_ID)
print("Transcription:", repr(transcription))
print("Generated tokens:", generated_ids.shape[-1])
print("Inference time:", round(elapsed, 3), "seconds")
print("Output CSV:", OUT)
print("Report:", REPORT)
print("Checkpoint updated.")
print("Status: COMPLETED")


In [ ]:

import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
H1_FILE = PROJECT / "features/asr/asr_smoke_test_096140003_H1.csv"
H2_FILE = PROJECT / "features/asr/asr_smoke_test_096140003_H2.csv"
REPORT = PROJECT / "reports/step03_whisper_beam_test.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert "model" in globals(), "Whisper model not loaded"
assert "processor" in globals(), "Whisper processor not loaded"
assert "input_features" in globals(), "Audio features not loaded"

assert H1_FILE.exists()

h1 = pd.read_csv(H1_FILE, keep_default_na=False).iloc[0]
assert h1["audio_id"] == "096140003"
assert h1["hypothesis_id"] == "H1"

print("=== STEP 3.2: H2 BEAM SEARCH TEST ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Audio ID:", h1["audio_id"])

model.eval()
torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    generated_ids_h2 = model.generate(
        input_features,
        do_sample=False,
        num_beams=5,
        max_new_tokens=128
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

transcription_h2 = processor.batch_decode(
    generated_ids_h2,
    skip_special_tokens=True
)[0].strip()

transcription_h1 = h1["transcription"]

h2 = pd.DataFrame([{
    "audio_id": "096140003",
    "hypothesis_id": "H2",
    "method": "beam_search",
    "num_beams": 5,
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "transcription": transcription_h2,
    "inference_sec": elapsed
}])

H2_FILE.parent.mkdir(parents=True, exist_ok=True)
H2_FILE.to_csv(H2_FILE, index=False)

report = {
    "audio_id": "096140003",
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "h1_transcription": transcription_h1,
    "h2_transcription": transcription_h2,
    "h1_h2_identical": transcription_h1 == transcription_h2,
    "h2_num_beams": 5,
    "h2_inference_sec": elapsed,
    "status": "completed",
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_h2_smoke_test"] = "completed"
progress["next_task"] = (
    "Compare H1/H2 and test sampling H3-H5"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1 VS H2 COMPARISON ===")
print("H1 (greedy):", repr(transcription_h1))
print("H2 (beam):", repr(transcription_h2))
print("Identical:", report["h1_h2_identical"])
print("Beam inference time:", round(elapsed, 3), "seconds")
print("H2 CSV:", H2_FILE)
print("Checkpoint updated.")


In [ ]:

import json
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
H1_FILE = PROJECT / "features/asr/asr_smoke_test_096140003_H1.csv"
H2_FILE = PROJECT / "features/asr/asr_smoke_test_096140003_H2.csv"
REPORT = PROJECT / "reports/step03_whisper_beam_test.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert "model" in globals(), "Whisper model not loaded"
assert "processor" in globals(), "Whisper processor not loaded"
assert "input_features" in globals(), "Audio features not loaded"

assert H1_FILE.exists()

# IMPORTANT: Preserve leading zeros in audio IDs.
h1_df = pd.read_csv(
    H1_FILE,
    dtype={"audio_id": "string"},
    keep_default_na=False
)

assert len(h1_df) == 1

h1 = h1_df.iloc[0]

assert str(h1["audio_id"]) == "096140003"
assert h1["hypothesis_id"] == "H1"
assert h1["model_id"] == config["model_id"]
assert h1["model_revision"] == config["model_revision"]

print("=== STEP 3.2: H2 BEAM SEARCH TEST ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Audio ID:", h1["audio_id"])

# Same model and same audio; only decoding changes.
model.eval()

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    generated_ids_h2 = model.generate(
        input_features,
        do_sample=False,
        num_beams=5,
        max_new_tokens=128
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

transcription_h2 = processor.batch_decode(
    generated_ids_h2,
    skip_special_tokens=True
)[0].strip()

transcription_h1 = str(h1["transcription"])

h2_df = pd.DataFrame([{
    "audio_id": "096140003",
    "hypothesis_id": "H2",
    "method": "beam_search",
    "num_beams": 5,
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "transcription": transcription_h2,
    "inference_sec": elapsed
}])

H2_FILE.parent.mkdir(parents=True, exist_ok=True)
h2_df.to_csv(H2_FILE, index=False, encoding="utf-8")

report = {
    "audio_id": "096140003",
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "h1_transcription": transcription_h1,
    "h2_transcription": transcription_h2,
    "h1_h2_identical": transcription_h1 == transcription_h2,
    "h2_num_beams": 5,
    "h2_inference_sec": elapsed,
    "status": "completed",
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_h2_smoke_test"] = "completed"
progress["next_task"] = "Compare H1/H2 and test sampling H3-H5"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1 VS H2 COMPARISON ===")
print("H1 (greedy):", repr(transcription_h1))
print("H2 (beam):", repr(transcription_h2))
print("Identical:", report["h1_h2_identical"])
print("Beam inference time:", round(elapsed, 3), "seconds")
print("H2 CSV:", H2_FILE)
print("Checkpoint updated.")


In [ ]:

import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch
import transformers

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
OUT_DIR = PROJECT / "features/asr"
REPORT = PROJECT / "reports/step03_sampling_smoke_test.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert "model" in globals(), "Whisper model not loaded"
assert "processor" in globals(), "Whisper processor not loaded"
assert "input_features" in globals(), "Audio features not loaded"

assert config["status"] == "proposed_not_frozen"

TARGET_ID = "096140003"

OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

# Read saved deterministic hypotheses
transcriptions = {}

for hypothesis_id in ["H1", "H2"]:
    path = OUT_DIR / f"asr_smoke_test_{TARGET_ID}_{hypothesis_id}.csv"

    df = pd.read_csv(
        path,
        dtype={"audio_id": "string"},
        keep_default_na=False
    )

    assert len(df) == 1
    assert df.iloc[0]["audio_id"] == TARGET_ID
    assert df.iloc[0]["hypothesis_id"] == hypothesis_id

    transcriptions[hypothesis_id] = str(
        df.iloc[0]["transcription"]
    )

print("=== STEP 3.2: STOCHASTIC SAMPLING TEST ===")
print("Model:", config["model_id"])
print("Audio:", TARGET_ID)

sampling_hypotheses = [
    h for h in config["hypotheses"]
    if h["method"] == "sampling"
]

assert len(sampling_hypotheses) == 3

records = []

model.eval()

for h in sampling_hypotheses:

    hypothesis_id = h["id"]
    seed = int(h["seed"])
    temperature = float(h["temperature"])

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            input_features,
            do_sample=True,
            num_beams=1,
            temperature=temperature,
            max_new_tokens=128
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    transcription = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    record = {
        "audio_id": TARGET_ID,
        "hypothesis_id": hypothesis_id,
        "method": "sampling",
        "model_id": config["model_id"],
        "model_revision": config["model_revision"],
        "do_sample": True,
        "num_beams": 1,
        "temperature": temperature,
        "seed": seed,
        "transcription": transcription,
        "generated_tokens": int(generated_ids.shape[-1]),
        "inference_sec": elapsed
    }

    records.append(record)
    transcriptions[hypothesis_id] = transcription

    output = OUT_DIR / f"asr_smoke_test_{TARGET_ID}_{hypothesis_id}.csv"

    pd.DataFrame([record]).to_csv(
        output,
        index=False,
        encoding="utf-8"
    )

    print(
        f"{hypothesis_id} | seed={seed} | "
        f"time={elapsed:.3f}s | {transcription!r}"
    )

unique_transcripts = len(set(transcriptions.values()))

report = {
    "audio_id": TARGET_ID,
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "hypotheses": transcriptions,
    "unique_transcriptions": unique_transcripts,
    "total_hypotheses": 5,
    "all_identical": unique_transcripts == 1,
    "status": "completed",
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_sampling_smoke_test"] = "completed"
progress["step3_smoke_unique_transcriptions"] = unique_transcripts
progress["next_task"] = (
    "Assess H1-H5 variability and finalize ASR pilot design"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1-H5 TRANSCRIPTION COMPARISON ===")

for key, value in transcriptions.items():
    print(f"{key}: {value!r}")

print("\nTotal hypotheses:", len(transcriptions))
print("Unique transcriptions:", unique_transcripts)
print("All identical:", report["all_identical"])
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
OUT = PROJECT / "data/metadata/asr_pilot_selection_10.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert len(pilot) == 400
assert pilot["audio_id"].is_unique
assert config["pilot_audio_count"] == 10

SMOKE_ID = "096140003"
SELECTION_SEED = 42

smoke = pilot.loc[
    pilot["audio_id"] == SMOKE_ID
].copy()

assert len(smoke) == 1

smoke_speaker = smoke.iloc[0]["speaker"]

# One audio per speaker for broader coverage
remaining = pilot.loc[
    pilot["speaker"] != smoke_speaker
].copy()

speakers = sorted(remaining["speaker"].unique())

rng = np.random.default_rng(SELECTION_SEED)

selected_speakers = rng.choice(
    speakers,
    size=9,
    replace=False
)

selected_rows = []

for speaker in selected_speakers:
    options = remaining.loc[
        remaining["speaker"] == speaker
    ].sort_values("pilot_index")

    selected_position = int(
        rng.integers(0, len(options))
    )

    selected_rows.append(
        options.iloc[selected_position]
    )

additional = pd.DataFrame(selected_rows)

selected = pd.concat(
    [smoke, additional],
    ignore_index=True
)

selected = selected.sort_values(
    "pilot_index"
).reset_index(drop=True)

assert len(selected) == 10
assert selected["audio_id"].is_unique
assert selected["speaker"].nunique() == 10
assert SMOKE_ID in set(selected["audio_id"])

selected["asr_pilot_order"] = range(1, 11)
selected["selection_method"] = (
    "one_recording_per_speaker_seed42"
)

if OUT.exists():
    old = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )

    assert old["audio_id"].tolist() == selected["audio_id"].tolist(), (
        "Existing pilot selection differs. "
        "Do not silently overwrite."
    )
else:
    selected.to_csv(OUT, index=False, encoding="utf-8")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_asr_pilot_selection_count"] = 10
progress["step3_asr_pilot_selection_seed"] = SELECTION_SEED
progress["step3_asr_pilot_selection_file"] = str(OUT)
progress["next_task"] = "Generate five ASR hypotheses for 10 pilot audios"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.2 ASR PILOT SELECTION ===")
print("Selected recordings:", len(selected))
print("Unique speakers:", selected["speaker"].nunique())
print("Random seed:", SELECTION_SEED)

print("\n=== SELECTED AUDIO IDs ===")
print(
    selected[
        ["asr_pilot_order", "pilot_index", "audio_id", "speaker"]
    ].to_string(index=False)
)

print("\nSelection file:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
SELECTION = PROJECT / "data/metadata/asr_pilot_selection_10.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/pilot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "proposed_not_frozen"

selection = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("asr_pilot_order")

assert len(selection) == 10
assert selection["audio_id"].is_unique
assert selection["speaker"].nunique() == 10

hypotheses = config["hypotheses"]
assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

MODEL_ID = config["model_id"]
REVISION = config["model_revision"]
device = "cuda:0"

assert torch.cuda.is_available()

# Find exactly the selected recordings.
wanted = set(selection["audio_id"])
audio_data = {}

pf = pq.ParquetFile(TRAIN)

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id, columns=["audio"]
    )

    for item in table.column("audio").to_pylist():
        audio_id = Path(item["path"]).stem

        if audio_id in wanted:
            assert audio_id not in audio_data
            audio_data[audio_id] = item["bytes"]

assert set(audio_data) == wanted

# Load pinned ASR model.
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    dtype=torch.float16
).to(device)

model.eval()

print("=== STEP 3.2: ASR PILOT GENERATION ===")
print("Model:", MODEL_ID)
print("Revision:", REVISION)
print("Selected audios:", len(selection))
print("Hypotheses per audio:", len(hypotheses))

for _, item in selection.iterrows():

    audio_id = str(item["audio_id"])
    speaker = str(item["speaker"])

    output = OUT_DIR / f"asr_pilot_{audio_id}_H1-H5.csv"

    if output.exists():
        previous = pd.read_csv(
            output,
            dtype={"audio_id": "string", "speaker": "string"},
            keep_default_na=False
        )

        assert len(previous) == 5
        assert previous["audio_id"].eq(audio_id).all()
        assert previous["hypothesis_id"].tolist() == [
            "H1", "H2", "H3", "H4", "H5"
        ]
        assert previous["model_revision"].eq(REVISION).all()
        assert previous["transcription"].map(
            lambda x: isinstance(x, str)
        ).all()

        print(f"SKIP completed: {audio_id}")
        continue

    y, sr = sf.read(
        io.BytesIO(audio_data[audio_id]),
        dtype="float32"
    )

    assert sr == 16000 and y.ndim == 1
    assert np.isfinite(y).all()

    inputs = processor(
        y,
        sampling_rate=sr,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device=device,
        dtype=torch.float16
    )

    records = []

    for h in hypotheses:

        hid = h["id"]
        method = h["method"]

        kwargs = {
            "max_new_tokens": 128,
            "num_beams": 1,
            "do_sample": False
        }

        seed = None
        temperature = None

        if method == "beam_search":
            kwargs["num_beams"] = h["num_beams"]

        elif method == "sampling":
            seed = int(h["seed"])
            temperature = float(h["temperature"])

            kwargs["do_sample"] = True
            kwargs["temperature"] = temperature

            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            generated_ids = model.generate(
                input_features,
                **kwargs
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        transcription = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0].strip()

        records.append({
            "audio_id": audio_id,
            "speaker": speaker,
            "hypothesis_id": hid,
            "method": method,
            "model_id": MODEL_ID,
            "model_revision": REVISION,
            "do_sample": kwargs["do_sample"],
            "num_beams": kwargs["num_beams"],
            "temperature": temperature,
            "seed": seed,
            "transcription": transcription,
            "duration_sec": len(y) / sr,
            "inference_sec": elapsed
        })

    result = pd.DataFrame(records)

    assert len(result) == 5
    assert result["hypothesis_id"].is_unique

    # Atomic save to protect against interrupted writes.
    temp = output.with_suffix(".tmp")
    result.to_csv(temp, index=False, encoding="utf-8")
    temp.replace(output)

    unique = result["transcription"].nunique()

    print(
        f"COMPLETE: {audio_id} | "
        f"unique transcripts: {unique}/5"
    )

# Verify all pilot files.
all_results = []

for audio_id in selection["audio_id"]:
    path = OUT_DIR / f"asr_pilot_{audio_id}_H1-H5.csv"

    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df["audio_id"].eq(audio_id).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    all_results.append(df)

combined = pd.concat(all_results, ignore_index=True)

assert len(combined) == 50

unique_per_audio = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_asr_pilot_generated_audios"] = 10
progress["step3_asr_pilot_generated_hypotheses"] = 50
progress["step3_asr_pilot_status"] = "generated_pending_qc"
progress["next_task"] = (
    "Analyze ASR transcription variability across 10 pilot audios"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ASR PILOT SUMMARY ===")
print("Audio count:", len(unique_per_audio))
print("Total hypotheses:", len(combined))
print("Audios with >1 transcription:",
      int((unique_per_audio > 1).sum()))
print("Mean unique transcripts:",
      round(unique_per_audio.mean(), 3))

print("\n=== UNIQUE TRANSCRIPTS PER AUDIO ===")
print(unique_per_audio.to_string())

print("\nOutput folder:", OUT_DIR)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ASR_DIR = PROJECT / "features/asr/pilot"
SELECTION = PROJECT / "data/metadata/asr_pilot_selection_10.csv"
OUT = PROJECT / "reports/step03_pilot_variability_per_audio.csv"
SUMMARY = PROJECT / "reports/step03_pilot_variability_summary.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

selection = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(selection) == 10

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        curr = [i] + [0] * len(b)
        for j, y in enumerate(b, start=1):
            curr[j] = min(
                prev[j] + 1,
                curr[j-1] + 1,
                prev[j-1] + (x != y)
            )
        prev = curr
    return prev[-1]

def pair_metrics(reference, hypothesis):
    ref_words = reference.split()
    hyp_words = hypothesis.split()

    wer = (
        edit_distance(ref_words, hyp_words) / len(ref_words)
        if ref_words else (0.0 if not hyp_words else np.nan)
    )

    ref_chars = list(reference.replace(" ", ""))
    hyp_chars = list(hypothesis.replace(" ", ""))

    cer = (
        edit_distance(ref_chars, hyp_chars) / len(ref_chars)
        if ref_chars else (0.0 if not hyp_chars else np.nan)
    )

    return wer, cer

rows = []
all_hypotheses = []

for _, selected in selection.iterrows():

    audio_id = selected["audio_id"]
    source = ASR_DIR / f"asr_pilot_{audio_id}_H1-H5.csv"

    df = pd.read_csv(
        source,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df["audio_id"].eq(audio_id).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    df["normalized"] = df["transcription"].map(normalize_text)
    all_hypotheses.append(df)

    h1 = df.iloc[0]["normalized"]

    metric_rows = [
        pair_metrics(h1, text)
        for text in df.iloc[1:]["normalized"]
    ]

    valid_wer = [x[0] for x in metric_rows if np.isfinite(x[0])]
    valid_cer = [x[1] for x in metric_rows if np.isfinite(x[1])]

    rows.append({
        "audio_id": audio_id,
        "speaker": selected["speaker"],
        "raw_unique": df["transcription"].nunique(),
        "lexical_unique": df["normalized"].nunique(),
        "mean_wer_vs_h1": (
            float(np.mean(valid_wer)) if valid_wer else np.nan
        ),
        "max_wer_vs_h1": (
            float(np.max(valid_wer)) if valid_wer else np.nan
        ),
        "mean_cer_vs_h1": (
            float(np.mean(valid_cer)) if valid_cer else np.nan
        ),
        "empty_transcriptions": int(
            df["normalized"].eq("").sum()
        )
    })

result = pd.DataFrame(rows)

assert len(result) == 10
assert result["audio_id"].is_unique

OUT.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT, index=False, encoding="utf-8")

combined = pd.concat(all_hypotheses, ignore_index=True)

summary = {
    "total_audio": len(result),
    "total_hypotheses": len(combined),
    "raw_variable_audio_count": int(
        (result["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (result["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(result["raw_unique"].mean()),
    "mean_lexical_unique": float(result["lexical_unique"].mean()),
    "mean_wer_vs_h1": float(result["mean_wer_vs_h1"].mean()),
    "mean_cer_vs_h1": float(result["mean_cer_vs_h1"].mean()),
    "empty_transcriptions": int(
        result["empty_transcriptions"].sum()
    ),
    "note": (
        "WER/CER compare ASR hypotheses against H1, "
        "not against a verified human transcription."
    ),
    "created_at": datetime.now(timezone.utc).isoformat()
}

SUMMARY.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_pilot_variability_analysis"] = "completed"
progress["next_task"] = (
    "Review lexical variability and determine full ASR design"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.2 PILOT VARIABILITY SUMMARY ===")
for key, value in summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== PER AUDIO VARIABILITY ===")
print(result.round(4).to_string(index=False))

print("\nReport CSV:", OUT)
print("Summary JSON:", SUMMARY)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
ASR_DIR = PROJECT / "features/asr/pilot"
SELECTION = PROJECT / "data/metadata/asr_pilot_selection_10.csv"
OUT = PROJECT / "reports/step03_pilot_hypothesis_content_audit.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

selection = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

records = []

for _, selected in selection.iterrows():

    audio_id = str(selected["audio_id"])
    path = ASR_DIR / f"asr_pilot_{audio_id}_H1-H5.csv"

    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df["audio_id"].eq(audio_id).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    record = {
        "audio_id": audio_id,
        "speaker": str(selected["speaker"])
    }

    for _, h in df.iterrows():
        record[h["hypothesis_id"]] = str(h["transcription"])

    records.append(record)

audit = pd.DataFrame(records)

assert len(audit) == 10
assert audit["audio_id"].is_unique

OUT.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT, index=False, encoding="utf-8")

print("=== STEP 3.2 HYPOTHESIS CONTENT AUDIT ===")

for _, row in audit.iterrows():
    unique = len(set(row[h] for h in ["H1", "H2", "H3", "H4", "H5"]))

    print("\n" + "=" * 65)
    print("Audio ID:", row["audio_id"])
    print("Speaker:", row["speaker"])
    print("Raw unique:", unique)

    for hid in ["H1", "H2", "H3", "H4", "H5"]:
        print(f"{hid}: {row[hid]!r}")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_pilot_content_audit_file"] = str(OUT)
progress["step3_pilot_content_audit_status"] = "generated_pending_review"
progress["next_task"] = "Review differences and freeze ASR configuration"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== CONTENT AUDIT SAVED ===")
print("Audio count:", len(audit))
print("Total hypotheses:", len(audit) * 5)
print("CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import matplotlib.pyplot as plt

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
ASR_FILE = (
    PROJECT /
    "features/asr/pilot/asr_pilot_009600118_H1-H5.csv"
)
OUT = PROJECT / "data/metadata/asr_critical_listening_009600118.csv"
FIG = PROJECT / "results/figures/asr_critical_009600118.png"

TARGET_ID = "009600118"

pf = pq.ParquetFile(TRAIN)
matches = []

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id, columns=["audio", "text", "speaker"]
    )

    audios = table.column("audio").to_pylist()
    texts = table.column("text").to_pylist()
    speakers = table.column("speaker").to_pylist()

    for audio, text, speaker in zip(
        audios, texts, speakers
    ):
        if Path(audio["path"]).stem == TARGET_ID:
            matches.append((audio["bytes"], text, speaker))

assert len(matches) == 1

audio_bytes, prompt_text, speaker = matches[0]

y, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

assert sr == 16000 and y.ndim == 1
assert np.isfinite(y).all()

asr = pd.read_csv(
    ASR_FILE,
    dtype={"audio_id": "string"},
    keep_default_na=False
)

assert len(asr) == 5
assert asr["audio_id"].eq(TARGET_ID).all()

print("=== CRITICAL ASR AUDIO INSPECTION ===")
print("Audio ID:", TARGET_ID)
print("Speaker:", speaker)
print("Duration:", round(len(y) / sr, 3), "seconds")
print("Reading prompt:", repr(prompt_text))

print("\n=== ASR HYPOTHESES ===")
for _, row in asr.iterrows():
    print(row["hypothesis_id"], ":", repr(row["transcription"]))

print("\n=== ORIGINAL AUDIO ===")
display(Audio(y, rate=sr, normalize=False))

times = np.arange(len(y)) / sr

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(times, y, linewidth=0.6)
ax.set_title("Critical ASR Inspection: 009600118")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Amplitude")
ax.set_xlim(0, len(y) / sr)
fig.tight_layout()

FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=150)
plt.show()
plt.close(fig)

# Create a pending qualitative listening audit.
record = {
    "audio_id": TARGET_ID,
    "speaker": str(speaker),
    "reading_prompt": prompt_text,
    "manual_listening_observation": "pending",
    "audible_words_observation": "pending",
    "noise_or_recording_observation": "pending",
    "asr_error_observation": "pending",
    "manual_verbatim_transcript_available": False
}

OUT.parent.mkdir(parents=True, exist_ok=True)

if OUT.exists():
    previous = pd.read_csv(
        OUT,
        dtype={"audio_id": "string"},
        keep_default_na=False
    )
    assert len(previous) == 1
    assert previous.iloc[0]["audio_id"] == TARGET_ID
    print("\nExisting audit preserved:", OUT)
else:
    pd.DataFrame([record]).to_csv(
        OUT,
        index=False,
        encoding="utf-8"
    )
    print("\nAudit template saved:", OUT)

print("Waveform saved:", FIG)
print("=== INSPECTION READY ===")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
AUDIT = PROJECT / "data/metadata/asr_critical_listening_009600118.csv"
REPORT = PROJECT / "reports/step03_critical_asr_review_009600118.md"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Cell 18 audit file not found"
assert CHECK.exists(), "Research checkpoint not found"

df = pd.read_csv(
    AUDIT,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(df) == 1
assert df.iloc[0]["audio_id"] == "009600118"

corrected_observation = (
    "Ey yet a possiple you are yen the wrong account."
)

df.loc[0, "manual_listening_observation"] = (
    "Pronunciation errors were audible."
)
df.loc[0, "audible_words_observation"] = corrected_observation
df.loc[0, "noise_or_recording_observation"] = (
    "Not assessed; no confirmed noise cause."
)
df.loc[0, "asr_error_observation"] = (
    "Substantial differences among H1-H5. "
    "H2 appeared closer to the researcher's "
    "approximate phonetic listening impression."
)
df.loc[0, "manual_verbatim_transcript_available"] = False
df["annotation_type"] = "informal_phonetic_listening_impression"
df["annotation_status"] = "reviewed"
df["updated_at"] = datetime.now(timezone.utc).isoformat()

df.to_csv(AUDIT, index=False, encoding="utf-8")

report = f"""# Critical ASR Listening Review

Audio ID: 009600118

## Researcher observation
Pronunciation errors were audible.

Corrected approximate phonetic listening impression:
{corrected_observation}

The researcher corrected a typing error:
'pissiple' -> 'possiple'.

This is not a verified verbatim transcript.

## ASR interpretation
H1-H5 produced substantially different transcripts.
H2 appeared closer to the researcher's listening impression.

No ground-truth WER or CER is reported.

## Methodological decisions
- Preserve all original ASR hypotheses.
- Do not treat reading prompts as verified transcripts.
- Do not select a winning hypothesis using this observation.
- Do not assert a definitive cause of the variability.
"""

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(report, encoding="utf-8")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_critical_asr_audio"] = "009600118"
progress["step3_critical_asr_listening_status"] = "reviewed"
progress["step3_critical_asr_report"] = str(REPORT)
progress["next_task"] = (
    "Review and freeze ASR hypothesis generation design"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== CORRECTED ASR LISTENING AUDIT SAVED ===")
print("Audio ID:", df.iloc[0]["audio_id"])
print("Corrected observation:", corrected_observation)
print("Verified verbatim transcript:", False)
print("Audit:", AUDIT)
print("Report:", REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG_FILE = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
SUMMARY_FILE = PROJECT / "reports/step03_pilot_variability_summary.json"
AUDIT_FILE = PROJECT / "data/metadata/asr_critical_listening_009600118.csv"
CHECK_FILE = PROJECT / "checkpoints/research_progress.json"

FREEZE_REPORT = PROJECT / "reports/step03_asr_configuration_freeze.json"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

config = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
summary = json.loads(SUMMARY_FILE.read_text(encoding="utf-8"))

assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)
assert config["status"] in [
    "proposed_not_frozen", "frozen"
]

hypotheses = config["hypotheses"]
assert len(hypotheses) == 5

expected = {
    "H1": ("greedy", 1, False),
    "H2": ("beam_search", 5, False),
    "H3": ("sampling", None, True),
    "H4": ("sampling", None, True),
    "H5": ("sampling", None, True)
}

for h in hypotheses:
    hid = h["id"]
    method, beams, sampling = expected[hid]

    assert h["method"] == method
    assert h["do_sample"] == sampling

    if beams is not None:
        assert h["num_beams"] == beams

sampling = [h for h in hypotheses if h["method"] == "sampling"]

assert [h["seed"] for h in sampling] == [101, 202, 303]
assert all(h["temperature"] == 0.3 for h in sampling)

assert summary["total_audio"] == 10
assert summary["total_hypotheses"] == 50
assert summary["lexically_variable_audio_count"] == 5
assert summary["empty_transcriptions"] == 0
assert AUDIT_FILE.exists()

if config["status"] == "proposed_not_frozen":
    config["status"] = "frozen"
    config["freeze_date"] = datetime.now(timezone.utc).isoformat()

    config["freeze_rationale"] = (
        "Pilot of 10 recordings produced lexical variation "
        "in 5 recordings; decoding parameters retained "
        "for subsequent evaluation."
    )

    config["runtime_generation_rules"] = {
        "max_new_tokens": 128,
        "english_only_no_task_argument": True,
        "english_only_no_language_argument": True,
        "num_beams_sampling": 1,
        "dtype": "float16",
        "device": "cuda:0",
        "seed_reset_per_audio_and_hypothesis": True,
        "preserve_raw_transcriptions": True,
        "no_prompt_text_as_context": True,
        "no_fluency_labels_for_hypothesis_selection": True
    }

    CONFIG_FILE.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

report = {
    "status": "FROZEN",
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "hypotheses": hypotheses,
    "pilot_audio_count": 10,
    "pilot_hypothesis_count": 50,
    "pilot_lexically_variable_audio_count": 5,
    "configuration_sha256": sha256(CONFIG_FILE),
    "pilot_summary_sha256": sha256(SUMMARY_FILE),
    "listening_audit_sha256": sha256(AUDIT_FILE),
    "freeze_date": config["freeze_date"]
}

FREEZE_REPORT.parent.mkdir(parents=True, exist_ok=True)
FREEZE_REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(
    CHECK_FILE.read_text(encoding="utf-8")
)

progress["step3_asr_config_status"] = "frozen"
progress["step3_asr_config_sha256"] = sha256(CONFIG_FILE)
progress["step3_asr_freeze_report"] = str(FREEZE_REPORT)
progress["next_task"] = (
    "Generate H1-H5 ASR hypotheses for 400 pilot recordings"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK_FILE.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.2 ASR CONFIGURATION FROZEN ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Hypotheses:", len(hypotheses))
print("Pilot audio:", summary["total_audio"])
print(
    "Pilot lexical variability:",
    summary["lexically_variable_audio_count"]
)
print("Configuration status:", config["status"])
print("Configuration SHA256:", sha256(CONFIG_FILE))
print("Freeze report:", FREEZE_REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0001-0100_parts"
FINAL = OUT_DIR / "asr_hypotheses_0001-0100.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot[
    pilot["pilot_index"].between(1, 100)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(1, 101))
assert batch["audio_id"].is_unique

hypotheses = config["hypotheses"]

assert len(hypotheses) == 5
assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

# Preserve frozen configuration integrity.
config_sha = hashlib.sha256(CONFIG.read_bytes()).hexdigest()

assert config_sha == (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
), "Frozen configuration has changed."

print("=== STEP 3.3: ASR BATCH 0001-0100 ===")
print("Model:", config["model_id"])
print("Audios:", len(batch))
print("Expected hypotheses:", len(batch) * len(hypotheses))
print("Config SHA256:", config_sha)

# Load original audio signals.
wanted = set(batch["audio_id"])
audio_data = {}

pf = pq.ParquetFile(TRAIN)

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id,
        columns=["audio"]
    )

    for audio in table.column("audio").to_pylist():
        audio_id = Path(audio["path"]).stem

        if audio_id in wanted:
            assert audio_id not in audio_data
            audio_data[audio_id] = audio["bytes"]

assert set(audio_data) == wanted

device = "cuda:0"
assert torch.cuda.is_available()

processor = WhisperProcessor.from_pretrained(
    config["model_id"],
    revision=config["model_revision"]
)

model = WhisperForConditionalGeneration.from_pretrained(
    config["model_id"],
    revision=config["model_revision"],
    dtype=torch.float16
).to(device)

model.eval()

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):
    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df.columns.tolist() == expected_columns
    assert df["pilot_index"].eq(int(row["pilot_index"])).all()
    assert df["audio_id"].eq(str(row["audio_id"])).all()
    assert df["speaker"].eq(str(row["speaker"])).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(config["model_revision"]).all()
    return df

# Generate and checkpoint one audio at a time.
for n, (_, row) in enumerate(batch.iterrows(), start=1):

    audio_id = str(row["audio_id"])
    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        print(f"[{n:03d}/100] SKIP {audio_id}")
        continue

    y, sr = sf.read(
        io.BytesIO(audio_data[audio_id]),
        dtype="float32"
    )

    assert sr == 16000 and y.ndim == 1
    assert np.isfinite(y).all()

    inputs = processor(
        y,
        sampling_rate=sr,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device=device,
        dtype=torch.float16
    )

    records = []

    for h in hypotheses:

        kwargs = {
            "do_sample": False,
            "num_beams": 1,
            "max_new_tokens": 128
        }

        seed = None
        temperature = None

        if h["method"] == "beam_search":
            kwargs["num_beams"] = int(h["num_beams"])

        elif h["method"] == "sampling":
            seed = int(h["seed"])
            temperature = float(h["temperature"])

            kwargs["do_sample"] = True
            kwargs["temperature"] = temperature

            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            generated_ids = model.generate(
                input_features,
                **kwargs
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        transcription = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0].strip()

        records.append({
            "pilot_index": int(row["pilot_index"]),
            "audio_id": audio_id,
            "speaker": str(row["speaker"]),
            "hypothesis_id": h["id"],
            "method": h["method"],
            "model_id": config["model_id"],
            "model_revision": config["model_revision"],
            "do_sample": kwargs["do_sample"],
            "num_beams": kwargs["num_beams"],
            "temperature": temperature,
            "seed": seed,
            "transcription": transcription,
            "duration_sec": len(y) / sr,
            "inference_sec": elapsed
        })

    result = pd.DataFrame(records, columns=expected_columns)

    # Atomic checkpoint write.
    tmp = part.with_suffix(".tmp")
    result.to_csv(tmp, index=False, encoding="utf-8")
    tmp.replace(part)

    validate_part(part, row)

    if n % 10 == 0 or n == 1:
        print(f"[{n:03d}/100] COMPLETE {audio_id}")

# Combine verified per-audio checkpoints.
pieces = []

for _, row in batch.iterrows():
    audio_id = str(row["audio_id"])
    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100

temp_final = FINAL.with_suffix(".tmp")
combined.to_csv(temp_final, index=False, encoding="utf-8")
temp_final.replace(FINAL)

sha = hashlib.sha256(FINAL.read_bytes()).hexdigest()

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0001-0100"] = "completed_pending_qc"
progress["step3_full_batch_0001-0100_sha256"] = sha
progress["step3_full_generated_audio_count"] = 100
progress["step3_full_generated_hypothesis_count"] = 500
progress["next_task"] = "Quality control for ASR batch 0001-0100"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0001-0100 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print("Unique transcripts mean:", round(unique_counts.mean(), 3))
print("Audios with raw variability:", int((unique_counts > 1).sum()))
print(
    "Empty transcriptions:",
    int(combined["transcription"].eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", sha)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0100.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

QC_REPORT = PROJECT / "reports/step03_batch_0001-0100_qc.json"
QC_TABLE = PROJECT / "reports/step03_batch_0001-0100_qc_per_audio.csv"

EXPECTED_CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)
EXPECTED_ASR_SHA = (
    "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == EXPECTED_CONFIG_SHA
assert sha256(ASR) == EXPECTED_ASR_SHA

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "frozen"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

expected = pilot.loc[
    pilot["pilot_index"].between(1, 100),
    ["pilot_index", "audio_id", "speaker"]
].sort_values("pilot_index")

df = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(expected) == 100
assert len(df) == 500
assert df["audio_id"].nunique() == 100

hypothesis_ids = ["H1", "H2", "H3", "H4", "H5"]

assert not df.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert df["model_id"].eq(config["model_id"]).all()
assert df["model_revision"].eq(config["model_revision"]).all()

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()

qc_rows = []

for _, item in expected.iterrows():

    audio_id = str(item["audio_id"])

    subset = df.loc[
        df["audio_id"] == audio_id
    ].sort_values("hypothesis_id")

    assert len(subset) == 5
    assert subset["hypothesis_id"].tolist() == hypothesis_ids
    assert subset["pilot_index"].eq(
        int(item["pilot_index"])
    ).all()
    assert subset["speaker"].eq(str(item["speaker"])).all()

    normalized = subset["transcription"].map(normalize_text)

    qc_rows.append({
        "pilot_index": int(item["pilot_index"]),
        "audio_id": audio_id,
        "speaker": str(item["speaker"]),
        "raw_unique": int(subset["transcription"].nunique()),
        "lexical_unique": int(normalized.nunique()),
        "empty_transcriptions": int(
            subset["transcription"].str.strip().eq("").sum()
        ),
        "max_transcription_chars": int(
            subset["transcription"].str.len().max()
        ),
        "duration_sec": float(subset["duration_sec"].iloc[0])
    })

qc = pd.DataFrame(qc_rows)

assert len(qc) == 100
assert int(qc["empty_transcriptions"].sum()) == 0
assert (qc["duration_sec"] > 0).all()

summary = {
    "batch": "0001-0100",
    "status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "raw_variable_audio_count": int(
        (qc["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (qc["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(qc["raw_unique"].mean()),
    "mean_lexical_unique": float(qc["lexical_unique"].mean()),
    "empty_transcriptions": int(
        qc["empty_transcriptions"].sum()
    ),
    "config_sha256": sha256(CONFIG),
    "asr_csv_sha256": sha256(ASR),
    "created_at": datetime.now(timezone.utc).isoformat()
}

QC_REPORT.parent.mkdir(parents=True, exist_ok=True)

QC_TABLE_TMP = QC_TABLE.with_suffix(".tmp")
qc.to_csv(QC_TABLE_TMP, index=False, encoding="utf-8")
QC_TABLE_TMP.replace(QC_TABLE)

QC_REPORT.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0001-0100"] = "qc_pass"
progress["step3_full_batch_0001-0100_qc_report"] = str(QC_REPORT)
progress["next_task"] = "Create Batch 1 backup, then Batch 2"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 1 QC SUMMARY ===")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n=== VARIABILITY DISTRIBUTION ===")
print(
    qc["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nQC report:", QC_REPORT)
print("QC table:", QC_TABLE)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
BACKUP = Path("/kaggle/working/asr_fluency_step03_batch_0001-0100.zip")
CHECK = PROJECT / "checkpoints/research_progress.json"

EXPECTED_CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
EXPECTED_ASR_SHA = "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3"

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

required_files = [
    "configs/asr_hypothesis_pilot_v1.json",
    "data/metadata/pilot_selection_400.csv",
    "data/metadata/asr_pilot_selection_10.csv",
    "data/metadata/asr_critical_listening_009600118.csv",
    "features/asr/full/asr_hypotheses_0001-0100.csv",
    "reports/step03_asr_configuration_freeze.json",
    "reports/step03_pilot_variability_summary.json",
    "reports/step03_pilot_variability_per_audio.csv",
    "reports/step03_pilot_hypothesis_content_audit.csv",
    "reports/step03_critical_asr_review_009600118.md",
    "reports/step03_batch_0001-0100_qc.json",
    "reports/step03_batch_0001-0100_qc_per_audio.csv",
    "checkpoints/research_progress.json",
]

for relative in required_files:
    assert (PROJECT / relative).is_file(), (
        f"Required file missing: {relative}"
    )

assert file_sha256(
    PROJECT / "configs/asr_hypothesis_pilot_v1.json"
) == EXPECTED_CONFIG_SHA

assert file_sha256(
    PROJECT / "features/asr/full/asr_hypotheses_0001-0100.csv"
) == EXPECTED_ASR_SHA

qc = json.loads(
    (PROJECT / "reports/step03_batch_0001-0100_qc.json")
    .read_text(encoding="utf-8")
)

assert qc["status"] == "PASS"
assert qc["audio_count"] == 100
assert qc["hypothesis_count"] == 500
assert qc["config_sha256"] == EXPECTED_CONFIG_SHA
assert qc["asr_csv_sha256"] == EXPECTED_ASR_SHA

part_dir = PROJECT / "features/asr/full/batch_0001-0100_parts"
part_files = sorted(part_dir.glob("asr_*_H1-H5.csv"))
assert len(part_files) == 100, (
    f"Expected 100 audio checkpoints, found {len(part_files)}"
)

# Update checkpoint before packaging it.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0001-0100"] = "qc_pass"
progress["step3_batch1_backup_status"] = "packaging"
progress["step3_batch1_backup_filename"] = BACKUP.name
progress["next_task"] = (
    "Restore/retain Batch 1 backup and start ASR Batch 0101-0200"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Include the previous smoke tests, pilot ASR outputs,
# and all per-audio checkpoints.
files = [PROJECT / p for p in required_files]

for directory in [
    PROJECT / "features/asr/pilot",
    PROJECT / "features/asr/full/batch_0001-0100_parts",
]:
    files.extend(
        p for p in directory.rglob("*") if p.is_file()
    )

files.extend(
    PROJECT.glob("features/asr/asr_smoke_test_*.csv")
)

for name in [
    "reports/step03_whisper_smoke_test.json",
    "reports/step03_whisper_beam_test.json",
    "reports/step03_sampling_smoke_test.json",
]:
    p = PROJECT / name
    if p.is_file():
        files.append(p)

files = sorted(set(files))
assert len(files) >= 113

manifest = {
    "backup_name": BACKUP.name,
    "batch": "0001-0100",
    "qc_status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "project_root": "acoustic_linguistic_research",
    "raw_parquet_included": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

temporary = BACKUP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    temporary, "w", compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in files:
        relative = path.relative_to(PROJECT).as_posix()
        archive_name = f"acoustic_linguistic_research/{relative}"
        manifest["files"][archive_name] = file_sha256(path)
        archive.write(path, arcname=archive_name)

    archive.writestr(
        "acoustic_linguistic_research/"
        "backup_manifest_batch_0001-0100.json",
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )

temporary.replace(BACKUP)

# Independently verify the ZIP contents.
with zipfile.ZipFile(BACKUP, "r") as archive:
    assert archive.testzip() is None

    for name, expected_hash in manifest["files"].items():
        actual_hash = hashlib.sha256(
            archive.read(name)
        ).hexdigest()

        assert actual_hash == expected_hash, (
            f"ZIP integrity failure: {name}"
        )

print("=== STEP 3.3 BATCH 1 BACKUP VERIFIED ===")
print("Backup:", BACKUP)
print("Files archived:", len(files))
print("Audio checkpoint files:", len(part_files))
print("ZIP size MB:", round(BACKUP.stat().st_size / 1024**2, 3))
print("ZIP SHA256:", file_sha256(BACKUP))
print("QC status:", qc["status"])
print("Archive integrity: PASS")
print("Next batch: 0101-0200")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0101-0200_parts"
FINAL = OUT_DIR / "asr_hypotheses_0101-0200.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == EXPECTED_CONFIG_SHA, (
    "Frozen ASR configuration has changed."
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

# Verify that Batch 1 is complete and unchanged.
BATCH1 = OUT_DIR / "asr_hypotheses_0001-0100.csv"
QC1 = PROJECT / "reports/step03_batch_0001-0100_qc.json"

assert QC1.exists() and BATCH1.exists()

qc1 = json.loads(QC1.read_text(encoding="utf-8"))
assert qc1["status"] == "PASS"
assert sha256(BATCH1) == qc1["asr_csv_sha256"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.loc[
    pilot["pilot_index"].between(101, 200)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(101, 201))
assert batch["audio_id"].is_unique
assert batch["speaker"].notna().all()

hypotheses = config["hypotheses"]

assert len(hypotheses) == 5
assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):
    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df.columns.tolist() == expected_columns
    assert df["pilot_index"].eq(int(row["pilot_index"])).all()
    assert df["audio_id"].eq(str(row["audio_id"])).all()
    assert df["speaker"].eq(str(row["speaker"])).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(config["model_revision"]).all()

    for _, record in df.iterrows():
        spec = next(
            h for h in hypotheses
            if h["id"] == record["hypothesis_id"]
        )

        assert record["method"] == spec["method"]
        assert str(record["transcription"]) != ""

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            assert int(record["seed"]) == int(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"
            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )
            assert int(record["num_beams"]) == expected_beams

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0

    return df

# Identify completed audio checkpoints before model loading.
pending = []
completed_before_run = 0

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        completed_before_run += 1
    else:
        pending.append(row)

print("=== STEP 3.3: ASR BATCH 0101-0200 ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Batch range: 101-200")
print("Total audios:", len(batch))
print("Completed checkpoints:", completed_before_run)
print("Pending audios:", len(pending))
print("Expected hypotheses:", 500)
print("Frozen config SHA256:", sha256(CONFIG))

if pending:
    assert TRAIN.exists(), "Raw train Parquet missing"
    assert torch.cuda.is_available(), "CUDA GPU unavailable"

    wanted = {str(row["audio_id"]) for row in pending}
    audio_data = {}

    pf = pq.ParquetFile(TRAIN)

    assert "audio" in pf.schema_arrow.names

    for group_id in range(pf.metadata.num_row_groups):
        table = pf.read_row_group(
            group_id,
            columns=["audio"]
        )

        for audio in table.column("audio").to_pylist():
            audio_id = Path(audio["path"]).stem

            if audio_id in wanted:
                assert audio_id not in audio_data
                audio_data[audio_id] = audio["bytes"]

    assert set(audio_data) == wanted

    device = "cuda:0"

    processor = WhisperProcessor.from_pretrained(
        config["model_id"],
        revision=config["model_revision"]
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        dtype=torch.float16
    ).to(device)

    model.eval()

    for count, row in enumerate(pending, start=1):
        audio_id = str(row["audio_id"])
        pilot_index = int(row["pilot_index"])

        y, sr = sf.read(
            io.BytesIO(audio_data[audio_id]),
            dtype="float32"
        )

        assert sr == 16000 and y.ndim == 1
        assert np.isfinite(y).all()

        inputs = processor(
            y,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16
        )

        records = []

        for h in hypotheses:
            kwargs = {
                "do_sample": False,
                "num_beams": 1,
                "max_new_tokens": 128
            }

            seed = None
            temperature = None

            if h["method"] == "beam_search":
                kwargs["num_beams"] = int(h["num_beams"])

            elif h["method"] == "sampling":
                seed = int(h["seed"])
                temperature = float(h["temperature"])

                kwargs["do_sample"] = True
                kwargs["temperature"] = temperature

                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():
                generated_ids = model.generate(
                    input_features,
                    **kwargs
                )

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            transcription = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            records.append({
                "pilot_index": pilot_index,
                "audio_id": audio_id,
                "speaker": str(row["speaker"]),
                "hypothesis_id": h["id"],
                "method": h["method"],
                "model_id": config["model_id"],
                "model_revision": config["model_revision"],
                "do_sample": kwargs["do_sample"],
                "num_beams": kwargs["num_beams"],
                "temperature": temperature,
                "seed": seed,
                "transcription": transcription,
                "duration_sec": len(y) / sr,
                "inference_sec": elapsed
            })

        result = pd.DataFrame(
            records,
            columns=expected_columns
        )

        part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
        tmp = part.with_suffix(".tmp")

        result.to_csv(
            tmp,
            index=False,
            encoding="utf-8"
        )
        tmp.replace(part)

        validate_part(part, row)

        if count == 1 or count % 10 == 0 or count == len(pending):
            print(
                f"[{count:03d}/{len(pending):03d}] "
                f"COMPLETE index={pilot_index} audio={audio_id}"
            )

# Final validation and assembly.
pieces = []

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"
    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100
assert combined["pilot_index"].nunique() == 100
assert not combined.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

tmp_final = FINAL.with_suffix(".tmp")

combined.to_csv(
    tmp_final,
    index=False,
    encoding="utf-8"
)
tmp_final.replace(FINAL)

final_sha = sha256(FINAL)

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0101-0200"] = "completed_pending_qc"
progress["step3_full_batch_0101-0200_sha256"] = final_sha
progress["step3_full_generated_audio_count"] = 200
progress["step3_full_generated_hypothesis_count"] = 1000
progress["next_task"] = (
    "Quality control for ASR batch 0101-0200"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0101-0200 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print("Mean unique transcripts:", round(unique_counts.mean(), 3))
print("Audios with raw variability:", int((unique_counts > 1).sum()))
print(
    "Empty transcriptions:",
    int(combined["transcription"].str.strip().eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", final_sha)
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0101-0200_parts"
FINAL = OUT_DIR / "asr_hypotheses_0101-0200.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == EXPECTED_CONFIG_SHA

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

# Verify Batch 1 remains unchanged.
BATCH1 = OUT_DIR / "asr_hypotheses_0001-0100.csv"
QC1 = PROJECT / "reports/step03_batch_0001-0100_qc.json"

qc1 = json.loads(QC1.read_text(encoding="utf-8"))
assert qc1["status"] == "PASS"
assert sha256(BATCH1) == qc1["asr_csv_sha256"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.loc[
    pilot["pilot_index"].between(101, 200)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(101, 201))
assert batch["audio_id"].is_unique

hypotheses = config["hypotheses"]

assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):
    df = pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "seed": "string"
        },
        keep_default_na=False
    )

    assert len(df) == 5
    assert df.columns.tolist() == expected_columns
    assert df["pilot_index"].eq(int(row["pilot_index"])).all()
    assert df["audio_id"].eq(str(row["audio_id"])).all()
    assert df["speaker"].eq(str(row["speaker"])).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(config["model_revision"]).all()

    for _, record in df.iterrows():
        spec = next(
            h for h in hypotheses
            if h["id"] == record["hypothesis_id"]
        )

        assert record["method"] == spec["method"]
        assert isinstance(record["transcription"], str)

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            # CSV may contain 101.0 instead of 101.
            assert float(record["seed"]) == float(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"

            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )

            assert int(record["num_beams"]) == expected_beams

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0

    return df

# Discover completed checkpoints before loading the model.
pending = []
completed_before_run = 0

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        completed_before_run += 1
    else:
        pending.append(row)

print("=== STEP 3.3 BATCH 2 RESUME ===")
print("Total audios:", len(batch))
print("Completed checkpoints:", completed_before_run)
print("Pending audios:", len(pending))
print("Frozen config SHA256:", sha256(CONFIG))

if pending:
    assert TRAIN.exists()
    assert torch.cuda.is_available()

    wanted = {str(row["audio_id"]) for row in pending}
    audio_data = {}

    pf = pq.ParquetFile(TRAIN)
    assert "audio" in pf.schema_arrow.names

    for group_id in range(pf.metadata.num_row_groups):
        table = pf.read_row_group(
            group_id,
            columns=["audio"]
        )

        for audio in table.column("audio").to_pylist():
            audio_id = Path(audio["path"]).stem

            if audio_id in wanted:
                assert audio_id not in audio_data
                audio_data[audio_id] = audio["bytes"]

    assert set(audio_data) == wanted

    device = "cuda:0"

    processor = WhisperProcessor.from_pretrained(
        config["model_id"],
        revision=config["model_revision"]
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        dtype=torch.float16
    ).to(device)

    model.eval()

    for count, row in enumerate(pending, start=1):
        audio_id = str(row["audio_id"])
        pilot_index = int(row["pilot_index"])

        y, sr = sf.read(
            io.BytesIO(audio_data[audio_id]),
            dtype="float32"
        )

        assert sr == 16000 and y.ndim == 1
        assert np.isfinite(y).all()

        inputs = processor(
            y,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16
        )

        records = []

        for h in hypotheses:
            kwargs = {
                "do_sample": False,
                "num_beams": 1,
                "max_new_tokens": 128
            }

            seed = None
            temperature = None

            if h["method"] == "beam_search":
                kwargs["num_beams"] = int(h["num_beams"])

            elif h["method"] == "sampling":
                seed = int(h["seed"])
                temperature = float(h["temperature"])

                kwargs["do_sample"] = True
                kwargs["temperature"] = temperature

                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():
                generated_ids = model.generate(
                    input_features,
                    **kwargs
                )

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            transcription = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            records.append({
                "pilot_index": pilot_index,
                "audio_id": audio_id,
                "speaker": str(row["speaker"]),
                "hypothesis_id": h["id"],
                "method": h["method"],
                "model_id": config["model_id"],
                "model_revision": config["model_revision"],
                "do_sample": kwargs["do_sample"],
                "num_beams": kwargs["num_beams"],
                "temperature": temperature,
                "seed": seed,
                "transcription": transcription,
                "duration_sec": len(y) / sr,
                "inference_sec": elapsed
            })

        result = pd.DataFrame(
            records,
            columns=expected_columns
        )

        part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
        tmp = part.with_suffix(".tmp")

        result.to_csv(
            tmp,
            index=False,
            encoding="utf-8"
        )
        tmp.replace(part)

        validate_part(part, row)

        if count == 1 or count % 10 == 0 or count == len(pending):
            print(
                f"[{count:03d}/{len(pending):03d}] "
                f"COMPLETE index={pilot_index} audio={audio_id}"
            )

pieces = []

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"
    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100
assert combined["pilot_index"].nunique() == 100
assert not combined.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

tmp_final = FINAL.with_suffix(".tmp")
combined.to_csv(tmp_final, index=False, encoding="utf-8")
tmp_final.replace(FINAL)

final_sha = sha256(FINAL)

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0101-0200"] = "completed_pending_qc"
progress["step3_full_batch_0101-0200_sha256"] = final_sha
progress["step3_full_generated_audio_count"] = 200
progress["step3_full_generated_hypothesis_count"] = 1000
progress["next_task"] = "Quality control for ASR batch 0101-0200"
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0101-0200 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print("Mean unique transcripts:", round(unique_counts.mean(), 3))
print("Audios with raw variability:", int((unique_counts > 1).sum()))
print(
    "Empty transcriptions:",
    int(combined["transcription"].str.strip().eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", final_sha)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
ASR1 = PROJECT / "features/asr/full/asr_hypotheses_0001-0100.csv"
ASR2 = PROJECT / "features/asr/full/asr_hypotheses_0101-0200.csv"

QC1 = PROJECT / "reports/step03_batch_0001-0100_qc.json"
QC2 = PROJECT / "reports/step03_batch_0101-0200_qc.json"
TABLE2 = PROJECT / "reports/step03_batch_0101-0200_qc_per_audio.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
ASR2_SHA = "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == CONFIG_SHA
assert sha256(ASR2) == ASR2_SHA

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "frozen"

# Confirm Batch 1 integrity remains intact.
qc1 = json.loads(QC1.read_text(encoding="utf-8"))
assert qc1["status"] == "PASS"
assert sha256(ASR1) == qc1["asr_csv_sha256"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

expected = pilot.loc[
    pilot["pilot_index"].between(101, 200),
    ["pilot_index", "audio_id", "speaker"]
].sort_values("pilot_index")

df = pd.read_csv(
    ASR2,
    dtype={
        "audio_id": "string",
        "speaker": "string",
        "seed": "string"
    },
    keep_default_na=False
)

assert len(expected) == 100
assert len(df) == 500
assert df["audio_id"].nunique() == 100
assert df["pilot_index"].nunique() == 100

hypotheses = config["hypotheses"]
specs = {h["id"]: h for h in hypotheses}

assert set(specs) == {"H1", "H2", "H3", "H4", "H5"}

assert not df.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert df["model_id"].eq(config["model_id"]).all()
assert df["model_revision"].eq(config["model_revision"]).all()

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

qc_rows = []

for _, item in expected.iterrows():
    audio_id = str(item["audio_id"])

    subset = df.loc[
        df["audio_id"] == audio_id
    ].sort_values("hypothesis_id")

    assert len(subset) == 5
    assert subset["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    assert subset["pilot_index"].eq(
        int(item["pilot_index"])
    ).all()

    assert subset["speaker"].eq(
        str(item["speaker"])
    ).all()

    for _, record in subset.iterrows():
        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            assert float(record["seed"]) == float(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"
            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )
            assert int(record["num_beams"]) == expected_beams

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0

    normalized = subset["transcription"].map(normalize_text)

    qc_rows.append({
        "pilot_index": int(item["pilot_index"]),
        "audio_id": audio_id,
        "speaker": str(item["speaker"]),
        "raw_unique": int(subset["transcription"].nunique()),
        "lexical_unique": int(normalized.nunique()),
        "empty_transcriptions": int(
            subset["transcription"].str.strip().eq("").sum()
        ),
        "max_transcription_chars": int(
            subset["transcription"].str.len().max()
        ),
        "duration_sec": float(subset["duration_sec"].iloc[0])
    })

qc = pd.DataFrame(qc_rows)

assert len(qc) == 100
assert int(qc["empty_transcriptions"].sum()) == 0
assert (qc["duration_sec"] > 0).all()

summary = {
    "batch": "0101-0200",
    "status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "raw_variable_audio_count": int(
        (qc["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (qc["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(qc["raw_unique"].mean()),
    "mean_lexical_unique": float(qc["lexical_unique"].mean()),
    "empty_transcriptions": int(
        qc["empty_transcriptions"].sum()
    ),
    "config_sha256": sha256(CONFIG),
    "asr_csv_sha256": sha256(ASR2),
    "created_at": datetime.now(timezone.utc).isoformat()
}

QC2.parent.mkdir(parents=True, exist_ok=True)

tmp_table = TABLE2.with_suffix(".tmp")
qc.to_csv(tmp_table, index=False, encoding="utf-8")
tmp_table.replace(TABLE2)

QC2.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0101-0200"] = "qc_pass"
progress["step3_full_batch_0101-0200_qc_report"] = str(QC2)
progress["next_task"] = "Create Batch 2 verified ZIP backup"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 2 QC SUMMARY ===")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n=== LEXICAL VARIABILITY DISTRIBUTION ===")
print(
    qc["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nQC report:", QC2)
print("QC table:", TABLE2)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
BACKUP = Path("/kaggle/working/asr_fluency_step03_batch_0101-0200.zip")
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
BATCH1_SHA = "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3"
BATCH2_SHA = "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

required = [
    "configs/asr_hypothesis_pilot_v1.json",
    "data/metadata/pilot_selection_400.csv",
    "data/metadata/asr_pilot_selection_10.csv",
    "data/metadata/asr_critical_listening_009600118.csv",
    "features/asr/full/asr_hypotheses_0001-0100.csv",
    "features/asr/full/asr_hypotheses_0101-0200.csv",
    "reports/step03_asr_configuration_freeze.json",
    "reports/step03_batch_0001-0100_qc.json",
    "reports/step03_batch_0001-0100_qc_per_audio.csv",
    "reports/step03_batch_0101-0200_qc.json",
    "reports/step03_batch_0101-0200_qc_per_audio.csv",
    "checkpoints/research_progress.json",
]

for rel in required:
    assert (PROJECT / rel).is_file(), f"Missing: {rel}"

assert sha256_file(PROJECT / required[0]) == CONFIG_SHA
assert sha256_file(PROJECT / required[4]) == BATCH1_SHA
assert sha256_file(PROJECT / required[5]) == BATCH2_SHA

qc1 = json.loads(
    (PROJECT / "reports/step03_batch_0001-0100_qc.json")
    .read_text(encoding="utf-8")
)
qc2 = json.loads(
    (PROJECT / "reports/step03_batch_0101-0200_qc.json")
    .read_text(encoding="utf-8")
)

assert qc1["status"] == "PASS"
assert qc2["status"] == "PASS"
assert qc1["asr_csv_sha256"] == BATCH1_SHA
assert qc2["asr_csv_sha256"] == BATCH2_SHA
assert qc2["audio_count"] == 100
assert qc2["hypothesis_count"] == 500

part_dir = PROJECT / "features/asr/full/batch_0101-0200_parts"
parts = sorted(part_dir.glob("asr_*_H1-H5.csv"))

assert len(parts) == 100, (
    f"Expected 100 Batch 2 checkpoints, found {len(parts)}"
)

# Mark the verified QC state before packaging.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0001-0100"] = "qc_pass"
progress["step3_full_batch_0101-0200"] = "qc_pass"
progress["step3_batch2_backup_filename"] = BACKUP.name
progress["step3_batch2_backup_status"] = "packaging"
progress["next_task"] = (
    "Verify Batch 2 backup and generate ASR batch 0201-0300"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

files = [PROJECT / rel for rel in required]
files.extend(parts)

# Preserve other small research files if available.
optional_dirs = [
    PROJECT / "features/asr/pilot",
]

for directory in optional_dirs:
    if directory.exists():
        files.extend(p for p in directory.rglob("*") if p.is_file())

optional_files = [
    "reports/step03_pilot_variability_summary.json",
    "reports/step03_pilot_variability_per_audio.csv",
    "reports/step03_pilot_hypothesis_content_audit.csv",
    "reports/step03_critical_asr_review_009600118.md",
]

for rel in optional_files:
    path = PROJECT / rel
    if path.is_file():
        files.append(path)

files = sorted(set(files))

manifest = {
    "archive": BACKUP.name,
    "batch": "0101-0200",
    "batch1_included": True,
    "batch2_qc": "PASS",
    "completed_audio_count": 200,
    "completed_hypothesis_count": 1000,
    "raw_parquet_included": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

tmp = BACKUP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    tmp,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in files:
        relative = path.relative_to(PROJECT).as_posix()
        name = f"acoustic_linguistic_research/{relative}"

        manifest["files"][name] = sha256_file(path)
        archive.write(path, arcname=name)

    archive.writestr(
        "acoustic_linguistic_research/"
        "backup_manifest_batch_0101-0200.json",
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )

tmp.replace(BACKUP)

with zipfile.ZipFile(BACKUP, "r") as archive:
    assert archive.testzip() is None

    saved_manifest = json.loads(
        archive.read(
            "acoustic_linguistic_research/"
            "backup_manifest_batch_0101-0200.json"
        )
    )

    assert saved_manifest["files"] == manifest["files"]

    for name, expected_sha in manifest["files"].items():
        assert sha256_bytes(archive.read(name)) == expected_sha, (
            f"Archive integrity failure: {name}"
        )

print("=== STEP 3.3 BATCH 2 BACKUP VERIFIED ===")
print("Backup:", BACKUP)
print("Files archived:", len(files))
print("Batch 2 audio checkpoints:", len(parts))
print("Total completed audios:", 200)
print("Total completed hypotheses:", 1000)
print("ZIP size MB:", round(BACKUP.stat().st_size / 1024**2, 3))
print("ZIP SHA256:", sha256_file(BACKUP))
print("Batch 1 QC:", qc1["status"])
print("Batch 2 QC:", qc2["status"])
print("Archive integrity: PASS")
print("Next batch: 0201-0300")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0201-0300_parts"
FINAL = OUT_DIR / "asr_hypotheses_0201-0300.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)

PREVIOUS_BATCHES = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == CONFIG_SHA, (
    "Frozen configuration SHA256 mismatch"
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

# Verify both previous batches before continuing.
for label, expected_hash in PREVIOUS_BATCHES.items():

    previous_csv = OUT_DIR / f"asr_hypotheses_{label}.csv"
    previous_qc = PROJECT / f"reports/step03_batch_{label}_qc.json"

    assert previous_csv.exists(), f"Missing: {previous_csv}"
    assert previous_qc.exists(), f"Missing: {previous_qc}"

    qc = json.loads(previous_qc.read_text(encoding="utf-8"))

    assert qc["status"] == "PASS"
    assert sha256(previous_csv) == expected_hash
    assert qc["asr_csv_sha256"] == expected_hash
    assert qc["config_sha256"] == CONFIG_SHA

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.loc[
    pilot["pilot_index"].between(201, 300)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(201, 301))
assert batch["audio_id"].is_unique
assert batch["speaker"].notna().all()

hypotheses = config["hypotheses"]

assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

specs = {h["id"]: h for h in hypotheses}

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):
    df = pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "seed": "string"
        },
        keep_default_na=False
    )

    assert len(df) == 5
    assert df.columns.tolist() == expected_columns
    assert df["pilot_index"].eq(int(row["pilot_index"])).all()
    assert df["audio_id"].eq(str(row["audio_id"])).all()
    assert df["speaker"].eq(str(row["speaker"])).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(
        config["model_revision"]
    ).all()

    assert not df.duplicated(
        ["audio_id", "hypothesis_id"]
    ).any()

    for _, record in df.iterrows():
        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]

        assert isinstance(record["transcription"], str)
        assert record["transcription"].strip() != ""

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1

            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )

            # Accept CSV representations "101" and "101.0".
            assert float(record["seed"]) == float(spec["seed"])

        else:
            assert str(record["do_sample"]).lower() == "false"

            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )

            assert int(record["num_beams"]) == expected_beams
            assert str(record["seed"]) == ""
            assert str(record["temperature"]) == ""

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0
        assert np.isfinite(float(record["inference_sec"]))
        assert float(record["inference_sec"]) >= 0

    return df

# Identify completed work before loading Whisper.
pending = []
completed = 0

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        completed += 1
    else:
        pending.append(row)

print("=== STEP 3.3: ASR BATCH 0201-0300 ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Frozen config SHA256:", sha256(CONFIG))
print("Total audios:", len(batch))
print("Completed checkpoints:", completed)
print("Pending audios:", len(pending))
print("Expected hypotheses:", 500)
print("Previous batches: QC PASS")

if pending:

    assert TRAIN.exists(), "Train Parquet missing"
    assert torch.cuda.is_available(), "CUDA GPU unavailable"

    wanted = {str(row["audio_id"]) for row in pending}
    audio_data = {}

    pf = pq.ParquetFile(TRAIN)

    assert "audio" in pf.schema_arrow.names

    for group_id in range(pf.metadata.num_row_groups):

        table = pf.read_row_group(
            group_id,
            columns=["audio"]
        )

        for audio in table.column("audio").to_pylist():

            audio_id = Path(audio["path"]).stem

            if audio_id in wanted:
                assert audio_id not in audio_data
                audio_data[audio_id] = audio["bytes"]

    assert set(audio_data) == wanted, (
        "Some selected audio files were not found"
    )

    device = "cuda:0"

    processor = WhisperProcessor.from_pretrained(
        config["model_id"],
        revision=config["model_revision"]
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        dtype=torch.float16
    ).to(device)

    model.eval()

    for count, row in enumerate(pending, start=1):

        audio_id = str(row["audio_id"])
        pilot_index = int(row["pilot_index"])

        y, sr = sf.read(
            io.BytesIO(audio_data[audio_id]),
            dtype="float32"
        )

        assert sr == 16000
        assert y.ndim == 1
        assert len(y) > 0
        assert np.isfinite(y).all()

        inputs = processor(
            y,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16
        )

        records = []

        for h in hypotheses:

            kwargs = {
                "do_sample": False,
                "num_beams": 1,
                "max_new_tokens": 128
            }

            seed = None
            temperature = None

            if h["method"] == "beam_search":

                kwargs["num_beams"] = int(h["num_beams"])

            elif h["method"] == "sampling":

                seed = int(h["seed"])
                temperature = float(h["temperature"])

                kwargs["do_sample"] = True
                kwargs["temperature"] = temperature

                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():

                generated_ids = model.generate(
                    input_features,
                    **kwargs
                )

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            transcription = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            records.append({
                "pilot_index": pilot_index,
                "audio_id": audio_id,
                "speaker": str(row["speaker"]),
                "hypothesis_id": h["id"],
                "method": h["method"],
                "model_id": config["model_id"],
                "model_revision": config["model_revision"],
                "do_sample": kwargs["do_sample"],
                "num_beams": kwargs["num_beams"],
                "temperature": temperature,
                "seed": seed,
                "transcription": transcription,
                "duration_sec": len(y) / sr,
                "inference_sec": elapsed
            })

        result = pd.DataFrame(
            records,
            columns=expected_columns
        )

        part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
        temp = part.with_suffix(".tmp")

        result.to_csv(
            temp,
            index=False,
            encoding="utf-8"
        )
        temp.replace(part)

        validate_part(part, row)

        if (
            count == 1
            or count % 10 == 0
            or count == len(pending)
        ):
            print(
                f"[{count:03d}/{len(pending):03d}] "
                f"COMPLETE index={pilot_index} "
                f"audio={audio_id}"
            )

# Assemble the complete Batch 3 in pilot-index order.
pieces = []

for _, row in batch.iterrows():

    audio_id = str(row["audio_id"])

    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"

    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100
assert combined["pilot_index"].nunique() == 100

assert not combined.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

temp_final = FINAL.with_suffix(".tmp")

combined.to_csv(
    temp_final,
    index=False,
    encoding="utf-8"
)
temp_final.replace(FINAL)

final_sha = sha256(FINAL)

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0201-0300"] = "completed_pending_qc"
progress["step3_full_batch_0201-0300_sha256"] = final_sha
progress["step3_full_generated_audio_count"] = 300
progress["step3_full_generated_hypothesis_count"] = 1500
progress["next_task"] = (
    "Quality control for ASR batch 0201-0300"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0201-0300 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print("Mean unique transcripts:", round(unique_counts.mean(), 3))
print(
    "Audios with raw variability:",
    int((unique_counts > 1).sum())
)
print(
    "Empty transcriptions:",
    int(combined["transcription"].str.strip().eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", final_sha)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
ASR_DIR = PROJECT / "features/asr/full"
REPORT_DIR = PROJECT / "reports"
CHECK = PROJECT / "checkpoints/research_progress.json"

ASR3 = ASR_DIR / "asr_hypotheses_0201-0300.csv"
QC3 = REPORT_DIR / "step03_batch_0201-0300_qc.json"
TABLE3 = REPORT_DIR / "step03_batch_0201-0300_qc_per_audio.csv"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
ASR3_SHA = "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51"

PREVIOUS = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def normalize_text(value):
    value = str(value).lower()
    value = re.sub(r"[^\w\s']", " ", value)
    return re.sub(r"\s+", " ", value).strip()

assert sha256(CONFIG) == CONFIG_SHA
assert sha256(ASR3) == ASR3_SHA

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "frozen"

# Verify earlier batches are unchanged.
for label, expected_sha in PREVIOUS.items():
    csv_path = ASR_DIR / f"asr_hypotheses_{label}.csv"
    qc_path = REPORT_DIR / f"step03_batch_{label}_qc.json"

    previous_qc = json.loads(
        qc_path.read_text(encoding="utf-8")
    )

    assert previous_qc["status"] == "PASS"
    assert sha256(csv_path) == expected_sha
    assert previous_qc["asr_csv_sha256"] == expected_sha

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

expected = pilot.loc[
    pilot["pilot_index"].between(201, 300),
    ["pilot_index", "audio_id", "speaker"]
].sort_values("pilot_index")

df = pd.read_csv(
    ASR3,
    dtype={
        "audio_id": "string",
        "speaker": "string",
        "seed": "string"
    },
    keep_default_na=False
)

assert len(expected) == 100
assert len(df) == 500
assert df["audio_id"].nunique() == 100
assert df["pilot_index"].nunique() == 100
assert not df.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert df["model_id"].eq(config["model_id"]).all()
assert df["model_revision"].eq(config["model_revision"]).all()

hypothesis_ids = ["H1", "H2", "H3", "H4", "H5"]
specs = {h["id"]: h for h in config["hypotheses"]}
assert set(specs) == set(hypothesis_ids)

qc_rows = []

for _, item in expected.iterrows():
    audio_id = str(item["audio_id"])

    subset = df.loc[
        df["audio_id"] == audio_id
    ].sort_values("hypothesis_id")

    assert len(subset) == 5
    assert subset["hypothesis_id"].tolist() == hypothesis_ids
    assert subset["pilot_index"].eq(
        int(item["pilot_index"])
    ).all()
    assert subset["speaker"].eq(
        str(item["speaker"])
    ).all()

    for _, record in subset.iterrows():
        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]
        assert str(record["transcription"]).strip() != ""

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            assert float(record["seed"]) == float(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"

            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )

            assert int(record["num_beams"]) == expected_beams
            assert str(record["temperature"]) == ""
            assert str(record["seed"]) == ""

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0
        assert np.isfinite(float(record["inference_sec"]))
        assert float(record["inference_sec"]) >= 0

    normalized = subset["transcription"].map(normalize_text)

    qc_rows.append({
        "pilot_index": int(item["pilot_index"]),
        "audio_id": audio_id,
        "speaker": str(item["speaker"]),
        "raw_unique": int(subset["transcription"].nunique()),
        "lexical_unique": int(normalized.nunique()),
        "empty_transcriptions": int(
            subset["transcription"].str.strip().eq("").sum()
        ),
        "max_transcription_chars": int(
            subset["transcription"].str.len().max()
        ),
        "duration_sec": float(subset["duration_sec"].iloc[0])
    })

qc = pd.DataFrame(qc_rows)

assert len(qc) == 100
assert int(qc["empty_transcriptions"].sum()) == 0
assert (qc["duration_sec"] > 0).all()
assert (qc["lexical_unique"] <= qc["raw_unique"]).all()

summary = {
    "batch": "0201-0300",
    "status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "raw_variable_audio_count": int(
        (qc["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (qc["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(qc["raw_unique"].mean()),
    "mean_lexical_unique": float(qc["lexical_unique"].mean()),
    "empty_transcriptions": int(
        qc["empty_transcriptions"].sum()
    ),
    "config_sha256": sha256(CONFIG),
    "asr_csv_sha256": sha256(ASR3),
    "created_at": datetime.now(timezone.utc).isoformat()
}

# Verify summary agrees with generation output.
assert summary["raw_variable_audio_count"] == 38
assert np.isclose(summary["mean_raw_unique"], 1.62)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

tmp_table = TABLE3.with_suffix(".tmp")
qc.to_csv(tmp_table, index=False, encoding="utf-8")
tmp_table.replace(TABLE3)

tmp_report = QC3.with_suffix(".tmp")
tmp_report.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
tmp_report.replace(QC3)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0201-0300"] = "qc_pass"
progress["step3_full_batch_0201-0300_qc_report"] = str(QC3)
progress["next_task"] = "Create Batch 3 verified ZIP backup"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 3 QC SUMMARY ===")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n=== LEXICAL VARIABILITY DISTRIBUTION ===")
print(
    qc["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nQC report:", QC3)
print("QC table:", TABLE3)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
BACKUP = Path("/kaggle/working/asr_fluency_step03_batch_0201-0300.zip")
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"

BATCH_SHAS = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f",
    "0201-0300": "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51"
}

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
assert sha256_file(CONFIG) == CONFIG_SHA

required = [
    "configs/asr_hypothesis_pilot_v1.json",
    "data/metadata/pilot_selection_400.csv",
    "data/metadata/asr_pilot_selection_10.csv",
    "data/metadata/asr_critical_listening_009600118.csv",
    "reports/step03_asr_configuration_freeze.json",
    "checkpoints/research_progress.json",
]

for label, expected_sha in BATCH_SHAS.items():
    csv_relative = f"features/asr/full/asr_hypotheses_{label}.csv"
    qc_relative = f"reports/step03_batch_{label}_qc.json"
    table_relative = f"reports/step03_batch_{label}_qc_per_audio.csv"

    required.extend([csv_relative, qc_relative, table_relative])

    csv_path = PROJECT / csv_relative
    qc_path = PROJECT / qc_relative

    assert csv_path.is_file()
    assert qc_path.is_file()
    assert sha256_file(csv_path) == expected_sha

    qc = json.loads(qc_path.read_text(encoding="utf-8"))

    assert qc["status"] == "PASS"
    assert qc["audio_count"] == 100
    assert qc["hypothesis_count"] == 500
    assert qc["config_sha256"] == CONFIG_SHA
    assert qc["asr_csv_sha256"] == expected_sha

for relative in required:
    assert (PROJECT / relative).is_file(), (
        f"Missing required file: {relative}"
    )

# Verify Batch 3 per-audio checkpoints.
PART_DIR = PROJECT / "features/asr/full/batch_0201-0300_parts"
parts = sorted(PART_DIR.glob("asr_*_H1-H5.csv"))

assert len(parts) == 100, (
    f"Expected 100 checkpoints, found {len(parts)}"
)

# Update the checkpoint before packaging.
progress = json.loads(CHECK.read_text(encoding="utf-8"))

for label in BATCH_SHAS:
    assert progress.get(
        f"step3_full_batch_{label}"
    ) == "qc_pass", f"QC state mismatch: {label}"

progress["step3_batch3_backup_filename"] = BACKUP.name
progress["step3_batch3_backup_status"] = "packaging"
progress["step3_full_generated_audio_count"] = 300
progress["step3_full_generated_hypothesis_count"] = 1500
progress["next_task"] = (
    "Verify Batch 3 backup and start ASR Batch 0301-0400"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

files = [PROJECT / relative for relative in required]
files.extend(parts)

# Include existing pilot reports and small ASR artifacts.
optional_files = [
    "reports/step03_pilot_variability_summary.json",
    "reports/step03_pilot_variability_per_audio.csv",
    "reports/step03_pilot_hypothesis_content_audit.csv",
    "reports/step03_critical_asr_review_009600118.md",
    "reports/step03_whisper_smoke_test.json",
    "reports/step03_whisper_beam_test.json",
    "reports/step03_sampling_smoke_test.json"
]

for relative in optional_files:
    path = PROJECT / relative
    if path.is_file():
        files.append(path)

for directory in [
    PROJECT / "features/asr/pilot"
]:
    if directory.exists():
        files.extend(
            p for p in directory.rglob("*") if p.is_file()
        )

files.extend(
    PROJECT.glob("features/asr/asr_smoke_test_*.csv")
)

files = sorted(set(files))

manifest = {
    "backup_name": BACKUP.name,
    "batch": "0201-0300",
    "batch_csv_ranges_included": list(BATCH_SHAS.keys()),
    "batch3_individual_checkpoints": 100,
    "completed_audio_count": 300,
    "completed_hypothesis_count": 1500,
    "qc_status": "PASS",
    "raw_parquet_included": False,
    "whisper_weights_included": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

temp_zip = BACKUP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    temp_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in files:
        relative = path.relative_to(PROJECT).as_posix()
        archive_name = (
            "acoustic_linguistic_research/" + relative
        )

        manifest["files"][archive_name] = sha256_file(path)
        archive.write(path, arcname=archive_name)

    manifest_name = (
        "acoustic_linguistic_research/"
        "backup_manifest_batch_0201-0300.json"
    )

    archive.writestr(
        manifest_name,
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        )
    )

temp_zip.replace(BACKUP)

# Verify all archived bytes against the manifest.
with zipfile.ZipFile(BACKUP, "r") as archive:

    assert archive.testzip() is None

    saved_manifest = json.loads(
        archive.read(manifest_name)
    )

    assert saved_manifest["files"] == manifest["files"]

    for name, expected_hash in manifest["files"].items():
        actual_hash = sha256_bytes(archive.read(name))
        assert actual_hash == expected_hash, (
            f"Archive integrity error: {name}"
        )

# Mark backup as verified in the live checkpoint.
# The archived checkpoint intentionally records "packaging".
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_batch3_backup_status"] = "verified"
progress["step3_batch3_backup_sha256"] = sha256_file(BACKUP)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 3 BACKUP VERIFIED ===")
print("Backup:", BACKUP)
print("Files archived:", len(files))
print("Batch 3 audio checkpoints:", len(parts))
print("Completed audios:", 300)
print("Completed hypotheses:", 1500)
print("ZIP size MB:", round(BACKUP.stat().st_size / 1024**2, 3))
print("ZIP SHA256:", sha256_file(BACKUP))
print("Batch 1 QC: PASS")
print("Batch 2 QC: PASS")
print("Batch 3 QC: PASS")
print("Archive integrity: PASS")
print("Next batch: 0301-0400")
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0301-0400_parts"
FINAL = OUT_DIR / "asr_hypotheses_0301-0400.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)

PREVIOUS_BATCHES = {
    "0001-0100": (
        "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3"
    ),
    "0101-0200": (
        "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"
    ),
    "0201-0300": (
        "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51"
    )
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == CONFIG_SHA, (
    "Frozen configuration SHA256 mismatch"
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

# Verify all previous batches.
for label, expected_hash in PREVIOUS_BATCHES.items():

    previous_csv = (
        OUT_DIR / f"asr_hypotheses_{label}.csv"
    )
    previous_qc = (
        PROJECT / f"reports/step03_batch_{label}_qc.json"
    )

    assert previous_csv.is_file()
    assert previous_qc.is_file()

    qc = json.loads(
        previous_qc.read_text(encoding="utf-8")
    )

    assert qc["status"] == "PASS"
    assert qc["audio_count"] == 100
    assert qc["hypothesis_count"] == 500
    assert qc["config_sha256"] == CONFIG_SHA
    assert qc["asr_csv_sha256"] == expected_hash
    assert sha256(previous_csv) == expected_hash

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.loc[
    pilot["pilot_index"].between(301, 400)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(301, 401))
assert batch["audio_id"].is_unique
assert batch["speaker"].notna().all()

hypotheses = config["hypotheses"]

assert len(hypotheses) == 5
assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

specs = {h["id"]: h for h in hypotheses}

assert [specs[h]["method"] for h in ["H1", "H2"]] == [
    "greedy", "beam_search"
]

for hid, seed in [("H3", 101), ("H4", 202), ("H5", 303)]:
    assert specs[hid]["method"] == "sampling"
    assert int(specs[hid]["seed"]) == seed
    assert float(specs[hid]["temperature"]) == 0.3

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):

    df = pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "seed": "string"
        },
        keep_default_na=False
    )

    assert df.columns.tolist() == expected_columns
    assert len(df) == 5
    assert df["pilot_index"].eq(
        int(row["pilot_index"])
    ).all()
    assert df["audio_id"].eq(
        str(row["audio_id"])
    ).all()
    assert df["speaker"].eq(
        str(row["speaker"])
    ).all()

    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    assert df["model_id"].eq(
        config["model_id"]
    ).all()

    assert df["model_revision"].eq(
        config["model_revision"]
    ).all()

    assert not df.duplicated(
        ["audio_id", "hypothesis_id"]
    ).any()

    for _, record in df.iterrows():

        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]
        assert isinstance(record["transcription"], str)
        assert record["transcription"].strip() != ""

        if spec["method"] == "sampling":

            assert str(
                record["do_sample"]
            ).lower() == "true"

            assert int(record["num_beams"]) == 1

            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )

            # Accept both "101" and "101.0".
            assert float(record["seed"]) == float(
                spec["seed"]
            )

        else:

            assert str(
                record["do_sample"]
            ).lower() == "false"

            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )

            assert int(
                record["num_beams"]
            ) == expected_beams

            assert str(record["temperature"]) == ""
            assert str(record["seed"]) == ""

        duration = float(record["duration_sec"])
        elapsed = float(record["inference_sec"])

        assert np.isfinite(duration) and duration > 0
        assert np.isfinite(elapsed) and elapsed >= 0

    return df


# Identify finished checkpoints before loading the model.
pending = []
completed = 0

for _, row in batch.iterrows():

    audio_id = str(row["audio_id"])
    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        completed += 1
    else:
        pending.append(row)

print("=== STEP 3.3: ASR BATCH 0301-0400 ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Config SHA256:", sha256(CONFIG))
print("Previous batches: QC PASS")
print("Total audios:", len(batch))
print("Completed checkpoints:", completed)
print("Pending audios:", len(pending))
print("Expected hypotheses:", 500)

if pending:

    assert TRAIN.exists(), "Train Parquet missing"
    assert torch.cuda.is_available(), "CUDA GPU unavailable"

    wanted = {
        str(row["audio_id"])
        for row in pending
    }

    audio_data = {}
    pf = pq.ParquetFile(TRAIN)

    # Use schema_arrow.names, NOT schema.names.
    assert "audio" in pf.schema_arrow.names

    for group_id in range(pf.metadata.num_row_groups):

        table = pf.read_row_group(
            group_id,
            columns=["audio"]
        )

        for audio in table.column("audio").to_pylist():

            audio_id = Path(audio["path"]).stem

            if audio_id in wanted:

                assert audio_id not in audio_data
                audio_data[audio_id] = audio["bytes"]

    assert set(audio_data) == wanted, (
        "Some selected audio recordings are missing."
    )

    device = "cuda:0"

    processor = WhisperProcessor.from_pretrained(
        config["model_id"],
        revision=config["model_revision"]
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        dtype=torch.float16
    ).to(device)

    model.eval()

    for count, row in enumerate(pending, start=1):

        audio_id = str(row["audio_id"])
        pilot_index = int(row["pilot_index"])

        y, sr = sf.read(
            io.BytesIO(audio_data[audio_id]),
            dtype="float32"
        )

        assert sr == 16000
        assert y.ndim == 1
        assert len(y) > 0
        assert np.isfinite(y).all()

        inputs = processor(
            y,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16
        )

        records = []

        for h in hypotheses:

            kwargs = {
                "do_sample": False,
                "num_beams": 1,
                "max_new_tokens": 128
            }

            seed = None
            temperature = None

            if h["method"] == "beam_search":

                kwargs["num_beams"] = int(
                    h["num_beams"]
                )

            elif h["method"] == "sampling":

                seed = int(h["seed"])
                temperature = float(h["temperature"])

                kwargs["do_sample"] = True
                kwargs["temperature"] = temperature

                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():

                generated_ids = model.generate(
                    input_features,
                    **kwargs
                )

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            transcription = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            records.append({
                "pilot_index": pilot_index,
                "audio_id": audio_id,
                "speaker": str(row["speaker"]),
                "hypothesis_id": h["id"],
                "method": h["method"],
                "model_id": config["model_id"],
                "model_revision": config["model_revision"],
                "do_sample": kwargs["do_sample"],
                "num_beams": kwargs["num_beams"],
                "temperature": temperature,
                "seed": seed,
                "transcription": transcription,
                "duration_sec": len(y) / sr,
                "inference_sec": elapsed
            })

        result = pd.DataFrame(
            records,
            columns=expected_columns
        )

        # Save each completed recording atomically.
        part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
        temp = part.with_suffix(".tmp")

        result.to_csv(
            temp,
            index=False,
            encoding="utf-8"
        )

        temp.replace(part)
        validate_part(part, row)

        if (
            count == 1
            or count % 10 == 0
            or count == len(pending)
        ):
            print(
                f"[{count:03d}/{len(pending):03d}] "
                f"COMPLETE index={pilot_index} "
                f"audio={audio_id}"
            )

# Combine all verified audio checkpoints.
pieces = []

for _, row in batch.iterrows():

    audio_id = str(row["audio_id"])
    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"

    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100
assert combined["pilot_index"].nunique() == 100

assert not combined.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

# Avoid accidentally replacing an existing final
# result with different bytes.
temp_final = FINAL.with_suffix(".tmp")

combined.to_csv(
    temp_final,
    index=False,
    encoding="utf-8"
)

if FINAL.exists():
    assert sha256(FINAL) == sha256(temp_final), (
        "Existing final Batch 4 CSV differs. "
        "Existing file preserved; investigate before overwrite."
    )
    temp_final.unlink()
else:
    temp_final.replace(FINAL)

final_sha = sha256(FINAL)

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0301-0400"] = "completed_pending_qc"
progress["step3_full_batch_0301-0400_sha256"] = final_sha
progress["step3_full_generated_audio_count"] = 400
progress["step3_full_generated_hypothesis_count"] = 2000
progress["next_task"] = (
    "Quality control for ASR Batch 0301-0400"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0301-0400 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print(
    "Mean unique transcripts:",
    round(unique_counts.mean(), 3)
)
print(
    "Audios with raw variability:",
    int((unique_counts > 1).sum())
)
print(
    "Empty transcriptions:",
    int(combined["transcription"].str.strip().eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", final_sha)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
ASR_DIR = PROJECT / "features/asr/full"
REPORT_DIR = PROJECT / "reports"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"

EXPECTED_SHAS = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f",
    "0201-0300": "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51",
    "0301-0400": "552a52ee21cbda8a07722f60baafee67114542316d3c602243ef5d0cbae1c777"
}

BATCH4_QC = REPORT_DIR / "step03_batch_0301-0400_qc.json"
BATCH4_TABLE = REPORT_DIR / "step03_batch_0301-0400_qc_per_audio.csv"
FULL_CSV = ASR_DIR / "asr_hypotheses_0001-0400.csv"
FULL_QC = REPORT_DIR / "step03_full_0400_integrity_qc.json"
FULL_TABLE = REPORT_DIR / "step03_full_0400_variability_per_audio.csv"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def normalize_text(value):
    value = str(value).lower()
    value = re.sub(r"[^\w\s']", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def atomic_csv(df, path):
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing file differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

def atomic_json(data, path):
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        for key in data:
            if key != "created_at":
                assert existing.get(key) == data[key], (
                    f"Existing report differs: {path}, field={key}"
                )
        return
    tmp = path.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(path)

assert sha256(CONFIG) == CONFIG_SHA
config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "frozen"

specs = {h["id"]: h for h in config["hypotheses"]}
HIDS = ["H1", "H2", "H3", "H4", "H5"]
assert list(specs.keys()) == HIDS

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("pilot_index")

assert len(pilot) == 400
assert pilot["pilot_index"].tolist() == list(range(1, 401))
assert pilot["audio_id"].is_unique

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method", "model_id",
    "model_revision", "do_sample", "num_beams",
    "temperature", "seed", "transcription",
    "duration_sec", "inference_sec"
]

pieces = []

for label, expected_hash in EXPECTED_SHAS.items():
    path = ASR_DIR / f"asr_hypotheses_{label}.csv"
    assert sha256(path) == expected_hash, (
        f"Hash mismatch: {label}"
    )

    df = pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "seed": "string"
        },
        keep_default_na=False
    )

    assert df.columns.tolist() == expected_columns
    assert len(df) == 500
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(
        config["model_revision"]
    ).all()

    pieces.append(df)

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 2000
assert combined["audio_id"].nunique() == 400
assert combined["pilot_index"].nunique() == 400
assert not combined.duplicated(
    ["pilot_index", "audio_id", "hypothesis_id"]
).any()

qc_rows = []

for _, item in pilot.iterrows():
    idx = int(item["pilot_index"])
    audio_id = str(item["audio_id"])
    speaker = str(item["speaker"])

    subset = combined.loc[
        combined["pilot_index"] == idx
    ].sort_values("hypothesis_id")

    assert len(subset) == 5
    assert subset["hypothesis_id"].tolist() == HIDS
    assert subset["audio_id"].eq(audio_id).all()
    assert subset["speaker"].eq(speaker).all()

    durations = subset["duration_sec"].astype(float)
    assert np.isfinite(durations).all()
    assert (durations > 0).all()
    assert np.allclose(durations, durations.iloc[0])

    for _, record in subset.iterrows():
        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]
        assert str(record["transcription"]).strip() != ""

        assert np.isfinite(float(record["inference_sec"]))
        assert float(record["inference_sec"]) >= 0

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            assert float(record["seed"]) == float(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"
            beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )
            assert int(record["num_beams"]) == beams
            assert str(record["seed"]) == ""
            assert str(record["temperature"]) == ""

    norm = subset["transcription"].map(normalize_text)

    qc_rows.append({
        "pilot_index": idx,
        "audio_id": audio_id,
        "speaker": speaker,
        "batch": f"{((idx - 1) // 100) * 100 + 1:04d}-"
                 f"{((idx - 1) // 100 + 1) * 100:04d}",
        "raw_unique": int(subset["transcription"].nunique()),
        "lexical_unique": int(norm.nunique()),
        "empty_transcriptions": int(
            subset["transcription"].str.strip().eq("").sum()
        ),
        "duration_sec": float(durations.iloc[0])
    })

qc_table = pd.DataFrame(qc_rows)
assert len(qc_table) == 400
assert (qc_table["lexical_unique"] <= qc_table["raw_unique"]).all()
assert qc_table["empty_transcriptions"].sum() == 0

batch4 = qc_table.loc[
    qc_table["pilot_index"].between(301, 400)
].copy()

batch4_summary = {
    "batch": "0301-0400",
    "status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "raw_variable_audio_count": int((batch4["raw_unique"] > 1).sum()),
    "lexically_variable_audio_count": int(
        (batch4["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(batch4["raw_unique"].mean()),
    "mean_lexical_unique": float(batch4["lexical_unique"].mean()),
    "empty_transcriptions": 0,
    "config_sha256": CONFIG_SHA,
    "asr_csv_sha256": EXPECTED_SHAS["0301-0400"],
    "created_at": datetime.now(timezone.utc).isoformat()
}

assert batch4_summary["raw_variable_audio_count"] == 78
assert np.isclose(batch4_summary["mean_raw_unique"], 2.81)

# Confirm previous QC summaries match current recomputation.
for label in ["0001-0100", "0101-0200", "0201-0300"]:
    old_path = REPORT_DIR / f"step03_batch_{label}_qc.json"
    old_qc = json.loads(old_path.read_text(encoding="utf-8"))
    subset = qc_table.loc[qc_table["batch"] == label]

    assert old_qc["status"] == "PASS"
    assert old_qc["asr_csv_sha256"] == EXPECTED_SHAS[label]
    assert old_qc["raw_variable_audio_count"] == int(
        (subset["raw_unique"] > 1).sum()
    )
    assert old_qc["lexically_variable_audio_count"] == int(
        (subset["lexical_unique"] > 1).sum()
    )

REPORT_DIR.mkdir(parents=True, exist_ok=True)

atomic_csv(batch4, BATCH4_TABLE)
atomic_json(batch4_summary, BATCH4_QC)
atomic_csv(qc_table, FULL_TABLE)

# Preserve original ASR transcript content and generation order.
atomic_csv(combined, FULL_CSV)

full_summary = {
    "status": "PASS",
    "audio_count": 400,
    "hypothesis_count": 2000,
    "batch_count": 4,
    "raw_variable_audio_count": int(
        (qc_table["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (qc_table["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(qc_table["raw_unique"].mean()),
    "mean_lexical_unique": float(qc_table["lexical_unique"].mean()),
    "empty_transcriptions": 0,
    "config_sha256": CONFIG_SHA,
    "batch_csv_sha256": EXPECTED_SHAS,
    "combined_csv_sha256": sha256(FULL_CSV),
    "created_at": datetime.now(timezone.utc).isoformat()
}

atomic_json(full_summary, FULL_QC)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0301-0400"] = "qc_pass"
progress["step3_full_batch_0301-0400_qc_report"] = str(BATCH4_QC)
progress["step3_full_0400_integrity_status"] = "qc_pass"
progress["step3_full_0400_combined_sha256"] = sha256(FULL_CSV)
progress["step3_full_generated_audio_count"] = 400
progress["step3_full_generated_hypothesis_count"] = 2000
progress["next_task"] = (
    "Create verified final ASR ZIP backup, then proceed to "
    "linguistic feature extraction"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 4 QC SUMMARY ===")
for key, value in batch4_summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== ALL 400 AUDIO INTEGRITY SUMMARY ===")
for key, value in full_summary.items():
    if key not in ["created_at", "batch_csv_sha256"]:
        print(f"{key}: {value}")

print("\n=== LEXICAL VARIABILITY BY BATCH ===")
print(
    qc_table.groupby("batch").agg(
        audio_count=("audio_id", "count"),
        raw_variable=("raw_unique", lambda s: int((s > 1).sum())),
        lexical_variable=("lexical_unique", lambda s: int((s > 1).sum()))
    ).to_string()
)

print("\n=== BATCH 4 LEXICAL DISTRIBUTION ===")
print(
    batch4["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nCombined CSV:", FULL_CSV)
print("Full QC report:", FULL_QC)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
BACKUP = Path("/kaggle/working/asr_fluency_step03_complete.zip")
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
FULL_SHA = "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"

BATCH_SHAS = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f",
    "0201-0300": "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51",
    "0301-0400": "552a52ee21cbda8a07722f60baafee67114542316d3c602243ef5d0cbae1c777"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

config_file = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
full_file = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
full_qc_file = PROJECT / "reports/step03_full_0400_integrity_qc.json"

assert sha256(config_file) == CONFIG_SHA
assert sha256(full_file) == FULL_SHA

full_qc = json.loads(full_qc_file.read_text(encoding="utf-8"))

assert full_qc["status"] == "PASS"
assert full_qc["audio_count"] == 400
assert full_qc["hypothesis_count"] == 2000
assert full_qc["config_sha256"] == CONFIG_SHA
assert full_qc["combined_csv_sha256"] == FULL_SHA

# Verify four original batch files and QC reports.
for label, expected_sha in BATCH_SHAS.items():
    csv_file = (
        PROJECT / f"features/asr/full/asr_hypotheses_{label}.csv"
    )
    qc_file = PROJECT / f"reports/step03_batch_{label}_qc.json"

    assert sha256(csv_file) == expected_sha

    qc = json.loads(qc_file.read_text(encoding="utf-8"))
    assert qc["status"] == "PASS"
    assert qc["audio_count"] == 100
    assert qc["hypothesis_count"] == 500
    assert qc["asr_csv_sha256"] == expected_sha

# Verify all 400 individual checkpoints exist.
part_files = []

for label in BATCH_SHAS:
    directory = (
        PROJECT / f"features/asr/full/batch_{label}_parts"
    )
    parts = sorted(directory.glob("asr_*_H1-H5.csv"))

    assert len(parts) == 100, (
        f"Checkpoint count mismatch in {label}: {len(parts)}"
    )
    part_files.extend(parts)

assert len(part_files) == 400

# Check original acoustic results from STEP 2.
acoustic_file = (
    PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
)

EXPECTED_ACOUSTIC_SHA = (
    "4f477cd0fc0f6cf4382bc92fa39bdae734f22a9440e949b592143db2e261dced"
)

assert acoustic_file.is_file()
assert sha256(acoustic_file) == EXPECTED_ACOUSTIC_SHA

# Validate full ASR CSV once again.
asr = pd.read_csv(
    full_file,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(asr) == 2000
assert asr["audio_id"].nunique() == 400
assert asr["hypothesis_id"].nunique() == 5
assert not asr.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

# Update checkpoint before writing the archive.
progress = json.loads(CHECK.read_text(encoding="utf-8"))

for label in BATCH_SHAS:
    assert progress.get(
        f"step3_full_batch_{label}"
    ) == "qc_pass"

progress["step3_full_0400_integrity_status"] = "qc_pass"
progress["step3_full_generated_audio_count"] = 400
progress["step3_full_generated_hypothesis_count"] = 2000
progress["step3_complete_backup_status"] = "packaging"
progress["step3_complete_backup_filename"] = BACKUP.name
progress["next_task"] = (
    "Verify final STEP 3 backup and start STEP 4 "
    "linguistic feature extraction"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Archive all small research artifacts, not raw audio or models.
INCLUDE_DIRS = [
    "configs",
    "checkpoints",
    "data/metadata",
    "features/acoustic",
    "features/asr",
    "reports",
    "results"
]

EXCLUDE_SUFFIXES = {
    ".parquet", ".wav", ".mp3", ".flac",
    ".pt", ".pth", ".safetensors", ".tmp",
    ".zip"
}

files = set()

for relative_dir in INCLUDE_DIRS:
    directory = PROJECT / relative_dir

    if not directory.exists():
        continue

    for path in directory.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() in EXCLUDE_SUFFIXES:
            continue
        files.add(path)

files = sorted(files)

assert config_file in files
assert full_file in files
assert full_qc_file in files
assert acoustic_file in files
assert CHECK in files
assert all(path in files for path in part_files)

manifest = {
    "archive_name": BACKUP.name,
    "research_stage": "STEP 3 complete",
    "status": "QC_PASS",
    "audio_count": 400,
    "hypothesis_count": 2000,
    "individual_asr_checkpoints": 400,
    "frozen_config_sha256": CONFIG_SHA,
    "combined_asr_sha256": FULL_SHA,
    "acoustic_features_sha256": EXPECTED_ACOUSTIC_SHA,
    "raw_audio_included": False,
    "model_weights_included": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

temp_zip = BACKUP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    temp_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in files:
        relative = path.relative_to(PROJECT).as_posix()
        archive_name = (
            "acoustic_linguistic_research/" + relative
        )

        manifest["files"][archive_name] = sha256(path)
        archive.write(path, arcname=archive_name)

    manifest_name = (
        "acoustic_linguistic_research/"
        "backup_manifest_step03_complete.json"
    )

    archive.writestr(
        manifest_name,
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )

# Verify temporary ZIP BEFORE publishing it.
with zipfile.ZipFile(temp_zip, "r") as archive:
    assert archive.testzip() is None

    saved_manifest = json.loads(
        archive.read(manifest_name)
    )
    assert saved_manifest["files"] == manifest["files"]

    for name, expected_hash in manifest["files"].items():
        actual_hash = hashlib.sha256(
            archive.read(name)
        ).hexdigest()
        assert actual_hash == expected_hash, (
            f"Archive integrity failure: {name}"
        )

temp_zip.replace(BACKUP)

# Mark verified status in the live checkpoint.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_complete_backup_status"] = "verified"
progress["step3_complete_backup_sha256"] = sha256(BACKUP)
progress["completed_steps"] = sorted(
    set(progress.get("completed_steps", [])) | {3}
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3 FINAL BACKUP VERIFIED ===")
print("Backup:", BACKUP)
print("Files archived:", len(files))
print("Individual audio checkpoints:", len(part_files))
print("Total audios:", 400)
print("Total hypotheses:", 2000)
print("Full ASR QC:", full_qc["status"])
print("STEP 2 acoustic features: INCLUDED")
print("ZIP size MB:", round(BACKUP.stat().st_size / 1024**2, 3))
print("ZIP SHA256:", sha256(BACKUP))
print("Archive integrity: PASS")
print("STEP 3 status: COMPLETE")
print("Next task:", progress["next_task"])


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
CONFIG = PROJECT / "configs/linguistic_features_v1.json"
OUTPUT = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
REPORT = PROJECT / "reports/step04_linguistic_extraction_qc.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

EXPECTED_ASR_SHA = (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(SOURCE) == EXPECTED_ASR_SHA

# Preserve apostrophes inside words, and normalize case.
# This is a simple lexical tokenizer, not linguistic parsing.
TOKEN_PATTERN = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE
)

feature_names = [
    "word_count",
    "unique_word_count",
    "type_token_ratio",
    "mean_word_length",
    "character_count",
    "adjacent_repeat_count",
    "adjacent_repeat_ratio"
]

feature_config = {
    "name": "linguistic_features_v1",
    "status": "proposed_not_frozen",
    "source_asr_sha256": EXPECTED_ASR_SHA,
    "normalization": "unicode_casefold",
    "tokenization": "unicode_words_internal_apostrophes",
    "punctuation": "excluded_from_token_counts",
    "features": feature_names,
    "empty_text_policy": "zero_counts_nan_ratios",
    "notes": [
        "Features describe ASR output, not verified speech.",
        "No prompt or fluency label is used.",
        "TTR is dependent on utterance length.",
        "Adjacent repetition is a transcript proxy."
    ]
}

CONFIG.parent.mkdir(parents=True, exist_ok=True)

if CONFIG.exists():
    existing = json.loads(CONFIG.read_text(encoding="utf-8"))
    assert existing == feature_config, (
        "Existing linguistic config differs; not overwriting."
    )
else:
    CONFIG.write_text(
        json.dumps(feature_config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

asr = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(asr) == 2000
assert asr["audio_id"].nunique() == 400
assert not asr.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert asr["transcription"].str.strip().ne("").all()

def extract_features(text):
    words = [
        word.casefold()
        for word in TOKEN_PATTERN.findall(str(text))
    ]

    n = len(words)
    unique_n = len(set(words))
    chars = sum(len(word) for word in words)

    repeats = sum(
        words[i] == words[i-1]
        for i in range(1, n)
    )

    return {
        "word_count": n,
        "unique_word_count": unique_n,
        "type_token_ratio": unique_n / n if n else np.nan,
        "mean_word_length": chars / n if n else np.nan,
        "character_count": chars,
        "adjacent_repeat_count": repeats,
        "adjacent_repeat_ratio": (
            repeats / (n - 1) if n > 1 else np.nan
        )
    }

extracted = pd.DataFrame(
    [extract_features(t) for t in asr["transcription"]]
)

result = pd.concat(
    [
        asr[
            ["pilot_index", "audio_id", "speaker", "hypothesis_id"]
        ].reset_index(drop=True),
        extracted
    ],
    axis=1
)

assert len(result) == 2000
assert result["audio_id"].nunique() == 400
assert not result.duplicated(
    ["audio_id", "hypothesis_id"]
).any()
assert result["word_count"].ge(0).all()
assert (
    result["unique_word_count"] <= result["word_count"]
).all()
assert result["type_token_ratio"].dropna().between(0, 1).all()
assert result["adjacent_repeat_ratio"].dropna().between(0, 1).all()

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

tmp = OUTPUT.with_suffix(".tmp")
result.to_csv(tmp, index=False, encoding="utf-8")

if OUTPUT.exists():
    assert sha256(OUTPUT) == sha256(tmp), (
        "Existing output differs; preserved for investigation."
    )
    tmp.unlink()
else:
    tmp.replace(OUTPUT)

# Measure linguistic feature variation for the same audio.
variation = result.groupby("audio_id")[feature_names].agg(
    lambda col: col.nunique(dropna=False)
)

variable_counts = {
    feature: int((variation[feature] > 1).sum())
    for feature in feature_names
}

report = {
    "status": "PASS",
    "feature_config_status": "proposed_not_frozen",
    "audio_count": 400,
    "hypothesis_count": 2000,
    "feature_count": len(feature_names),
    "zero_word_transcripts": int(
        result["word_count"].eq(0).sum()
    ),
    "mean_word_count": float(result["word_count"].mean()),
    "feature_variable_audio_counts": variable_counts,
    "source_asr_sha256": EXPECTED_ASR_SHA,
    "output_sha256": sha256(OUTPUT),
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_linguistic_extraction"] = "completed_pending_review"
progress["step4_linguistic_features_sha256"] = sha256(OUTPUT)
progress["next_task"] = (
    "Review linguistic features and assess suitability "
    "for fluency prediction"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.1 LINGUISTIC EXTRACTION SUMMARY ===")
print("Status:", report["status"])
print("Audio count:", report["audio_count"])
print("Hypotheses:", report["hypothesis_count"])
print("Features:", report["feature_count"])
print("Mean word count:", round(report["mean_word_count"], 3))
print("Zero-word transcripts:", report["zero_word_transcripts"])

print("\n=== FEATURE VARIABILITY ACROSS H1-H5 ===")
for name, count in variable_counts.items():
    print(f"{name}: {count}/400 audios")

print("\n=== FEATURE DESCRIPTIVE STATISTICS ===")
print(
    result[feature_names]
    .describe()
    .round(4)
    .to_string()
)

print("\nCSV:", OUTPUT)
print("SHA256:", sha256(OUTPUT))
print("Configuration status:", feature_config["status"])
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
FEATURES = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

AUDIT = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
REPORT = PROJECT / "reports/step04_linguistic_anomaly_summary.json"

EXPECTED_ASR_SHA = "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
EXPECTED_FEATURE_SHA = "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(ASR) == EXPECTED_ASR_SHA
assert sha256(FEATURES) == EXPECTED_FEATURE_SHA

asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

keys = ["pilot_index", "audio_id", "speaker", "hypothesis_id"]

assert len(asr) == len(features) == 2000
assert not asr.duplicated(keys).any()
assert not features.duplicated(keys).any()

merged = features.merge(
    asr[keys + ["transcription", "duration_sec"]],
    on=keys,
    how="inner",
    validate="one_to_one"
)

assert len(merged) == 2000

# Diagnostic thresholds only; not exclusion criteria.
merged["flag_long_transcript"] = merged["word_count"] >= 20
merged["flag_low_ttr"] = merged["type_token_ratio"] < 0.5
merged["flag_long_mean_word"] = merged["mean_word_length"] > 10
merged["flag_high_repetition"] = merged["adjacent_repeat_count"] >= 2

flags = [
    "flag_long_transcript",
    "flag_low_ttr",
    "flag_long_mean_word",
    "flag_high_repetition"
]

merged["any_flag"] = merged[flags].any(axis=1)

anomalies = merged.loc[merged["any_flag"]].copy()

# Preserve original transcriptions and all numeric values.
AUDIT.parent.mkdir(parents=True, exist_ok=True)

temp = AUDIT.with_suffix(".tmp")
anomalies.to_csv(temp, index=False, encoding="utf-8")

if AUDIT.exists():
    assert sha256(AUDIT) == sha256(temp), (
        "Existing anomaly audit differs; no overwrite."
    )
    temp.unlink()
else:
    temp.replace(AUDIT)

one_word_count = int(merged["word_count"].eq(1).sum())
missing_repeat_ratio = int(
    merged["adjacent_repeat_ratio"].isna().sum()
)

assert one_word_count == missing_repeat_ratio, (
    "Unexpected missing adjacent_repeat_ratio values."
)

summary = {
    "status": "generated_pending_review",
    "total_hypotheses": 2000,
    "flagged_hypotheses": int(len(anomalies)),
    "flagged_unique_audios": int(anomalies["audio_id"].nunique()),
    "long_transcripts_ge_20": int(merged["flag_long_transcript"].sum()),
    "low_ttr_lt_0_5": int(merged["flag_low_ttr"].sum()),
    "long_mean_word_gt_10": int(merged["flag_long_mean_word"].sum()),
    "high_repetition_ge_2": int(merged["flag_high_repetition"].sum()),
    "one_word_transcripts": one_word_count,
    "missing_repeat_ratio": missing_repeat_ratio,
    "source_asr_sha256": sha256(ASR),
    "source_features_sha256": sha256(FEATURES),
    "audit_csv_sha256": sha256(AUDIT),
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_linguistic_anomaly_audit"] = "generated_pending_review"
progress["step4_linguistic_anomaly_report"] = str(REPORT)
progress["next_task"] = (
    "Inspect extreme ASR transcripts before freezing linguistic features"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.2 ANOMALY SUMMARY ===")
for key, value in summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== TOP 10 LONGEST TRANSCRIPTIONS ===")
longest = merged.sort_values(
    ["word_count", "pilot_index"],
    ascending=[False, True]
).head(10)

for _, row in longest.iterrows():
    print("\nAudio:", row["audio_id"],
          "| Hypothesis:", row["hypothesis_id"],
          "| Words:", row["word_count"],
          "| TTR:", round(row["type_token_ratio"], 4))
    print("Transcript:", repr(row["transcription"][:1000]))

print("\n=== TOP 10 LOWEST TTR TRANSCRIPTIONS ===")
lowest = merged.sort_values(
    ["type_token_ratio", "pilot_index"],
    ascending=[True, True]
).head(10)

for _, row in lowest.iterrows():
    print(
        row["audio_id"],
        row["hypothesis_id"],
        "words=", row["word_count"],
        "TTR=", round(row["type_token_ratio"], 4),
        "repeats=", row["adjacent_repeat_count"]
    )
    print("Transcript:", repr(row["transcription"][:500]))

print("\nAudit CSV:", AUDIT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
FEATURES = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

AUDIT = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
REPORT = PROJECT / "reports/step04_linguistic_anomaly_summary.json"

EXPECTED_ASR_SHA = "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
EXPECTED_FEATURE_SHA = "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(ASR) == EXPECTED_ASR_SHA
assert sha256(FEATURES) == EXPECTED_FEATURE_SHA

asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

features = pd.read_csv(
    FEATURES,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

numeric_cols = [
    "word_count",
    "unique_word_count",
    "type_token_ratio",
    "mean_word_length",
    "character_count",
    "adjacent_repeat_count",
    "adjacent_repeat_ratio"
]

# Convert CSV blanks into actual NaN values.
for col in numeric_cols:
    features[col] = pd.to_numeric(
        features[col],
        errors="coerce"
    )

keys = ["pilot_index", "audio_id", "speaker", "hypothesis_id"]

assert len(asr) == len(features) == 2000
assert not asr.duplicated(keys).any()
assert not features.duplicated(keys).any()

merged = features.merge(
    asr[keys + ["transcription", "duration_sec"]],
    on=keys,
    how="inner",
    validate="one_to_one"
)

assert len(merged) == 2000

# Verify the missing-value policy.
one_word_count = int(merged["word_count"].eq(1).sum())
missing_repeat_ratio = int(
    merged["adjacent_repeat_ratio"].isna().sum()
)

assert one_word_count == missing_repeat_ratio, (
    f"Missing ratio mismatch: "
    f"one_word={one_word_count}, "
    f"missing={missing_repeat_ratio}"
)

assert merged["word_count"].notna().all()
assert merged["adjacent_repeat_count"].notna().all()

# Diagnostic flags only — no exclusions.
merged["flag_long_transcript"] = (
    merged["word_count"] >= 20
)
merged["flag_low_ttr"] = (
    merged["type_token_ratio"] < 0.5
)
merged["flag_long_mean_word"] = (
    merged["mean_word_length"] > 10
)
merged["flag_high_repetition"] = (
    merged["adjacent_repeat_count"] >= 2
)

flags = [
    "flag_long_transcript",
    "flag_low_ttr",
    "flag_long_mean_word",
    "flag_high_repetition"
]

merged["any_flag"] = merged[flags].any(axis=1)

anomalies = merged.loc[
    merged["any_flag"]
].copy()

AUDIT.parent.mkdir(parents=True, exist_ok=True)

temp = AUDIT.with_suffix(".tmp")
anomalies.to_csv(
    temp,
    index=False,
    encoding="utf-8"
)

# The earlier failed Cell 34 may already have created
# an audit CSV. Verify its content before reuse.
if AUDIT.exists():
    assert sha256(AUDIT) == sha256(temp), (
        "Existing anomaly audit differs. "
        "Original file preserved."
    )
    temp.unlink()
else:
    temp.replace(AUDIT)

summary = {
    "status": "generated_pending_review",
    "total_hypotheses": 2000,
    "flagged_hypotheses": int(len(anomalies)),
    "flagged_unique_audios": int(
        anomalies["audio_id"].nunique()
    ),
    "long_transcripts_ge_20": int(
        merged["flag_long_transcript"].sum()
    ),
    "low_ttr_lt_0_5": int(
        merged["flag_low_ttr"].sum()
    ),
    "long_mean_word_gt_10": int(
        merged["flag_long_mean_word"].sum()
    ),
    "high_repetition_ge_2": int(
        merged["flag_high_repetition"].sum()
    ),
    "one_word_transcripts": one_word_count,
    "missing_repeat_ratio": missing_repeat_ratio,
    "source_asr_sha256": sha256(ASR),
    "source_features_sha256": sha256(FEATURES),
    "audit_csv_sha256": sha256(AUDIT),
    "created_at": datetime.now(
        timezone.utc
    ).isoformat()
}

REPORT.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step4_linguistic_anomaly_audit"] = (
    "generated_pending_review"
)
progress["step4_linguistic_anomaly_report"] = str(REPORT)
progress["next_task"] = (
    "Inspect extreme ASR transcripts before "
    "freezing linguistic features"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== STEP 4.2 ANOMALY SUMMARY ===")
for key, value in summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== TOP 10 LONGEST TRANSCRIPTIONS ===")
longest = merged.sort_values(
    ["word_count", "pilot_index"],
    ascending=[False, True]
).head(10)

for _, row in longest.iterrows():
    print(
        "\nAudio:", row["audio_id"],
        "| Hypothesis:", row["hypothesis_id"],
        "| Words:", row["word_count"],
        "| TTR:", round(row["type_token_ratio"], 4)
    )
    print(
        "Transcript:",
        repr(row["transcription"][:1000])
    )

print("\n=== TOP 10 LOWEST TTR TRANSCRIPTIONS ===")
lowest = merged.sort_values(
    ["type_token_ratio", "pilot_index"],
    ascending=[True, True]
).head(10)

for _, row in lowest.iterrows():
    print(
        row["audio_id"],
        row["hypothesis_id"],
        "words=", row["word_count"],
        "TTR=", round(row["type_token_ratio"], 4),
        "repeats=", row["adjacent_repeat_count"]
    )
    print(
        "Transcript:",
        repr(row["transcription"][:500])
    )

print("\n=== MISSING VALUE VALIDATION ===")
print("One-word transcripts:", one_word_count)
print("Missing adjacent_repeat_ratio:", missing_repeat_ratio)
print("Validation: PASS")
print("Audit CSV:", AUDIT)
print("Checkpoint updated.")


In [ ]:

import io
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import numpy as np

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ASR_FILE = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
FEATURE_FILE = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
ANOMALY_FILE = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
TRAIN_FILE = PROJECT / "data/raw/speechocean762_train.parquet"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "reports/step04_targeted_anomaly_inspection.csv"

keys = [
    "pilot_index", "audio_id", "speaker", "hypothesis_id"
]

asr = pd.read_csv(
    ASR_FILE,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

features = pd.read_csv(
    FEATURE_FILE,
    dtype={"audio_id": "string", "speaker": "string"}
)

anomalies = pd.read_csv(
    ANOMALY_FILE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(asr) == len(features) == 2000
assert len(anomalies) == 13

flagged_ids = sorted(anomalies["audio_id"].unique())

assert len(flagged_ids) == 5
assert "030210164" in flagged_ids

merged = asr.merge(
    features[
        keys + [
            "word_count",
            "unique_word_count",
            "type_token_ratio",
            "mean_word_length",
            "adjacent_repeat_count"
        ]
    ],
    on=keys,
    how="inner",
    validate="one_to_one"
)

selected = merged.loc[
    merged["audio_id"].isin(flagged_ids)
].sort_values(["audio_id", "hypothesis_id"]).copy()

assert len(selected) == 25

# Retrieve reading prompts and audio for the flagged cases.
pf = pq.ParquetFile(TRAIN_FILE)

audio_info = {}

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id,
        columns=["audio", "text"]
    )

    audios = table.column("audio").to_pylist()
    prompts = table.column("text").to_pylist()

    for audio, prompt in zip(audios, prompts):
        audio_id = Path(audio["path"]).stem

        if audio_id in flagged_ids:
            assert audio_id not in audio_info
            audio_info[audio_id] = {
                "audio_bytes": audio["bytes"],
                "reading_prompt": str(prompt)
            }

assert set(audio_info) == set(flagged_ids)

selected["reading_prompt"] = selected["audio_id"].map(
    lambda aid: audio_info[aid]["reading_prompt"]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
selected.to_csv(OUT, index=False, encoding="utf-8")

print("=== STEP 4.2 TARGETED ANOMALY INSPECTION ===")
print("Flagged unique audios:", len(flagged_ids))
print("Hypotheses for inspection:", len(selected))

for aid in flagged_ids:
    subset = selected.loc[selected["audio_id"] == aid]

    print("\n" + "=" * 70)
    print("Audio ID:", aid)
    print("Reading prompt:",
          repr(audio_info[aid]["reading_prompt"]))

    for _, row in subset.iterrows():
        print(
            f"\n{row['hypothesis_id']} "
            f"| words={int(row['word_count'])} "
            f"| TTR={row['type_token_ratio']:.4f} "
            f"| mean_length={row['mean_word_length']:.2f}"
        )
        print("Transcript:", repr(row["transcription"]))

print("\n=== CRITICAL AUDIO PLAYBACK: 030210164 ===")

critical = audio_info["030210164"]
y, sr = sf.read(
    io.BytesIO(critical["audio_bytes"]),
    dtype="float32"
)

assert sr == 16000
assert y.ndim == 1
assert np.isfinite(y).all()

print("Duration:", round(len(y) / sr, 3), "seconds")
print("Prompt:", repr(critical["reading_prompt"]))

display(Audio(y, rate=sr, normalize=False))

# Preserve the review as pending until listening is completed.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_targeted_anomaly_inspection"] = (
    "generated_pending_listening_review"
)
progress["step4_targeted_anomaly_file"] = str(OUT)
progress["next_task"] = (
    "Review audio 030210164 and classify ASR repetition anomaly"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== INSPECTION SAVED ===")
print("CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import numpy as np

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ASR_FILE = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
FEATURE_FILE = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
ANOMALY_FILE = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
TRAIN_FILE = PROJECT / "data/raw/speechocean762_train.parquet"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "reports/step04_targeted_anomaly_inspection.csv"

keys = [
    "pilot_index", "audio_id", "speaker", "hypothesis_id"
]

asr = pd.read_csv(
    ASR_FILE,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

features = pd.read_csv(
    FEATURE_FILE,
    dtype={"audio_id": "string", "speaker": "string"}
)

anomalies = pd.read_csv(
    ANOMALY_FILE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(asr) == len(features) == 2000
assert len(anomalies) == 13

flagged_ids = sorted(anomalies["audio_id"].unique())

assert len(flagged_ids) == 5
assert "030210164" in flagged_ids

merged = asr.merge(
    features[
        keys + [
            "word_count",
            "unique_word_count",
            "type_token_ratio",
            "mean_word_length",
            "adjacent_repeat_count"
        ]
    ],
    on=keys,
    how="inner",
    validate="one_to_one"
)

selected = merged.loc[
    merged["audio_id"].isin(flagged_ids)
].sort_values(["audio_id", "hypothesis_id"]).copy()

assert len(selected) == 25

# Retrieve reading prompts and audio for the flagged cases.
pf = pq.ParquetFile(TRAIN_FILE)

audio_info = {}

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id,
        columns=["audio", "text"]
    )

    audios = table.column("audio").to_pylist()
    prompts = table.column("text").to_pylist()

    for audio, prompt in zip(audios, prompts):
        audio_id = Path(audio["path"]).stem

        if audio_id in flagged_ids:
            assert audio_id not in audio_info
            audio_info[audio_id] = {
                "audio_bytes": audio["bytes"],
                "reading_prompt": str(prompt)
            }

assert set(audio_info) == set(flagged_ids)

selected["reading_prompt"] = selected["audio_id"].map(
    lambda aid: audio_info[aid]["reading_prompt"]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
selected.to_csv(OUT, index=False, encoding="utf-8")

print("=== STEP 4.2 TARGETED ANOMALY INSPECTION ===")
print("Flagged unique audios:", len(flagged_ids))
print("Hypotheses for inspection:", len(selected))

for aid in flagged_ids:
    subset = selected.loc[selected["audio_id"] == aid]

    print("\n" + "=" * 70)
    print("Audio ID:", aid)
    print("Reading prompt:",
          repr(audio_info[aid]["reading_prompt"]))

    for _, row in subset.iterrows():
        print(
            f"\n{row['hypothesis_id']} "
            f"| words={int(row['word_count'])} "
            f"| TTR={row['type_token_ratio']:.4f} "
            f"| mean_length={row['mean_word_length']:.2f}"
        )
        print("Transcript:", repr(row["transcription"]))

print("\n=== CRITICAL AUDIO PLAYBACK: 030210164 ===")

critical = audio_info["030210164"]
y, sr = sf.read(
    io.BytesIO(critical["audio_bytes"]),
    dtype="float32"
)

assert sr == 16000
assert y.ndim == 1
assert np.isfinite(y).all()

print("Duration:", round(len(y) / sr, 3), "seconds")
print("Prompt:", repr(critical["reading_prompt"]))

display(Audio(y, rate=sr, normalize=False))

# Preserve the review as pending until listening is completed.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_targeted_anomaly_inspection"] = (
    "generated_pending_listening_review"
)
progress["step4_targeted_anomaly_file"] = str(OUT)
progress["next_task"] = (
    "Review audio 030210164 and classify ASR repetition anomaly"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== INSPECTION SAVED ===")
print("CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

INSPECTION = PROJECT / "reports/step04_targeted_anomaly_inspection.csv"
ANOMALIES = PROJECT / "reports/step04_linguistic_anomaly_audit.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "reports/step04_anomaly_classification_v1.csv"
REPORT = PROJECT / "reports/step04_anomaly_classification_v1.json"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

inspection = pd.read_csv(
    INSPECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

anomalies = pd.read_csv(
    ANOMALIES,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(inspection) == 25
assert len(anomalies) == 13

ids = sorted(inspection["audio_id"].unique())
assert len(ids) == 5

classification = {
    "030210137": {
        "category": "prompt_consistent_repetition",
        "evidence": "prompt_and_all_five_asr_hypotheses",
        "review_status": "audio_not_listened",
        "notes": (
            "Repeated 'care' matches reading prompt. "
            "Actual delivery not independently verified by listening."
        )
    },
    "030210164": {
        "category": "confirmed_asr_phrase_repetition_loop",
        "evidence": "user_audio_listening_plus_asr_hypotheses",
        "review_status": "user_listening_reviewed",
        "notes": (
            "User heard the target words with difficult accent. "
            "H2 repeated 'God bless you' 32 times, inconsistent "
            "with the audio. No verbatim human transcript certified."
        )
    },
    "096340009": {
        "category": "non_english_script_and_character_repetition",
        "evidence": "asr_text_inspection",
        "review_status": "audio_not_listened",
        "notes": (
            "Chinese-script output and character repetitions. "
            "Listening verification pending."
        )
    },
    "096340012": {
        "category": "mixed_script_and_repetitive_decoding",
        "evidence": "asr_text_inspection",
        "review_status": "audio_not_listened",
        "notes": (
            "Mixed English, romanized and Chinese characters. "
            "Listening verification pending."
        )
    },
    "096340017": {
        "category": "non_english_script_repetition_all_hypotheses",
        "evidence": "asr_text_inspection",
        "review_status": "audio_not_listened",
        "notes": (
            "All five hypotheses contain repeated Chinese-script "
            "characters. Listening verification pending."
        )
    }
}

assert set(ids) == set(classification)

rows = []

for aid in ids:
    subset = inspection.loc[inspection["audio_id"] == aid]
    item = classification[aid]

    rows.append({
        "audio_id": aid,
        "pilot_index": int(subset["pilot_index"].iloc[0]),
        "speaker": str(subset["speaker"].iloc[0]),
        "reading_prompt": str(subset["reading_prompt"].iloc[0]),
        "flagged_hypotheses": int(
            (anomalies["audio_id"] == aid).sum()
        ),
        "total_hypotheses": 5,
        **item,
        "primary_analysis_include": True,
        "sensitivity_analysis_candidate": (
            aid != "030210137"
        )
    })

result = pd.DataFrame(rows)

OUT.parent.mkdir(parents=True, exist_ok=True)

tmp = OUT.with_suffix(".tmp")
result.to_csv(tmp, index=False, encoding="utf-8")

if OUT.exists():
    assert sha256(OUT) == sha256(tmp), (
        "Existing classification differs; original preserved."
    )
    tmp.unlink()
else:
    tmp.replace(OUT)

report = {
    "status": "preliminary_classification_saved",
    "audio_count": 5,
    "flagged_hypotheses": 13,
    "confirmed_phrase_repetition_loop_audio_count": 1,
    "audio_listening_pending_count": 4,
    "primary_analysis_excluded_audio_count": 0,
    "sensitivity_candidates": int(
        result["sensitivity_analysis_candidate"].sum()
    ),
    "classification_csv_sha256": sha256(OUT),
    "source_inspection_sha256": sha256(INSPECTION),
    "source_anomaly_sha256": sha256(ANOMALIES),
    "created_at": datetime.now(timezone.utc).isoformat()
}

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    for key in report:
        if key != "created_at":
            assert previous.get(key) == report[key]
else:
    tmp_report = REPORT.with_suffix(".tmp")
    tmp_report.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))

progress["step4_anomaly_classification"] = (
    "preliminary_saved"
)
progress["step4_anomaly_classification_sha256"] = sha256(OUT)
progress["next_task"] = (
    "Review feature suitability and define robust "
    "ASR anomaly indicators before model training"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.3 ANOMALY CLASSIFICATION SAVED ===")
print(result[
    [
        "audio_id",
        "category",
        "review_status",
        "primary_analysis_include",
        "sensitivity_analysis_candidate"
    ]
].to_string(index=False))

print("\nConfirmed repetition-loop audios:",
      report["confirmed_phrase_repetition_loop_audio_count"])
print("Pending audio listening:",
      report["audio_listening_pending_count"])
print("Primary analysis exclusions:",
      report["primary_analysis_excluded_audio_count"])
print("Sensitivity candidates:",
      report["sensitivity_candidates"])
print("CSV:", OUT)
print("SHA256:", sha256(OUT))
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUTPUT = (
    PROJECT /
    "features/linguistic/prompt_asr_disagreement_pilot_40.csv"
)
REPORT = (
    PROJECT /
    "reports/step04_prompt_asr_disagreement_pilot_40.json"
)

EXPECTED_ASR_SHA = (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(ASR) == EXPECTED_ASR_SHA
assert TRAIN.is_file()

# Use the same basic tokenizer as STEP 4.1.
TOKEN_PATTERN = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE
)

def tokens(text):
    return [
        word.casefold()
        for word in TOKEN_PATTERN.findall(str(text))
    ]

def edit_distance(a, b):
    previous = list(range(len(b) + 1))

    for i, item_a in enumerate(a, start=1):
        current = [i]

        for j, item_b in enumerate(b, start=1):
            current.append(
                min(
                    current[j - 1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (item_a != item_b)
                )
            )

        previous = current

    return previous[-1]

def normalized_words(text):
    return tokens(text)

def normalized_chars(text):
    return " ".join(tokens(text))

asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("pilot_index")

assert len(asr) == 2000
assert len(pilot) == 400

# Deterministic pilot: 10 recordings from each 100-audio batch.
# These are chosen without using human fluency labels.
selected_parts = []

for start in [1, 101, 201, 301]:
    block = pilot.loc[
        pilot["pilot_index"].between(start, start + 99)
    ]

    selected_parts.append(
        block.iloc[np.linspace(0, 99, 10, dtype=int)]
    )

selected = pd.concat(
    selected_parts,
    ignore_index=True
)

assert len(selected) == 40
assert selected["audio_id"].nunique() == 40

wanted = set(selected["audio_id"].astype(str))
prompts = {}

pf = pq.ParquetFile(TRAIN)

assert "audio" in pf.schema_arrow.names
assert "text" in pf.schema_arrow.names

for rg in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        rg,
        columns=["audio", "text"]
    )

    audio_rows = table.column("audio").to_pylist()
    text_rows = table.column("text").to_pylist()

    for audio, prompt in zip(audio_rows, text_rows):
        aid = Path(audio["path"]).stem

        if aid in wanted:
            assert aid not in prompts
            prompts[aid] = str(prompt)

assert set(prompts) == wanted

subset = asr.loc[
    asr["audio_id"].isin(wanted)
].copy()

assert len(subset) == 200

records = []

for _, row in subset.iterrows():
    aid = str(row["audio_id"])
    prompt = prompts[aid]
    transcript = str(row["transcription"])

    p_words = normalized_words(prompt)
    h_words = normalized_words(transcript)

    p_chars = normalized_chars(prompt)
    h_chars = normalized_chars(transcript)

    assert len(p_words) > 0
    assert len(p_chars) > 0

    word_distance = edit_distance(p_words, h_words)
    char_distance = edit_distance(p_chars, h_chars)

    records.append({
        "pilot_index": int(row["pilot_index"]),
        "audio_id": aid,
        "speaker": str(row["speaker"]),
        "hypothesis_id": row["hypothesis_id"],
        "reading_prompt": prompt,
        "transcription": transcript,
        "prompt_word_count": len(p_words),
        "asr_word_count": len(h_words),
        "prompt_asr_word_edit_distance": word_distance,
        "prompt_asr_word_disagreement": (
            word_distance / len(p_words)
        ),
        "prompt_asr_char_edit_distance": char_distance,
        "prompt_asr_char_disagreement": (
            char_distance / len(p_chars)
        ),
        "prompt_asr_exact_match": (
            p_words == h_words
        )
    })

result = pd.DataFrame(records).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(result) == 200
assert result["audio_id"].nunique() == 40
assert not result.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

temp = OUTPUT.with_suffix(".tmp")
result.to_csv(temp, index=False, encoding="utf-8")

if OUTPUT.exists():
    assert sha256(OUTPUT) == sha256(temp)
    temp.unlink()
else:
    temp.replace(OUTPUT)

summary = {
    "status": "pilot_completed_pending_review",
    "audio_count": 40,
    "hypothesis_count": 200,
    "mean_prompt_asr_word_disagreement": float(
        result["prompt_asr_word_disagreement"].mean()
    ),
    "median_prompt_asr_word_disagreement": float(
        result["prompt_asr_word_disagreement"].median()
    ),
    "mean_prompt_asr_char_disagreement": float(
        result["prompt_asr_char_disagreement"].mean()
    ),
    "exact_match_hypotheses": int(
        result["prompt_asr_exact_match"].sum()
    ),
    "source_asr_sha256": sha256(ASR),
    "output_sha256": sha256(OUTPUT),
    "interpretation": (
        "Prompt-ASR disagreement, not validated WER/CER "
        "against a verbatim human transcript."
    ),
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    for key, value in summary.items():
        if key != "created_at":
            assert previous.get(key) == value
else:
    temp_report = REPORT.with_suffix(".tmp")
    temp_report.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_prompt_asr_pilot"] = "completed_pending_review"
progress["step4_prompt_asr_pilot_sha256"] = sha256(OUTPUT)
progress["next_task"] = (
    "Review prompt-ASR disagreement pilot before "
    "deciding whether to include prompt alignment features"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.4 PROMPT-ASR PILOT SUMMARY ===")
for key, value in summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== DISAGREEMENT DISTRIBUTION ===")
print(
    result[
        [
            "prompt_asr_word_disagreement",
            "prompt_asr_char_disagreement"
        ]
    ].describe().round(4).to_string()
)

print("\n=== TOP 10 HIGHEST DISAGREEMENTS ===")
top = result.sort_values(
    "prompt_asr_word_disagreement",
    ascending=False
).head(10)

for _, row in top.iterrows():
    print(
        "\nAudio:", row["audio_id"],
        "| H:", row["hypothesis_id"],
        "| word disagreement:",
        round(row["prompt_asr_word_disagreement"], 4)
    )
    print("Prompt:", repr(row["reading_prompt"]))
    print("ASR:", repr(row["transcription"][:300]))

print("\nCSV:", OUTPUT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = (
    PROJECT /
    "features/linguistic/prompt_asr_disagreement_pilot_40.csv"
)
REPORT = (
    PROJECT /
    "reports/step04_prompt_asr_robustness_pilot_40.json"
)
PER_AUDIO = (
    PROJECT /
    "reports/step04_prompt_asr_robustness_per_audio_40.csv"
)
CHECK = PROJECT / "checkpoints/research_progress.json"

EXPECTED_SHA = (
    "bff38db4e4b5f1ee711091d0a621aac42a8214e1ad10fab9d18f6b2c29794143"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(SOURCE) == EXPECTED_SHA

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 200
assert df["audio_id"].nunique() == 40
assert df["hypothesis_id"].value_counts().to_dict() == {
    h: 40 for h in ["H1", "H2", "H3", "H4", "H5"]
}

WORD = "prompt_asr_word_disagreement"
CHAR = "prompt_asr_char_disagreement"

assert df[[WORD, CHAR]].notna().all().all()
assert (df[[WORD, CHAR]] >= 0).all().all()

# The previously confirmed phrase repetition-loop case.
outlier_mask = (
    df["audio_id"].eq("030210164") &
    df["hypothesis_id"].eq("H2")
)

assert int(outlier_mask.sum()) == 1
assert np.isclose(
    df.loc[outlier_mask, WORD].iloc[0], 16.0
)

def descriptive_stats(series):
    s = pd.to_numeric(series, errors="raise")
    return {
        "n": int(len(s)),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "std": float(s.std()),
        "q25": float(s.quantile(0.25)),
        "q75": float(s.quantile(0.75)),
        "iqr": float(s.quantile(0.75) - s.quantile(0.25)),
        "p95": float(s.quantile(0.95)),
        "max": float(s.max())
    }

overall_word = descriptive_stats(df[WORD])
overall_char = descriptive_stats(df[CHAR])

# Sensitivity calculation only; original data is retained.
without_one = df.loc[~outlier_mask].copy()

without_outlier_word = descriptive_stats(without_one[WORD])
without_outlier_char = descriptive_stats(without_one[CHAR])

by_hypothesis = {}

for hid in ["H1", "H2", "H3", "H4", "H5"]:
    subset = df.loc[df["hypothesis_id"] == hid]

    by_hypothesis[hid] = {
        "word": descriptive_stats(subset[WORD]),
        "character": descriptive_stats(subset[CHAR]),
        "exact_matches": int(
            subset["prompt_asr_exact_match"]
            .astype(str).str.lower().eq("true").sum()
        )
    }

# Each audio has five hypotheses. Use this to measure
# within-audio disagreement variability.
per_audio = (
    df.groupby(["audio_id", "speaker", "pilot_index"])
    .agg(
        word_mean=(WORD, "mean"),
        word_min=(WORD, "min"),
        word_max=(WORD, "max"),
        word_std=(WORD, "std"),
        char_mean=(CHAR, "mean"),
        char_min=(CHAR, "min"),
        char_max=(CHAR, "max"),
        char_std=(CHAR, "std")
    )
    .reset_index()
)

per_audio["word_range"] = (
    per_audio["word_max"] - per_audio["word_min"]
)
per_audio["char_range"] = (
    per_audio["char_max"] - per_audio["char_min"]
)

assert len(per_audio) == 40

variable_word_audios = int(
    (per_audio["word_range"] > 1e-12).sum()
)
variable_char_audios = int(
    (per_audio["char_range"] > 1e-12).sum()
)

REPORT.parent.mkdir(parents=True, exist_ok=True)

tmp_csv = PER_AUDIO.with_suffix(".tmp")
per_audio.to_csv(tmp_csv, index=False, encoding="utf-8")

if PER_AUDIO.exists():
    assert sha256(PER_AUDIO) == sha256(tmp_csv), (
        "Existing per-audio output differs."
    )
    tmp_csv.unlink()
else:
    tmp_csv.replace(PER_AUDIO)

report = {
    "status": "robustness_pilot_complete",
    "source_sha256": EXPECTED_SHA,
    "audio_count": 40,
    "hypothesis_count": 200,
    "word_disagreement_all": overall_word,
    "char_disagreement_all": overall_char,
    "outlier_audio_id": "030210164",
    "outlier_hypothesis_id": "H2",
    "word_disagreement_without_one_outlier":
        without_outlier_word,
    "char_disagreement_without_one_outlier":
        without_outlier_char,
    "by_hypothesis": by_hypothesis,
    "audios_with_variable_word_disagreement":
        variable_word_audios,
    "audios_with_variable_char_disagreement":
        variable_char_audios,
    "per_audio_csv_sha256": sha256(PER_AUDIO),
    "interpretation": (
        "Prompt-ASR disagreement, not ground-truth WER/CER. "
        "Outlier removal is diagnostic only, not a dataset edit."
    ),
    "created_at": datetime.now(timezone.utc).isoformat()
}

if REPORT.exists():
    previous = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    for key, value in report.items():
        if key != "created_at":
            assert previous.get(key) == value
else:
    temp_report = REPORT.with_suffix(".tmp")
    temp_report.write_text(
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    temp_report.replace(REPORT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step4_prompt_asr_robustness"] = (
    "pilot_complete_pending_review"
)
progress["step4_prompt_asr_robustness_report"] = str(REPORT)
progress["next_task"] = (
    "Review robust prompt-ASR measures and "
    "finalize linguistic feature design"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== STEP 4.5 ROBUSTNESS PILOT SUMMARY ===")

for name, stats in [
    ("WORD ALL", overall_word),
    ("WORD WITHOUT ONE OUTLIER", without_outlier_word),
    ("CHAR ALL", overall_char),
    ("CHAR WITHOUT ONE OUTLIER", without_outlier_char)
]:
    print("\n", name)
    for key in ["n", "mean", "median", "iqr", "p95", "max"]:
        print(f"{key}: {stats[key]:.4f}")

print("\n=== H1-H5 COMPARISON ===")
for hid, stats in by_hypothesis.items():
    print(
        hid,
        "word_mean=", round(stats["word"]["mean"], 4),
        "word_median=", round(stats["word"]["median"], 4),
        "char_mean=", round(stats["character"]["mean"], 4),
        "exact_matches=", stats["exact_matches"]
    )

print("\n=== WITHIN-AUDIO VARIABILITY ===")
print("Audios with variable word disagreement:",
      variable_word_audios)
print("Audios with variable char disagreement:",
      variable_char_audios)

print("\nReport:", REPORT)
print("Per-audio CSV:", PER_AUDIO)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
LING = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
ACOUSTIC = PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
PROMPT_PILOT = (
    PROJECT / "features/linguistic/prompt_asr_disagreement_pilot_40.csv"
)
ROBUSTNESS = (
    PROJECT / "reports/step04_prompt_asr_robustness_pilot_40.json"
)
CONFIG = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

EXPECTED = {
    str(ASR): "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80",
    str(LING): "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459",
    str(ACOUSTIC): "4f477cd0fc0f6cf4382bc92fa39bdae734f22a9440e949b592143db2e261dced",
    str(PROMPT_PILOT): "bff38db4e4b5f1ee711091d0a621aac42a8214e1ad10fab9d18f6b2c29794143"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in EXPECTED.items():
    path = Path(filename)
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected_hash

robustness = json.loads(
    ROBUSTNESS.read_text(encoding="utf-8")
)

assert robustness["status"] == "robustness_pilot_complete"
assert robustness["audio_count"] == 40
assert robustness["hypothesis_count"] == 200

linguistic_features = [
    "word_count",
    "unique_word_count",
    "type_token_ratio",
    "mean_word_length",
    "character_count",
    "adjacent_repeat_count",
    "adjacent_repeat_ratio"
]

prompt_features = [
    "prompt_asr_word_disagreement",
    "prompt_asr_char_disagreement"
]

model_specs = {
    "A": {
        "description": "Acoustic-only baseline",
        "feature_groups": ["acoustic"]
    },
    "B1": {
        "description": "Transcript-only baseline",
        "feature_groups": ["linguistic"]
    },
    "B2": {
        "description": "Prompt-aware transcript ablation",
        "feature_groups": ["linguistic", "prompt"]
    },
    "C1": {
        "description": "Acoustic + transcript fusion",
        "feature_groups": ["acoustic", "linguistic"]
    },
    "C2": {
        "description": "Acoustic + prompt-aware fusion",
        "feature_groups": ["acoustic", "linguistic", "prompt"]
    }
}

protocol = {
    "name": "fluency_feature_design_v1",
    "status": "design_predeclared_pending_full_prompt_extraction",
    "dataset": "mispeech/speechocean762",
    "selected_audio_count": 400,
    "asr_hypothesis_count": 2000,
    "target": "human_fluency_score",
    "acoustic_feature_count": 14,
    "acoustic_feature_names": (
        "read exact 14 column names from STEP 2 artifact"
    ),
    "linguistic_feature_names": linguistic_features,
    "prompt_feature_names": prompt_features,
    "models": model_specs,
    "primary_models": ["A", "B1", "C1"],
    "secondary_ablation_models": ["B2", "C2"],
    "validation_rules": {
        "speaker_disjoint": True,
        "same_audio_hypotheses_same_partition": True,
        "preprocessing_fit_on_training_only": True,
        "hyperparameter_selection_within_training_data": True,
        "human_labels_not_used_for_feature_selection_now": True,
        "sealed_official_test_split_untouched": True
    },
    "missing_value_policy": {
        "adjacent_repeat_ratio": (
            "NaN for one-word transcripts; "
            "impute using training-fold statistics only"
        ),
        "other_features": (
            "check and handle inside training folds"
        )
    },
    "scientific_cautions": [
        "Reading prompt is not a verbatim human transcript.",
        "Prompt-ASR disagreement is not ground-truth WER or CER.",
        "Transcript lexical diversity does not directly measure vocabulary proficiency.",
        "Transcript repetition may reflect ASR errors rather than speech disfluency.",
        "ASR anomalies remain in primary analysis.",
        "Sensitivity exclusions require explicit documented rules.",
        "Prompt-aware gains may reflect access to target sentence information."
    ],
    "next_required_task": (
        "Extract two prompt-ASR disagreement features "
        "for all 400 audios and 2000 hypotheses"
    ),
    "source_sha256": {
        "asr": sha256(ASR),
        "linguistic": sha256(LING),
        "acoustic": sha256(ACOUSTIC),
        "prompt_pilot": sha256(PROMPT_PILOT)
    }
}

CONFIG.parent.mkdir(parents=True, exist_ok=True)

if CONFIG.exists():
    previous = json.loads(CONFIG.read_text(encoding="utf-8"))
    assert previous == protocol, (
        "Feature design config differs; existing file preserved."
    )
else:
    temp = CONFIG.with_suffix(".tmp")
    temp.write_text(
        json.dumps(protocol, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(CONFIG)

progress = json.loads(CHECK.read_text(encoding="utf-8"))

progress["step4_feature_design"] = "predeclared"
progress["step4_feature_design_sha256"] = sha256(CONFIG)
progress["next_task"] = protocol["next_required_task"]
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.6 FEATURE DESIGN SAVED ===")
print("Status:", protocol["status"])
print("Primary models:", protocol["primary_models"])
print("Ablation models:", protocol["secondary_ablation_models"])
print("Acoustic features:", protocol["acoustic_feature_count"])
print("Linguistic features:", len(linguistic_features))
print("Prompt features:", len(prompt_features))
print("Speaker-disjoint validation: REQUIRED")
print("Sealed official test: UNTOUCHED")
print("Design config:", CONFIG)
print("SHA256:", sha256(CONFIG))
print("Next task:", progress["next_task"])


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
PILOT40 = PROJECT / "features/linguistic/prompt_asr_disagreement_pilot_40.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "features/linguistic/prompt_asr_disagreement_0001-0400.csv"
REPORT = PROJECT / "reports/step04_prompt_asr_full_0400_qc.json"

EXPECTED_ASR_SHA = "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
EXPECTED_DESIGN_SHA = "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
EXPECTED_PILOT_SHA = "bff38db4e4b5f1ee711091d0a621aac42a8214e1ad10fab9d18f6b2c29794143"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected_sha in [
    (ASR, EXPECTED_ASR_SHA),
    (DESIGN, EXPECTED_DESIGN_SHA),
    (PILOT40, EXPECTED_PILOT_SHA)
]:
    assert path.is_file(), f"Missing file: {path}"
    assert sha256(path) == expected_sha, f"SHA mismatch: {path}"

assert TRAIN.is_file()
assert PILOT.is_file()

design = json.loads(DESIGN.read_text(encoding="utf-8"))
assert design["selected_audio_count"] == 400
assert design["asr_hypothesis_count"] == 2000
assert design["prompt_feature_names"] == [
    "prompt_asr_word_disagreement",
    "prompt_asr_char_disagreement"
]

TOKEN_PATTERN = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE
)

def tokens(text):
    return [
        word.casefold()
        for word in TOKEN_PATTERN.findall(str(text))
    ]

def edit_distance(a, b):
    previous = list(range(len(b) + 1))

    for i, item_a in enumerate(a, start=1):
        current = [i]

        for j, item_b in enumerate(b, start=1):
            current.append(
                min(
                    current[j - 1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (item_a != item_b)
                )
            )

        previous = current

    return previous[-1]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("pilot_index")

asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(pilot) == 400
assert pilot["pilot_index"].tolist() == list(range(1, 401))
assert pilot["audio_id"].is_unique
assert pilot["audio_id"].notna().all()
assert pilot["speaker"].notna().all()

assert len(asr) == 2000
assert asr["audio_id"].nunique() == 400
assert asr["pilot_index"].nunique() == 400
assert asr["transcription"].str.strip().ne("").all()

keys = ["pilot_index", "audio_id", "speaker"]

verified = asr.merge(
    pilot[keys],
    on=keys,
    how="inner",
    validate="many_to_one"
)

assert len(verified) == 2000
assert not verified.duplicated(
    keys + ["hypothesis_id"]
).any()

HIDS = ["H1", "H2", "H3", "H4", "H5"]

assert verified.groupby("audio_id")["hypothesis_id"].apply(
    lambda s: sorted(s.tolist()) == HIDS
).all()

# Retrieve only prompts, not decoded audio.
wanted = set(pilot["audio_id"].astype(str))
prompts = {}

pf = pq.ParquetFile(TRAIN)

assert "audio" in pf.schema_arrow.names
assert "text" in pf.schema_arrow.names

for rg in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        rg,
        columns=["audio", "text"]
    )

    audio_rows = table.column("audio").to_pylist()
    text_rows = table.column("text").to_pylist()

    for audio, prompt in zip(audio_rows, text_rows):
        aid = Path(audio["path"]).stem

        if aid in wanted:
            assert aid not in prompts, f"Duplicate prompt: {aid}"
            prompts[aid] = str(prompt)

assert set(prompts) == wanted, (
    f"Missing prompts: {sorted(wanted - set(prompts))[:10]}"
)

print("=== STEP 4.7 FULL EXTRACTION START ===")
print("Audios:", len(prompts))
print("Hypotheses:", len(verified))
print("Feature design SHA256:", sha256(DESIGN))

records = []

for _, row in verified.iterrows():
    aid = str(row["audio_id"])
    prompt = prompts[aid]
    transcript = str(row["transcription"])

    p_words = tokens(prompt)
    h_words = tokens(transcript)

    p_chars = " ".join(p_words)
    h_chars = " ".join(h_words)

    assert len(p_words) > 0, f"Empty prompt tokens: {aid}"
    assert len(p_chars) > 0

    word_distance = edit_distance(p_words, h_words)
    char_distance = edit_distance(p_chars, h_chars)

    records.append({
        "pilot_index": int(row["pilot_index"]),
        "audio_id": aid,
        "speaker": str(row["speaker"]),
        "hypothesis_id": str(row["hypothesis_id"]),
        "reading_prompt": prompt,
        "transcription": transcript,
        "prompt_word_count": len(p_words),
        "asr_word_count": len(h_words),
        "prompt_asr_word_edit_distance": word_distance,
        "prompt_asr_word_disagreement": (
            word_distance / len(p_words)
        ),
        "prompt_asr_char_edit_distance": char_distance,
        "prompt_asr_char_disagreement": (
            char_distance / len(p_chars)
        ),
        "prompt_asr_exact_match": p_words == h_words
    })

result = pd.DataFrame(records).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(result) == 2000
assert result["audio_id"].nunique() == 400
assert not result.duplicated(
    keys + ["hypothesis_id"]
).any()

WORD = "prompt_asr_word_disagreement"
CHAR = "prompt_asr_char_disagreement"

assert result[[WORD, CHAR]].notna().all().all()
assert (result[[WORD, CHAR]] >= 0).all().all()

# Verify 40-audio pilot results match exactly in
# identifiers, prompts, transcriptions, and metrics.
old = pd.read_csv(
    PILOT40,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(old) == 200

compare_keys = keys + ["hypothesis_id"]

matched = old.merge(
    result,
    on=compare_keys,
    suffixes=("_pilot", "_full"),
    how="inner",
    validate="one_to_one"
)

assert len(matched) == 200

for col in ["reading_prompt", "transcription"]:
    assert (
        matched[f"{col}_pilot"] ==
        matched[f"{col}_full"]
    ).all(), f"Pilot mismatch: {col}"

metric_columns = [
    "prompt_word_count",
    "asr_word_count",
    "prompt_asr_word_edit_distance",
    WORD,
    "prompt_asr_char_edit_distance",
    CHAR
]

for col in metric_columns:
    a = pd.to_numeric(matched[f"{col}_pilot"])
    b = pd.to_numeric(matched[f"{col}_full"])

    assert np.allclose(a, b, rtol=0, atol=1e-12), (
        f"Pilot metric mismatch: {col}"
    )

assert (
    matched["prompt_asr_exact_match_pilot"].astype(str).str.lower()
    ==
    matched["prompt_asr_exact_match_full"].astype(str).str.lower()
).all()

# Save without overwriting differing previous results.
OUT.parent.mkdir(parents=True, exist_ok=True)

temp = OUT.with_suffix(".tmp")
result.to_csv(temp, index=False, encoding="utf-8")

if OUT.exists():
    assert sha256(OUT) == sha256(temp), (
        "Existing full prompt output differs; original preserved."
    )
    temp.unlink()
else:
    temp.replace(OUT)

# Within-audio feature variability.
variation = result.groupby("audio_id").agg(
    word_range=(WORD, lambda s: float(s.max() - s.min())),
    char_range=(CHAR, lambda s: float(s.max() - s.min()))
)

summary = {
    "status": "PASS",
    "audio_count": 400,
    "hypothesis_count": 2000,
    "feature_count": 2,
    "pilot_40_consistency": "PASS",
    "mean_word_disagreement": float(result[WORD].mean()),
    "median_word_disagreement": float(result[WORD].median()),
    "mean_char_disagreement": float(result[CHAR].mean()),
    "median_char_disagreement": float(result[CHAR].median()),
    "exact_match_hypotheses": int(
        result["prompt_asr_exact_match"].sum()
    ),
    "audios_with_variable_word_disagreement": int(
        (variation["word_range"] > 1e-12).sum()
    ),
    "audios_with_variable_char_disagreement": int(
        (variation["char_range"] > 1e-12).sum()
    ),
    "max_word_disagreement": float(result[WORD].max()),
    "max_char_disagreement": float(result[CHAR].max()),
    "source_asr_sha256": EXPECTED_ASR_SHA,
    "design_sha256": EXPECTED_DESIGN_SHA,
    "output_sha256": sha256(OUT),
    "interpretation": (
        "Reading prompt versus ASR output disagreement; "
        "not ground-truth WER or CER."
    )
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == summary, (
        "Existing QC report differs; original preserved."
    )
else:
    tmp_report = REPORT.with_suffix(".tmp")
    tmp_report.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_prompt_asr_full"] = "qc_pass"
progress["step4_prompt_asr_full_sha256"] = sha256(OUT)
progress["step4_prompt_asr_pilot_consistency"] = "PASS"
progress["next_task"] = (
    "Review full prompt-ASR features, merge feature groups "
    "and prepare speaker-disjoint modeling"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== STEP 4.7 FULL PROMPT-ASR SUMMARY ===")
for name, value in summary.items():
    print(f"{name}: {value}")

print("\n=== DISAGREEMENT DISTRIBUTION ===")
print(
    result[[WORD, CHAR]]
    .describe()
    .round(4)
    .to_string()
)

print("\n=== H1-H5 COMPARISON ===")
print(
    result.groupby("hypothesis_id")[[WORD, CHAR]]
    .agg(["mean", "median"])
    .round(4)
    .to_string()
)

print("\n=== PER-BATCH COMPARISON ===")
result["batch"] = (
    ((result["pilot_index"] - 1) // 100) + 1
)
print(
    result.groupby("batch")[[WORD, CHAR]]
    .agg(["mean", "median"])
    .round(4)
    .to_string()
)

print("\nCSV:", OUT)
print("SHA256:", sha256(OUT))
print("QC report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ACOUSTIC = PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
LINGUISTIC = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
PROMPT = PROJECT / "features/linguistic/prompt_asr_disagreement_0001-0400.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

REPORT = PROJECT / "reports/step04_modeling_schema_audit.json"

EXPECTED = {
    ACOUSTIC: "4f477cd0fc0f6cf4382bc92fa39bdae734f22a9440e949b592143db2e261dced",
    LINGUISTIC: "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459",
    PROMPT: "8b52805a91144c5ee0a379aeb2954595f55ec085203ac38861845c5fbe33c25c",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path.name}"

def read_csv(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

acoustic = read_csv(ACOUSTIC)
linguistic = read_csv(LINGUISTIC)
prompt = read_csv(PROMPT)

assert len(acoustic) == 400
assert len(linguistic) == 2000
assert len(prompt) == 2000

assert acoustic["audio_id"].nunique() == 400
assert linguistic["audio_id"].nunique() == 400
assert prompt["audio_id"].nunique() == 400

assert not acoustic["audio_id"].duplicated().any()

KEYS = ["pilot_index", "audio_id", "speaker", "hypothesis_id"]

assert all(c in linguistic.columns for c in KEYS)
assert all(c in prompt.columns for c in KEYS)

assert not linguistic.duplicated(KEYS).any()
assert not prompt.duplicated(KEYS).any()

lk = linguistic[KEYS].sort_values(KEYS).reset_index(drop=True)
pk = prompt[KEYS].sort_values(KEYS).reset_index(drop=True)

assert lk.equals(pk), "Linguistic/prompt key mismatch"

assert set(acoustic["audio_id"]) == set(linguistic["audio_id"])

# Check identity mapping without assuming acoustic schema.
identity_cols = [
    c for c in ["pilot_index", "audio_id", "speaker"]
    if c in acoustic.columns
]

identity = linguistic[
    ["pilot_index", "audio_id", "speaker"]
].drop_duplicates()

matched = acoustic[identity_cols].merge(
    identity,
    on="audio_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_acoustic", "_reference")
)

assert len(matched) == 400

for c in ["pilot_index", "speaker"]:
    if c in acoustic.columns:
        left = matched[f"{c}_acoustic"].astype(str)
        right = matched[f"{c}_reference"].astype(str)
        assert left.equals(right), f"{c} identity mismatch"

design = json.loads(DESIGN.read_text(encoding="utf-8"))

ling_features = design["linguistic_feature_names"]
prompt_features = design["prompt_feature_names"]

assert len(ling_features) == 7
assert len(prompt_features) == 2
assert all(c in linguistic.columns for c in ling_features)
assert all(c in prompt.columns for c in prompt_features)

# Inspect available acoustic columns without guessing
# which are predictors or labels.
schema = {
    "status": "PASS_pending_acoustic_feature_selection",
    "acoustic_rows": len(acoustic),
    "linguistic_rows": len(linguistic),
    "prompt_rows": len(prompt),
    "unique_audio_count": 400,
    "speaker_count": int(linguistic["speaker"].nunique()),
    "acoustic_columns": acoustic.columns.tolist(),
    "acoustic_dtypes": {
        c: str(acoustic[c].dtype)
        for c in acoustic.columns
    },
    "acoustic_missing_counts": {
        c: int(acoustic[c].isna().sum())
        for c in acoustic.columns
    },
    "linguistic_features": ling_features,
    "prompt_features": prompt_features,
    "identity_check": "PASS",
    "hypothesis_key_check": "PASS",
    "source_sha256": {
        str(p.relative_to(PROJECT)): sha256(p)
        for p in EXPECTED
    },
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    existing = json.loads(REPORT.read_text(encoding="utf-8"))
    for key, value in schema.items():
        if key != "created_at":
            assert existing.get(key) == value, (
                f"Existing schema report mismatch: {key}"
            )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(schema, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_modeling_schema_audit"] = "pass_pending_feature_mapping"
progress["next_task"] = (
    "Confirm 14 acoustic feature columns and human "
    "fluency label mapping before merging modeling dataset"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.8 MODELING SCHEMA AUDIT ===")
print("Status:", schema["status"])
print("Acoustic rows:", len(acoustic))
print("Linguistic rows:", len(linguistic))
print("Prompt rows:", len(prompt))
print("Unique audios:", schema["unique_audio_count"])
print("Speakers:", schema["speaker_count"])
print("Identity check: PASS")
print("Hypothesis key check: PASS")

print("\n=== ACOUSTIC COLUMNS AND DTYPES ===")
for col in acoustic.columns:
    print(
        f"{col} | {acoustic[col].dtype} "
        f"| missing={acoustic[col].isna().sum()}"
    )

print("\n=== ACOUSTIC SAMPLE ===")
print(acoustic.head(3).to_string(index=False))

print("\n=== LINGUISTIC FEATURE NAMES ===")
print(ling_features)

print("\n=== PROMPT FEATURE NAMES ===")
print(prompt_features)

print("\nSchema report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ACOUSTIC = PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
LING = PROJECT / "features/linguistic/linguistic_features_0001-0400.csv"
PROMPT = PROJECT / "features/linguistic/prompt_asr_disagreement_0001-0400.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
REPORT = PROJECT / "reports/step04_modeling_dataset_qc.json"

EXPECTED = {
    ACOUSTIC: "4f477cd0fc0f6cf4382bc92fa39bdae734f22a9440e949b592143db2e261dced",
    LING: "ed9585aa09defb10cd57d40f98356ca1de7fbecaf6e5251c0dcd035ab0395459",
    PROMPT: "8b52805a91144c5ee0a379aeb2954595f55ec085203ac38861845c5fbe33c25c",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path}"

def read_csv(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

acoustic = read_csv(ACOUSTIC)
ling = read_csv(LING)
prompt = read_csv(PROMPT)

design = json.loads(DESIGN.read_text(encoding="utf-8"))

ids = ["pilot_index", "audio_id", "speaker"]
keys = ids + ["hypothesis_id"]

ling_features = design["linguistic_feature_names"]
prompt_features = design["prompt_feature_names"]

# Explicitly exclude target and identifiers.
acoustic_features = [
    c for c in acoustic.columns
    if c not in ids + ["fluency"]
]

assert len(acoustic_features) == 14
assert len(ling_features) == 7
assert len(prompt_features) == 2

all_features = (
    acoustic_features + ling_features + prompt_features
)

assert len(all_features) == 23
assert len(set(all_features)) == 23
assert "fluency" not in all_features

assert len(acoustic) == 400
assert len(ling) == 2000
assert len(prompt) == 2000

assert acoustic["audio_id"].is_unique
assert not acoustic.duplicated(ids).any()
assert not ling.duplicated(keys).any()
assert not prompt.duplicated(keys).any()

assert set(acoustic["audio_id"]) == set(ling["audio_id"])
assert set(ling["audio_id"]) == set(prompt["audio_id"])

# Select only approved feature columns.
base = acoustic[
    ids + ["fluency"] + acoustic_features
].copy()

ling_data = ling[
    keys + ling_features
].copy()

prompt_data = prompt[
    keys + prompt_features
].copy()

# Strict one-to-one merge for hypothesis-level features.
hyp = ling_data.merge(
    prompt_data,
    on=keys,
    how="inner",
    validate="one_to_one"
)

assert len(hyp) == 2000

# Each audio has five hypotheses.
modeling = hyp.merge(
    base,
    on=ids,
    how="inner",
    validate="many_to_one"
)

assert len(modeling) == 2000
assert modeling["audio_id"].nunique() == 400
assert modeling["speaker"].nunique() == 20
assert not modeling.duplicated(keys).any()

assert modeling.groupby("audio_id")["hypothesis_id"].apply(
    lambda s: sorted(s.tolist()) == [
        "H1", "H2", "H3", "H4", "H5"
    ]
).all()

# Verify the target is constant within an audio.
assert modeling.groupby("audio_id")["fluency"].nunique().eq(1).all()
assert modeling["fluency"].notna().all()

# Compare every target value against its original acoustic row.
target_check = modeling[
    ids + ["fluency"]
].drop_duplicates()

assert len(target_check) == 400

original_targets = base[
    ids + ["fluency"]
].sort_values(ids).reset_index(drop=True)

merged_targets = target_check.sort_values(
    ids
).reset_index(drop=True)

pd.testing.assert_frame_equal(
    original_targets,
    merged_targets,
    check_dtype=False
)

# All predictors must be numeric.
for col in all_features:
    assert pd.api.types.is_numeric_dtype(modeling[col]), (
        f"Non-numeric predictor: {col}"
    )

numeric = modeling[all_features].to_numpy(dtype=float)
assert not np.isinf(numeric).any()

# Keep missing values for training-fold-only imputation.
missing = {
    col: int(modeling[col].isna().sum())
    for col in all_features
    if modeling[col].isna().sum() > 0
}

# Final explicit column order.
ordered_columns = (
    keys + ["fluency"] +
    acoustic_features +
    ling_features +
    prompt_features
)

modeling = modeling[
    ordered_columns
].sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert modeling.shape == (2000, 28)

# Verify every acoustic feature remains identical across H1-H5.
for col in acoustic_features:
    counts = modeling.groupby("audio_id")[col].nunique(
        dropna=False
    )
    assert counts.eq(1).all(), (
        f"Acoustic feature differs across hypotheses: {col}"
    )

OUT.parent.mkdir(parents=True, exist_ok=True)

temp = OUT.with_suffix(".tmp")
modeling.to_csv(
    temp,
    index=False,
    encoding="utf-8"
)

if OUT.exists():
    assert sha256(OUT) == sha256(temp), (
        "Existing modeling dataset differs; original preserved."
    )
    temp.unlink()
else:
    temp.replace(OUT)

report = {
    "status": "PASS",
    "rows": 2000,
    "columns": 28,
    "unique_audios": 400,
    "unique_speakers": 20,
    "hypotheses_per_audio": 5,
    "target": "fluency",
    "acoustic_features": acoustic_features,
    "linguistic_features": ling_features,
    "prompt_features": prompt_features,
    "total_predictors": 23,
    "missing_predictor_values": missing,
    "target_integrity": "PASS",
    "acoustic_invariance": "PASS",
    "source_sha256": {
        str(p.relative_to(PROJECT)): sha256(p)
        for p in EXPECTED
    },
    "output_sha256": sha256(OUT)
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    previous = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert previous == report, (
        "Existing QC report differs; original preserved."
    )
else:
    temp_report = REPORT.with_suffix(".tmp")
    temp_report.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step4_modeling_dataset"] = "qc_pass"
progress["step4_modeling_dataset_sha256"] = sha256(OUT)
progress["next_task"] = (
    "Audit human fluency label distribution and speaker groups "
    "before constructing speaker-disjoint validation splits"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 4.9 MODELING DATASET QC ===")
print("Status:", report["status"])
print("Rows:", report["rows"])
print("Columns:", report["columns"])
print("Unique audios:", report["unique_audios"])
print("Unique speakers:", report["unique_speakers"])
print("Target:", report["target"])
print("Acoustic features:", len(acoustic_features))
print("Linguistic features:", len(ling_features))
print("Prompt features:", len(prompt_features))
print("Total predictors:", report["total_predictors"])
print("Target integrity:", report["target_integrity"])
print("Acoustic invariance:", report["acoustic_invariance"])

print("\n=== MISSING PREDICTOR VALUES ===")
print(missing)

print("\n=== HUMAN FLUENCY DISTRIBUTION (400 UNIQUE AUDIOS) ===")
print(
    modeling.drop_duplicates("audio_id")["fluency"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n=== MODELING DATASET SAMPLE ===")
print(
    modeling[
        keys + [
            "fluency",
            "duration_sec",
            "word_count",
            "prompt_asr_word_disagreement"
        ]
    ].head(10).to_string(index=False)
)

print("\nCSV:", OUT)
print("SHA256:", sha256(OUT))
print("QC report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
REPORT_DIR = PROJECT / "reports"
SPLIT_DIR = PROJECT / "data/splits"
CHECK = PROJECT / "checkpoints/research_progress.json"

SPEAKER_CSV = REPORT_DIR / "step05_speaker_fluency_audit.csv"
FOLD_CSV = SPLIT_DIR / "speaker_groupkfold5_candidate.csv"
REPORT = REPORT_DIR / "step05_speaker_split_candidate_qc.json"

EXPECTED_SHA = (
    "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(SOURCE) == EXPECTED_SHA

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert df["fluency"].notna().all()

# One record per audio; H1-H5 must not inflate counts.
audio = df[
    ["pilot_index", "audio_id", "speaker", "fluency"]
].drop_duplicates().sort_values(
    "pilot_index"
).reset_index(drop=True)

assert len(audio) == 400
assert audio["audio_id"].is_unique
assert audio.groupby("speaker")["fluency"].count().sum() == 400

# Speaker-level descriptive statistics.
speaker_stats = (
    audio.groupby("speaker")
    .agg(
        audio_count=("audio_id", "count"),
        mean_fluency=("fluency", "mean"),
        median_fluency=("fluency", "median"),
        sd_fluency=("fluency", "std"),
        min_fluency=("fluency", "min"),
        max_fluency=("fluency", "max")
    )
    .reset_index()
)

score_counts = pd.crosstab(
    audio["speaker"],
    audio["fluency"]
)

score_counts.columns = [
    f"score_{int(c)}_count"
    for c in score_counts.columns
]

speaker_stats = speaker_stats.merge(
    score_counts.reset_index(),
    on="speaker",
    validate="one_to_one"
)

# Candidate GroupKFold: validation speakers are unseen in training.
gkf = GroupKFold(n_splits=5)
fold_rows = []
assignments = []

for fold_number, (train_idx, val_idx) in enumerate(
    gkf.split(
        audio,
        y=audio["fluency"],
        groups=audio["speaker"]
    ),
    start=1
):
    train = audio.iloc[train_idx]
    val = audio.iloc[val_idx]

    train_speakers = set(train["speaker"])
    val_speakers = set(val["speaker"])

    assert train_speakers.isdisjoint(val_speakers)
    assert set(train["audio_id"]).isdisjoint(
        set(val["audio_id"])
    )

    fold_rows.append({
        "fold": fold_number,
        "train_speakers": len(train_speakers),
        "val_speakers": len(val_speakers),
        "train_audios": len(train),
        "val_audios": len(val),
        "train_mean": float(train["fluency"].mean()),
        "val_mean": float(val["fluency"].mean()),
        "train_sd": float(train["fluency"].std()),
        "val_sd": float(val["fluency"].std()),
        "train_min": int(train["fluency"].min()),
        "train_max": int(train["fluency"].max()),
        "val_min": int(val["fluency"].min()),
        "val_max": int(val["fluency"].max()),
        "val_speaker_ids": ",".join(sorted(val_speakers))
    })

    for _, record in val.iterrows():
        assignments.append({
            "pilot_index": int(record["pilot_index"]),
            "audio_id": str(record["audio_id"]),
            "speaker": str(record["speaker"]),
            "fluency": int(record["fluency"]),
            "validation_fold": fold_number
        })

fold_stats = pd.DataFrame(fold_rows)
assignment_df = pd.DataFrame(assignments).sort_values(
    "pilot_index"
).reset_index(drop=True)

assert len(assignment_df) == 400
assert assignment_df["audio_id"].is_unique
assert assignment_df["validation_fold"].between(1, 5).all()

# Every speaker must appear in exactly one validation fold.
assert assignment_df.groupby("speaker")[
    "validation_fold"
].nunique().eq(1).all()

# Every audio's five hypotheses must inherit the same fold.
check = df[
    ["audio_id", "speaker", "hypothesis_id"]
].merge(
    assignment_df[
        ["audio_id", "speaker", "validation_fold"]
    ],
    on=["audio_id", "speaker"],
    how="inner",
    validate="many_to_one"
)

assert len(check) == 2000
assert check.groupby("audio_id")[
    "validation_fold"
].nunique().eq(1).all()

REPORT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

def save_csv_safely(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing CSV differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv_safely(speaker_stats, SPEAKER_CSV)
save_csv_safely(assignment_df, FOLD_CSV)

summary = {
    "status": "candidate_pending_review",
    "source_sha256": EXPECTED_SHA,
    "unique_audios": 400,
    "unique_speakers": 20,
    "number_of_folds": 5,
    "speaker_disjoint": True,
    "audio_hypotheses_same_fold": True,
    "official_test_split_used": False,
    "overall_mean_fluency": float(audio["fluency"].mean()),
    "overall_sd_fluency": float(audio["fluency"].std()),
    "speaker_mean_min": float(
        speaker_stats["mean_fluency"].min()
    ),
    "speaker_mean_max": float(
        speaker_stats["mean_fluency"].max()
    ),
    "fold_details": fold_rows,
    "speaker_csv_sha256": sha256(SPEAKER_CSV),
    "fold_csv_sha256": sha256(FOLD_CSV),
    "interpretation": (
        "Candidate 5-fold speaker-disjoint partition. "
        "No model fitting or outcome-based feature selection."
    )
}

if REPORT.exists():
    previous = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert previous == summary, (
        "Existing report differs; preserved."
    )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_speaker_split"] = "candidate_pending_review"
progress["step5_speaker_split_sha256"] = sha256(FOLD_CSV)
progress["next_task"] = (
    "Review speaker-level fluency distribution and "
    "candidate GroupKFold assignment before model training"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.1 SPEAKER AUDIT ===")
print("Unique audios:", len(audio))
print("Unique speakers:", audio["speaker"].nunique())
print("Overall mean fluency:", round(audio["fluency"].mean(), 4))
print("Overall SD fluency:", round(audio["fluency"].std(), 4))

print("\n=== SPEAKER-LEVEL FLUENCY ===")
print(
    speaker_stats[
        [
            "speaker", "audio_count", "mean_fluency",
            "sd_fluency", "min_fluency", "max_fluency"
        ]
    ].round(3).to_string(index=False)
)

print("\n=== CANDIDATE 5-FOLD DISTRIBUTION ===")
print(
    fold_stats.round(3).to_string(index=False)
)

print("\n=== SPEAKER-DISJOINT VALIDATION ===")
print("Train/validation speaker overlap: 0")
print("Each audio assigned to one fold: PASS")
print("H1-H5 remain in same fold: PASS")
print("Official test split: UNTOUCHED")
print("Split status: CANDIDATE, NOT YET FROZEN")
print("Fold CSV:", FOLD_CSV)
print("SHA256:", sha256(FOLD_CSV))
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

DATASET = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
CANDIDATE = PROJECT / "data/splits/speaker_groupkfold5_candidate.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

FROZEN = PROJECT / "data/splits/speaker_groupkfold5_frozen.csv"
PROTOCOL = PROJECT / "configs/speaker_disjoint_cv_v1.json"

DATA_SHA = "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746"
CANDIDATE_SHA = "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7"
DESIGN_SHA = "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(DATASET) == DATA_SHA
assert sha256(CANDIDATE) == CANDIDATE_SHA
assert sha256(DESIGN) == DESIGN_SHA

modeling = pd.read_csv(
    DATASET,
    dtype={"audio_id": "string", "speaker": "string"}
)

candidate = pd.read_csv(
    CANDIDATE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(modeling) == 2000
assert len(candidate) == 400
assert candidate["audio_id"].is_unique

assert candidate["validation_fold"].between(1, 5).all()
assert set(candidate["validation_fold"]) == set(range(1, 6))

keys = ["pilot_index", "audio_id", "speaker", "fluency"]

audio = modeling[keys].drop_duplicates()

assert len(audio) == 400
assert audio["audio_id"].is_unique
assert audio["speaker"].nunique() == 20

# Verify the candidate contains exactly the same audios,
# speaker IDs and original human labels.
audit = audio.merge(
    candidate[keys + ["validation_fold"]],
    on=keys,
    how="outer",
    indicator=True,
    validate="one_to_one"
)

assert len(audit) == 400
assert audit["_merge"].eq("both").all()

# One speaker must belong to one validation fold only.
assert candidate.groupby("speaker")[
    "validation_fold"
].nunique().eq(1).all()

fold_details = []

for fold in range(1, 6):

    val = candidate.loc[
        candidate["validation_fold"] == fold
    ]
    train = candidate.loc[
        candidate["validation_fold"] != fold
    ]

    val_speakers = set(val["speaker"])
    train_speakers = set(train["speaker"])

    assert len(val_speakers) == 4
    assert len(train_speakers) == 16
    assert len(val) == 80
    assert len(train) == 320
    assert train_speakers.isdisjoint(val_speakers)

    fold_details.append({
        "fold": fold,
        "train_audio_count": len(train),
        "validation_audio_count": len(val),
        "train_speaker_count": len(train_speakers),
        "validation_speaker_count": len(val_speakers),
        "validation_speakers": sorted(val_speakers),
        "train_mean_fluency": float(train["fluency"].mean()),
        "validation_mean_fluency": float(val["fluency"].mean())
    })

# Verify hypothesis-level assignment without changing it.
assigned = modeling.merge(
    candidate[["audio_id", "validation_fold"]],
    on="audio_id",
    how="inner",
    validate="many_to_one"
)

assert len(assigned) == 2000
assert assigned.groupby("audio_id")[
    "validation_fold"
].nunique().eq(1).all()
assert assigned.groupby("audio_id")[
    "hypothesis_id"
].nunique().eq(5).all()

# Freeze the exact CSV bytes; never regenerate this split.
FROZEN.parent.mkdir(parents=True, exist_ok=True)

if FROZEN.exists():
    assert sha256(FROZEN) == CANDIDATE_SHA, (
        "Frozen split differs. Original file preserved."
    )
else:
    tmp = FROZEN.with_suffix(".tmp")
    tmp.write_bytes(CANDIDATE.read_bytes())
    assert sha256(tmp) == CANDIDATE_SHA
    tmp.replace(FROZEN)

protocol = {
    "name": "speaker_disjoint_cv_v1",
    "status": "frozen",
    "dataset": "mispeech/speechocean762",
    "selected_audio_count": 400,
    "unique_speakers": 20,
    "number_of_folds": 5,
    "validation_speakers_per_fold": 4,
    "validation_audios_per_fold": 80,
    "group_key": "speaker",
    "split_method": "GroupKFold",
    "split_source": "precomputed_candidate_csv",
    "sklearn_version_at_freeze": sklearn.__version__,
    "random_seed_required_for_reusing_saved_split": False,
    "speaker_disjoint": True,
    "same_audio_hypotheses_same_fold": True,
    "official_test_split_untouched": True,
    "modeling_dataset_sha256": DATA_SHA,
    "frozen_split_sha256": CANDIDATE_SHA,
    "feature_design_sha256": DESIGN_SHA,
    "primary_models": ["A", "B1", "C1"],
    "secondary_ablation_models": ["B2", "C2"],
    "training_rules": {
        "all_validation_speakers_unseen_in_training": True,
        "fit_imputation_on_training_only": True,
        "fit_scaling_on_training_only": True,
        "tune_hyperparameters_with_training_speakers_only": True,
        "model_A_one_row_per_audio": True,
        "model_B_C_repeated_hypotheses_grouped_by_audio": True,
        "evaluate_each_hypothesis_separately": True,
        "no_validation_label_based_feature_selection": True
    },
    "evaluation_plan": {
        "primary_metrics": ["MAE", "RMSE"],
        "additional_metrics": [
            "speaker_level_MAE",
            "per_fold_MAE",
            "prediction_range_across_H1_H5",
            "prediction_sd_across_H1_H5"
        ],
        "acoustic_model_expected_asr_invariance": True
    },
    "fold_details": fold_details
}

PROTOCOL.parent.mkdir(parents=True, exist_ok=True)

if PROTOCOL.exists():
    previous = json.loads(
        PROTOCOL.read_text(encoding="utf-8")
    )
    assert previous == protocol, (
        "Existing frozen CV protocol differs; preserved."
    )
else:
    tmp = PROTOCOL.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(protocol, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(PROTOCOL)

progress = json.loads(CHECK.read_text(encoding="utf-8"))

progress["step5_speaker_split"] = "frozen"
progress["step5_speaker_split_sha256"] = sha256(FROZEN)
progress["step5_cv_protocol_sha256"] = sha256(PROTOCOL)
progress["next_task"] = (
    "Define leakage-safe training and evaluation pipelines "
    "for models A, B1, B2, C1 and C2"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.2 SPEAKER CV FROZEN ===")
print("Status:", protocol["status"])
print("Audios:", protocol["selected_audio_count"])
print("Speakers:", protocol["unique_speakers"])
print("Folds:", protocol["number_of_folds"])
print("Validation speakers per fold:", 4)
print("Validation audios per fold:", 80)
print("Train/validation speaker overlap: ZERO")
print("H1-H5 partition integrity: PASS")
print("Official test split: UNTOUCHED")

print("\n=== FROZEN FOLD SUMMARY ===")
for item in fold_details:
    print(
        f"Fold {item['fold']}: "
        f"val_speakers={item['validation_speakers']} "
        f"val_mean={item['validation_mean_fluency']:.3f}"
    )

print("\nFrozen split:", FROZEN)
print("Split SHA256:", sha256(FROZEN))
print("Protocol:", PROTOCOL)
print("Protocol SHA256:", sha256(PROTOCOL))
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

DATA = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = PROJECT / "data/splits/speaker_groupkfold5_frozen.csv"
CV_CONFIG = PROJECT / "configs/speaker_disjoint_cv_v1.json"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

RESULTS = PROJECT / "results/step05"
PREDICTIONS = RESULTS / "baseline_acoustic_oof_predictions.csv"
FOLD_RESULTS = RESULTS / "baseline_acoustic_fold_metrics.csv"
SPEAKER_RESULTS = RESULTS / "baseline_acoustic_speaker_metrics.csv"
REPORT = PROJECT / "reports/step05_baseline_acoustic_ridge_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    CV_CONFIG: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected_hash in EXPECTED.items():
    assert sha256(path) == expected_hash, (
        f"Source SHA256 mismatch: {path}"
    )

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv_config = json.loads(CV_CONFIG.read_text(encoding="utf-8"))

assert cv_config["status"] == "frozen"
assert cv_config["number_of_folds"] == 5
assert cv_config["speaker_disjoint"] is True

data = pd.read_csv(
    DATA,
    dtype={"audio_id": "string", "speaker": "string"}
)

split = pd.read_csv(
    SPLIT,
    dtype={"audio_id": "string", "speaker": "string"}
)

ids = ["pilot_index", "audio_id", "speaker", "fluency"]

audio = data[ids].drop_duplicates()

assert len(audio) == 400
assert audio["audio_id"].is_unique

acoustic_features = [
    col for col in data.columns
    if col not in (
        ids
        + ["hypothesis_id"]
        + design["linguistic_feature_names"]
        + design["prompt_feature_names"]
    )
]

assert len(acoustic_features) == 14
assert len(set(acoustic_features)) == 14
assert "fluency" not in acoustic_features

# Reduce to one acoustic record per audio.
acoustic = data.loc[
    data["hypothesis_id"] == "H1",
    ids + acoustic_features
].copy()

assert len(acoustic) == 400

df = acoustic.merge(
    split[ids + ["validation_fold"]],
    on=ids,
    how="inner",
    validate="one_to_one"
)

assert len(df) == 400
assert df["validation_fold"].between(1, 5).all()

prediction_rows = []
fold_rows = []

# Fixed alpha for the initial benchmark.
RIDGE_ALPHA = 1.0

for fold in range(1, 6):

    train = df.loc[
        df["validation_fold"] != fold
    ].copy()

    val = df.loc[
        df["validation_fold"] == fold
    ].copy()

    assert len(train) == 320
    assert len(val) == 80

    assert set(train["speaker"]).isdisjoint(
        set(val["speaker"])
    )

    X_train = train[acoustic_features]
    X_val = val[acoustic_features]

    y_train = train["fluency"].to_numpy(dtype=float)
    y_val = val["fluency"].to_numpy(dtype=float)

    # M0: training-fold mean only.
    pred_mean = np.full(
        len(val),
        y_train.mean(),
        dtype=float
    )

    # A: imputation and scaling learned on training only.
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=RIDGE_ALPHA))
    ])

    model.fit(X_train, y_train)
    pred_acoustic = model.predict(X_val)

    assert np.isfinite(pred_acoustic).all()

    fold_rows.append({
        "fold": fold,
        "train_audio_count": len(train),
        "validation_audio_count": len(val),
        "train_speaker_count": train["speaker"].nunique(),
        "validation_speaker_count": val["speaker"].nunique(),
        "train_mean_fluency": float(y_train.mean()),
        "validation_mean_fluency": float(y_val.mean()),
        "mean_baseline_mae": float(
            mean_absolute_error(y_val, pred_mean)
        ),
        "mean_baseline_rmse": float(
            np.sqrt(mean_squared_error(y_val, pred_mean))
        ),
        "acoustic_ridge_mae": float(
            mean_absolute_error(y_val, pred_acoustic)
        ),
        "acoustic_ridge_rmse": float(
            np.sqrt(mean_squared_error(y_val, pred_acoustic))
        )
    })

    for i, (_, row) in enumerate(val.iterrows()):
        prediction_rows.append({
            "pilot_index": int(row["pilot_index"]),
            "audio_id": str(row["audio_id"]),
            "speaker": str(row["speaker"]),
            "validation_fold": fold,
            "fluency": float(y_val[i]),
            "mean_baseline_prediction": float(pred_mean[i]),
            "acoustic_ridge_prediction": float(pred_acoustic[i])
        })

predictions = pd.DataFrame(prediction_rows).sort_values(
    "pilot_index"
).reset_index(drop=True)

fold_metrics = pd.DataFrame(fold_rows)

assert len(predictions) == 400
assert predictions["audio_id"].is_unique
assert predictions["speaker"].nunique() == 20
assert predictions["validation_fold"].nunique() == 5
assert predictions["fluency"].notna().all()

# Each speaker has predictions from exactly one fold.
assert predictions.groupby("speaker")[
    "validation_fold"
].nunique().eq(1).all()

y = predictions["fluency"].to_numpy()

overall_metrics = {}

for name, col in [
    ("M0_mean_baseline", "mean_baseline_prediction"),
    ("A_acoustic_ridge", "acoustic_ridge_prediction")
]:
    pred = predictions[col].to_numpy()

    overall_metrics[name] = {
        "MAE": float(mean_absolute_error(y, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y, pred)))
    }

speaker_rows = []

for speaker, group in predictions.groupby("speaker"):
    speaker_rows.append({
        "speaker": str(speaker),
        "fold": int(group["validation_fold"].iloc[0]),
        "audio_count": len(group),
        "mean_fluency": float(group["fluency"].mean()),
        "mean_baseline_mae": float(
            mean_absolute_error(
                group["fluency"],
                group["mean_baseline_prediction"]
            )
        ),
        "acoustic_ridge_mae": float(
            mean_absolute_error(
                group["fluency"],
                group["acoustic_ridge_prediction"]
            )
        )
    })

speaker_metrics = pd.DataFrame(speaker_rows)

assert len(speaker_metrics) == 20

RESULTS.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv_safely(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")

    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing result differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv_safely(predictions, PREDICTIONS)
save_csv_safely(fold_metrics, FOLD_RESULTS)
save_csv_safely(speaker_metrics, SPEAKER_RESULTS)

report = {
    "status": "PASS",
    "stage": "initial_fixed_alpha_benchmark",
    "model_A": "Ridge",
    "ridge_alpha": RIDGE_ALPHA,
    "hyperparameter_tuning_performed": False,
    "validation": "frozen_5fold_speaker_disjoint",
    "unique_audio_count": 400,
    "unique_speaker_count": 20,
    "acoustic_feature_count": 14,
    "prediction_count_per_model": 400,
    "metrics": overall_metrics,
    "predictions_sha256": sha256(PREDICTIONS),
    "fold_metrics_sha256": sha256(FOLD_RESULTS),
    "speaker_metrics_sha256": sha256(SPEAKER_RESULTS),
    "modeling_dataset_sha256": sha256(DATA),
    "frozen_split_sha256": sha256(SPLIT),
    "note": (
        "Predictions are out-of-fold for unseen speakers. "
        "Ridge alpha is fixed at 1.0, without tuning. "
        "No official test data were used."
    )
}

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == report, (
        "Existing benchmark report differs."
    )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_initial_acoustic_benchmark"] = "qc_pass"
progress["step5_initial_acoustic_predictions_sha256"] = sha256(PREDICTIONS)
progress["next_task"] = (
    "Review acoustic Ridge and mean baseline results "
    "before transcript-only Ridge experiments"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.3 INITIAL ACOUSTIC BENCHMARK ===")
print("Status:", report["status"])
print("Validation: frozen 5-fold speaker-disjoint")
print("Unique audios:", 400)
print("Features:", len(acoustic_features))
print("Ridge alpha:", RIDGE_ALPHA)
print("Hyperparameter tuning: NO")

print("\n=== OVERALL OUT-OF-FOLD METRICS ===")
for name, metrics in overall_metrics.items():
    print(
        f"{name}: "
        f"MAE={metrics['MAE']:.4f}, "
        f"RMSE={metrics['RMSE']:.4f}"
    )

print("\n=== PER-FOLD METRICS ===")
print(fold_metrics.round(4).to_string(index=False))

print("\n=== PER-SPEAKER MAE ===")
print(speaker_metrics.round(4).to_string(index=False))

print("\nPredictions CSV:", PREDICTIONS)
print("Predictions SHA256:", sha256(PREDICTIONS))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
DATA = PROJECT / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = PROJECT / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = PROJECT / "configs/fluency_feature_design_v1.json"
CV = PROJECT / "configs/speaker_disjoint_cv_v1.json"
ACOUSTIC_PRED = PROJECT / "results/step05/baseline_acoustic_oof_predictions.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

RESULT_DIR = PROJECT / "results/step05"
PRED_OUT = RESULT_DIR / "b1_transcript_ridge_oof_predictions.csv"
FOLD_OUT = RESULT_DIR / "b1_transcript_ridge_fold_metrics.csv"
SPEAKER_OUT = RESULT_DIR / "b1_transcript_ridge_speaker_metrics.csv"
REPORT = PROJECT / "reports/step05_b1_transcript_ridge_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    ACOUSTIC_PRED: "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path}"

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv = json.loads(CV.read_text(encoding="utf-8"))

assert cv["status"] == "frozen"
assert cv["speaker_disjoint"] is True

FEATURES = design["linguistic_feature_names"]
assert len(FEATURES) == 7

data = pd.read_csv(
    DATA,
    dtype={"audio_id": "string", "speaker": "string"}
)
split = pd.read_csv(
    SPLIT,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(data) == 2000
assert len(split) == 400
assert all(c in data.columns for c in FEATURES)

keys = ["pilot_index", "audio_id", "speaker", "fluency"]

df = data.merge(
    split[keys + ["validation_fold"]],
    on=keys,
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()

predictions = []
fold_metrics = []

ALPHA = 1.0

for fold in range(1, 6):

    train = df.loc[df["validation_fold"] != fold].copy()
    val = df.loc[df["validation_fold"] == fold].copy()

    assert len(train) == 1600
    assert len(val) == 400
    assert train["audio_id"].nunique() == 320
    assert val["audio_id"].nunique() == 80
    assert train["speaker"].nunique() == 16
    assert val["speaker"].nunique() == 4

    assert set(train["speaker"]).isdisjoint(set(val["speaker"]))
    assert set(train["audio_id"]).isdisjoint(set(val["audio_id"]))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ALPHA))
    ])

    model.fit(
        train[FEATURES],
        train["fluency"].to_numpy(dtype=float)
    )

    pred = model.predict(val[FEATURES])
    assert np.isfinite(pred).all()

    result = val[
        keys + ["hypothesis_id", "validation_fold"]
    ].copy()
    result["b1_prediction"] = pred
    predictions.append(result)

    # Each fold has 80 audios and exactly five
    # predictions per audio.
    fold_metrics.append({
        "fold": fold,
        "train_speakers": 16,
        "validation_speakers": 4,
        "train_unique_audios": 320,
        "validation_unique_audios": 80,
        "validation_hypotheses": 400,
        "MAE": float(mean_absolute_error(val["fluency"], pred)),
        "RMSE": float(np.sqrt(
            mean_squared_error(val["fluency"], pred)
        ))
    })

oof = pd.concat(predictions, ignore_index=True).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

fold_table = pd.DataFrame(fold_metrics)

assert len(oof) == 2000
assert not oof.duplicated(["audio_id", "hypothesis_id"]).any()
assert oof["audio_id"].nunique() == 400

# Absolute errors for every hypothesis.
oof["absolute_error"] = (
    oof["fluency"] - oof["b1_prediction"]
).abs()

# Per-audio instability of B1 prediction across H1-H5.
audio_stability = oof.groupby("audio_id")["b1_prediction"].agg(
    ["min", "max", "std"]
)

audio_stability["prediction_range"] = (
    audio_stability["max"] - audio_stability["min"]
)

# Per-hypothesis accuracy on the same 400 audios.
hypothesis_metrics = {}
for hid, group in oof.groupby("hypothesis_id"):
    hypothesis_metrics[hid] = {
        "MAE": float(mean_absolute_error(
            group["fluency"], group["b1_prediction"]
        )),
        "RMSE": float(np.sqrt(mean_squared_error(
            group["fluency"], group["b1_prediction"]
        )))
    }

# Speaker-level MAE: 20 independent speaker groups.
speaker_rows = []
for speaker, group in oof.groupby("speaker"):
    speaker_rows.append({
        "speaker": str(speaker),
        "fold": int(group["validation_fold"].iloc[0]),
        "unique_audios": group["audio_id"].nunique(),
        "hypothesis_rows": len(group),
        "MAE": float(group["absolute_error"].mean())
    })

speaker_table = pd.DataFrame(speaker_rows)
assert len(speaker_table) == 20

overall = {
    "MAE": float(mean_absolute_error(
        oof["fluency"], oof["b1_prediction"]
    )),
    "RMSE": float(np.sqrt(mean_squared_error(
        oof["fluency"], oof["b1_prediction"]
    ))),
    "mean_prediction_range": float(
        audio_stability["prediction_range"].mean()
    ),
    "median_prediction_range": float(
        audio_stability["prediction_range"].median()
    ),
    "mean_prediction_sd": float(
        audio_stability["std"].mean()
    ),
    "audios_with_nonzero_prediction_range": int(
        (audio_stability["prediction_range"] > 1e-10).sum()
    )
}

# Compare with previously saved acoustic-only OOF predictions.
a = pd.read_csv(
    ACOUSTIC_PRED,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(a) == 400
assert a["audio_id"].is_unique

comparison = oof.merge(
    a[["audio_id", "acoustic_ridge_prediction"]],
    on="audio_id",
    how="inner",
    validate="many_to_one"
)

assert len(comparison) == 2000
assert comparison["fluency"].notna().all()

a_mae = float(mean_absolute_error(
    comparison["fluency"],
    comparison["acoustic_ridge_prediction"]
))

# The acoustic value is repeated five times for
# comparison, not five independent observations.
assert np.isclose(a_mae, 0.9399, atol=0.0001)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing result differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv(oof, PRED_OUT)
save_csv(fold_table, FOLD_OUT)
save_csv(speaker_table, SPEAKER_OUT)

report = {
    "status": "PASS",
    "model": "B1_transcript_only_ridge",
    "alpha": ALPHA,
    "hyperparameter_tuning": False,
    "feature_names": FEATURES,
    "feature_count": 7,
    "speaker_disjoint_folds": 5,
    "train_preprocessing_only": True,
    "training_hypotheses_per_audio": 5,
    "unique_validation_audios": 400,
    "validation_hypothesis_rows": 2000,
    "overall_metrics": overall,
    "per_hypothesis_metrics": hypothesis_metrics,
    "reference_acoustic_MAE": a_mae,
    "source_modeling_sha256": sha256(DATA),
    "source_frozen_split_sha256": sha256(SPLIT),
    "predictions_sha256": sha256(PRED_OUT),
    "fold_metrics_sha256": sha256(FOLD_OUT),
    "speaker_metrics_sha256": sha256(SPEAKER_OUT),
    "note": (
        "All five hypotheses used as repeated training examples. "
        "Each audio has equal total weight because each has "
        "five hypotheses. Validation speakers remain unseen. "
        "Hypothesis rows are not independent observations."
    )
}

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == report, "Existing report differs"
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_b1_transcript_ridge"] = "qc_pass"
progress["step5_b1_predictions_sha256"] = sha256(PRED_OUT)
progress["next_task"] = (
    "Review B1 accuracy and prediction instability; "
    "then evaluate prompt-aware B2 and fusion C1/C2"
)
progress["updated_at"] = (
    __import__("datetime").datetime.now(
        __import__("datetime").timezone.utc
    ).isoformat()
)

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.4 B1 TRANSCRIPT RIDGE SUMMARY ===")
print("Status:", report["status"])
print("Unique audios:", overall and 400)
print("Speakers:", 20)
print("OOF hypothesis predictions:", len(oof))
print("Features:", len(FEATURES))
print("Alpha:", ALPHA)
print("Speaker-disjoint validation: PASS")

print("\n=== OVERALL COMPARISON ===")
print(f"A acoustic MAE: {a_mae:.4f}")
print(f"B1 transcript MAE: {overall['MAE']:.4f}")
print(f"B1 transcript RMSE: {overall['RMSE']:.4f}")

print("\n=== H1-H5 INDIVIDUAL METRICS ===")
for hid, metrics in sorted(hypothesis_metrics.items()):
    print(
        f"{hid}: MAE={metrics['MAE']:.4f}, "
        f"RMSE={metrics['RMSE']:.4f}"
    )

print("\n=== PREDICTION INSTABILITY ===")
for key in [
    "mean_prediction_range",
    "median_prediction_range",
    "mean_prediction_sd",
    "audios_with_nonzero_prediction_range"
]:
    print(f"{key}: {overall[key]}")

print("\n=== FOLD METRICS ===")
print(fold_table.round(4).to_string(index=False))

print("\n=== SPEAKER METRICS ===")
print(speaker_table.round(4).to_string(index=False))

print("\nPredictions:", PRED_OUT)
print("SHA256:", sha256(PRED_OUT))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = PROJECT / "results/step05/b1_transcript_ridge_oof_predictions.csv"
MODEL_REPORT = PROJECT / "reports/step05_b1_transcript_ridge_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "results/step05"
AUDIO_OUT = OUT_DIR / "b1_prediction_instability_per_audio.csv"
REPORT = PROJECT / "reports/step05_b1_outlier_influence_audit.json"

EXPECTED_SHA = (
    "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(SOURCE) == EXPECTED_SHA

previous = json.loads(MODEL_REPORT.read_text(encoding="utf-8"))
assert previous["status"] == "PASS"

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert not df.duplicated(["audio_id", "hypothesis_id"]).any()

required = ["fluency", "b1_prediction", "validation_fold"]
assert df[required].notna().all().all()

df["signed_error"] = df["b1_prediction"] - df["fluency"]
df["absolute_error_check"] = df["signed_error"].abs()
df["squared_error"] = df["signed_error"] ** 2

def metrics(frame):
    return {
        "n": int(len(frame)),
        "MAE": float(frame["absolute_error_check"].mean()),
        "RMSE": float(np.sqrt(frame["squared_error"].mean()))
    }

all_metrics = metrics(df)

assert np.isclose(
    all_metrics["MAE"],
    previous["overall_metrics"]["MAE"],
    atol=1e-10
)
assert np.isclose(
    all_metrics["RMSE"],
    previous["overall_metrics"]["RMSE"],
    atol=1e-10
)

critical_mask = (
    df["audio_id"].eq("030210164") &
    df["hypothesis_id"].eq("H2")
)

assert critical_mask.sum() == 1

critical = df.loc[critical_mask].iloc[0]
without_one = df.loc[~critical_mask].copy()

# Diagnostic only: 1999 predictions, unequal
# hypothesis counts for the affected audio.
without_one_metrics = metrics(without_one)

# Audio-level sensitivity: remove all five
# hypotheses for the affected audio.
without_audio = df.loc[
    ~df["audio_id"].eq("030210164")
].copy()
without_audio_metrics = metrics(without_audio)

total_sse = float(df["squared_error"].sum())
critical_sse = float(critical["squared_error"])

critical_sse_share = critical_sse / total_sse

# Complete, equal-weight 400-audio stability table.
audio = (
    df.groupby(
        ["pilot_index", "audio_id", "speaker", "validation_fold"],
        as_index=False
    )
    .agg(
        human_fluency=("fluency", "first"),
        prediction_min=("b1_prediction", "min"),
        prediction_max=("b1_prediction", "max"),
        prediction_mean=("b1_prediction", "mean"),
        prediction_sd=("b1_prediction", "std"),
        mean_absolute_error=("absolute_error_check", "mean"),
        max_absolute_error=("absolute_error_check", "max"),
        hypothesis_count=("hypothesis_id", "count")
    )
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

audio["prediction_range"] = (
    audio["prediction_max"] - audio["prediction_min"]
)

audio = audio.sort_values(
    ["pilot_index"]
).reset_index(drop=True)

top_errors = df.sort_values(
    "squared_error", ascending=False
).head(10)

top_ranges = audio.sort_values(
    "prediction_range", ascending=False
).head(10)

# Separate analysis for the beam-search hypothesis.
h2 = df.loc[df["hypothesis_id"] == "H2"]
h2_without = h2.loc[
    ~h2["audio_id"].eq("030210164")
]

# Fold 2 includes this critical audio.
critical_fold = int(critical["validation_fold"])
fold_df = df.loc[
    df["validation_fold"] == critical_fold
]

fold_without_one = fold_df.loc[
    ~critical_mask.loc[fold_df.index]
]

OUT_DIR.mkdir(parents=True, exist_ok=True)

tmp = AUDIO_OUT.with_suffix(".tmp")
audio.to_csv(tmp, index=False, encoding="utf-8")

if AUDIO_OUT.exists():
    assert sha256(AUDIO_OUT) == sha256(tmp), (
        "Existing audio instability table differs."
    )
    tmp.unlink()
else:
    tmp.replace(AUDIO_OUT)

report = {
    "status": "PASS",
    "model": "B1_transcript_only_ridge",
    "original_predictions_unchanged": True,
    "full_metrics": all_metrics,
    "without_one_h2_diagnostic": without_one_metrics,
    "without_entire_audio_diagnostic": without_audio_metrics,
    "critical_audio_id": "030210164",
    "critical_hypothesis_id": "H2",
    "critical_speaker": str(critical["speaker"]),
    "critical_fold": critical_fold,
    "critical_human_fluency": float(critical["fluency"]),
    "critical_prediction": float(critical["b1_prediction"]),
    "critical_absolute_error": float(critical["absolute_error_check"]),
    "critical_squared_error": critical_sse,
    "critical_sse_share": critical_sse_share,
    "h2_full": metrics(h2),
    "h2_without_critical": metrics(h2_without),
    "critical_fold_full": metrics(fold_df),
    "critical_fold_without_one": metrics(fold_without_one),
    "mean_prediction_range": float(audio["prediction_range"].mean()),
    "median_prediction_range": float(audio["prediction_range"].median()),
    "p95_prediction_range": float(
        audio["prediction_range"].quantile(0.95)
    ),
    "max_prediction_range": float(audio["prediction_range"].max()),
    "variable_prediction_audios": int(
        (audio["prediction_range"] > 1e-10).sum()
    ),
    "source_sha256": sha256(SOURCE),
    "audio_table_sha256": sha256(AUDIO_OUT),
    "interpretation": (
        "Post-hoc influence diagnostics only. "
        "No observations excluded from primary model evaluation. "
        "The same fitted OOF models are used."
    )
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    saved = json.loads(REPORT.read_text(encoding="utf-8"))
    assert saved == report, "Existing report differs."
else:
    temp_report = REPORT.with_suffix(".tmp")
    temp_report.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp_report.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_b1_outlier_audit"] = "qc_pass"
progress["step5_b1_outlier_report_sha256"] = sha256(REPORT)
progress["next_task"] = (
    "Review B1 outlier influence and proceed to B2/C1/C2 "
    "with the unchanged frozen speaker-disjoint folds"
)

from datetime import datetime, timezone
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.5 B1 OUTLIER INFLUENCE AUDIT ===")
print("Status: PASS")
print("Critical audio:", report["critical_audio_id"])
print("Critical hypothesis:", report["critical_hypothesis_id"])
print("Speaker:", report["critical_speaker"])
print("Fold:", report["critical_fold"])
print("Human fluency:", report["critical_human_fluency"])
print("B1 prediction:", round(report["critical_prediction"], 4))
print("Absolute error:", round(report["critical_absolute_error"], 4))
print("Squared error share (%):", round(100 * critical_sse_share, 2))

print("\n=== GLOBAL DIAGNOSTIC COMPARISON ===")
for name, values in [
    ("All 2000 predictions", all_metrics),
    ("Without one H2 prediction", without_one_metrics),
    ("Without entire critical audio", without_audio_metrics)
]:
    print(
        f"{name}: n={values['n']}, "
        f"MAE={values['MAE']:.4f}, RMSE={values['RMSE']:.4f}"
    )

print("\n=== H2 AND CRITICAL FOLD ===")
for name, values in [
    ("H2 full", metrics(h2)),
    ("H2 without one", metrics(h2_without)),
    ("Critical fold full", metrics(fold_df)),
    ("Critical fold without one", metrics(fold_without_one))
]:
    print(
        f"{name}: n={values['n']}, "
        f"MAE={values['MAE']:.4f}, RMSE={values['RMSE']:.4f}"
    )

print("\n=== TOP 10 INDIVIDUAL ERRORS ===")
print(
    top_errors[
        ["audio_id", "speaker", "hypothesis_id",
         "fluency", "b1_prediction",
         "absolute_error_check", "squared_error"]
    ].round(4).to_string(index=False)
)

print("\n=== TOP 10 PREDICTION RANGES ===")
print(
    top_ranges[
        ["audio_id", "speaker", "prediction_range",
         "prediction_sd", "mean_absolute_error"]
    ].round(4).to_string(index=False)
)

print("\n=== STABILITY SUMMARY ===")
print(
    "Mean range:",
    round(report["mean_prediction_range"], 4)
)
print(
    "Median range:",
    round(report["median_prediction_range"], 4)
)
print(
    "95th percentile range:",
    round(report["p95_prediction_range"], 4)
)
print(
    "Maximum range:",
    round(report["max_prediction_range"], 4)
)
print(
    "Variable prediction audios:",
    report["variable_prediction_audios"]
)

print("\nSaved audio table:", AUDIO_OUT)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
CV = P / "configs/speaker_disjoint_cv_v1.json"
A_PRED = P / "results/step05/baseline_acoustic_oof_predictions.csv"
B1_PRED = P / "results/step05/b1_transcript_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step05"
PRED = OUTDIR / "c1_fusion_ridge_oof_predictions.csv"
FOLDS = OUTDIR / "c1_fusion_ridge_fold_metrics.csv"
AUDIO = OUTDIR / "c1_fusion_prediction_stability_per_audio.csv"
REPORT = P / "reports/step05_c1_fusion_ridge_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    A_PRED: "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
    B1_PRED: "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file()
    assert sha256(path) == expected, f"SHA mismatch: {path}"

design = json.loads(DESIGN.read_text(encoding="utf-8"))
protocol = json.loads(CV.read_text(encoding="utf-8"))

assert protocol["status"] == "frozen"
assert design["models"]["C1"]["feature_groups"] == [
    "acoustic", "linguistic"
]

data = pd.read_csv(
    DATA, dtype={"audio_id": "string", "speaker": "string"}
)
split = pd.read_csv(
    SPLIT, dtype={"audio_id": "string", "speaker": "string"}
)

ID = ["pilot_index", "audio_id", "speaker"]
KEYS = ID + ["hypothesis_id"]
LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]

ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

FEATURES = ACOUSTIC + LING

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(FEATURES) == 21
assert "fluency" not in FEATURES
assert not set(FEATURES).intersection(KEYS + PROMPT)

assert len(data) == 2000
assert len(split) == 400
assert not data.duplicated(KEYS).any()

df = data.merge(
    split[ID + ["fluency", "validation_fold"]],
    on=ID + ["fluency"],
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()

ALPHA = 1.0
pred_parts = []
fold_rows = []

for fold in range(1, 6):
    train = df.loc[df["validation_fold"] != fold]
    val = df.loc[df["validation_fold"] == fold]

    assert len(train) == 1600
    assert len(val) == 400
    assert train["audio_id"].nunique() == 320
    assert val["audio_id"].nunique() == 80

    assert set(train["speaker"]).isdisjoint(set(val["speaker"]))
    assert set(train["audio_id"]).isdisjoint(set(val["audio_id"]))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ALPHA))
    ])

    model.fit(
        train[FEATURES],
        train["fluency"].to_numpy(dtype=float)
    )

    predictions = model.predict(val[FEATURES])
    assert np.isfinite(predictions).all()

    part = val[KEYS + ["fluency", "validation_fold"]].copy()
    part["c1_prediction"] = predictions
    pred_parts.append(part)

    fold_rows.append({
        "fold": fold,
        "validation_audios": 80,
        "validation_hypotheses": 400,
        "MAE": float(mean_absolute_error(
            val["fluency"], predictions
        )),
        "RMSE": float(np.sqrt(mean_squared_error(
            val["fluency"], predictions
        )))
    })

oof = pd.concat(pred_parts, ignore_index=True).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(oof) == 2000
assert oof["audio_id"].nunique() == 400
assert not oof.duplicated(["audio_id", "hypothesis_id"]).any()

# Verify C1 rows against earlier acoustic and transcript OOF.
a = pd.read_csv(
    A_PRED, dtype={"audio_id": "string", "speaker": "string"}
)
b = pd.read_csv(
    B1_PRED, dtype={"audio_id": "string", "speaker": "string"}
)

comp = oof.merge(
    a[ID + ["fluency", "validation_fold", "acoustic_ridge_prediction"]],
    on=ID + ["fluency", "validation_fold"],
    validate="many_to_one"
).merge(
    b[KEYS + ["fluency", "validation_fold", "b1_prediction"]],
    on=KEYS + ["fluency", "validation_fold"],
    validate="one_to_one"
)

assert len(comp) == 2000

def metrics(y, pred):
    return {
        "MAE": float(mean_absolute_error(y, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y, pred)))
    }

comparison = {
    "A_acoustic": metrics(
        comp["fluency"], comp["acoustic_ridge_prediction"]
    ),
    "B1_transcript": metrics(
        comp["fluency"], comp["b1_prediction"]
    ),
    "C1_fusion": metrics(
        comp["fluency"], comp["c1_prediction"]
    )
}

audio = (
    oof.groupby(ID + ["validation_fold"], as_index=False)
    .agg(
        fluency=("fluency", "first"),
        prediction_min=("c1_prediction", "min"),
        prediction_max=("c1_prediction", "max"),
        prediction_sd=("c1_prediction", "std"),
        hypothesis_count=("hypothesis_id", "count")
    )
)

audio["prediction_range"] = (
    audio["prediction_max"] - audio["prediction_min"]
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

hyp_metrics = {
    h: metrics(
        g["fluency"], g["c1_prediction"]
    )
    for h, g in oof.groupby("hypothesis_id")
}

stability = {
    "mean_range": float(audio["prediction_range"].mean()),
    "median_range": float(audio["prediction_range"].median()),
    "p95_range": float(audio["prediction_range"].quantile(0.95)),
    "max_range": float(audio["prediction_range"].max()),
    "variable_audio_count": int(
        (audio["prediction_range"] > 1e-10).sum()
    )
}

critical = oof.loc[
    oof["audio_id"].eq("030210164") &
    oof["hypothesis_id"].eq("H2")
]
assert len(critical) == 1

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(temp), (
            f"Existing file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_csv(oof, PRED)
save_csv(pd.DataFrame(fold_rows), FOLDS)
save_csv(audio, AUDIO)

report = {
    "status": "PASS",
    "model": "C1_acoustic_linguistic_fusion_ridge",
    "alpha": ALPHA,
    "hyperparameter_tuning": False,
    "feature_count": 21,
    "acoustic_feature_count": 14,
    "linguistic_feature_count": 7,
    "validation": "frozen_5fold_speaker_disjoint",
    "OOF_prediction_rows": 2000,
    "unique_audios": 400,
    "comparison": comparison,
    "per_hypothesis_metrics": hyp_metrics,
    "stability": stability,
    "critical_audio_H2_prediction": float(
        critical["c1_prediction"].iloc[0]
    ),
    "source_dataset_sha256": sha256(DATA),
    "source_split_sha256": sha256(SPLIT),
    "predictions_sha256": sha256(PRED),
    "fold_metrics_sha256": sha256(FOLDS),
    "audio_stability_sha256": sha256(AUDIO),
    "note": (
        "No clipping or hyperparameter tuning. "
        "Training-only imputation and scaling. "
        "All five ASR hypotheses remain in the same speaker fold."
    )
}

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == report, (
        "Existing report differs; original preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_c1_fusion_ridge"] = "qc_pass"
progress["step5_c1_predictions_sha256"] = sha256(PRED)
progress["next_task"] = (
    "Compare C1 fusion accuracy and ASR-induced instability "
    "against models A and B1"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.6 C1 FUSION RIDGE SUMMARY ===")
print("Status: PASS")
print("Features:", len(FEATURES))
print("Unique audios: 400")
print("OOF predictions:", len(oof))
print("Speaker-disjoint CV: PASS")
print("Ridge alpha:", ALPHA)

print("\n=== MODEL COMPARISON ===")
for name, m in comparison.items():
    print(f"{name}: MAE={m['MAE']:.4f}, RMSE={m['RMSE']:.4f}")

print("\n=== C1 H1-H5 METRICS ===")
for h, m in sorted(hyp_metrics.items()):
    print(f"{h}: MAE={m['MAE']:.4f}, RMSE={m['RMSE']:.4f}")

print("\n=== C1 PREDICTION STABILITY ===")
for key, value in stability.items():
    print(f"{key}: {value}")

print("\n=== C1 CRITICAL ASR CASE ===")
print("Audio: 030210164")
print("Hypothesis: H2")
print("Human fluency:", float(critical["fluency"].iloc[0]))
print("C1 prediction:", float(critical["c1_prediction"].iloc[0]))

print("\n=== C1 PER-FOLD METRICS ===")
print(pd.DataFrame(fold_rows).round(4).to_string(index=False))

print("\nPrediction CSV:", PRED)
print("SHA256:", sha256(PRED))
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
A_FILE = P / "results/step05/baseline_acoustic_oof_predictions.csv"
B_FILE = P / "results/step05/b1_transcript_ridge_oof_predictions.csv"
C_FILE = P / "results/step05/c1_fusion_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUT = P / "results/step05/c1_vs_a_paired_audio_audit.csv"
REPORT = P / "reports/step05_c1_outlier_paired_audit.json"

EXPECTED = {
    A_FILE: "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
    B_FILE: "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    C_FILE: "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert sha256(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

a, b, c = read(A_FILE), read(B_FILE), read(C_FILE)

assert len(a) == 400
assert len(b) == len(c) == 2000

KEYS = [
    "pilot_index", "audio_id", "speaker",
    "fluency", "validation_fold"
]
HKEYS = KEYS + ["hypothesis_id"]

assert not a.duplicated(KEYS).any()
assert not b.duplicated(HKEYS).any()
assert not c.duplicated(HKEYS).any()

merged = c[HKEYS + ["c1_prediction"]].merge(
    b[HKEYS + ["b1_prediction"]],
    on=HKEYS,
    validate="one_to_one"
).merge(
    a[KEYS + ["acoustic_ridge_prediction"]],
    on=KEYS,
    validate="many_to_one"
)

assert len(merged) == 2000
assert merged["audio_id"].nunique() == 400
assert merged.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

for name, col in [
    ("A", "acoustic_ridge_prediction"),
    ("B1", "b1_prediction"),
    ("C1", "c1_prediction")
]:
    merged[f"{name}_error"] = merged[col] - merged["fluency"]
    merged[f"{name}_ae"] = merged[f"{name}_error"].abs()
    merged[f"{name}_se"] = merged[f"{name}_error"] ** 2

def metrics(frame, model):
    return {
        "n": int(len(frame)),
        "MAE": float(frame[f"{model}_ae"].mean()),
        "RMSE": float(np.sqrt(frame[f"{model}_se"].mean()))
    }

all_metrics = {
    name: metrics(merged, name)
    for name in ["A", "B1", "C1"]
}

CRITICAL_ID = "030210164"
critical_mask = (
    merged["audio_id"].eq(CRITICAL_ID) &
    merged["hypothesis_id"].eq("H2")
)

assert critical_mask.sum() == 1

critical = merged.loc[critical_mask].iloc[0]
without_one = merged.loc[~critical_mask]
without_audio = merged.loc[
    ~merged["audio_id"].eq(CRITICAL_ID)
]

diagnostics = {
    "all_predictions": all_metrics,
    "without_one_H2": {
        name: metrics(without_one, name)
        for name in ["A", "B1", "C1"]
    },
    "without_entire_audio": {
        name: metrics(without_audio, name)
        for name in ["A", "B1", "C1"]
    }
}

sse_share = {
    name: float(
        critical[f"{name}_se"] /
        merged[f"{name}_se"].sum()
    )
    for name in ["B1", "C1"]
}

# Paired audio-level error: each of 400 audios
# contributes once with five equally weighted hypotheses.
audio = merged.groupby(
    KEYS, as_index=False
).agg(
    a_mae=("A_ae", "mean"),
    b1_mae=("B1_ae", "mean"),
    c1_mae=("C1_ae", "mean"),
    a_mse=("A_se", "mean"),
    b1_mse=("B1_se", "mean"),
    c1_mse=("C1_se", "mean"),
    b1_min=("b1_prediction", "min"),
    b1_max=("b1_prediction", "max"),
    c1_min=("c1_prediction", "min"),
    c1_max=("c1_prediction", "max")
)

assert len(audio) == 400
assert audio["audio_id"].is_unique

audio["c1_minus_a_absolute_error"] = (
    audio["c1_mae"] - audio["a_mae"]
)
audio["b1_range"] = audio["b1_max"] - audio["b1_min"]
audio["c1_range"] = audio["c1_max"] - audio["c1_min"]

win_counts = {
    "C1_better_than_A": int(
        (audio["c1_minus_a_absolute_error"] < -1e-10).sum()
    ),
    "C1_worse_than_A": int(
        (audio["c1_minus_a_absolute_error"] > 1e-10).sum()
    ),
    "ties": int(
        (audio["c1_minus_a_absolute_error"].abs() <= 1e-10).sum()
    )
}
assert sum(win_counts.values()) == 400

stability = {}

for name in ["b1", "c1"]:
    ranges = audio[f"{name}_range"]
    without_critical = audio.loc[
        ~audio["audio_id"].eq(CRITICAL_ID),
        f"{name}_range"
    ]
    stability[name.upper()] = {
        "mean_range_all": float(ranges.mean()),
        "median_range_all": float(ranges.median()),
        "p95_range_all": float(ranges.quantile(0.95)),
        "mean_range_without_critical_audio": float(
            without_critical.mean()
        ),
        "p95_range_without_critical_audio": float(
            without_critical.quantile(0.95)
        )
    }

audio = audio.sort_values("pilot_index").reset_index(drop=True)

OUT.parent.mkdir(parents=True, exist_ok=True)

tmp = OUT.with_suffix(".tmp")
audio.to_csv(tmp, index=False, encoding="utf-8")

if OUT.exists():
    assert sha256(OUT) == sha256(tmp)
    tmp.unlink()
else:
    tmp.replace(OUT)

report = {
    "status": "PASS",
    "scope": "posthoc_diagnostic_no_refitting",
    "unique_audios": 400,
    "hypothesis_rows": 2000,
    "critical_audio": CRITICAL_ID,
    "critical_hypothesis": "H2",
    "critical_true_fluency": float(critical["fluency"]),
    "critical_predictions": {
        name: float(critical[col])
        for name, col in [
            ("A", "acoustic_ridge_prediction"),
            ("B1", "b1_prediction"),
            ("C1", "c1_prediction")
        ]
    },
    "critical_squared_error_shares": sse_share,
    "diagnostics": diagnostics,
    "paired_audio_win_counts": win_counts,
    "stability": stability,
    "paired_audio_csv_sha256": sha256(OUT),
    "source_sha256": {
        path.name: sha256(path)
        for path in EXPECTED
    },
    "interpretation": (
        "No retraining, no primary-analysis exclusions. "
        "MAE comparisons use equal audio weighting. "
        "Speaker-clustered uncertainty not yet estimated."
    )
}

REPORT.parent.mkdir(parents=True, exist_ok=True)

if REPORT.exists():
    previous = json.loads(REPORT.read_text(encoding="utf-8"))
    assert previous == report, (
        "Existing report differs; original preserved."
    )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_c1_outlier_paired_audit"] = "qc_pass"
progress["step5_c1_outlier_paired_audit_sha256"] = sha256(REPORT)
progress["next_task"] = (
    "Review C1 paired audit and run B2 prompt-aware Ridge"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.7 C1 PAIRED AUDIT SUMMARY ===")
print("Status: PASS")
print("Unique audios:", len(audio))
print("Hypothesis predictions:", len(merged))
print("Critical audio:", CRITICAL_ID)
print("Critical H2 predictions:", report["critical_predictions"])
print("B1 squared-error share (%):", round(100 * sse_share["B1"], 2))
print("C1 squared-error share (%):", round(100 * sse_share["C1"], 2))

print("\n=== DIAGNOSTIC ACCURACY COMPARISON ===")
for condition, models in diagnostics.items():
    print("\n", condition)
    for name, values in models.items():
        print(
            f"{name}: n={values['n']}, "
            f"MAE={values['MAE']:.4f}, "
            f"RMSE={values['RMSE']:.4f}"
        )

print("\n=== PAIRED AUDIO-LEVEL COMPARISON ===")
for name, count in win_counts.items():
    print(f"{name}: {count}")

print("\n=== STABILITY WITH AND WITHOUT CRITICAL AUDIO ===")
for name, values in stability.items():
    print(name)
    for metric, value in values.items():
        print(f"  {metric}: {value:.4f}")

print("\n=== WORST 10 C1 VS A AUDIO DIFFERENCES ===")
print(
    audio.nlargest(
        10, "c1_minus_a_absolute_error"
    )[
        [
            "audio_id", "speaker",
            "a_mae", "c1_mae",
            "c1_minus_a_absolute_error",
            "b1_range", "c1_range"
        ]
    ].round(4).to_string(index=False)
)

print("\nAudio comparison CSV:", OUT)
print("SHA256:", sha256(OUT))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
CV = P / "configs/speaker_disjoint_cv_v1.json"
B1 = P / "results/step05/b1_transcript_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step05"
PRED = OUTDIR / "b2_prompt_aware_ridge_oof_predictions.csv"
FOLDS = OUTDIR / "b2_prompt_aware_ridge_fold_metrics.csv"
AUDIO = OUTDIR / "b2_prompt_aware_stability_per_audio.csv"
REPORT = P / "reports/step05_b2_prompt_aware_ridge_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    B1: "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path}"

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv = json.loads(CV.read_text(encoding="utf-8"))

assert cv["status"] == "frozen"
assert cv["speaker_disjoint"]
assert design["models"]["B2"]["feature_groups"] == [
    "linguistic", "prompt"
]

FEATURES = (
    design["linguistic_feature_names"]
    + design["prompt_feature_names"]
)
assert len(FEATURES) == 9
assert len(set(FEATURES)) == 9

KEYS = ["pilot_index", "audio_id", "speaker",
        "hypothesis_id", "fluency"]
AUDIO_KEYS = ["pilot_index", "audio_id", "speaker", "fluency"]

data = pd.read_csv(
    DATA, dtype={"audio_id": "string", "speaker": "string"}
)
split = pd.read_csv(
    SPLIT, dtype={"audio_id": "string", "speaker": "string"}
)
b1 = pd.read_csv(
    B1, dtype={"audio_id": "string", "speaker": "string"}
)

assert len(data) == len(b1) == 2000
assert len(split) == 400
assert not data.duplicated(KEYS).any()
assert not b1.duplicated(KEYS).any()
assert not split.duplicated(AUDIO_KEYS).any()

df = data.merge(
    split[AUDIO_KEYS + ["validation_fold"]],
    on=AUDIO_KEYS,
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

pred_parts = []
fold_rows = []
ALPHA = 1.0

for fold in range(1, 6):
    train = df.loc[df["validation_fold"] != fold]
    val = df.loc[df["validation_fold"] == fold]

    assert len(train) == 1600
    assert len(val) == 400
    assert train["speaker"].nunique() == 16
    assert val["speaker"].nunique() == 4
    assert set(train["speaker"]).isdisjoint(set(val["speaker"]))
    assert set(train["audio_id"]).isdisjoint(set(val["audio_id"]))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ALPHA))
    ])

    model.fit(
        train[FEATURES],
        train["fluency"].to_numpy(dtype=float)
    )

    prediction = model.predict(val[FEATURES])
    assert np.isfinite(prediction).all()

    part = val[KEYS + ["validation_fold"]].copy()
    part["b2_prediction"] = prediction
    pred_parts.append(part)

    err = prediction - val["fluency"].to_numpy(dtype=float)
    fold_rows.append({
        "fold": fold,
        "train_audios": 320,
        "validation_audios": 80,
        "validation_hypotheses": 400,
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2)))
    })

oof = pd.concat(pred_parts, ignore_index=True).sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(oof) == 2000
assert not oof.duplicated(KEYS).any()

comp = oof.merge(
    b1[KEYS + ["validation_fold", "b1_prediction"]],
    on=KEYS + ["validation_fold"],
    how="inner",
    validate="one_to_one"
)
assert len(comp) == 2000

def metrics(frame, col):
    err = frame[col].to_numpy(dtype=float) - frame[
        "fluency"
    ].to_numpy(dtype=float)
    return {
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2)))
    }

comparison = {
    "B1": metrics(comp, "b1_prediction"),
    "B2": metrics(comp, "b2_prediction")
}

hypothesis_metrics = {
    h: metrics(group, "b2_prediction")
    for h, group in oof.groupby("hypothesis_id")
}

audio = oof.groupby(
    AUDIO_KEYS + ["validation_fold"], as_index=False
).agg(
    prediction_min=("b2_prediction", "min"),
    prediction_max=("b2_prediction", "max"),
    prediction_sd=("b2_prediction", "std"),
    hypothesis_count=("hypothesis_id", "count")
)

audio["prediction_range"] = (
    audio["prediction_max"] - audio["prediction_min"]
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

stability = {
    "mean_range": float(audio["prediction_range"].mean()),
    "median_range": float(audio["prediction_range"].median()),
    "p95_range": float(audio["prediction_range"].quantile(0.95)),
    "max_range": float(audio["prediction_range"].max()),
    "variable_audio_count": int(
        (audio["prediction_range"] > 1e-10).sum()
    )
}

critical = oof.loc[
    oof["audio_id"].eq("030210164") &
    oof["hypothesis_id"].eq("H2")
]
assert len(critical) == 1

without_critical = comp.loc[
    ~(comp["audio_id"].eq("030210164") &
      comp["hypothesis_id"].eq("H2"))
]

sensitivity = {
    "B1_without_one_H2": metrics(
        without_critical, "b1_prediction"
    ),
    "B2_without_one_H2": metrics(
        without_critical, "b2_prediction"
    )
}

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing result differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv(oof, PRED)
save_csv(pd.DataFrame(fold_rows), FOLDS)
save_csv(audio.sort_values("pilot_index"), AUDIO)

report = {
    "status": "PASS",
    "model": "B2_prompt_aware_ridge",
    "alpha": ALPHA,
    "hyperparameter_tuning": False,
    "features": FEATURES,
    "feature_count": 9,
    "speaker_disjoint_folds": 5,
    "unique_audios": 400,
    "OOF_predictions": 2000,
    "comparison": comparison,
    "per_hypothesis_metrics": hypothesis_metrics,
    "stability": stability,
    "critical_H2_prediction": float(
        critical["b2_prediction"].iloc[0]
    ),
    "sensitivity": sensitivity,
    "source_data_sha256": sha256(DATA),
    "frozen_split_sha256": sha256(SPLIT),
    "predictions_sha256": sha256(PRED),
    "fold_metrics_sha256": sha256(FOLDS),
    "audio_stability_sha256": sha256(AUDIO),
    "notes": (
        "Fixed-alpha Ridge, no clipping or tuning. "
        "Training-only imputation and scaling. "
        "Prompt disagreement is not ground-truth WER/CER. "
        "All 400 audios remain in primary evaluation."
    )
}

if REPORT.exists():
    old = json.loads(REPORT.read_text(encoding="utf-8"))
    assert old == report, "Existing report differs"
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_b2_prompt_aware_ridge"] = "qc_pass"
progress["step5_b2_predictions_sha256"] = sha256(PRED)
progress["next_task"] = (
    "Review B2 prompt-aware accuracy and instability; "
    "then evaluate C2 prompt-aware fusion"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.8 B2 PROMPT-AWARE RIDGE ===")
print("Status: PASS")
print("Features:", len(FEATURES))
print("OOF predictions:", len(oof))
print("Speaker-disjoint folds: 5")
print("Alpha:", ALPHA)

print("\n=== B1 VS B2 ACCURACY ===")
for name, values in comparison.items():
    print(
        f"{name}: MAE={values['MAE']:.4f}, "
        f"RMSE={values['RMSE']:.4f}"
    )

print("\n=== B2 H1-H5 METRICS ===")
for h, values in sorted(hypothesis_metrics.items()):
    print(
        f"{h}: MAE={values['MAE']:.4f}, "
        f"RMSE={values['RMSE']:.4f}"
    )

print("\n=== B2 PREDICTION STABILITY ===")
for key, value in stability.items():
    print(f"{key}: {value}")

print("\n=== B2 CRITICAL H2 CASE ===")
print("Audio: 030210164")
print("Human fluency:", float(critical["fluency"].iloc[0]))
print("B2 prediction:", float(critical["b2_prediction"].iloc[0]))

print("\n=== SENSITIVITY: WITHOUT ONE H2 PREDICTION ===")
for name, values in sensitivity.items():
    print(
        f"{name}: MAE={values['MAE']:.4f}, "
        f"RMSE={values['RMSE']:.4f}"
    )

print("\n=== B2 PER-FOLD METRICS ===")
print(pd.DataFrame(fold_rows).round(4).to_string(index=False))

print("\nPrediction CSV:", PRED)
print("SHA256:", sha256(PRED))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
CV = P / "configs/speaker_disjoint_cv_v1.json"
C1 = P / "results/step05/c1_fusion_ridge_oof_predictions.csv"
B2 = P / "results/step05/b2_prompt_aware_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step05"
PRED = OUTDIR / "c2_prompt_aware_fusion_ridge_oof_predictions.csv"
FOLDS = OUTDIR / "c2_prompt_aware_fusion_fold_metrics.csv"
AUDIO = OUTDIR / "c2_prompt_aware_fusion_stability_per_audio.csv"
REPORT = P / "reports/step05_c2_prompt_aware_fusion_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    C1: "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    B2: "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path, expected in EXPECTED.items():
    assert path.exists(), f"Missing file: {path}"
    assert sha256(path) == expected, f"SHA mismatch: {path}"

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv = json.loads(CV.read_text(encoding="utf-8"))

assert cv["status"] == "frozen"
assert cv["speaker_disjoint"] is True
assert design["models"]["C2"]["feature_groups"] == [
    "acoustic", "linguistic", "prompt"
]

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(DATA)
split = read(SPLIT)
c1 = read(C1)
b2 = read(B2)

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]
FULL_KEYS = KEYS + ["fluency", "validation_fold"]

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]
ACOUSTIC = [
    x for x in data.columns
    if x not in KEYS + ["fluency"] + LING + PROMPT
]
FEATURES = ACOUSTIC + LING + PROMPT

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(PROMPT) == 2
assert len(FEATURES) == 23
assert len(set(FEATURES)) == 23
assert "fluency" not in FEATURES

assert len(data) == 2000
assert len(split) == 400
assert len(c1) == len(b2) == 2000

assert not data.duplicated(KEYS).any()
assert not split.duplicated(IDS).any()

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

ALPHA = 1.0
parts = []
fold_rows = []

for fold in range(1, 6):
    train = df[df["validation_fold"] != fold]
    val = df[df["validation_fold"] == fold]

    assert len(train) == 1600
    assert len(val) == 400
    assert train["audio_id"].nunique() == 320
    assert val["audio_id"].nunique() == 80
    assert train["speaker"].nunique() == 16
    assert val["speaker"].nunique() == 4

    assert set(train["speaker"]).isdisjoint(set(val["speaker"]))
    assert set(train["audio_id"]).isdisjoint(set(val["audio_id"]))

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=ALPHA))
    ])

    model.fit(
        train[FEATURES],
        train["fluency"].to_numpy(dtype=float)
    )

    yhat = model.predict(val[FEATURES])
    assert np.isfinite(yhat).all()

    result = val[FULL_KEYS].copy()
    result["c2_prediction"] = yhat
    parts.append(result)

    err = yhat - val["fluency"].to_numpy(dtype=float)

    fold_rows.append({
        "fold": fold,
        "train_speakers": 16,
        "validation_speakers": 4,
        "train_unique_audios": 320,
        "validation_unique_audios": 80,
        "validation_hypotheses": 400,
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2)))
    })

oof = pd.concat(parts, ignore_index=True)
oof = oof.sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(oof) == 2000
assert not oof.duplicated(KEYS).any()

# Strictly align with previously stored C1 and B2 predictions.
comparison = oof.merge(
    c1[FULL_KEYS + ["c1_prediction"]],
    on=FULL_KEYS,
    how="inner",
    validate="one_to_one"
).merge(
    b2[FULL_KEYS + ["b2_prediction"]],
    on=FULL_KEYS,
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 2000

def metrics(frame, prediction_column):
    error = (
        frame[prediction_column].to_numpy(dtype=float)
        - frame["fluency"].to_numpy(dtype=float)
    )
    return {
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error ** 2)))
    }

model_comparison = {
    "B2": metrics(comparison, "b2_prediction"),
    "C1": metrics(comparison, "c1_prediction"),
    "C2": metrics(comparison, "c2_prediction")
}

hypothesis_metrics = {
    hid: metrics(group, "c2_prediction")
    for hid, group in oof.groupby("hypothesis_id")
}

audio = oof.groupby(
    IDS + ["fluency", "validation_fold"],
    as_index=False
).agg(
    prediction_min=("c2_prediction", "min"),
    prediction_max=("c2_prediction", "max"),
    prediction_sd=("c2_prediction", "std"),
    prediction_mean=("c2_prediction", "mean"),
    hypothesis_count=("hypothesis_id", "count")
)

audio["prediction_range"] = (
    audio["prediction_max"] - audio["prediction_min"]
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

stability = {
    "mean_range": float(audio["prediction_range"].mean()),
    "median_range": float(audio["prediction_range"].median()),
    "p95_range": float(audio["prediction_range"].quantile(0.95)),
    "max_range": float(audio["prediction_range"].max()),
    "mean_prediction_sd": float(audio["prediction_sd"].mean()),
    "variable_audio_count": int(
        (audio["prediction_range"] > 1e-10).sum()
    )
}

CRITICAL_ID = "030210164"

critical_mask = (
    comparison["audio_id"].eq(CRITICAL_ID)
    & comparison["hypothesis_id"].eq("H2")
)

assert critical_mask.sum() == 1
critical = comparison.loc[critical_mask].iloc[0]

without_one = comparison.loc[~critical_mask]
without_audio = comparison[
    comparison["audio_id"] != CRITICAL_ID
]

sensitivity = {
    "without_one_H2": {
        name: metrics(without_one, f"{name.lower()}_prediction")
        for name in ["B2", "C1", "C2"]
    },
    "without_entire_audio": {
        name: metrics(without_audio, f"{name.lower()}_prediction")
        for name in ["B2", "C1", "C2"]
    }
}

c2_error = (
    comparison["c2_prediction"] - comparison["fluency"]
)
c2_critical_sse_share = float(
    (critical["c2_prediction"] - critical["fluency"]) ** 2
    / np.square(c2_error).sum()
)

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")

    if path.exists():
        assert sha256(path) == sha256(temp), (
            f"Existing result differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_csv(oof, PRED)
save_csv(pd.DataFrame(fold_rows), FOLDS)
save_csv(audio.sort_values("pilot_index"), AUDIO)

report = {
    "status": "PASS",
    "model": "C2_acoustic_linguistic_prompt_aware_ridge",
    "feature_count": 23,
    "acoustic_features": 14,
    "linguistic_features": 7,
    "prompt_features": 2,
    "alpha": ALPHA,
    "hyperparameter_tuning_performed": False,
    "prediction_clipping_performed": False,
    "speaker_disjoint_cv": "frozen_5fold",
    "unique_audio_count": 400,
    "OOF_predictions": 2000,
    "comparison": model_comparison,
    "per_hypothesis_metrics": hypothesis_metrics,
    "stability": stability,
    "critical_audio_H2": {
        "human_fluency": float(critical["fluency"]),
        "B2_prediction": float(critical["b2_prediction"]),
        "C1_prediction": float(critical["c1_prediction"]),
        "C2_prediction": float(critical["c2_prediction"]),
        "C2_squared_error_share": c2_critical_sse_share
    },
    "posthoc_sensitivity": sensitivity,
    "source_dataset_sha256": sha256(DATA),
    "frozen_split_sha256": sha256(SPLIT),
    "predictions_sha256": sha256(PRED),
    "fold_metrics_sha256": sha256(FOLDS),
    "audio_stability_sha256": sha256(AUDIO),
    "notes": (
        "Unclipped fixed-alpha Ridge predictions. "
        "Train-fold-only median imputation and scaling. "
        "Each audio contributes exactly five hypotheses. "
        "Prompt-ASR disagreement is not verified WER/CER. "
        "No primary-analysis observations excluded."
    )
}

if REPORT.exists():
    saved = json.loads(REPORT.read_text(encoding="utf-8"))
    assert saved == report, (
        "Existing C2 report differs; original preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step5_c2_prompt_aware_fusion"] = "qc_pass"
progress["step5_c2_predictions_sha256"] = sha256(PRED)
progress["next_task"] = (
    "Compare five fixed-alpha models and estimate "
    "speaker-clustered uncertainty for paired differences"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 5.9 C2 PROMPT-AWARE FUSION ===")
print("Status: PASS")
print("Features:", len(FEATURES))
print("Unique audios:", 400)
print("OOF predictions:", len(oof))
print("Speaker-disjoint CV: PASS")
print("Ridge alpha:", ALPHA)

print("\n=== B2 VS C1 VS C2 ACCURACY ===")
for name, m in model_comparison.items():
    print(
        f"{name}: MAE={m['MAE']:.4f}, "
        f"RMSE={m['RMSE']:.4f}"
    )

print("\n=== C2 H1-H5 METRICS ===")
for hid, m in sorted(hypothesis_metrics.items()):
    print(
        f"{hid}: MAE={m['MAE']:.4f}, "
        f"RMSE={m['RMSE']:.4f}"
    )

print("\n=== C2 PREDICTION STABILITY ===")
for key, value in stability.items():
    print(f"{key}: {value:.4f}")

print("\n=== C2 CRITICAL H2 CASE ===")
for key, value in report["critical_audio_H2"].items():
    print(f"{key}: {value}")

print("\n=== SENSITIVITY WITHOUT ONE H2 ===")
for name, m in sensitivity["without_one_H2"].items():
    print(
        f"{name}: MAE={m['MAE']:.4f}, "
        f"RMSE={m['RMSE']:.4f}"
    )

print("\n=== C2 PER-FOLD METRICS ===")
print(pd.DataFrame(fold_rows).round(4).to_string(index=False))

print("\nPrediction CSV:", PRED)
print("SHA256:", sha256(PRED))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
DIR = P / "results/step05"
CHECK = P / "checkpoints/research_progress.json"

SOURCES = {
    "A": DIR / "baseline_acoustic_oof_predictions.csv",
    "B1": DIR / "b1_transcript_ridge_oof_predictions.csv",
    "B2": DIR / "b2_prompt_aware_ridge_oof_predictions.csv",
    "C1": DIR / "c1_fusion_ridge_oof_predictions.csv",
    "C2": DIR / "c2_prompt_aware_fusion_ridge_oof_predictions.csv"
}

EXPECTED = {
    "A": "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
    "B1": "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    "B2": "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
    "C1": "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    "C2": "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

PRED_COLUMNS = {
    "A": "acoustic_ridge_prediction",
    "B1": "b1_prediction",
    "B2": "b2_prediction",
    "C1": "c1_prediction",
    "C2": "c2_prediction"
}

OUT = DIR / "speaker_cluster_bootstrap_10000_ci.csv"
REPORT = P / "reports/step06_speaker_cluster_bootstrap_v1.json"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for model, path in SOURCES.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == EXPECTED[model], (
        f"SHA mismatch for {model}"
    )

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

frames = {name: read(path) for name, path in SOURCES.items()}

KEYS = [
    "pilot_index", "audio_id", "speaker",
    "fluency", "validation_fold"
]
HKEYS = KEYS + ["hypothesis_id"]

assert len(frames["A"]) == 400

for name in ["B1", "B2", "C1", "C2"]:
    assert len(frames[name]) == 2000
    assert not frames[name].duplicated(HKEYS).any()

base = frames["B1"][
    HKEYS + [PRED_COLUMNS["B1"]]
].copy()

for name in ["B2", "C1", "C2"]:
    base = base.merge(
        frames[name][HKEYS + [PRED_COLUMNS[name]]],
        on=HKEYS,
        how="inner",
        validate="one_to_one"
    )

base = base.merge(
    frames["A"][KEYS + [PRED_COLUMNS["A"]]],
    on=KEYS,
    how="inner",
    validate="many_to_one"
)

assert len(base) == 2000
assert base["audio_id"].nunique() == 400
assert base["speaker"].nunique() == 20
assert base.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert base.groupby("speaker")["validation_fold"].nunique().eq(1).all()
assert base.groupby("speaker")["audio_id"].nunique().eq(20).all()

# One row per audio: average the five hypothesis errors
# BEFORE speaker-level resampling.


# Explicit error columns avoid ambiguity in aggregation.
for name, col in PRED_COLUMNS.items():
    base[f"{name}_ae"] = (
        base[col] - base["fluency"]
    ).abs()
    base[f"{name}_se"] = (
        base[col] - base["fluency"]
    ) ** 2

agg = {}
for name in PRED_COLUMNS:
    agg[f"{name}_ae"] = (f"{name}_ae", "mean")
    agg[f"{name}_se"] = (f"{name}_se", "mean")
    agg[f"{name}_min"] = (PRED_COLUMNS[name], "min")
    agg[f"{name}_max"] = (PRED_COLUMNS[name], "max")

audio = base.groupby(
    KEYS, as_index=False
).agg(**agg)

assert len(audio) == 400
assert audio["audio_id"].is_unique

for name in PRED_COLUMNS:
    audio[f"{name}_range"] = (
        audio[f"{name}_max"] - audio[f"{name}_min"]
    )

# A must be invariant to ASR hypotheses.
assert np.allclose(audio["A_range"], 0, atol=1e-12)

MODELS = ["A", "B1", "B2", "C1", "C2"]
SPEAKERS = sorted(audio["speaker"].unique())
assert len(SPEAKERS) == 20

# Each speaker has 20 audios, so equal speaker weights
# are also equal audio weights in this pilot.
speaker = audio.groupby("speaker").agg(
    **{
        f"{name}_{metric}": (f"{name}_{metric}", "mean")
        for name in MODELS
        for metric in ["ae", "se", "range"]
    }
).reindex(SPEAKERS)

assert speaker.shape[0] == 20
assert speaker.notna().all().all()

N_BOOT = 10000
SEED = 20261026
rng = np.random.default_rng(SEED)

# Matrix: iterations x sampled speakers.
indices = rng.integers(
    low=0,
    high=20,
    size=(N_BOOT, 20)
)

bootstrap = {}
observed = {}

for name in MODELS:
    for metric in ["ae", "se", "range"]:
        values = speaker[f"{name}_{metric}"].to_numpy(
            dtype=float
        )

        avg = values[indices].mean(axis=1)

        if metric == "se":
            bootstrap[(name, "RMSE")] = np.sqrt(avg)
            observed[(name, "RMSE")] = float(
                np.sqrt(values.mean())
            )
        elif metric == "ae":
            bootstrap[(name, "MAE")] = avg
            observed[(name, "MAE")] = float(values.mean())
        else:
            bootstrap[(name, "mean_range")] = avg
            observed[(name, "mean_range")] = float(
                values.mean()
            )

# Verify observed metrics agree with previous reports.
expected_metrics = {
    "A": (0.9399, 1.2066),
    "B1": (1.2426, 1.9972),
    "B2": (1.1761, 1.9518),
    "C1": (0.8972, 1.3735),
    "C2": (0.7782, 1.2461)
}

for name, (mae, rmse) in expected_metrics.items():
    assert np.isclose(observed[(name, "MAE")], mae, atol=0.0001)
    assert np.isclose(observed[(name, "RMSE")], rmse, atol=0.0001)

def ci(values):
    return [
        float(np.quantile(values, 0.025)),
        float(np.quantile(values, 0.975))
    ]

rows = []

for name in MODELS:
    for metric in ["MAE", "RMSE", "mean_range"]:
        low, high = ci(bootstrap[(name, metric)])
        rows.append({
            "comparison": name,
            "metric": metric,
            "observed": observed[(name, metric)],
            "ci_2_5": low,
            "ci_97_5": high
        })

# Paired differences are calculated within the SAME
# bootstrap speaker draws for both models.
PAIRINGS = [
    ("C1", "A"),
    ("C2", "A"),
    ("C2", "C1"),
    ("B2", "B1"),
    ("C1", "B1"),
    ("C2", "B2")
]

for model, reference in PAIRINGS:
    for metric in ["MAE", "RMSE", "mean_range"]:
        diff = (
            bootstrap[(model, metric)]
            - bootstrap[(reference, metric)]
        )
        low, high = ci(diff)

        rows.append({
            "comparison": f"{model}_minus_{reference}",
            "metric": metric,
            "observed": (
                observed[(model, metric)]
                - observed[(reference, metric)]
            ),
            "ci_2_5": low,
            "ci_97_5": high
        })

results = pd.DataFrame(rows)

assert len(results) == 33
assert results[["observed", "ci_2_5", "ci_97_5"]].notna().all().all()

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

tmp = OUT.with_suffix(".tmp")
results.to_csv(tmp, index=False, encoding="utf-8")

if OUT.exists():
    assert sha256(OUT) == sha256(tmp), (
        "Existing bootstrap CSV differs; preserved."
    )
    tmp.unlink()
else:
    tmp.replace(OUT)

report = {
    "status": "PASS",
    "method": "paired_speaker_cluster_percentile_bootstrap",
    "bootstrap_iterations": N_BOOT,
    "random_seed": SEED,
    "confidence_level": 0.95,
    "resampling_unit": "speaker",
    "unique_speakers": 20,
    "unique_audios": 400,
    "hypotheses_per_audio": 5,
    "source_split": "frozen_5fold_speaker_disjoint",
    "all_oof_predictions_unchanged": True,
    "all_outliers_retained": True,
    "refitting_during_bootstrap": False,
    "models": MODELS,
    "comparison_pairs": [
        f"{m}_minus_{r}" for m, r in PAIRINGS
    ],
    "results_sha256": sha256(OUT),
    "source_sha256": {
        name: sha256(path)
        for name, path in SOURCES.items()
    },
    "interpretation": (
        "Paired speaker-cluster bootstrap CIs conditional "
        "on previously fitted OOF models; not nested-CV "
        "or refitting uncertainty. "
        "Negative error difference favors the first model. "
        "Bootstrap intervals are descriptive, especially "
        "with only 20 speaker clusters."
    )
}

if REPORT.exists():
    old = json.loads(REPORT.read_text(encoding="utf-8"))
    assert old == report, (
        "Existing bootstrap report differs; preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_speaker_cluster_bootstrap"] = "qc_pass"
progress["step6_bootstrap_results_sha256"] = sha256(OUT)
progress["next_task"] = (
    "Review speaker-cluster bootstrap uncertainty; "
    "perform robustness and final model comparison"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.1 SPEAKER-CLUSTER BOOTSTRAP ===")
print("Status: PASS")
print("Bootstrap iterations:", N_BOOT)
print("Speaker clusters:", len(SPEAKERS))
print("Confidence interval: 95%")
print("Refitting: NO")
print("All 400 audios retained: YES")

print("\n=== MODEL ACCURACY AND 95% CI ===")
print(
    results[
        results["comparison"].isin(MODELS) &
        results["metric"].isin(["MAE", "RMSE"])
    ].round(4).to_string(index=False)
)

print("\n=== PAIRED MODEL DIFFERENCES AND 95% CI ===")
print(
    results[
        results["comparison"].str.contains("_minus_") &
        results["metric"].isin(["MAE", "RMSE"])
    ].round(4).to_string(index=False)
)

print("\n=== PREDICTION STABILITY AND 95% CI ===")
print(
    results[results["metric"] == "mean_range"]
    .round(4).to_string(index=False)
)

print("\nCSV:", OUT)
print("SHA256:", sha256(OUT))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile
import os

ROOT = Path("/kaggle/working/acoustic_linguistic_research")
WORK = Path("/kaggle/working")
ZIP = WORK / "asr_fluency_step01-05_complete_with_step06_bootstrap.zip"

assert ROOT.is_dir(), "Research project directory missing"

EXPECTED_CRITICAL = {
    "data/modeling/modeling_dataset_0001-0400.csv":
        "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    "data/splits/speaker_groupkfold5_frozen.csv":
        "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    "configs/speaker_disjoint_cv_v1.json":
        "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    "features/asr/full/asr_hypotheses_0001-0400.csv":
        "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80",
    "results/step05/baseline_acoustic_oof_predictions.csv":
        "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
    "results/step05/b1_transcript_ridge_oof_predictions.csv":
        "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    "results/step05/b2_prompt_aware_ridge_oof_predictions.csv":
        "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
    "results/step05/c1_fusion_ridge_oof_predictions.csv":
        "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv":
        "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9",
    "results/step05/speaker_cluster_bootstrap_10000_ci.csv":
        "5ef3d6dd01eb05758ccb8b4f77293396a81e988b9b9e33152cb666c6b48d1d93"
}

def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=== VERIFY CRITICAL RESEARCH FILES ===")

for rel, expected in EXPECTED_CRITICAL.items():
    path = ROOT / rel
    assert path.is_file(), f"CRITICAL FILE MISSING: {rel}"
    assert digest(path) == expected, f"SHA MISMATCH: {rel}"
    print("PASS:", rel)

MAX_RAW_PARQUET_BYTES = 250 * 1024 * 1024
INCLUDED = []
OMITTED = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path.is_symlink():
        continue

    rel = path.relative_to(ROOT)
    rel_str = rel.as_posix()
    size = path.stat().st_size

    # Never package sealed official test data.
    if (
        "speechocean762_test" in rel_str.lower()
        or "official_test" in rel_str.lower()
    ):
        OMITTED.append((path, "sealed official test excluded"))
        continue

    # Raw parquet can be large; keep its SHA even if omitted.
    if (
        rel_str.startswith("data/raw/")
        and path.suffix.lower() == ".parquet"
        and size > MAX_RAW_PARQUET_BYTES
    ):
        OMITTED.append((path, "raw parquet exceeds 250 MB"))
        continue

    # Exclude temporary/cache files, not research deliverables.
    if (
        "__pycache__" in path.parts
        or ".ipynb_checkpoints" in path.parts
        or path.suffix in [".tmp", ".pyc"]
    ):
        OMITTED.append((path, "temporary or cache file"))
        continue

    INCLUDED.append((path, f"project/{rel_str}"))

# Add STEP 3 complete checkpoint, if available.
STEP3 = WORK / "asr_fluency_step03_complete.zip"
if STEP3.is_file():
    INCLUDED.append(
        (STEP3, "extra/asr_fluency_step03_complete.zip")
    )

# Package research source code/notebooks if saved separately.
for path in sorted(WORK.iterdir()):
    if (
        path.is_file()
        and path.suffix.lower() in [".py", ".ipynb", ".r", ".R"]
        and path != ZIP
    ):
        arc = f"extra/source_files/{path.name}"
        if arc not in [a for _, a in INCLUDED]:
            INCLUDED.append((path, arc))

assert INCLUDED
assert len({arc for _, arc in INCLUDED}) == len(INCLUDED)

entries = []
for path, arc in INCLUDED:
    entries.append({
        "archive_path": arc,
        "size_bytes": path.stat().st_size,
        "sha256": digest(path)
    })

omitted_entries = []
for path, reason in OMITTED:
    omitted_entries.append({
        "original_path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": digest(path),
        "reason": reason
    })

manifest = {
    "archive_purpose": "Restore PhD ASR fluency research through STEP 5",
    "step05_status": "COMPLETE",
    "step06_1_bootstrap_status": "COMPLETE",
    "step06_2_sensitivity_status": "NOT YET CONFIRMED",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "mispeech/speechocean762",
    "dataset_revision": "06385584fad212b26134c656fdd3ccf9f093f33e",
    "asr_model": "openai/whisper-small.en",
    "asr_model_revision": "e8727524f962ee844a7319d92be39ac1bd25655a",
    "unique_audio_count": 400,
    "speaker_count": 20,
    "asr_hypothesis_count": 2000,
    "cv": "frozen 5-fold speaker-disjoint",
    "ridge_alpha": 1.0,
    "official_test_used": False,
    "critical_file_hashes": EXPECTED_CRITICAL,
    "included_files": entries,
    "omitted_files": omitted_entries
}

restore = """ASR FLUENCY RESEARCH — RESTORE GUIDE

1. Upload this ZIP to a new Kaggle session.
2. Extract its project/ contents into:
   /kaggle/working/acoustic_linguistic_research/
3. Verify each restored file using MANIFEST.json SHA256 values.
4. STEP 5 is COMPLETE. Do not rerun its training cells.
5. STEP 6.1 bootstrap results are included.
6. Continue with STEP 6.2 anomaly sensitivity analysis.
7. Restore raw SpeechOcean762 train parquet separately if absent.
8. The official sealed test partition must remain untouched.
9. Original ASR model weights are not required to analyze
   already saved transcripts and OOF predictions.
10. Kaggle notebook cell history is not automatically captured
    by this archive unless a notebook/script exists as a file.
    Export the Kaggle notebook separately for full reproducibility.

IMPORTANT:
- Reading prompts are NOT verified ground-truth transcripts.
- Prompt-ASR disagreement is NOT WER or CER.
- All primary results retain the 400 selected audios.
- Sensitivity exclusions are diagnostic only.
- All predictions are from fixed-alpha Ridge (alpha=1.0).
- Review MANIFEST.json for any excluded large files.
"""

tmp_zip = ZIP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    tmp_zip, "w", compression=zipfile.ZIP_DEFLATED,
    compresslevel=6, allowZip64=True
) as z:
    for path, arc in INCLUDED:
        z.write(path, arcname=arc)

    z.writestr(
        "MANIFEST.json",
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )
    z.writestr("RESTORE_INSTRUCTIONS.txt", restore)

# Verify archive CRC and SHA256 for every included file.
with zipfile.ZipFile(tmp_zip, "r") as z:
    assert z.testzip() is None, "ZIP CRC ERROR"
    assert "MANIFEST.json" in z.namelist()
    assert "RESTORE_INSTRUCTIONS.txt" in z.namelist()

    for entry in entries:
        h = hashlib.sha256()
        with z.open(entry["archive_path"]) as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        assert h.hexdigest() == entry["sha256"], (
            "ZIP CONTENT HASH MISMATCH: "
            + entry["archive_path"]
        )

if ZIP.exists():
    assert digest(ZIP) == digest(tmp_zip), (
        "An archive with this name already exists and differs. "
        "Original archive preserved."
    )
    tmp_zip.unlink()
else:
    os.replace(tmp_zip, ZIP)

print("\n=== STEP 5 RESEARCH BACKUP COMPLETE ===")
print("Status: PASS")
print("STEP 5: COMPLETE")
print("STEP 6.1: COMPLETE")
print("Files included:", len(entries))
print("Files omitted:", len(omitted_entries))
print("STEP 3 archive included:", STEP3.is_file())
print("ZIP integrity: PASS")
print("All included SHA256 checks: PASS")
print("ZIP:", ZIP)
print("Size (MB):", round(ZIP.stat().st_size / 1024**2, 2))
print("ZIP SHA256:", digest(ZIP))

print("\n=== OMITTED FILES ===")
for item in omitted_entries:
    print(item["reason"], "|", item["original_path"])

print("\nNEXT: Download this ZIP and save it outside Kaggle.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
CHECK = P / "checkpoints/research_progress.json"

SOURCES = {
    "A": ("baseline_acoustic_oof_predictions.csv",
          "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
          "acoustic_ridge_prediction"),
    "B1": ("b1_transcript_ridge_oof_predictions.csv",
           "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
           "b1_prediction"),
    "B2": ("b2_prompt_aware_ridge_oof_predictions.csv",
           "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
           "b2_prediction"),
    "C1": ("c1_fusion_ridge_oof_predictions.csv",
           "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
           "c1_prediction"),
    "C2": ("c2_prompt_aware_fusion_ridge_oof_predictions.csv",
           "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9",
           "c2_prediction")
}

AUDIT = P / "reports/step04_anomaly_classification_v1.csv"
AUDIT_SHA = "80122fe23528b9a221836ac1f738876a1ae503baa8dd3354cce329952e8952b0"

OUT = R / "step06_anomaly_sensitivity_400_399_396.csv"
PAIRED = R / "step06_anomaly_sensitivity_paired_differences.csv"
REPORT = P / "reports/step06_anomaly_sensitivity_v1.json"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

assert sha256(AUDIT) == AUDIT_SHA

audit = read(AUDIT)

EXPECTED_CANDIDATES = {
    "030210164", "096340009",
    "096340012", "096340017"
}

# Verify the four previously declared anomaly candidates.
possible_flags = [
    c for c in audit.columns
    if "candidate" in c.lower()
    and ("sensitivity" in c.lower() or "anomaly" in c.lower())
]

if "sensitivity_analysis_candidate" in audit.columns:
    flag = "sensitivity_analysis_candidate"
elif len(possible_flags) == 1:
    flag = possible_flags[0]
else:
    raise ValueError(
        f"Review audit flag column: {audit.columns.tolist()}"
    )

values = audit[flag].astype(str).str.strip().str.lower()

candidate_ids = set(
    audit.loc[
        values.isin(["true", "1", "yes"]),
        "audio_id"
    ].astype(str)
)

assert candidate_ids == EXPECTED_CANDIDATES, (
    f"Audit candidates mismatch: {candidate_ids}"
)

KEYS = [
    "pilot_index", "audio_id", "speaker",
    "fluency", "validation_fold"
]
HKEYS = KEYS + ["hypothesis_id"]

frames = {}

for name, (filename, expected, col) in SOURCES.items():
    path = R / filename
    assert sha256(path) == expected, f"SHA mismatch: {name}"
    frames[name] = read(path)

base = frames["B1"][
    HKEYS + [SOURCES["B1"][2]]
].copy()

for name in ["B2", "C1", "C2"]:
    base = base.merge(
        frames[name][HKEYS + [SOURCES[name][2]]],
        on=HKEYS,
        validate="one_to_one"
    )

base = base.merge(
    frames["A"][KEYS + [SOURCES["A"][2]]],
    on=KEYS,
    validate="many_to_one"
)

assert len(base) == 2000
assert base["audio_id"].nunique() == 400
assert base.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

MODELS = list(SOURCES)
aggregations = {}

for name in MODELS:
    pred_col = SOURCES[name][2]

    base[f"{name}_ae"] = (
        base[pred_col] - base["fluency"]
    ).abs()

    base[f"{name}_se"] = (
        base[pred_col] - base["fluency"]
    ) ** 2

    aggregations[f"{name}_ae"] = (f"{name}_ae", "mean")
    aggregations[f"{name}_se"] = (f"{name}_se", "mean")
    aggregations[f"{name}_min"] = (pred_col, "min")
    aggregations[f"{name}_max"] = (pred_col, "max")

audio = base.groupby(
    KEYS, as_index=False
).agg(**aggregations)

assert len(audio) == 400
assert audio["audio_id"].is_unique

for name in MODELS:
    audio[f"{name}_range"] = (
        audio[f"{name}_max"] - audio[f"{name}_min"]
    )

assert np.allclose(audio["A_range"], 0, atol=1e-12)

CONDITIONS = {
    "primary_all_400": set(),
    "diagnostic_without_critical_399": {"030210164"},
    "diagnostic_without_4_candidates_396": candidate_ids
}

rows = []
pairs = []

PAIRINGS = [
    ("C2", "A"),
    ("C2", "C1"),
    ("B2", "B1")
]

for condition, excluded in CONDITIONS.items():
    sub = audio.loc[
        ~audio["audio_id"].isin(excluded)
    ].copy()

    assert len(sub) == 400 - len(excluded)

    for name in MODELS:
        rows.append({
            "condition": condition,
            "model": name,
            "audio_count": len(sub),
            "hypothesis_count": len(sub) * 5,
            "speaker_count": sub["speaker"].nunique(),
            "MAE": float(sub[f"{name}_ae"].mean()),
            "RMSE": float(
                np.sqrt(sub[f"{name}_se"].mean())
            ),
            "mean_prediction_range": float(
                sub[f"{name}_range"].mean()
            ),
            "median_prediction_range": float(
                sub[f"{name}_range"].median()
            ),
            "p95_prediction_range": float(
                sub[f"{name}_range"].quantile(0.95)
            )
        })

    for left, right in PAIRINGS:
        pairs.append({
            "condition": condition,
            "comparison": f"{left}_minus_{right}",
            "MAE_difference": float(
                (sub[f"{left}_ae"] -
                 sub[f"{right}_ae"]).mean()
            ),
            "mean_range_difference": float(
                (sub[f"{left}_range"] -
                 sub[f"{right}_range"]).mean()
            )
        })

result = pd.DataFrame(rows)
paired = pd.DataFrame(pairs)

assert len(result) == 15
assert len(paired) == 9

EXPECTED_PRIMARY = {
    "A": (0.9399, 1.2066),
    "B1": (1.2426, 1.9972),
    "B2": (1.1761, 1.9518),
    "C1": (0.8972, 1.3735),
    "C2": (0.7782, 1.2461)
}

for name, (mae, rmse) in EXPECTED_PRIMARY.items():
    row = result.loc[
        result["condition"].eq("primary_all_400") &
        result["model"].eq(name)
    ].iloc[0]

    assert np.isclose(row["MAE"], mae, atol=1e-4)
    assert np.isclose(row["RMSE"], rmse, atol=1e-4)

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")

    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing file differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

safe_csv(result, OUT)
safe_csv(paired, PAIRED)

report = {
    "status": "PASS",
    "analysis": "preflagged_asr_anomaly_sensitivity",
    "primary_audio_count": 400,
    "primary_observations_retained": True,
    "refitting_performed": False,
    "candidate_audio_ids": sorted(candidate_ids),
    "audit_sha256": sha256(AUDIT),
    "conditions": {
        name: {
            "excluded_audio_ids": sorted(ids),
            "remaining_audio_count": 400 - len(ids)
        }
        for name, ids in CONDITIONS.items()
    },
    "main_results_sha256": sha256(OUT),
    "paired_results_sha256": sha256(PAIRED),
    "source_prediction_sha256": {
        name: sha256(R / spec[0])
        for name, spec in SOURCES.items()
    },
    "interpretation": (
        "Post-hoc sensitivity analysis using previously "
        "flagged ASR anomalies. No retraining and no "
        "primary-analysis exclusions. Reading prompt "
        "disagreement is not verified WER or CER."
    )
}

if REPORT.exists():
    old = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert old == report, (
        "Existing sensitivity report differs."
    )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step6_anomaly_sensitivity"] = "qc_pass"
progress["step6_anomaly_sensitivity_sha256"] = sha256(OUT)
progress["next_task"] = (
    "Prepare integrated final model, uncertainty and "
    "robustness result tables"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.2 ANOMALY SENSITIVITY AUDIT ===")
print("Status: PASS")
print("Pre-flagged candidates:", sorted(candidate_ids))
print("Primary observations removed: NONE")
print("Model refitting: NO")

print("\n=== PRIMARY VS SENSITIVITY METRICS ===")
print(
    result[
        [
            "condition", "model", "audio_count",
            "MAE", "RMSE", "mean_prediction_range"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== PAIRED MODEL DIFFERENCES ===")
print(paired.round(4).to_string(index=False))

print("\nResults:", OUT)
print("SHA256:", sha256(OUT))
print("Paired results:", PAIRED)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
FINAL = P / "results/step06_final_tables"
CHECK = P / "checkpoints/research_progress.json"

BOOT = R / "speaker_cluster_bootstrap_10000_ci.csv"
SENS = R / "step06_anomaly_sensitivity_400_399_396.csv"
PAIRED = R / "step06_anomaly_sensitivity_paired_differences.csv"
SENS_REPORT = P / "reports/step06_anomaly_sensitivity_v1.json"

EXPECTED_BOOT = (
    "5ef3d6dd01eb05758ccb8b4f77293396a81e988b9b9e33152cb666c6b48d1d93"
)
EXPECTED_SENS = (
    "83b34da4aa34634319b59c94391967565e3f3c0fd438fb5a80d48131ca5563d3"
)

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert sha(BOOT) == EXPECTED_BOOT
assert sha(SENS) == EXPECTED_SENS

sens_report = json.loads(
    SENS_REPORT.read_text(encoding="utf-8")
)
assert sens_report["status"] == "PASS"
assert sens_report["main_results_sha256"] == sha(SENS)
assert sens_report["paired_results_sha256"] == sha(PAIRED)

boot = pd.read_csv(BOOT)
sens = pd.read_csv(SENS)
paired = pd.read_csv(PAIRED)

MODELS = ["A", "B1", "B2", "C1", "C2"]
METRICS = ["MAE", "RMSE", "mean_range"]

assert len(boot) == 33
assert len(sens) == 15
assert len(paired) == 9

# 1. Primary model metrics with cluster-bootstrap CIs.
primary = sens.loc[
    sens["condition"] == "primary_all_400"
].set_index("model")

assert set(primary.index) == set(MODELS)

def get_boot(name, metric):
    row = boot.loc[
        boot["comparison"].eq(name)
        & boot["metric"].eq(metric)
    ]
    assert len(row) == 1
    return row.iloc[0]

main_rows = []

feature_counts = {
    "A": 14,
    "B1": 7,
    "B2": 9,
    "C1": 21,
    "C2": 23
}

for model in MODELS:
    row = primary.loc[model]

    mae = get_boot(model, "MAE")
    rmse = get_boot(model, "RMSE")
    stability = get_boot(model, "mean_range")

    assert np.isclose(
        float(row["MAE"]), float(mae["observed"]),
        atol=1e-10
    )
    assert np.isclose(
        float(row["RMSE"]), float(rmse["observed"]),
        atol=1e-10
    )
    assert np.isclose(
        float(row["mean_prediction_range"]),
        float(stability["observed"]),
        atol=1e-10
    )

    main_rows.append({
        "model": model,
        "feature_count": feature_counts[model],
        "MAE": float(mae["observed"]),
        "MAE_CI_low": float(mae["ci_2_5"]),
        "MAE_CI_high": float(mae["ci_97_5"]),
        "RMSE": float(rmse["observed"]),
        "RMSE_CI_low": float(rmse["ci_2_5"]),
        "RMSE_CI_high": float(rmse["ci_97_5"]),
        "mean_prediction_range": float(
            stability["observed"]
        ),
        "range_CI_low": float(stability["ci_2_5"]),
        "range_CI_high": float(stability["ci_97_5"])
    })

main_table = pd.DataFrame(main_rows)

# 2. Paired differences and bootstrap CIs.
COMPARISONS = [
    "C1_minus_A",
    "C2_minus_A",
    "C2_minus_C1",
    "B2_minus_B1",
    "C1_minus_B1",
    "C2_minus_B2"
]

comparison_rows = []

for name in COMPARISONS:
    for metric in METRICS:
        row = get_boot(name, metric)

        comparison_rows.append({
            "comparison": name,
            "metric": metric,
            "difference": float(row["observed"]),
            "CI_low": float(row["ci_2_5"]),
            "CI_high": float(row["ci_97_5"]),
            "CI_excludes_zero": bool(
                float(row["ci_2_5"]) > 0
                or float(row["ci_97_5"]) < 0
            )
        })

comparison_table = pd.DataFrame(comparison_rows)
assert len(comparison_table) == 18

# 3. Sensitivity matrix, unchanged from source.
condition_order = [
    "primary_all_400",
    "diagnostic_without_critical_399",
    "diagnostic_without_4_candidates_396"
]

sensitivity_table = sens.copy()
sensitivity_table["condition"] = pd.Categorical(
    sensitivity_table["condition"],
    categories=condition_order,
    ordered=True
)
sensitivity_table["model"] = pd.Categorical(
    sensitivity_table["model"],
    categories=MODELS,
    ordered=True
)
sensitivity_table = sensitivity_table.sort_values(
    ["condition", "model"]
).reset_index(drop=True)

assert len(sensitivity_table) == 15

# 4. Save deterministic, immutable results.
FINAL.mkdir(parents=True, exist_ok=True)

OUTPUTS = {
    "table01_primary_models_with_ci.csv": main_table,
    "table02_paired_differences_with_ci.csv": comparison_table,
    "table03_anomaly_sensitivity.csv": sensitivity_table,
    "table04_sensitivity_paired_differences.csv": paired
}

def safe_save(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp,
        index=False,
        encoding="utf-8",
        float_format="%.12g"
    )

    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing output differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

for filename, frame in OUTPUTS.items():
    safe_save(frame, FINAL / filename)

manifest = {
    "status": "PASS",
    "analysis": "integrated_fixed_alpha_ridge_results",
    "unique_audios": 400,
    "unique_speakers": 20,
    "asr_hypotheses": 2000,
    "models": MODELS,
    "speaker_disjoint_cv": "frozen_5fold",
    "bootstrap_clusters": 20,
    "bootstrap_iterations": 10000,
    "confidence_interval_method": (
        "paired_speaker_cluster_percentile_bootstrap"
    ),
    "all_primary_observations_retained": True,
    "model_refitting": False,
    "hyperparameter_tuning": False,
    "ridge_alpha": 1.0,
    "original_source_sha256": {
        "bootstrap": sha(BOOT),
        "sensitivity": sha(SENS),
        "sensitivity_paired": sha(PAIRED),
        "sensitivity_report": sha(SENS_REPORT)
    },
    "generated_tables_sha256": {
        filename: sha(FINAL / filename)
        for filename in OUTPUTS
    },
    "limitations": [
        "Pilot sample contains only 20 speakers.",
        "Bootstrap intervals are conditional on saved OOF predictions.",
        "ASR hypotheses from one audio are correlated.",
        "Prompt-ASR disagreement is not verified WER/CER.",
        "Reading prompts are not verified verbatim transcripts.",
        "Sensitivity exclusions are diagnostic, not primary.",
        "Official test split remains untouched.",
        "Results use fixed alpha=1.0 without hyperparameter tuning."
    ]
}

MANIFEST = FINAL / "step06_integrated_results_manifest.json"

if MANIFEST.exists():
    old = json.loads(
        MANIFEST.read_text(encoding="utf-8")
    )
    assert old == manifest, (
        "Existing manifest differs; preserved."
    )
else:
    tmp = MANIFEST.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(MANIFEST)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step6_integrated_results"] = "qc_pass"
progress["step6_integrated_manifest_sha256"] = sha(MANIFEST)
progress["next_task"] = (
    "Review consolidated results, evaluate remaining "
    "methodological limitations and prepare research reporting"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()
CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.3 INTEGRATED RESULTS ===")
print("Status: PASS")
print("Models:", len(MODELS))
print("Primary audios:", 400)
print("Speakers:", 20)
print("Primary observations excluded: NONE")

print("\n=== TABLE 1: PRIMARY MODELS WITH 95% CI ===")
print(main_table.round(4).to_string(index=False))

print("\n=== TABLE 2: PAIRED DIFFERENCES WITH 95% CI ===")
print(
    comparison_table.round(4).to_string(index=False)
)

print("\n=== TABLE 3: ANOMALY SENSITIVITY ===")
print(
    sensitivity_table[
        [
            "condition", "model", "audio_count",
            "MAE", "RMSE", "mean_prediction_range"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== SAVED RESEARCH TABLES ===")
for filename in OUTPUTS:
    path = FINAL / filename
    print(filename, "| SHA256:", sha(path))

print("\nManifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Checkpoint updated.")


In [ ]:

import hashlib
import json
import re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
CHECK = P / "checkpoints/research_progress.json"

ASR = P / "features/asr/full/asr_hypotheses_0001-0400.csv"

FILES = {
    "B1": R / "b1_transcript_ridge_oof_predictions.csv",
    "B2": R / "b2_prompt_aware_ridge_oof_predictions.csv",
    "C1": R / "c1_fusion_ridge_oof_predictions.csv",
    "C2": R / "c2_prompt_aware_fusion_ridge_oof_predictions.csv"
}

EXPECTED = {
    ASR: "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80",
    FILES["B1"]: "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    FILES["B2"]: "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
    FILES["C1"]: "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    FILES["C2"]: "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

OUT = P / "results/step06_final_tables/step06_asr_variability_per_audio.csv"
SUMMARY = P / "results/step06_final_tables/step06_variability_association.csv"
REPORT = P / "reports/step06_asr_variability_association_v1.json"

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

asr = read(ASR)
assert len(asr) == 2000

TOKEN_PATTERN = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE
)

def lexical_normalize(text):
    return " ".join(
        w.casefold() for w in TOKEN_PATTERN.findall(str(text))
    )

asr["lexical_text"] = asr["transcription"].map(lexical_normalize)
asr["raw_text"] = asr["transcription"].astype(str).str.strip()

ID = ["pilot_index", "audio_id", "speaker"]
KEYS = ID + ["hypothesis_id"]

assert not asr.duplicated(KEYS).any()
assert asr.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

audio = asr.groupby(ID, as_index=False).agg(
    raw_unique=("raw_text", "nunique"),
    lexical_unique=("lexical_text", "nunique"),
    hypothesis_count=("hypothesis_id", "count")
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

audio["raw_variable"] = audio["raw_unique"] > 1
audio["lexical_variable"] = audio["lexical_unique"] > 1

PRED_COLS = {
    "B1": "b1_prediction",
    "B2": "b2_prediction",
    "C1": "c1_prediction",
    "C2": "c2_prediction"
}

for model, path in FILES.items():
    df = read(path)
    col = PRED_COLS[model]
    df[col] = pd.to_numeric(df[col], errors="raise")

    assert len(df) == 2000
    assert not df.duplicated(KEYS).any()

    stat = df.groupby(ID, as_index=False).agg(
        pred_min=(col, "min"),
        pred_max=(col, "max"),
        pred_sd=(col, "std"),
        rows=(col, "count")
    )
    assert len(stat) == 400
    assert stat["rows"].eq(5).all()

    stat[f"{model}_range"] = stat["pred_max"] - stat["pred_min"]
    stat[f"{model}_sd"] = stat["pred_sd"]

    audio = audio.merge(
        stat[ID + [f"{model}_range", f"{model}_sd"]],
        on=ID,
        validate="one_to_one"
    )

assert len(audio) == 400
assert audio["speaker"].nunique() == 20

# Independently reproduce previously reported counts.
assert int(audio["raw_variable"].sum()) == 224
assert int(audio["lexical_variable"].sum()) == 172

observations = []

for model in PRED_COLS:
    col = f"{model}_range"

    rho, naive_p = spearmanr(
        audio["lexical_unique"].to_numpy(),
        audio[col].to_numpy()
    )

    group_means = audio.groupby(
        "lexical_variable"
    )[col].mean()

    observations.append({
        "model": model,
        "n_audios": 400,
        "spearman_rho_lexical_unique_vs_range": float(rho),
        "naive_p_not_for_inference": float(naive_p),
        "mean_range_lexically_invariant": float(
            group_means.loc[False]
        ),
        "mean_range_lexically_variable": float(
            group_means.loc[True]
        ),
        "mean_range_all": float(audio[col].mean()),
        "nonzero_range_audios": int(
            (audio[col] > 1e-10).sum()
        )
    })

summary = pd.DataFrame(observations)

# Invariant normalized transcripts should have exactly
# invariant linguistic features and model predictions.
# Verify this structural expectation; any violation
# requires investigation.
invariant = audio.loc[~audio["lexical_variable"]]

for model in PRED_COLS:
    assert np.allclose(
        invariant[f"{model}_range"],
        0,
        atol=1e-9
    ), f"Unexpected variability in {model} for invariant text"

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(audio.sort_values("pilot_index"), OUT)
safe_csv(summary, SUMMARY)

report = {
    "status": "PASS",
    "analysis": "asr_lexical_variability_vs_prediction_range",
    "audio_count": 400,
    "speaker_count": 20,
    "raw_variable_audios": int(audio["raw_variable"].sum()),
    "lexically_variable_audios": int(
        audio["lexical_variable"].sum()
    ),
    "correlation_type": "Spearman descriptive",
    "naive_p_values_for_inference": False,
    "speaker_clustered_uncertainty_estimated": False,
    "models_retrained": False,
    "all_original_observations_retained": True,
    "source_sha256": {
        str(path.relative_to(P)): sha(path)
        for path in EXPECTED
    },
    "audio_csv_sha256": sha(OUT),
    "summary_csv_sha256": sha(SUMMARY),
    "interpretation": (
        "Observed association, not causal evidence. "
        "Zero prediction range for lexically invariant "
        "hypotheses is expected from deterministic features. "
        "Speaker-clustered uncertainty is still required."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_asr_variability_association"] = "qc_pass"
progress["step6_asr_variability_association_sha256"] = sha(SUMMARY)
progress["next_task"] = (
    "Review ASR variability associations and assess "
    "speaker-clustered uncertainty and interpretation"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.4 ASR VARIABILITY ASSOCIATION ===")
print("Status: PASS")
print("Audios:", len(audio))
print("Speakers:", audio["speaker"].nunique())
print("Raw-variable audios:", int(audio["raw_variable"].sum()))
print("Lexically-variable audios:", int(audio["lexical_variable"].sum()))
print("Model refitting: NO")

print("\n=== VARIABILITY VS PREDICTION RANGE ===")
print(summary.round(4).to_string(index=False))

print("\n=== LEXICAL UNIQUENESS DISTRIBUTION ===")
print(
    audio["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n=== TOP 10 C2 PREDICTION RANGES ===")
print(
    audio.nlargest(10, "C2_range")[
        ["audio_id", "speaker", "lexical_unique",
         "B1_range", "C1_range", "C2_range"]
    ].round(4).to_string(index=False)
)

print("\nAudio CSV:", OUT)
print("Summary CSV:", SUMMARY)
print("Report:", REPORT)
print("Checkpoint updated.")



In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = P / "results/step06_final_tables/step06_asr_variability_per_audio.csv"
CHECK = P / "checkpoints/research_progress.json"

OUT = P / "results/step06_final_tables/step06_variable_subset_bootstrap.csv"
REPORT = P / "reports/step06_variable_subset_bootstrap_v1.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert SOURCE.is_file()
df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 400
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20

MODELS = ["B1", "B2", "C1", "C2"]
for model in MODELS:
    assert f"{model}_range" in df.columns

assert "lexical_unique" in df.columns

# The main analysis is restricted to genuinely
# variable normalized ASR hypotheses.
variable = df.loc[df["lexical_unique"] > 1].copy()

assert len(variable) == 172
assert variable["audio_id"].is_unique

speaker_ids = sorted(df["speaker"].astype(str).unique())
assert len(speaker_ids) == 20

variable_speakers = sorted(variable["speaker"].astype(str).unique())

# Speaker bootstrap of the complete observed variable
# subset, keeping all selected audios of a speaker together.
groups = {
    s: variable.loc[
        variable["speaker"] == s
    ].copy()
    for s in speaker_ids
}

N_BOOT = 10000
SEED = 20261026
rng = np.random.default_rng(SEED)

draws = rng.integers(0, len(speaker_ids), size=(N_BOOT, 20))

ANOMALIES = {
    "030210164",
    "096340009",
    "096340012",
    "096340017"
}

def rho_or_nan(frame, model):
    x = frame["lexical_unique"].to_numpy(dtype=float)
    y = frame[f"{model}_range"].to_numpy(dtype=float)

    if len(frame) < 5:
        return float("nan")

    if np.unique(x).size < 2 or np.unique(y).size < 2:
        return float("nan")

    return float(spearmanr(x, y).statistic)

rows = []

for model in MODELS:

    observed = rho_or_nan(variable, model)
    assert np.isfinite(observed)

    samples = []

    for draw in draws:
        pieces = [
            groups[speaker_ids[int(i)]]
            for i in draw
            if len(groups[speaker_ids[int(i)]]) > 0
        ]

        if not pieces:
            continue

        sample = pd.concat(pieces, ignore_index=True)
        r = rho_or_nan(sample, model)

        if np.isfinite(r):
            samples.append(r)

    samples = np.asarray(samples, dtype=float)
    assert len(samples) >= 9500, (
        f"Too few valid bootstrap draws: {model}"
    )

    filtered = variable.loc[
        ~variable["audio_id"].isin(ANOMALIES)
    ]

    diagnostic_rho = rho_or_nan(filtered, model)

    rows.append({
        "model": model,
        "subset": "lexically_variable_172",
        "unique_audios": len(variable),
        "speaker_clusters_present": len(variable_speakers),
        "observed_spearman_rho": observed,
        "bootstrap_ci_low": float(
            np.quantile(samples, 0.025)
        ),
        "bootstrap_ci_high": float(
            np.quantile(samples, 0.975)
        ),
        "valid_bootstrap_draws": len(samples),
        "mean_prediction_range": float(
            variable[f"{model}_range"].mean()
        ),
        "diagnostic_n_without_4": len(filtered),
        "diagnostic_rho_without_4": diagnostic_rho
    })

results = pd.DataFrame(rows)

assert len(results) == 4
assert results["observed_spearman_rho"].notna().all()

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

temp = OUT.with_suffix(".tmp")
results.to_csv(
    temp,
    index=False,
    encoding="utf-8",
    float_format="%.12g"
)

if OUT.exists():
    assert sha(OUT) == sha(temp), (
        "Existing results differ; original preserved."
    )
    temp.unlink()
else:
    temp.replace(OUT)

report = {
    "status": "PASS",
    "analysis": "lexically_variable_subset_speaker_bootstrap",
    "source_sha256": sha(SOURCE),
    "audio_count_full": 400,
    "audio_count_variable": 172,
    "speaker_clusters_total": 20,
    "speaker_clusters_with_variable_audio": len(variable_speakers),
    "bootstrap_iterations": N_BOOT,
    "random_seed": SEED,
    "confidence_interval_method": (
        "speaker_cluster_percentile_bootstrap"
    ),
    "bootstrap_model_refitting": False,
    "excluded_from_primary_model_evaluation": False,
    "diagnostic_anomaly_audio_ids": sorted(ANOMALIES),
    "output_sha256": sha(OUT),
    "limitations": [
        "Analysis is conditional on having lexically variable ASR outputs.",
        "Spearman association does not establish causality.",
        "The same ASR outputs construct both lexical features and prediction variation.",
        "Bootstrap resamples speakers, not independent hypothesis rows.",
        "Only 20 speakers are available in the full pilot.",
        "No fitted model is retrained during bootstrap.",
        "Four-anomaly exclusion is sensitivity analysis only."
    ]
}

if REPORT.exists():
    old = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert old == report, (
        "Existing report differs; original preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_variable_subset_bootstrap"] = "qc_pass"
progress["step6_variable_subset_bootstrap_sha256"] = sha(OUT)
progress["next_task"] = (
    "Interpret within-variable-audio ASR association, "
    "then assess feature contributions and reporting limitations"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.5 VARIABLE-SUBSET BOOTSTRAP ===")
print("Status: PASS")
print("Full audios: 400")
print("Lexically-variable audios:", len(variable))
print("Speakers represented:", len(variable_speakers))
print("Bootstrap iterations:", N_BOOT)
print("Model refitting: NO")

print("\n=== WITHIN-VARIABLE CORRELATION AND 95% CI ===")
print(
    results[
        [
            "model", "unique_audios",
            "observed_spearman_rho",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "valid_bootstrap_draws"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== SENSITIVITY WITHOUT FOUR ANOMALY CANDIDATES ===")
print(
    results[
        [
            "model", "diagnostic_n_without_4",
            "observed_spearman_rho",
            "diagnostic_rho_without_4"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== VARIABLE-SUBSET MEAN PREDICTION RANGE ===")
print(
    results[
        ["model", "mean_prediction_range"]
    ].round(4).to_string(index=False)
)

print("\nOutput:", OUT)
print("SHA256:", sha(OUT))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
SAVED = P / "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step06_final_tables"
COEFS = OUTDIR / "step06_c2_standardized_coefficients.csv"
CONTRIB = OUTDIR / "step06_c2_critical_h2_vs_h1_contributions.csv"
REPORT = P / "reports/step06_c2_feature_contribution_audit.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    SAVED: "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file()
    assert sha(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(DATA)
split = read(SPLIT)
saved = read(SAVED)

design = json.loads(DESIGN.read_text(encoding="utf-8"))

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]
IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]

ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

FEATURES = ACOUSTIC + LING + PROMPT

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(PROMPT) == 2
assert len(FEATURES) == 23

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    validate="many_to_one"
)

assert len(df) == 2000
assert len(saved) == 2000

key_full = KEYS + ["fluency", "validation_fold"]
assert not saved.duplicated(key_full).any()

coef_rows = []
pred_parts = []
critical_contributions = []

CRITICAL_AUDIO = "030210164"

for fold in range(1, 6):

    train = df[df["validation_fold"] != fold]
    val = df[df["validation_fold"] == fold]

    assert len(train) == 1600
    assert len(val) == 400
    assert set(train["speaker"]).isdisjoint(
        set(val["speaker"])
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    model.fit(train[FEATURES], train["fluency"])

    predictions = model.predict(val[FEATURES])

    part = val[key_full].copy()
    part["reconstructed_prediction"] = predictions
    pred_parts.append(part)

    ridge = model.named_steps["ridge"]
    coefs = ridge.coef_

    for feature, coef in zip(FEATURES, coefs):
        if feature in ACOUSTIC:
            group = "acoustic"
        elif feature in LING:
            group = "linguistic"
        else:
            group = "prompt"

        coef_rows.append({
            "fold": fold,
            "feature": feature,
            "feature_group": group,
            "standardized_coefficient": float(coef),
            "absolute_coefficient": float(abs(coef))
        })

    if CRITICAL_AUDIO not in set(val["audio_id"]):
        continue

    critical = val.loc[
        val["audio_id"] == CRITICAL_AUDIO
    ].sort_values("hypothesis_id")

    assert len(critical) == 5
    assert critical["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    # Same fitted fold model for H1 and H2.
    X1 = critical.loc[
        critical["hypothesis_id"] == "H1", FEATURES
    ]
    X2 = critical.loc[
        critical["hypothesis_id"] == "H2", FEATURES
    ]

    imputer = model.named_steps["imputer"]
    scaler = model.named_steps["scaler"]

    z1 = scaler.transform(imputer.transform(X1))[0]
    z2 = scaler.transform(imputer.transform(X2))[0]

    pred1 = float(model.predict(X1)[0])
    pred2 = float(model.predict(X2)[0])

    delta = (z2 - z1) * coefs

    assert np.isclose(
        delta.sum(), pred2 - pred1, atol=1e-9
    ), "Coefficient decomposition mismatch"

    for j, feature in enumerate(FEATURES):
        group = (
            "acoustic" if feature in ACOUSTIC else
            "linguistic" if feature in LING else "prompt"
        )

        critical_contributions.append({
            "fold": fold,
            "audio_id": CRITICAL_AUDIO,
            "comparison": "H2_minus_H1",
            "feature": feature,
            "feature_group": group,
            "h1_standardized_value": float(z1[j]),
            "h2_standardized_value": float(z2[j]),
            "standardized_coefficient": float(coefs[j]),
            "prediction_delta_contribution": float(delta[j]),
            "absolute_delta_contribution": float(abs(delta[j]))
        })

reconstructed = pd.concat(pred_parts, ignore_index=True)

verified = reconstructed.merge(
    saved[key_full + ["c2_prediction"]],
    on=key_full,
    how="inner",
    validate="one_to_one"
)

assert len(verified) == 2000

differences = np.abs(
    verified["reconstructed_prediction"]
    - verified["c2_prediction"]
)

MAX_DIFFERENCE = float(differences.max())

assert MAX_DIFFERENCE < 1e-7, (
    f"Reconstruction mismatch: {MAX_DIFFERENCE}"
)

coef_df = pd.DataFrame(coef_rows)
contrib_df = pd.DataFrame(critical_contributions)

assert len(coef_df) == 5 * 23
assert len(contrib_df) == 23
assert contrib_df["fold"].nunique() == 1
assert int(contrib_df["fold"].iloc[0]) == 2

critical_predictions = verified.loc[
    verified["audio_id"] == CRITICAL_AUDIO
].set_index("hypothesis_id")

critical_h1 = float(
    critical_predictions.loc["H1", "c2_prediction"]
)
critical_h2 = float(
    critical_predictions.loc["H2", "c2_prediction"]
)

assert np.isclose(
    critical_h2, -26.126716638013125, atol=1e-7
)

delta_total = float(
    contrib_df["prediction_delta_contribution"].sum()
)

assert np.isclose(
    critical_h2 - critical_h1,
    delta_total, atol=1e-7
)

group_contributions = (
    contrib_df.groupby("feature_group")[
        "prediction_delta_contribution"
    ].sum()
)

# Same audio means acoustic features cannot explain
# the difference between H1 and H2 predictions.
assert np.isclose(
    group_contributions["acoustic"],
    0,
    atol=1e-9
)

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp, index=False, encoding="utf-8",
        float_format="%.12g"
    )

    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing output differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_csv(coef_df, COEFS)
save_csv(contrib_df, CONTRIB)

report = {
    "status": "PASS",
    "analysis": "C2_ridge_feature_contribution_and_reconstruction",
    "models_retrained_for_reconstruction": True,
    "hyperparameters_changed": False,
    "speaker_disjoint_folds": 5,
    "OOF_predictions_verified": 2000,
    "maximum_reconstruction_difference": MAX_DIFFERENCE,
    "feature_count": 23,
    "critical_audio": CRITICAL_AUDIO,
    "critical_fold": 2,
    "critical_H1_prediction": critical_h1,
    "critical_H2_prediction": critical_h2,
    "H2_minus_H1_prediction_delta": delta_total,
    "group_contributions": {
        name: float(value)
        for name, value in group_contributions.items()
    },
    "coefficients_sha256": sha(COEFS),
    "contributions_sha256": sha(CONTRIB),
    "original_prediction_sha256": sha(SAVED),
    "interpretation": (
        "Exact linear decomposition of one H2-minus-H1 "
        "prediction difference under the same fitted Ridge "
        "model. Coefficient contributions are model-specific, "
        "not causal feature effects. Correlated features "
        "limit independent feature importance interpretation."
    )
}

if REPORT.exists():
    previous = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert previous == report, (
        "Existing report differs; original preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_c2_feature_contribution"] = "qc_pass"
progress["step6_c2_feature_contribution_sha256"] = sha(CONTRIB)
progress["next_task"] = (
    "Review critical feature contributions and "
    "coefficient consistency across speaker folds"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.6 C2 FEATURE CONTRIBUTION AUDIT ===")
print("Status: PASS")
print("Reconstructed OOF predictions:", len(verified))
print("Max prediction difference:", MAX_DIFFERENCE)
print("Critical audio:", CRITICAL_AUDIO)
print("Critical fold: 2")

print("\n=== CRITICAL H1 VS H2 ===")
print("H1 prediction:", round(critical_h1, 4))
print("H2 prediction:", round(critical_h2, 4))
print("H2 - H1 delta:", round(delta_total, 4))

print("\n=== CONTRIBUTION BY FEATURE GROUP ===")
print(group_contributions.round(4).to_string())

print("\n=== TOP 12 FEATURES EXPLAINING H2-H1 DELTA ===")
print(
    contrib_df.nlargest(
        12, "absolute_delta_contribution"
    )[
        [
            "feature", "feature_group",
            "h1_standardized_value",
            "h2_standardized_value",
            "standardized_coefficient",
            "prediction_delta_contribution"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== C2 TOP COEFFICIENTS ACROSS FIVE FOLDS ===")
coef_summary = coef_df.groupby(
    ["feature", "feature_group"]
).agg(
    mean_coefficient=("standardized_coefficient", "mean"),
    sd_coefficient=("standardized_coefficient", "std"),
    mean_absolute_coefficient=("absolute_coefficient", "mean")
).reset_index()

print(
    coef_summary.nlargest(
        12, "mean_absolute_coefficient"
    ).round(4).to_string(index=False)
)

print("\nCoefficients CSV:", COEFS)
print("Contributions CSV:", CONTRIB)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
COEFS = P / "results/step06_final_tables/step06_c2_standardized_coefficients.csv"
CONTRIB = P / "results/step06_final_tables/step06_c2_critical_h2_vs_h1_contributions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step06_final_tables"
STABILITY = OUTDIR / "step06_c2_coefficient_stability.csv"
CORRELATIONS = OUTDIR / "step06_c2_fold2_feature_correlations.csv"
EXTREMES = OUTDIR / "step06_c2_critical_h2_feature_extremes.csv"
REPORT = P / "reports/step06_c2_coefficient_stability_audit.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file()
    assert sha(path) == expected, f"SHA mismatch: {path}"

assert COEFS.is_file()
assert CONTRIB.is_file()

coefs = pd.read_csv(COEFS)
contrib = pd.read_csv(CONTRIB)

assert len(coefs) == 115
assert len(contrib) == 23
assert coefs["fold"].nunique() == 5
assert coefs.groupby("feature")["fold"].nunique().eq(5).all()

# Analyze coefficient consistency across folds.
rows = []

for (feature, group), part in coefs.groupby(
    ["feature", "feature_group"]
):
    values = part.sort_values("fold")[
        "standardized_coefficient"
    ].to_numpy(dtype=float)

    nonzero = values[np.abs(values) > 1e-10]

    positive = int((values > 1e-10).sum())
    negative = int((values < -1e-10).sum())

    rows.append({
        "feature": feature,
        "feature_group": group,
        "mean_coefficient": float(values.mean()),
        "sd_coefficient": float(values.std(ddof=1)),
        "min_coefficient": float(values.min()),
        "max_coefficient": float(values.max()),
        "mean_absolute_coefficient": float(
            np.abs(values).mean()
        ),
        "positive_folds": positive,
        "negative_folds": negative,
        "sign_consistent": bool(
            len(nonzero) == 5 and
            (positive == 5 or negative == 5)
        )
    })

coef_stability = pd.DataFrame(rows)
assert len(coef_stability) == 23

data = pd.read_csv(
    DATA,
    dtype={"audio_id": "string", "speaker": "string"}
)
split = pd.read_csv(
    SPLIT,
    dtype={"audio_id": "string", "speaker": "string"}
)
design = json.loads(DESIGN.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]
ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]
FEATURES = ACOUSTIC + LING + PROMPT

assert len(FEATURES) == 23

df = data.merge(
    split[IDS + ["validation_fold"]],
    on=IDS,
    validate="many_to_one"
)

assert len(df) == 2000

# Use fold 2 training data only.
train = df.loc[df["validation_fold"] != 2]
val = df.loc[df["validation_fold"] == 2]

assert len(train) == 1600
assert len(val) == 400
assert set(train["speaker"]).isdisjoint(set(val["speaker"]))

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = scaler.fit_transform(
    imputer.fit_transform(train[FEATURES])
)

# These are the same preprocessing steps as C2,
# without fitting a regression model.
correlation = np.corrcoef(X_train, rowvar=False)

corr_rows = []

for i in range(len(FEATURES)):
    for j in range(i + 1, len(FEATURES)):
        r = float(correlation[i, j])

        if np.isfinite(r):
            corr_rows.append({
                "feature_1": FEATURES[i],
                "feature_2": FEATURES[j],
                "pearson_r": r,
                "absolute_r": abs(r)
            })

corr_df = pd.DataFrame(corr_rows).sort_values(
    "absolute_r", ascending=False
).reset_index(drop=True)

# Eigenvalue-based condition number of the
# standardized design matrix, descriptive only.
singular_values = np.linalg.svd(
    X_train, compute_uv=False
)

positive_sv = singular_values[
    singular_values > 1e-10
]

condition_number = float(
    singular_values.max() / positive_sv.min()
)

rank = int(np.linalg.matrix_rank(X_train))

# Critical H2, standardized using fold 2 train statistics.
critical = val.loc[
    val["audio_id"].eq("030210164") &
    val["hypothesis_id"].eq("H2")
]

assert len(critical) == 1

z = scaler.transform(
    imputer.transform(critical[FEATURES])
)[0]

extremes = pd.DataFrame({
    "feature": FEATURES,
    "critical_H2_train_standardized_value": z,
    "absolute_standardized_value": np.abs(z)
}).sort_values(
    "absolute_standardized_value",
    ascending=False
).reset_index(drop=True)

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp, index=False, encoding="utf-8",
        float_format="%.12g"
    )
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing file differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv(coef_stability, STABILITY)
save_csv(corr_df, CORRELATIONS)
save_csv(extremes, EXTREMES)

report = {
    "status": "PASS",
    "analysis": "C2_coefficient_stability_multicollinearity",
    "fold_count": 5,
    "feature_count": 23,
    "sign_consistent_features": int(
        coef_stability["sign_consistent"].sum()
    ),
    "fold2_training_rows": 1600,
    "fold2_training_speakers": 16,
    "feature_matrix_rank": rank,
    "feature_matrix_condition_number": condition_number,
    "feature_pairs_abs_correlation_ge_0_9": int(
        (corr_df["absolute_r"] >= 0.9).sum()
    ),
    "critical_H2_max_abs_z": float(
        extremes["absolute_standardized_value"].max()
    ),
    "model_refitting": False,
    "coefficients_source_sha256": sha(COEFS),
    "contributions_source_sha256": sha(CONTRIB),
    "outputs_sha256": {
        path.name: sha(path)
        for path in [STABILITY, CORRELATIONS, EXTREMES]
    },
    "limitations": [
        "Coefficients describe conditional fitted model weights.",
        "Correlated predictors limit isolated feature interpretation.",
        "Only five training folds were examined.",
        "Fold 2 correlation analysis is diagnostic.",
        "No causal feature importance can be inferred."
    ]
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_c2_coefficient_stability"] = "qc_pass"
progress["step6_c2_coefficient_stability_sha256"] = sha(REPORT)
progress["next_task"] = (
    "Review coefficient consistency, correlated predictors "
    "and extreme standardized feature values"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.7 COEFFICIENT STABILITY AUDIT ===")
print("Status: PASS")
print("Features:", len(FEATURES))
print(
    "Sign-consistent features:",
    report["sign_consistent_features"]
)
print("Fold 2 design rank:", rank)
print("Fold 2 condition number:", condition_number)
print(
    "Feature pairs |r| >= 0.9:",
    report["feature_pairs_abs_correlation_ge_0_9"]
)

print("\n=== TOP 15 COEFFICIENTS ===")
print(
    coef_stability.nlargest(
        15, "mean_absolute_coefficient"
    ).round(4).to_string(index=False)
)

print("\n=== TOP 15 FEATURE CORRELATIONS ===")
print(corr_df.head(15).round(4).to_string(index=False))

print("\n=== CRITICAL H2 EXTREME FEATURES ===")
print(extremes.head(12).round(4).to_string(index=False))

print("\nReport:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
C2 = P / "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv"
A = P / "results/step05/baseline_acoustic_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step06_final_tables"
DETAIL = OUTDIR / "step06_ood_diagnostic_per_hypothesis.csv"
SUMMARY = OUTDIR / "step06_ood_diagnostic_summary.csv"
REPORT = P / "reports/step06_ood_feature_audit_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    C2: "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9",
    A: "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file()
    assert sha(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(DATA)
split = read(SPLIT)
c2 = read(C2)
acoustic_pred = read(A)
design = json.loads(DESIGN.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]
FULL = KEYS + ["fluency", "validation_fold"]

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]
ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]
FEATURES = ACOUSTIC + LING + PROMPT

assert len(FEATURES) == 23

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    validate="many_to_one"
)

df = df.merge(
    c2[FULL + ["c2_prediction"]],
    on=FULL,
    validate="one_to_one"
)

df = df.merge(
    acoustic_pred[
        IDS + ["fluency", "validation_fold",
               "acoustic_ridge_prediction"]
    ],
    on=IDS + ["fluency", "validation_fold"],
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20

parts = []

for fold in range(1, 6):
    train = df.loc[df["validation_fold"] != fold]
    val = df.loc[df["validation_fold"] == fold].copy()

    assert len(train) == 1600
    assert len(val) == 400
    assert set(train["speaker"]).isdisjoint(
        set(val["speaker"])
    )

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    Xtrain = imputer.fit_transform(train[FEATURES])
    scaler.fit(Xtrain)

    Xval = imputer.transform(val[FEATURES])
    Z = scaler.transform(Xval)

    assert Z.shape == (400, 23)
    assert np.isfinite(Z).all()

    abs_z = np.abs(Z)
    ap = abs_z[:, :14]
    lp = abs_z[:, 14:]

    val["max_abs_z_all"] = abs_z.max(axis=1)
    val["max_abs_z_acoustic"] = ap.max(axis=1)
    val["max_abs_z_text_prompt"] = lp.max(axis=1)
    val["count_abs_z_gt5"] = (abs_z > 5).sum(axis=1)
    val["count_text_prompt_z_gt5"] = (lp > 5).sum(axis=1)
    val["any_text_prompt_z_gt5"] = (
        val["max_abs_z_text_prompt"] > 5
    )

    worst_idx = lp.argmax(axis=1)
    text_features = FEATURES[14:]
    val["largest_text_prompt_z_feature"] = [
        text_features[int(i)] for i in worst_idx
    ]

    parts.append(
        val[
            FULL + [
                "c2_prediction",
                "acoustic_ridge_prediction",
                "max_abs_z_all",
                "max_abs_z_acoustic",
                "max_abs_z_text_prompt",
                "count_abs_z_gt5",
                "count_text_prompt_z_gt5",
                "any_text_prompt_z_gt5",
                "largest_text_prompt_z_feature"
            ]
        ]
    )

detail = pd.concat(parts, ignore_index=True)
detail = detail.sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(detail) == 2000
assert not detail.duplicated(KEYS).any()

detail["C2_absolute_error"] = (
    detail["c2_prediction"] - detail["fluency"]
).abs()
detail["A_absolute_error"] = (
    detail["acoustic_ridge_prediction"] -
    detail["fluency"]
).abs()

detail["C2_squared_error"] = (
    detail["c2_prediction"] - detail["fluency"]
) ** 2

detail["A_squared_error"] = (
    detail["acoustic_ridge_prediction"] -
    detail["fluency"]
) ** 2

# Primary values must reproduce existing fixed results.
assert np.isclose(
    detail["C2_absolute_error"].mean(),
    0.7782, atol=1e-4
)
assert np.isclose(
    detail["A_absolute_error"].mean(),
    0.9399, atol=1e-4
)

detail["diagnostic_group"] = np.where(
    detail["any_text_prompt_z_gt5"],
    "text_prompt_abs_z_gt5",
    "text_prompt_abs_z_le5"
)

summary_rows = []

for group, sub in detail.groupby("diagnostic_group"):
    summary_rows.append({
        "diagnostic_group": group,
        "hypotheses": int(len(sub)),
        "unique_audios": int(sub["audio_id"].nunique()),
        "unique_speakers": int(sub["speaker"].nunique()),
        "C2_MAE": float(sub["C2_absolute_error"].mean()),
        "C2_RMSE": float(
            np.sqrt(sub["C2_squared_error"].mean())
        ),
        "A_MAE": float(sub["A_absolute_error"].mean()),
        "A_RMSE": float(
            np.sqrt(sub["A_squared_error"].mean())
        ),
        "C2_minus_A_MAE": float(
            (sub["C2_absolute_error"] -
             sub["A_absolute_error"]).mean()
        ),
        "maximum_text_prompt_abs_z": float(
            sub["max_abs_z_text_prompt"].max()
        )
    })

summary = pd.DataFrame(summary_rows)

assert summary["hypotheses"].sum() == 2000

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def safe_save(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp, index=False, encoding="utf-8",
        float_format="%.12g"
    )
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing output differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

safe_save(detail, DETAIL)
safe_save(summary, SUMMARY)

critical = detail.loc[
    detail["audio_id"].eq("030210164") &
    detail["hypothesis_id"].eq("H2")
]

assert len(critical) == 1
assert critical["max_abs_z_text_prompt"].iloc[0] > 50

report = {
    "status": "PASS",
    "analysis": "training_fold_standardized_feature_extremes",
    "threshold_abs_z": 5.0,
    "threshold_role": "descriptive_diagnostic_not_formal_OOD",
    "total_hypotheses": 2000,
    "unique_audios": 400,
    "unique_speakers": 20,
    "hypotheses_above_threshold": int(
        detail["any_text_prompt_z_gt5"].sum()
    ),
    "audios_above_threshold": int(
        detail.loc[
            detail["any_text_prompt_z_gt5"],
            "audio_id"
        ].nunique()
    ),
    "model_refitting": False,
    "all_predictions_unchanged": True,
    "all_primary_observations_retained": True,
    "detail_sha256": sha(DETAIL),
    "summary_sha256": sha(SUMMARY),
    "source_sha256": {
        path.name: sha(path) for path in EXPECTED
    },
    "limitations": [
        "Z-score threshold of 5 is an exploratory diagnostic.",
        "All H1-H5 hypotheses of one audio are correlated.",
        "Feature extremeness and prediction errors are associated, not causally identified.",
        "Training distribution depends on each frozen fold.",
        "No additional model fitting or hyperparameter tuning."
    ]
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_ood_feature_audit"] = "qc_pass"
progress["step6_ood_feature_audit_sha256"] = sha(SUMMARY)
progress["next_task"] = (
    "Assess feature-extreme prevalence and error association; "
    "complete Step 6 synthesis and checkpoint"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.8 FEATURE EXTREMENESS AUDIT ===")
print("Status: PASS")
print("Hypotheses:", len(detail))
print("Unique audios:", detail["audio_id"].nunique())
print("Models retrained: NO")
print("Primary observations excluded: NONE")

print("\n=== EXTREME FEATURE PREVALENCE ===")
print("Hypotheses with text/prompt |z| > 5:",
      report["hypotheses_above_threshold"])
print("Audios with text/prompt |z| > 5:",
      report["audios_above_threshold"])

print("\n=== ERROR BY FEATURE EXTREMENESS ===")
print(summary.round(4).to_string(index=False))

print("\n=== TOP 12 EXTREME HYPOTHESES ===")
print(
    detail.nlargest(
        12, "max_abs_z_text_prompt"
    )[
        [
            "audio_id", "speaker", "hypothesis_id",
            "max_abs_z_text_prompt",
            "largest_text_prompt_z_feature",
            "C2_absolute_error",
            "A_absolute_error"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== CRITICAL H2 CHECK ===")
print(
    critical[
        [
            "max_abs_z_text_prompt",
            "C2_absolute_error",
            "A_absolute_error"
        ]
    ].round(4).to_string(index=False)
)

print("\nDetail CSV:", DETAIL)
print("Summary CSV:", SUMMARY)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import zipfile

ROOT = Path("/kaggle/working/acoustic_linguistic_research")
WORK = Path("/kaggle/working")
ARCHIVE = WORK / "asr_fluency_step01-06_complete.zip"

REQUIRED = [
    "data/modeling/modeling_dataset_0001-0400.csv",
    "data/splits/speaker_groupkfold5_frozen.csv",
    "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv",
    "results/step05/speaker_cluster_bootstrap_10000_ci.csv",
    "results/step05/step06_anomaly_sensitivity_400_399_396.csv",
    "results/step06_final_tables/table01_primary_models_with_ci.csv",
    "results/step06_final_tables/table02_paired_differences_with_ci.csv",
    "results/step06_final_tables/step06_variable_subset_bootstrap.csv",
    "results/step06_final_tables/step06_c2_critical_h2_vs_h1_contributions.csv",
    "results/step06_final_tables/step06_c2_coefficient_stability.csv",
    "results/step06_final_tables/step06_ood_diagnostic_per_hypothesis.csv",
    "results/step06_final_tables/step06_ood_diagnostic_summary.csv",
    "reports/step06_ood_feature_audit_v1.json",
]

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert ROOT.is_dir()

for rel in REQUIRED:
    assert (ROOT / rel).is_file(), f"Required file missing: {rel}"

# Verify previously frozen reference files.
KNOWN_HASHES = {
    "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv":
        "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9",
    "results/step05/speaker_cluster_bootstrap_10000_ci.csv":
        "5ef3d6dd01eb05758ccb8b4f77293396a81e988b9b9e33152cb666c6b48d1d93",
    "results/step05/step06_anomaly_sensitivity_400_399_396.csv":
        "83b34da4aa34634319b59c94391967565e3f3c0fd438fb5a80d48131ca5563d3",
    "results/step06_final_tables/table01_primary_models_with_ci.csv":
        "22512d4fdd54feec3f6658f99248653e8c9fb99e110887db64578aadfb43b4c0",
    "results/step06_final_tables/step06_variable_subset_bootstrap.csv":
        "257f53dd5eb50979bf4f3cb7a75de48be361baff049239e650c20f38f3d7bab9"
}

for rel, expected in KNOWN_HASHES.items():
    assert sha256(ROOT / rel) == expected, f"SHA mismatch: {rel}"

included = []
omitted = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path.is_symlink():
        continue

    rel = path.relative_to(ROOT).as_posix()

    if "official_test" in rel.lower() or "speechocean762_test" in rel.lower():
        omitted.append((path, "Sealed official test excluded"))
        continue

    if rel.startswith("data/raw/") and path.suffix == ".parquet":
        omitted.append((path, "Raw parquet stored externally"))
        continue

    if "__pycache__" in path.parts or path.suffix in [".tmp", ".pyc"]:
        continue

    included.append((path, "project/" + rel))

# Include the previous complete STEP 5 checkpoint.
OLD_BACKUP = WORK / "asr_fluency_step01-05_complete_with_step06_bootstrap.zip"
if OLD_BACKUP.is_file():
    included.append((OLD_BACKUP, "extra/step01-05_previous_checkpoint.zip"))

STEP3 = WORK / "asr_fluency_step03_complete.zip"
if STEP3.is_file():
    included.append((STEP3, "extra/asr_fluency_step03_complete.zip"))

# Include saved notebooks/scripts if present.
for path in sorted(WORK.iterdir()):
    if path.is_file() and path.suffix.lower() in [".py", ".ipynb", ".r"]:
        included.append((path, "extra/source_files/" + path.name))

assert len({name for _, name in included}) == len(included)

entries = [
    {
        "archive_path": name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path)
    }
    for path, name in included
]

manifest = {
    "project": "ASR fluency prediction accuracy and stability",
    "status": "STEP_1_TO_6_COMPLETE",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "mispeech/speechocean762",
    "dataset_revision": "06385584fad212b26134c656fdd3ccf9f093f33e",
    "audio_count": 400,
    "speaker_count": 20,
    "hypothesis_count": 2000,
    "cv": "5-fold speaker-disjoint",
    "models": ["M0", "A", "B1", "B2", "C1", "C2"],
    "analysis_complete": [
        "Fixed-alpha Ridge evaluation",
        "Speaker-clustered bootstrap",
        "ASR anomaly sensitivity",
        "Integrated research tables",
        "ASR variability association",
        "Variable-subset bootstrap",
        "Critical feature contribution",
        "Coefficient stability and multicollinearity",
        "Feature extremeness audit"
    ],
    "official_test_untouched": True,
    "included_files": entries,
    "omitted_files": [
        {
            "original_path": str(path),
            "reason": reason,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path)
        }
        for path, reason in omitted
    ],
    "research_cautions": [
        "Reading prompt is not a verified transcription",
        "Prompt-ASR disagreement is not ground-truth WER/CER",
        "Only 20 independent speaker clusters",
        "Hypotheses from the same audio are correlated",
        "Bootstrap does not refit models",
        "Sensitivity analyses do not replace the primary 400-audio analysis",
        "Extreme feature threshold is exploratory"
    ]
}

restore = """RESTORE STEP 1-6 RESEARCH

1. Upload this ZIP to Kaggle.
2. Extract project/ files to:
   /kaggle/working/acoustic_linguistic_research/
3. Check MANIFEST.json for file hashes and omitted files.
4. STEP 1-6 experiments are complete.
5. Do not retrain the fixed-alpha Ridge models unnecessarily.
6. Keep the official test partition sealed.
7. Continue with manuscript preparation, further validation
   planning, or reviewer-requested analysis.
8. Export the Kaggle notebook separately as .ipynb.
9. The original train Parquet is excluded; retrieve the
   pinned train revision if raw audio is needed.
"""

temp = WORK / (ARCHIVE.name + ".tmp")

with zipfile.ZipFile(
    temp, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True
) as z:
    for path, name in included:
        z.write(path, arcname=name)

    z.writestr(
        "MANIFEST.json",
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )
    z.writestr("RESTORE_INSTRUCTIONS.txt", restore)

with zipfile.ZipFile(temp) as z:
    assert z.testzip() is None

    for entry in entries:
        h = hashlib.sha256()
        with z.open(entry["archive_path"]) as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)

        assert h.hexdigest() == entry["sha256"], (
            f"Archive hash mismatch: {entry['archive_path']}"
        )

if ARCHIVE.exists():
    # A completed earlier backup is preserved.
    assert sha256(ARCHIVE) == sha256(temp), (
        "Archive already exists with different content. "
        "Original preserved; rename the new backup before saving."
    )
    temp.unlink()
else:
    os.replace(temp, ARCHIVE)

print("=== STEP 6 FINAL CHECKPOINT ===")
print("Status: PASS")
print("STEP 5: COMPLETE")
print("STEP 6: COMPLETE")
print("Included files:", len(entries))
print("Omitted files:", len(omitted))
print("ZIP integrity: PASS")
print("SHA256 verification: PASS")
print("Archive:", ARCHIVE)
print("Size MB:", round(ARCHIVE.stat().st_size / 1024**2, 2))
print("SHA256:", sha256(ARCHIVE))

print("\n=== OMITTED FILES ===")
for path, reason in omitted:
    print(reason, "|", path)

print("\nNEXT: Download ZIP and export Kaggle notebook.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")

DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
CV = P / "configs/speaker_disjoint_cv_v1.json"

OUT = P / "results/step07_methodology"
REPORT = OUT / "step07_01_dataset_integrity.json"
SPEAKERS = OUT / "step07_01_speaker_summary.csv"
FOLDS = OUT / "step07_01_fold_summary.csv"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    CV: "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha(path) == expected, f"SHA mismatch: {path}"

data = pd.read_csv(
    DATA,
    dtype={"audio_id": "string", "speaker": "string"}
)

split = pd.read_csv(
    SPLIT,
    dtype={"audio_id": "string", "speaker": "string"}
)

design = json.loads(DESIGN.read_text(encoding="utf-8"))
cv = json.loads(CV.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]

assert len(data) == 2000
assert len(split) == 400
assert data["audio_id"].nunique() == 400
assert split["audio_id"].nunique() == 400
assert data["speaker"].nunique() == 20
assert split["speaker"].nunique() == 20

assert not data.duplicated(KEYS).any()
assert not split.duplicated(IDS).any()

assert data.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert data.groupby("audio_id")["fluency"].nunique().eq(1).all()
assert data.groupby("audio_id")["speaker"].nunique().eq(1).all()

assert set(data["audio_id"]) == set(split["audio_id"])

merged = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS,
    how="inner",
    suffixes=("_data", "_split"),
    validate="many_to_one"
)

assert len(merged) == 2000
assert np.allclose(
    merged["fluency_data"],
    merged["fluency_split"]
)

assert merged["validation_fold"].nunique() == 5

assert merged.groupby("speaker")[
    "validation_fold"
].nunique().eq(1).all()

assert merged.groupby("audio_id")[
    "validation_fold"
].nunique().eq(1).all()

assert set(merged["validation_fold"]) == {1, 2, 3, 4, 5}

# One independent observation per original audio.
audio = split.copy()

assert audio.groupby("speaker")["audio_id"].nunique().eq(20).all()

speaker_summary = (
    audio.groupby(["speaker", "validation_fold"])
    .agg(
        audios=("audio_id", "nunique"),
        fluency_mean=("fluency", "mean"),
        fluency_sd=("fluency", "std"),
        fluency_min=("fluency", "min"),
        fluency_max=("fluency", "max")
    )
    .reset_index()
    .sort_values(["validation_fold", "speaker"])
)

fold_summary = (
    audio.groupby("validation_fold")
    .agg(
        speakers=("speaker", "nunique"),
        audios=("audio_id", "nunique"),
        fluency_mean=("fluency", "mean"),
        fluency_sd=("fluency", "std"),
        fluency_min=("fluency", "min"),
        fluency_max=("fluency", "max")
    )
    .reset_index()
)

assert len(speaker_summary) == 20
assert len(fold_summary) == 5
assert fold_summary["audios"].eq(80).all()
assert fold_summary["speakers"].eq(4).all()

# Verify that the five folds contain disjoint speakers.
fold_speakers = {
    int(fold): set(group["speaker"])
    for fold, group in audio.groupby("validation_fold")
}

for fold_a in range(1, 6):
    for fold_b in range(fold_a + 1, 6):
        assert fold_speakers[fold_a].isdisjoint(
            fold_speakers[fold_b]
        )

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]

ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(PROMPT) == 2

missing = {
    col: {
        "count": int(data[col].isna().sum()),
        "percent": float(data[col].isna().mean() * 100)
    }
    for col in ACOUSTIC + LING + PROMPT
    if data[col].isna().any()
}

feature_groups = {
    "acoustic": ACOUSTIC,
    "linguistic": LING,
    "prompt": PROMPT
}

REPORT_DATA = {
    "status": "PASS",
    "step": "7.1",
    "analysis": "dataset_split_and_feature_integrity",
    "dataset": "mispeech/speechocean762",
    "dataset_revision": "06385584fad212b26134c656fdd3ccf9f093f33e",
    "dataset_partition_analyzed": "train",
    "official_test_access_checked_by_this_cell": False,
    "audio_count": int(audio["audio_id"].nunique()),
    "speaker_count": int(audio["speaker"].nunique()),
    "hypothesis_rows": int(len(data)),
    "hypotheses_per_audio": 5,
    "fold_count": 5,
    "validation_speakers_per_fold": 4,
    "validation_audios_per_fold": 80,
    "human_fluency_min": float(audio["fluency"].min()),
    "human_fluency_max": float(audio["fluency"].max()),
    "human_fluency_mean": float(audio["fluency"].mean()),
    "human_fluency_sd": float(audio["fluency"].std()),
    "feature_groups": feature_groups,
    "missing_feature_values": missing,
    "speaker_disjoint": True,
    "audio_disjoint": True,
    "target_consistency": True,
    "all_original_audios_retained": True,
    "models_retrained": False,
    "source_sha256": {
        path.name: sha(path)
        for path in EXPECTED
    },
    "methodology_notes": [
        "Five ASR hypotheses share one human fluency label.",
        "The effective independent sampling units are speakers.",
        "Fold mean labels may differ by speaker composition.",
        "Dataset revision is documented from frozen experiment records.",
        "This cell does not independently verify initial sample randomization.",
        "This cell does not independently prove the official test was never opened.",
        "Prompt text is not verified ground-truth transcription."
    ]
}

OUT.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp,
        index=False,
        encoding="utf-8",
        float_format="%.12g"
    )

    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing report differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(speaker_summary, SPEAKERS)
safe_csv(fold_summary, FOLDS)

REPORT_DATA["speaker_summary_sha256"] = sha(SPEAKERS)
REPORT_DATA["fold_summary_sha256"] = sha(FOLDS)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == REPORT_DATA, "Existing JSON differs"
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(
            REPORT_DATA,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("=== STEP 7.1 DATASET INTEGRITY AUDIT ===")
print("Status: PASS")
print("Original audios:", len(audio))
print("Speakers:", audio["speaker"].nunique())
print("ASR hypotheses:", len(data))
print("Speaker-disjoint folds: PASS")
print("Target consistency: PASS")

print("\n=== FLUENCY DISTRIBUTION ===")
print(
    "Min:", REPORT_DATA["human_fluency_min"],
    "Max:", REPORT_DATA["human_fluency_max"],
    "Mean:", round(REPORT_DATA["human_fluency_mean"], 4),
    "SD:", round(REPORT_DATA["human_fluency_sd"], 4)
)

print("\n=== FOLD SUMMARY ===")
print(fold_summary.round(4).to_string(index=False))

print("\n=== SPEAKER SUMMARY ===")
print(speaker_summary.round(4).to_string(index=False))

print("\n=== FEATURE MISSINGNESS ===")
print(json.dumps(missing, indent=2))

print("\nReport:", REPORT)
print("SHA256:", sha(REPORT))
print("Dataset and split unchanged: YES")
print("Next: STEP 7.2 leakage audit")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
C2 = P / "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv"
OUT = P / "results/step07_methodology"
REPORT = OUT / "step07_02_leakage_audit.json"
FOLD_CSV = OUT / "step07_02_fold_preprocessing_checks.csv"

HASHES = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    C2: "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path, expected in HASHES.items():
    assert path.is_file()
    assert sha(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(DATA)
split = read(SPLIT)
saved = read(C2)
design = json.loads(DESIGN.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]
FULL = KEYS + ["fluency", "validation_fold"]

LING = list(design["linguistic_feature_names"])
PROMPT = list(design["prompt_feature_names"])

ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

FEATURES = ACOUSTIC + LING + PROMPT

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(PROMPT) == 2
assert len(FEATURES) == 23
assert len(set(FEATURES)) == 23

FORBIDDEN = {
    "fluency", "audio_id", "speaker", "pilot_index",
    "hypothesis_id", "validation_fold",
    "c2_prediction", "human_score", "human_fluency"
}

assert FORBIDDEN.isdisjoint(FEATURES)

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    validate="many_to_one",
    how="inner"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20

assert not df.duplicated(KEYS).any()
assert df.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()
assert df.groupby("audio_id")["validation_fold"].nunique().eq(1).all()
assert df.groupby("speaker")["validation_fold"].nunique().eq(1).all()

assert len(saved) == 2000
assert not saved.duplicated(KEYS).any()

fold_results = []
predictions = []

for fold in range(1, 6):
    train = df.loc[df["validation_fold"] != fold].copy()
    val = df.loc[df["validation_fold"] == fold].copy()

    assert len(train) == 1600
    assert len(val) == 400

    train_speakers = set(train["speaker"])
    val_speakers = set(val["speaker"])

    train_audios = set(train["audio_id"])
    val_audios = set(val["audio_id"])

    assert train_speakers.isdisjoint(val_speakers)
    assert train_audios.isdisjoint(val_audios)

    assert len(train_speakers) == 16
    assert len(val_speakers) == 4
    assert len(train_audios) == 320
    assert len(val_audios) == 80

    X_train = train[FEATURES]
    X_val = val[FEATURES]

    assert all(
        pd.api.types.is_numeric_dtype(X_train[col])
        for col in FEATURES
    )

    # Independently compute train-only median statistics.
    train_medians = X_train.median(axis=0).to_numpy(dtype=float)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    model.fit(X_train, train["fluency"])

    imputer = model.named_steps["imputer"]
    scaler = model.named_steps["scaler"]

    assert np.allclose(
        imputer.statistics_,
        train_medians,
        rtol=1e-12,
        atol=1e-12
    )

    train_imputed = imputer.transform(X_train)

    independent_mean = train_imputed.mean(axis=0)
    independent_var = train_imputed.var(axis=0)

    assert np.allclose(
        scaler.mean_,
        independent_mean,
        rtol=1e-12,
        atol=1e-12
    )
    assert np.allclose(
        scaler.var_,
        independent_var,
        rtol=1e-12,
        atol=1e-12
    )

    # Compare against a different hypothetical preprocessing:
    # fitting on all data. This is diagnostic only;
    # those statistics are NEVER used for prediction.
    train_missing = int(X_train.isna().sum().sum())
    val_missing = int(X_val.isna().sum().sum())

    pred = model.predict(X_val)
    assert np.isfinite(pred).all()

    part = val[FULL].copy()
    part["reconstructed_prediction"] = pred
    predictions.append(part)

    fold_results.append({
        "fold": fold,
        "train_speakers": len(train_speakers),
        "validation_speakers": len(val_speakers),
        "train_audios": len(train_audios),
        "validation_audios": len(val_audios),
        "train_hypotheses": len(train),
        "validation_hypotheses": len(val),
        "speaker_overlap": len(train_speakers & val_speakers),
        "audio_overlap": len(train_audios & val_audios),
        "training_missing_cells": train_missing,
        "validation_missing_cells": val_missing,
        "train_only_median_verified": True,
        "train_only_scaler_verified": True
    })

reconstructed = pd.concat(predictions, ignore_index=True)

comparison = reconstructed.merge(
    saved[FULL + ["c2_prediction"]],
    on=FULL,
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 2000

delta = np.abs(
    comparison["reconstructed_prediction"].to_numpy()
    - comparison["c2_prediction"].to_numpy()
)

max_delta = float(delta.max())
assert max_delta < 1e-7, (
    f"Reconstruction mismatch: {max_delta}"
)

fold_table = pd.DataFrame(fold_results)

assert len(fold_table) == 5
assert fold_table["speaker_overlap"].eq(0).all()
assert fold_table["audio_overlap"].eq(0).all()

OUT.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")

    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing CSV differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(fold_table, FOLD_CSV)

report = {
    "status": "PASS",
    "step": "7.2",
    "audit": "speaker_split_feature_and_preprocessing_leakage",
    "speaker_disjoint": True,
    "audio_disjoint": True,
    "hypotheses_grouped_by_audio": True,
    "forbidden_predictor_columns_excluded": True,
    "imputation_fit_on_training_fold": True,
    "scaler_fit_on_training_fold": True,
    "human_fluency_is_training_target_only": True,
    "validation_labels_not_used_for_preprocessing": True,
    "ridge_alpha": 1.0,
    "oof_predictions_reconstructed": 2000,
    "max_prediction_absolute_difference": max_delta,
    "fold_results_sha256": sha(FOLD_CSV),
    "source_sha256": {
        path.name: sha(path)
        for path in HASHES
    },
    "scope_limitations": [
        "Confirms leakage controls for reconstructed C2 pipeline.",
        "Does not independently audit original ASR training data.",
        "Does not verify possible overlap with Whisper pretraining.",
        "Does not establish provenance of source audio or prompts.",
        "Does not audit original sample selection procedure.",
        "Does not independently prove sealed test access history.",
        "Does not establish whether prompt features are permitted operationally."
    ]
}

if REPORT.exists():
    old = json.loads(REPORT.read_text(encoding="utf-8"))
    assert old == report, "Existing report differs"
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("=== STEP 7.2 DATA LEAKAGE AUDIT ===")
print("Status: PASS")
print("Feature count:", len(FEATURES))
print("Forbidden predictors: EXCLUDED")
print("Speaker overlap: ZERO")
print("Audio overlap: ZERO")
print("Train-only imputation: PASS")
print("Train-only scaling: PASS")
print("C2 OOF reconstruction: PASS")
print("Maximum prediction difference:", max_delta)

print("\n=== FOLD-BY-FOLD LEAKAGE CHECKS ===")
print(fold_table.to_string(index=False))

print("\n=== FEATURE GROUPS ===")
print("Acoustic:", len(ACOUSTIC))
print("Linguistic:", len(LING))
print("Prompt-aware:", len(PROMPT))

print("\n=== SCOPE LIMITATIONS ===")
for item in report["scope_limitations"]:
    print("-", item)

print("\nReport:", REPORT)
print("SHA256:", sha(REPORT))
print("Fold CSV:", FOLD_CSV)
print("Next: STEP 7.3 model-comparison fairness")


In [ ]:

import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
D = P / "data/modeling/modeling_dataset_0001-0400.csv"
S = P / "data/splits/speaker_groupkfold5_frozen.csv"
CONFIG = P / "configs/fluency_feature_design_v1.json"
OUT = P / "results/step07_methodology"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE_HASHES = {
    D: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    S: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    CONFIG: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

ORIGINALS = {
    "B1": ("b1_transcript_ridge_oof_predictions.csv",
           "b1_prediction",
           "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc"),
    "B2": ("b2_prompt_aware_ridge_oof_predictions.csv",
           "b2_prediction",
           "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189"),
    "C1": ("c1_fusion_ridge_oof_predictions.csv",
           "c1_prediction",
           "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e"),
    "C2": ("c2_prompt_aware_fusion_ridge_oof_predictions.csv",
           "c2_prediction",
           "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9")
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path, expected in SOURCE_HASHES.items():
    assert path.is_file() and sha(path) == expected

def read(path):
    return pd.read_csv(
        path, dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(D)
split = read(S)
design = json.loads(CONFIG.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]
FULL = KEYS + ["fluency", "validation_fold"]

LING = list(design["linguistic_feature_names"])
PROMPT = list(design["prompt_feature_names"])
ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

FEATURE_MAP = {
    "B1": LING,
    "B2": LING + PROMPT,
    "C1": ACOUSTIC + LING,
    "C2": ACOUSTIC + LING + PROMPT
}

assert [len(FEATURE_MAP[m]) for m in ORIGINALS] == [7, 9, 21, 23]

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    how="inner",
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert not df.duplicated(KEYS).any()

saved = {}

for name, (filename, col, expected_hash) in ORIGINALS.items():
    path = P / "results/step05" / filename
    assert sha(path) == expected_hash

    original = read(path)
    assert len(original) == 2000
    assert not original.duplicated(KEYS).any()

    saved[name] = original[FULL + [col]]

rows = []
all_predictions = []
max_reconstruction = {}

for name, features in FEATURE_MAP.items():

    model_parts = []

    for fold in range(1, 6):
        tr = df[df["validation_fold"] != fold]
        va = df[df["validation_fold"] == fold]

        assert len(tr) == 1600 and len(va) == 400
        assert tr["audio_id"].nunique() == 320
        assert va["audio_id"].nunique() == 80
        assert set(tr["speaker"]).isdisjoint(set(va["speaker"]))
        assert set(tr["audio_id"]).isdisjoint(set(va["audio_id"]))

        for alpha in [1.0, 5.0]:

            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=alpha))
            ])

            pipe.fit(tr[features], tr["fluency"])
            prediction = pipe.predict(va[features])
            assert np.isfinite(prediction).all()

            part = va[FULL].copy()
            part["model"] = name
            part["alpha"] = alpha
            part["prediction"] = prediction

            model_parts.append(part)

    result = pd.concat(model_parts, ignore_index=True)

    for alpha, sub in result.groupby("alpha"):

        assert len(sub) == 2000
        assert not sub.duplicated(KEYS).any()

        error = (
            sub["prediction"].to_numpy(dtype=float)
            - sub["fluency"].to_numpy(dtype=float)
        )

        ranges = sub.groupby(
            IDS
        )["prediction"].agg(
            lambda x: x.max() - x.min()
        )

        assert len(ranges) == 400

        rows.append({
            "model": name,
            "alpha": float(alpha),
            "features": len(features),
            "OOF_predictions": len(sub),
            "MAE": float(np.abs(error).mean()),
            "RMSE": float(np.sqrt(np.square(error).mean())),
            "mean_prediction_range": float(ranges.mean()),
            "p95_prediction_range": float(ranges.quantile(0.95)),
            "max_prediction_range": float(ranges.max())
        })

        if alpha == 1.0:

            original_col = ORIGINALS[name][1]
            matched = sub.merge(
                saved[name],
                on=FULL,
                how="inner",
                validate="one_to_one"
            )
            assert len(matched) == 2000

            delta = np.abs(
                matched["prediction"].to_numpy()
                - matched[original_col].to_numpy()
            )

            maximum = float(delta.max())
            assert maximum < 1e-7, (
                f"{name} reconstruction mismatch: {maximum}"
            )
            max_reconstruction[name] = maximum

    all_predictions.append(result)

summary = pd.DataFrame(rows)
predictions = pd.concat(all_predictions, ignore_index=True)

assert len(summary) == 8
assert len(predictions) == 16000
assert len(max_reconstruction) == 4

# A is not refitted; its frozen benchmark is retained.
A_MAE = 0.9399
A_RMSE = 1.2066

summary["MAE_minus_A"] = summary["MAE"] - A_MAE
summary["RMSE_minus_A"] = summary["RMSE"] - A_RMSE

paired_rows = []
for alpha in [1.0, 5.0]:
    sub = summary.set_index(["model", "alpha"])

    c2 = sub.loc[("C2", alpha)]
    c1 = sub.loc[("C1", alpha)]

    paired_rows.append({
        "alpha": alpha,
        "C2_minus_C1_MAE": float(c2["MAE"] - c1["MAE"]),
        "C2_minus_C1_RMSE": float(c2["RMSE"] - c1["RMSE"]),
        "C2_minus_A_MAE": float(c2["MAE"] - A_MAE),
        "C2_minus_A_RMSE": float(c2["RMSE"] - A_RMSE),
        "C2_minus_C1_mean_range": float(
            c2["mean_prediction_range"] -
            c1["mean_prediction_range"]
        )
    })

paired = pd.DataFrame(paired_rows)

SUMMARY = OUT / "step07_03_regularization_sensitivity.csv"
PRED = OUT / "step07_03_regularization_oof_predictions.csv"
PAIRED = OUT / "step07_03_regularization_comparisons.csv"
REPORT = OUT / "step07_03_regularization_fairness.json"

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp, index=False, encoding="utf-8",
        float_format="%.12g"
    )
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing output differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(summary, SUMMARY)
safe_csv(predictions, PRED)
safe_csv(paired, PAIRED)

report = {
    "status": "PASS",
    "step": "7.3",
    "analysis": "ridge_regularization_scale_sensitivity",
    "original_alpha": 1.0,
    "relative_penalty_matched_alpha": 5.0,
    "audio_training_rows_for_A": 320,
    "hypothesis_training_rows_for_text_models": 1600,
    "unique_audios": 400,
    "unique_speakers": 20,
    "hypotheses_per_audio": 5,
    "original_OOF_reconstruction_max_difference":
        max_reconstruction,
    "new_model_selection_performed": False,
    "official_test_used": False,
    "primary_original_predictions_unchanged": True,
    "source_hashes": {
        str(path.relative_to(P)): sha(path)
        for path in SOURCE_HASHES
    },
    "output_hashes": {
        path.name: sha(path)
        for path in [SUMMARY, PRED, PAIRED]
    },
    "interpretation": (
        "Alpha 5 is an equal-per-audio objective-normalization "
        "sensitivity check, not a tuned optimum. "
        "For acoustic duplicate rows it matches the alpha 1 "
        "penalty-to-loss ratio of 320 original audio rows. "
        "Transcript features differ across hypotheses, so "
        "this does not make all model assumptions identical."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing report differs"
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("=== STEP 7.3 REGULARIZATION FAIRNESS AUDIT ===")
print("Status: PASS")
print("Original Ridge alpha:", 1.0)
print("Normalized sensitivity alpha:", 5.0)
print("Original STEP 5 predictions changed: NO")

print("\n=== ORIGINAL PREDICTION RECONSTRUCTION ===")
for name, maximum in max_reconstruction.items():
    print(name, "max absolute difference:", maximum)

print("\n=== ALPHA 1 VS ALPHA 5 ===")
print(
    summary[
        ["model", "alpha", "MAE", "RMSE",
         "mean_prediction_range", "MAE_minus_A", "RMSE_minus_A"]
    ].round(4).to_string(index=False)
)

print("\n=== C2 PAIRED COMPARISONS ===")
print(paired.round(4).to_string(index=False))

print("\nSummary CSV:", SUMMARY)
print("Predictions CSV:", PRED)
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Next: interpret fairness sensitivity before Step 7.4")


In [ ]:

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
OUT = P / "results/step07_methodology"
CHECK = P / "checkpoints/research_progress.json"

DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
PROMPT = P / "features/linguistic/prompt_asr_disagreement_0001-0400.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"

MODELS = {
    "B1": ("b1_transcript_ridge_oof_predictions.csv",
           "b1_prediction"),
    "B2": ("b2_prompt_aware_ridge_oof_predictions.csv",
           "b2_prediction"),
    "C1": ("c1_fusion_ridge_oof_predictions.csv",
           "c1_prediction"),
    "C2": ("c2_prompt_aware_fusion_ridge_oof_predictions.csv",
           "c2_prediction")
}

HASHES = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    PROMPT: "8b52805a91144c5ee0a379aeb2954595f55ec085203ac38861845c5fbe33c25c",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

MODEL_HASHES = {
    "B1": "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    "B2": "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
    "C1": "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    "C2": "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

for path, expected in HASHES.items():
    assert path.is_file() and sha(path) == expected

design = json.loads(DESIGN.read_text(encoding="utf-8"))
features = list(design["prompt_feature_names"])

assert len(features) == 2

data = read(DATA)
prompt = read(PROMPT)

KEYS = ["pilot_index", "audio_id", "speaker", "hypothesis_id"]
IDS = ["pilot_index", "audio_id", "speaker"]
FULL = KEYS + ["fluency", "validation_fold"]

assert len(data) == len(prompt) == 2000
assert not data.duplicated(KEYS).any()
assert not prompt.duplicated(KEYS).any()
assert prompt[features].notna().all().all()
assert np.isfinite(prompt[features].to_numpy(dtype=float)).all()
assert (prompt[features] >= 0).all().all()

aligned = data[KEYS + ["fluency"] + features].merge(
    prompt[KEYS + features],
    on=KEYS,
    how="inner",
    validate="one_to_one",
    suffixes=("_model", "_source")
)

assert len(aligned) == 2000

for feature in features:
    assert np.allclose(
        aligned[f"{feature}_model"],
        aligned[f"{feature}_source"],
        rtol=0, atol=1e-10
    )

# Original frozen OOF predictions, never retrained.
base = None

for name, (filename, pred_col) in MODELS.items():
    path = R / filename
    assert sha(path) == MODEL_HASHES[name]

    frame = read(path)
    assert len(frame) == 2000

    if base is None:
        base = frame[FULL + [pred_col]].copy()
    else:
        base = base.merge(
            frame[FULL + [pred_col]],
            on=FULL,
            how="inner",
            validate="one_to_one"
        )

assert len(base) == 2000

base = base.merge(
    data[KEYS + features],
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

assert len(base) == 2000
assert base["audio_id"].nunique() == 400
assert base["speaker"].nunique() == 20

for name, (_, col) in MODELS.items():
    base[f"{name}_ae"] = (base[col] - base["fluency"]).abs()
    base[f"{name}_se"] = (base[col] - base["fluency"]) ** 2

# Each audio is represented by five hypotheses.
agg = {}
for f in features:
    agg[f"{f}_mean"] = (f, "mean")
    agg[f"{f}_max"] = (f, "max")
    agg[f"{f}_range"] = (
        f, lambda x: x.max() - x.min()
    )

for name, (_, col) in MODELS.items():
    agg[f"{name}_ae"] = (f"{name}_ae", "mean")
    agg[f"{name}_se"] = (f"{name}_se", "mean")
    agg[f"{name}_range"] = (
        col, lambda x: x.max() - x.min()
    )

audio = base.groupby(
    IDS + ["fluency", "validation_fold"],
    as_index=False
).agg(**agg)

assert len(audio) == 400
assert audio["audio_id"].is_unique

# Audio-level paired error differences.
audio["B2_minus_B1_AE"] = (
    audio["B2_ae"] - audio["B1_ae"]
)
audio["C2_minus_C1_AE"] = (
    audio["C2_ae"] - audio["C1_ae"]
)
audio["B2_minus_B1_range"] = (
    audio["B2_range"] - audio["B1_range"]
)
audio["C2_minus_C1_range"] = (
    audio["C2_range"] - audio["C1_range"]
)

word_feature = "prompt_asr_word_disagreement"
char_feature = "prompt_asr_char_disagreement"

assert word_feature in features
assert char_feature in features

# Quartiles are DESCRIPTIVE strata defined on these 400
# audio means; no model fitting or threshold optimization.
audio["word_disagreement_quartile"] = pd.qcut(
    audio[f"{word_feature}_mean"].rank(method="first"),
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

strata = audio.groupby(
    "word_disagreement_quartile",
    observed=True
).agg(
    audios=("audio_id", "count"),
    speakers=("speaker", "nunique"),
    mean_word_disagreement=(
        f"{word_feature}_mean", "mean"
    ),
    mean_char_disagreement=(
        f"{char_feature}_mean", "mean"
    ),
    B2_minus_B1_MAE=("B2_minus_B1_AE", "mean"),
    C2_minus_C1_MAE=("C2_minus_C1_AE", "mean"),
    B2_minus_B1_mean_range=(
        "B2_minus_B1_range", "mean"
    ),
    C2_minus_C1_mean_range=(
        "C2_minus_C1_range", "mean"
    )
).reset_index()

assert len(strata) == 4
assert strata["audios"].sum() == 400

descriptive = {
    "word_disagreement_mean": float(base[word_feature].mean()),
    "word_disagreement_median": float(base[word_feature].median()),
    "word_disagreement_max": float(base[word_feature].max()),
    "char_disagreement_mean": float(base[char_feature].mean()),
    "char_disagreement_median": float(base[char_feature].median()),
    "char_disagreement_max": float(base[char_feature].max()),
    "exact_prompt_matches": int(
        ((base[word_feature] == 0) &
         (base[char_feature] == 0)).sum()
    ),
    "audio_word_disagreement_variable": int(
        (audio[f"{word_feature}_range"] > 1e-12).sum()
    ),
    "audio_char_disagreement_variable": int(
        (audio[f"{char_feature}_range"] > 1e-12).sum()
    )
}

assert descriptive["audio_word_disagreement_variable"] == 131
assert descriptive["audio_char_disagreement_variable"] == 159

primary = {}
for name in MODELS:
    primary[name] = {
        "MAE": float(audio[f"{name}_ae"].mean()),
        "RMSE": float(np.sqrt(audio[f"{name}_se"].mean())),
        "mean_range": float(audio[f"{name}_range"].mean())
    }

expected_mae = {
    "B1": 1.2426, "B2": 1.1761,
    "C1": 0.8972, "C2": 0.7782
}
for name, value in expected_mae.items():
    assert np.isclose(
        primary[name]["MAE"], value, atol=1e-4
    )

OUT.mkdir(parents=True, exist_ok=True)

AUDIO_OUT = OUT / "step07_04_prompt_audit_per_audio.csv"
STRATA_OUT = OUT / "step07_04_prompt_disagreement_strata.csv"
REPORT_OUT = OUT / "step07_04_prompt_validity_audit.json"

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp, index=False, encoding="utf-8",
        float_format="%.12g"
    )
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing output differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(audio, AUDIO_OUT)
safe_csv(strata, STRATA_OUT)

report = {
    "status": "PASS",
    "step": "7.4",
    "analysis": "prompt_feature_validity_and_ablation",
    "original_audio_count": 400,
    "hypothesis_count": 2000,
    "speaker_count": 20,
    "feature_names": features,
    "feature_values_match_original_source": True,
    "descriptive_statistics": descriptive,
    "original_model_metrics": primary,
    "observational_only": True,
    "model_retraining": False,
    "ground_truth_wer_or_cer": False,
    "reading_prompt_is_verified_transcript": False,
    "operational_prompt_availability_verified": False,
    "quartiles": "equal_audio_count_descriptive",
    "source_sha256": {
        str(path.relative_to(P)): sha(path)
        for path in HASHES
    },
    "output_sha256": {
        AUDIO_OUT.name: sha(AUDIO_OUT),
        STRATA_OUT.name: sha(STRATA_OUT)
    },
    "limitations": [
        "Disagreement with a reading prompt is not ASR WER/CER.",
        "Deviations may reflect learner speech or ASR errors.",
        "Prompt feature availability in deployment is unverified.",
        "Quartile comparisons are descriptive, not causal.",
        "Feature-group gains are not isolated causal contributions.",
        "All five ASR hypotheses share one human label."
    ]
}

if REPORT_OUT.exists():
    assert json.loads(
        REPORT_OUT.read_text(encoding="utf-8")
    ) == report
else:
    temp = REPORT_OUT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT_OUT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step7_04_prompt_validity_audit"] = "qc_pass"
progress["step7_04_prompt_validity_sha256"] = sha(REPORT_OUT)
progress["next_task"] = (
    "Review prompt-aware feature interpretation "
    "and assess statistical claim boundaries in STEP 7.5"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 7.4 PROMPT FEATURE VALIDITY AUDIT ===")
print("Status: PASS")
print("Original observations retained: YES")
print("Model retraining: NO")
print("Ground-truth WER/CER: NO")

print("\n=== PROMPT DISAGREEMENT DISTRIBUTION ===")
for key, value in descriptive.items():
    print(f"{key}: {value:.4f}"
          if isinstance(value, float) else f"{key}: {value}")

print("\n=== ORIGINAL MODEL METRICS ===")
for name, value in primary.items():
    print(
        f"{name}: MAE={value['MAE']:.4f}, "
        f"RMSE={value['RMSE']:.4f}, "
        f"mean_range={value['mean_range']:.4f}"
    )

print("\n=== PROMPT DISAGREEMENT QUARTILES ===")
print(strata.round(4).to_string(index=False))

print("\n=== IMPORTANT INTERPRETATION ===")
print("Reading prompt != verified human transcript")
print("Prompt disagreement != ground-truth WER/CER")
print("Operational prompt availability: NOT VERIFIED")
print("Feature ablations: OBSERVATIONAL")

print("\nReport:", REPORT_OUT)
print("SHA256:", sha(REPORT_OUT))
print("Audio CSV:", AUDIO_OUT)
print("Quartile CSV:", STRATA_OUT)
print("Next: STEP 7.5 statistical claims audit")


In [ ]:

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
OUT = P / "results/step07_methodology"
OUT.mkdir(parents=True, exist_ok=True)

BOOT = R / "speaker_cluster_bootstrap_10000_ci.csv"
REG = OUT / "step07_03_regularization_sensitivity.csv"
PROMPT = OUT / "step07_04_prompt_disagreement_strata.csv"
CHECK = P / "checkpoints/research_progress.json"

CLAIMS = OUT / "step07_05_statistical_claims.csv"
REPORT = OUT / "step07_05_statistical_claims_audit.json"

EXPECTED = {
    BOOT: "5ef3d6dd01eb05758ccb8b4f77293396a81e988b9b9e33152cb666c6b48d1d93",
    PROMPT: "2d1f90d7d6d87a82a181bba300c14bee14e9c468ce0dd98b5a37877ab3709326"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert sha(BOOT) == EXPECTED[BOOT]

# PROMPT hash above identifies the audit JSON, not the CSV.
PROMPT_REPORT = OUT / "step07_04_prompt_validity_audit.json"
assert sha(PROMPT_REPORT) == EXPECTED[PROMPT_REPORT] if (
    PROMPT_REPORT in EXPECTED
) else sha(PROMPT_REPORT) == EXPECTED[PROMPT]

prompt_report = json.loads(
    PROMPT_REPORT.read_text(encoding="utf-8")
)
assert prompt_report["status"] == "PASS"
assert sha(PROMPT) == prompt_report["output_sha256"][PROMPT.name]

reg_report_path = OUT / "step07_03_regularization_fairness.json"
reg_report = json.loads(
    reg_report_path.read_text(encoding="utf-8")
)
assert reg_report["status"] == "PASS"
assert sha(REG) == reg_report["output_hashes"][REG.name]

boot = pd.read_csv(BOOT)
reg = pd.read_csv(REG)
strata = pd.read_csv(PROMPT)

assert len(boot) == 33
assert len(reg) == 8
assert len(strata) == 4

rows = []

COMPARISONS = [
    ("C1_minus_A", "MAE"),
    ("C2_minus_A", "MAE"),
    ("C2_minus_A", "RMSE"),
    ("C2_minus_C1", "MAE"),
    ("C2_minus_C1", "RMSE"),
    ("C2_minus_C1", "mean_range"),
    ("B2_minus_B1", "MAE"),
    ("B2_minus_B1", "mean_range"),
    ("C1_minus_B1", "MAE"),
    ("C2_minus_B2", "MAE")
]

for comparison, metric in COMPARISONS:
    matches = boot[
        boot["comparison"].eq(comparison) &
        boot["metric"].eq(metric)
    ]
    assert len(matches) == 1
    r = matches.iloc[0]

    lower = float(r["ci_2_5"])
    upper = float(r["ci_97_5"])
    diff = float(r["observed"])

    status = (
        "CI_excludes_zero"
        if lower > 0 or upper < 0
        else "CI_includes_zero"
    )

    rows.append({
        "claim": f"{comparison}_{metric}",
        "analysis": "original_fixed_alpha_1",
        "metric": metric,
        "observed_difference": diff,
        "CI_low": lower,
        "CI_high": upper,
        "evidence_status": status,
        "interpretation_scope": (
            "Paired 20-speaker bootstrap; "
            "conditional on frozen OOF predictions"
        )
    })

# Alpha=5 is secondary methodological sensitivity.
alpha5 = reg[reg["alpha"].eq(5.0)].set_index("model")
assert set(alpha5.index) == {"B1", "B2", "C1", "C2"}

for left, right in [("C2", "C1"), ("B2", "B1")]:
    for metric in ["MAE", "RMSE", "mean_prediction_range"]:
        difference = float(
            alpha5.loc[left, metric] -
            alpha5.loc[right, metric]
        )

        rows.append({
            "claim": f"{left}_minus_{right}_{metric}",
            "analysis": "alpha_5_sensitivity",
            "metric": metric,
            "observed_difference": difference,
            "CI_low": np.nan,
            "CI_high": np.nan,
            "evidence_status": "descriptive_only",
            "interpretation_scope": (
                "Secondary alpha normalization check; "
                "no speaker bootstrap CI computed"
            )
        })

# Prompt quartiles are descriptive post-hoc strata.
for _, row in strata.iterrows():
    rows.append({
        "claim": f"{row['word_disagreement_quartile']}_C2_minus_C1_MAE",
        "analysis": "prompt_quartile",
        "metric": "MAE",
        "observed_difference": float(
            row["C2_minus_C1_MAE"]
        ),
        "CI_low": np.nan,
        "CI_high": np.nan,
        "evidence_status": "descriptive_only",
        "interpretation_scope": (
            "Quartiles of observed audio-level "
            "prompt disagreement; not causal"
        )
    })

claims = pd.DataFrame(rows)
assert len(claims) == 20

# Validate selected core claim direction and CI status.
core = claims.set_index("claim")

assert (
    core.loc["C2_minus_A_MAE", "evidence_status"]
    == "CI_excludes_zero"
)
assert core.loc["C2_minus_A_MAE", "CI_high"] < 0

assert (
    core.loc["C2_minus_A_RMSE", "evidence_status"]
    == "CI_includes_zero"
)

assert (
    core.loc["C2_minus_C1_mean_range", "evidence_status"]
    == "CI_excludes_zero"
)
assert core.loc["C2_minus_C1_mean_range", "CI_low"] > 0

temp = CLAIMS.with_suffix(".tmp")
claims.to_csv(
    temp, index=False, encoding="utf-8",
    float_format="%.12g"
)

if CLAIMS.exists():
    assert sha(CLAIMS) == sha(temp)
    temp.unlink()
else:
    temp.replace(CLAIMS)

report = {
    "status": "PASS",
    "step": "7.5",
    "analysis": "statistical_claims_and_generalization_audit",
    "original_audio_count": 400,
    "independent_speaker_clusters": 20,
    "asr_hypothesis_rows": 2000,
    "bootstrap_iterations": 10000,
    "bootstrap_refitting": False,
    "multiple_comparison_adjustment": False,
    "formal_familywise_significance_claims": False,
    "alpha5_CI_available": False,
    "official_test_evaluated": False,
    "claim_count": len(claims),
    "source_sha256": {
        "bootstrap": sha(BOOT),
        "regularization": sha(REG),
        "prompt_strata": sha(PROMPT)
    },
    "claims_sha256": sha(CLAIMS),
    "approved_interpretation": [
        "C2 shows lower observed MAE than A and C1.",
        "The paired speaker-bootstrap MAE intervals exclude zero.",
        "C2 does not demonstrate an RMSE advantage over A "
        "in the original alpha=1 evaluation.",
        "C2 has greater mean prediction range than C1.",
        "Alpha=5 supports descriptive regularization robustness.",
        "Prompt disagreement does not measure verified ASR WER/CER."
    ],
    "limitations": [
        "Only 20 speaker clusters.",
        "Bootstrap conditional on fixed OOF predictions.",
        "Multiple comparisons were not multiplicity-adjusted.",
        "No external dataset validation.",
        "Official test partition remains sealed.",
        "Post-hoc model and subset comparisons are exploratory.",
        "Prompt feature deployment availability is not established."
    ]
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step7_05_statistical_claims"] = "qc_pass"
progress["step7_05_statistical_claims_sha256"] = sha(CLAIMS)
progress["next_task"] = (
    "STEP 7.6 reviewer readiness and final "
    "methodological decisions"
)
CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 7.5 STATISTICAL CLAIMS AUDIT ===")
print("Status: PASS")
print("Claim records:", len(claims))
print("Speaker clusters:", 20)
print("Multiple-testing adjustment: NO")
print("Official test evaluated: NO")

print("\n=== ORIGINAL MODEL CLAIMS ===")
print(
    claims[
        claims["analysis"].eq("original_fixed_alpha_1")
    ][
        ["claim", "observed_difference",
         "CI_low", "CI_high", "evidence_status"]
    ].round(4).to_string(index=False)
)

print("\n=== SECONDARY ALPHA=5 CLAIMS ===")
print(
    claims[
        claims["analysis"].eq("alpha_5_sensitivity")
    ][
        ["claim", "observed_difference",
         "evidence_status"]
    ].round(4).to_string(index=False)
)

print("\n=== PROMPT QUARTILE CLAIMS ===")
print(
    claims[
        claims["analysis"].eq("prompt_quartile")
    ][
        ["claim", "observed_difference",
         "evidence_status"]
    ].round(4).to_string(index=False)
)

print("\n=== LIMITATIONS ===")
for item in report["limitations"]:
    print("-", item)

print("\nClaims CSV:", CLAIMS)
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Next: STEP 7.6 reviewer readiness")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step07_methodology"
CHECK = P / "checkpoints/research_progress.json"

SOURCES = {
    "7.1": OUT / "step07_01_dataset_integrity.json",
    "7.2": OUT / "step07_02_leakage_audit.json",
    "7.3": OUT / "step07_03_regularization_fairness.json",
    "7.4": OUT / "step07_04_prompt_validity_audit.json",
    "7.5": OUT / "step07_05_statistical_claims_audit.json",
}

KNOWN_HASHES = {
    "7.1": "9b48a723f15fc080c86e23a9e3b61b6183917818de4badc79770080a46eb8a31",
    "7.2": "d0deb85ae53f034664f870132913cea3498b26a22caed7c042a66e8a41d900d0",
    "7.3": "7a10748ff0e91cd199ea0b5242aae28da1f7f7aba5deeec03f86ae00b14a701f",
    "7.4": "2d1f90d7d6d87a82a181bba300c14bee14e9c468ce0dd98b5a37877ab3709326",
    "7.5": "f0b9f8aedcf1e72ea467cff21bde9d33836b43bde3c5a81cb8a0863f86ad2f8a"
}

REGISTRY = OUT / "step07_06_reviewer_decision_registry.csv"
REPORT = OUT / "step07_06_reviewer_readiness.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

audits = {}

for step, path in SOURCES.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha(path) == KNOWN_HASHES[step], (
        f"SHA mismatch: STEP {step}"
    )

    audits[step] = json.loads(
        path.read_text(encoding="utf-8")
    )
    assert audits[step]["status"] == "PASS"

assert audits["7.1"]["speaker_disjoint"]
assert audits["7.2"]["speaker_disjoint"]
assert audits["7.2"]["audio_disjoint"]
assert audits["7.2"]["imputation_fit_on_training_fold"]
assert audits["7.2"]["scaler_fit_on_training_fold"]

assert (
    audits["7.3"]["primary_original_predictions_unchanged"]
)
assert audits["7.4"]["ground_truth_wer_or_cer"] is False
assert audits["7.5"]["multiple_comparison_adjustment"] is False

# These are methodological review decisions,
# not new experimentally verified findings.
DECISIONS = [
    (
        "D01", "Dataset integrity", "verified",
        "Use 400 audios, 20 speakers and frozen splits.",
        "7.1", "Methods"
    ),
    (
        "D02", "Downstream leakage controls", "verified",
        "Report speaker-disjoint CV and train-only preprocessing.",
        "7.2", "Methods"
    ),
    (
        "D03", "OOF model reproducibility", "verified",
        "Retain saved predictions and reconstruction evidence.",
        "7.2", "Supplementary"
    ),
    (
        "D04", "Regularization normalization", "verified_with_scope",
        "Report alpha=1 primary and alpha=5 sensitivity; "
        "do not select alpha based on observed validation results.",
        "7.3", "Methods and Supplementary"
    ),
    (
        "D05", "Prompt-aware model interpretation",
        "verified_with_scope",
        "Treat B2/C2 as prompt-conditional secondary models; "
        "avoid ground-truth WER/CER terminology.",
        "7.4", "Methods and Discussion"
    ),
    (
        "D06", "Bootstrap uncertainty", "verified_with_scope",
        "Report paired speaker-cluster intervals; "
        "state no model refitting or multiplicity correction.",
        "7.5", "Results and Limitations"
    ),
    (
        "D07", "Original sample selection provenance",
        "requires_documentation",
        "Locate and preserve initial 20-speaker sampling code, "
        "selection criteria and random seed, if used.",
        "7.1", "Methods and Reproducibility"
    ),
    (
        "D08", "ASR decoding reproducibility",
        "requires_documentation",
        "Locate exact model revision and H1-H5 decoding "
        "parameters from original notebooks and configs.",
        "STEP 2", "Methods and Supplementary"
    ),
    (
        "D09", "ASR pretraining overlap",
        "unverified_external_limit",
        "Do not assert absence of overlap with ASR pretraining.",
        "7.2", "Limitations"
    ),
    (
        "D10", "Prompt availability in deployment",
        "requires_scope_statement",
        "Define the application as read-aloud scoring where "
        "the reading prompt is accessible when using B2/C2.",
        "7.4", "Methods and Discussion"
    ),
    (
        "D11", "External validation",
        "not_performed",
        "Describe conclusions as pilot results; "
        "do not generalize to other datasets or languages.",
        "7.5", "Limitations"
    ),
    (
        "D12", "Multiple statistical comparisons",
        "verified_with_scope",
        "Present intervals descriptively rather than claiming "
        "familywise-controlled statistical significance.",
        "7.5", "Results and Limitations"
    ),
    (
        "D13", "Anomaly exclusions",
        "verified_with_scope",
        "Use all 400 audios in primary analyses; "
        "report 399/396 results as post-hoc sensitivity.",
        "STEP 6.2", "Results and Supplementary"
    ),
    (
        "D14", "Primary research emphasis",
        "methodological_decision",
        "Present A/B1/C1 as prompt-independent primary "
        "comparisons and B2/C2 as prompt-aware secondary "
        "ablations; avoid post-hoc winner selection.",
        "7.4 and 7.5", "Article structure"
    )
]

registry = pd.DataFrame(
    DECISIONS,
    columns=[
        "decision_id", "issue", "status",
        "required_action_or_reporting_rule",
        "evidence", "manuscript_section"
    ]
)

assert len(registry) == 14
assert registry["decision_id"].is_unique

unresolved_statuses = {
    "requires_documentation",
    "requires_scope_statement",
    "unverified_external_limit",
    "not_performed"
}

unresolved = registry[
    registry["status"].isin(unresolved_statuses)
]

# Audit is complete, but submission readiness
# remains conditional on unresolved documentation.
readiness = "CONDITIONAL_NOT_SUBMISSION_READY"

OUT.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp, index=False, encoding="utf-8"
    )

    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing registry differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

safe_csv(registry, REGISTRY)

report = {
    "status": "PASS",
    "step": "7.6",
    "analysis": "reviewer_readiness_and_method_decisions",
    "methodological_audits_completed": list(SOURCES),
    "research_sample": {
        "speakers": 20,
        "audios": 400,
        "asr_hypotheses": 2000
    },
    "decision_count": len(registry),
    "unresolved_or_limited_items": unresolved[
        "decision_id"
    ].tolist(),
    "manuscript_readiness": readiness,
    "model_retraining": False,
    "official_test_used_in_this_cell": False,
    "original_results_changed": False,
    "source_report_sha256": {
        step: sha(path)
        for step, path in SOURCES.items()
    },
    "registry_sha256": sha(REGISTRY),
    "next_recommended_action": (
        "Resolve sample-selection and ASR-decoding "
        "documentation before final Methods drafting; "
        "prepare figures, literature review and manuscript."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing readiness report differs"
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(
            report, indent=2, ensure_ascii=False
        ),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step7_06_reviewer_readiness"] = "qc_pass"
progress["step7_06_registry_sha256"] = sha(REGISTRY)
progress["step7_methodology_status"] = "completed_with_open_items"
progress["next_task"] = (
    "Archive STEP 7 and resolve sampling/ASR "
    "provenance before manuscript finalization"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=== STEP 7.6 REVIEWER READINESS ===")
print("Status: PASS")
print("STEP 7.1-7.5 source audit checks: PASS")
print("Decision records:", len(registry))
print("Manuscript readiness:", readiness)

print("\n=== REVIEWER DECISION REGISTRY ===")
print(
    registry[
        ["decision_id", "issue", "status"]
    ].to_string(index=False)
)

print("\n=== UNRESOLVED OR LIMITED ITEMS ===")
print(
    unresolved[
        [
            "decision_id", "issue",
            "required_action_or_reporting_rule"
        ]
    ].to_string(index=False)
)

print("\nRegistry:", REGISTRY)
print("Registry SHA256:", sha(REGISTRY))
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Checkpoint updated.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import zipfile

ROOT = Path("/kaggle/working/acoustic_linguistic_research")
WORK = Path("/kaggle/working")
ARCHIVE = WORK / "asr_fluency_step01-07_complete.zip"
TEMP = WORK / "asr_fluency_step01-07_complete.zip.tmp"

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

REQUIRED = {
    "results/step07_methodology/step07_01_dataset_integrity.json":
        "9b48a723f15fc080c86e23a9e3b61b6183917818de4badc79770080a46eb8a31",
    "results/step07_methodology/step07_02_leakage_audit.json":
        "d0deb85ae53f034664f870132913cea3498b26a22caed7c042a66e8a41d900d0",
    "results/step07_methodology/step07_03_regularization_fairness.json":
        "7a10748ff0e91cd199ea0b5242aae28da1f7f7aba5deeec03f86ae00b14a701f",
    "results/step07_methodology/step07_04_prompt_validity_audit.json":
        "2d1f90d7d6d87a82a181bba300c14bee14e9c468ce0dd98b5a37877ab3709326",
    "results/step07_methodology/step07_05_statistical_claims_audit.json":
        "f0b9f8aedcf1e72ea467cff21bde9d33836b43bde3c5a81cb8a0863f86ad2f8a",
    "results/step07_methodology/step07_06_reviewer_readiness.json":
        "a74ad6f9c3fd7b54ba92575c90914054e0ddd59079d30686f436c3989e1c6c6f",
    "results/step07_methodology/step07_06_reviewer_decision_registry.csv":
        "eb998dbe6a6757940985202166be4793480fd56851a014bb6b6deddf4eeed08b"
}

assert ROOT.is_dir()

print("=== VERIFY STEP 7 AUDITS ===")
for rel, expected in REQUIRED.items():
    path = ROOT / rel
    assert path.is_file(), f"Missing: {rel}"
    assert sha256(path) == expected, f"SHA mismatch: {rel}"
    print("PASS:", rel)

readiness_path = ROOT / (
    "results/step07_methodology/"
    "step07_06_reviewer_readiness.json"
)
readiness = json.loads(readiness_path.read_text(encoding="utf-8"))

assert readiness["status"] == "PASS"
assert readiness["manuscript_readiness"] == (
    "CONDITIONAL_NOT_SUBMISSION_READY"
)

items = []
omitted = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path.is_symlink():
        continue

    rel = path.relative_to(ROOT).as_posix()
    lower = rel.lower()

    if "official_test" in lower or "speechocean762_test" in lower:
        omitted.append((path, "Sealed official test excluded"))
        continue

    if rel.startswith("data/raw/") and path.suffix.lower() == ".parquet":
        omitted.append((path, "Original train parquet excluded"))
        continue

    if (
        "__pycache__" in path.parts
        or ".ipynb_checkpoints" in path.parts
        or path.suffix.lower() in [".tmp", ".pyc"]
    ):
        continue

    items.append((path, "project/" + rel))

PREVIOUS = WORK / "asr_fluency_step01-06_complete.zip"
if PREVIOUS.is_file():
    items.append(
        (PREVIOUS, "extra/asr_fluency_step01-06_complete.zip")
    )

# Include available notebooks/scripts from Kaggle working directory.
for path in sorted(WORK.iterdir()):
    if (
        path.is_file()
        and path.suffix.lower() in [".ipynb", ".py", ".r"]
    ):
        items.append((path, "extra/source_files/" + path.name))

assert len(items) > 0
assert len(set(arc for _, arc in items)) == len(items)

entries = [
    {
        "archive_path": arc,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path)
    }
    for path, arc in items
]

manifest = {
    "project": "ASR fluency accuracy and stability research",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "completed_steps": [1, 2, 3, 4, 5, 6, 7],
    "step07_audits": "PASS",
    "manuscript_readiness": "CONDITIONAL_NOT_SUBMISSION_READY",
    "unresolved_items": readiness["unresolved_or_limited_items"],
    "dataset": "mispeech/speechocean762",
    "dataset_revision":
        "06385584fad212b26134c656fdd3ccf9f093f33e",
    "original_audios": 400,
    "speakers": 20,
    "asr_hypotheses": 2000,
    "primary_results_unchanged": True,
    "official_test_partition": "not evaluated",
    "included_files": entries,
    "omitted_files": [
        {
            "original_path": str(path),
            "reason": reason,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path)
        }
        for path, reason in omitted
    ]
}

instructions = """
RESTORE INSTRUCTIONS — STEP 1 TO STEP 7

1. Upload this ZIP to a Kaggle session.
2. Extract project/ into:
   /kaggle/working/acoustic_linguistic_research/
3. Verify SHA256 values using MANIFEST.json.
4. STEP 1-7 calculations and audits are completed.
5. Do not automatically retrain or overwrite original models.
6. Review D07 and D08 before final manuscript Methods.
7. Official test partition remains sealed.
8. The original train parquet is excluded.
9. Export the original Kaggle notebook separately as .ipynb.
10. Continue with provenance verification and STEP 8 figures.
"""

# If already created, validate instead of silently overwriting.
if ARCHIVE.exists():
    with zipfile.ZipFile(ARCHIVE, "r") as z:
        assert z.testzip() is None
        old_manifest = json.loads(
            z.read("MANIFEST.json").decode("utf-8")
        )

        required_archive_paths = {
            "project/" + rel for rel in REQUIRED
        }
        assert required_archive_paths.issubset(
            set(z.namelist())
        )

        for entry in old_manifest["included_files"]:
            h = hashlib.sha256()
            with z.open(entry["archive_path"]) as f:
                for block in iter(
                    lambda: f.read(1024 * 1024), b""
                ):
                    h.update(block)
            assert h.hexdigest() == entry["sha256"]

    print("\nExisting STEP 7 backup verified.")
else:
    with zipfile.ZipFile(
        TEMP, "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
        allowZip64=True
    ) as z:
        for path, arc in items:
            z.write(path, arcname=arc)

        z.writestr(
            "MANIFEST.json",
            json.dumps(manifest, indent=2, ensure_ascii=False)
        )
        z.writestr("RESTORE_INSTRUCTIONS.txt", instructions)

    with zipfile.ZipFile(TEMP, "r") as z:
        assert z.testzip() is None
        for entry in entries:
            h = hashlib.sha256()
            with z.open(entry["archive_path"]) as f:
                for block in iter(
                    lambda: f.read(1024 * 1024), b""
                ):
                    h.update(block)
            assert h.hexdigest() == entry["sha256"], (
                entry["archive_path"]
            )

    os.replace(TEMP, ARCHIVE)

with zipfile.ZipFile(ARCHIVE) as z:
    final_manifest = json.loads(
        z.read("MANIFEST.json").decode("utf-8")
    )
    file_count = len(final_manifest["included_files"])
    omitted_count = len(final_manifest["omitted_files"])

print("\n=== STEP 7 FINAL CHECKPOINT ===")
print("Status: PASS")
print("STEP 1-7: COMPLETE")
print("Manuscript readiness: CONDITIONAL")
print("Included files:", file_count)
print("Omitted files:", omitted_count)
print("ZIP integrity: PASS")
print("SHA256 verification: PASS")
print("Archive:", ARCHIVE)
print("Size MB:", round(ARCHIVE.stat().st_size / 1024**2, 2))
print("SHA256:", sha256(ARCHIVE))
print("Unresolved items:",
      final_manifest["unresolved_items"])
print("\nNEXT: Download ZIP and export Kaggle notebook.")


In [ ]:

from pathlib import Path
import hashlib
import json
import re

import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

INVENTORY = OUT / "step08_01_provenance_inventory.csv"
REPORT = OUT / "step08_01_provenance_inventory.json"

EXTENSIONS = {".json", ".py", ".ipynb", ".yaml", ".yml", ".toml", ".txt", ".md"}
MAX_FILE_SIZE = 5 * 1024 * 1024

PATTERNS = {
    "sampling": re.compile(
        r"sample|speaker|selection|random_state|"
        r"random.seed|default_rng|pilot_index|"
        r"train_test_split|stratified",
        re.IGNORECASE
    ),
    "asr": re.compile(
        r"whisper|decode|generate|num_beams|"
        r"temperature|top_k|top_p|"
        r"do_sample|language|task|"
        r"return_timestamps|forced_decoder_ids|"
        r"model_revision|model_id|"
        r"repetition_penalty|no_repeat_ngram",
        re.IGNORECASE
    ),
    "revision": re.compile(
        r"06385584fad212b26134c656fdd3ccf9f093f33e|"
        r"e8727524f962ee844a7319d92be39ac1bd25655a|"
        r"revision|commit_hash|seed",
        re.IGNORECASE
    )
}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_text(path):
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return None

files = []

for root in [P, Path("/kaggle/working")]:
    for path in root.glob("*") if root != P else root.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in EXTENSIONS:
            continue
        if path.stat().st_size > MAX_FILE_SIZE:
            continue
        if ".ipynb_checkpoints" in path.parts:
            continue
        files.append(path)

files = sorted(set(files))

records = []
searched_count = 0

for path in files:
    raw = safe_text(path)
    if raw is None:
        continue

    searched_count += 1

    if path.suffix.lower() == ".ipynb":
        try:
            notebook = json.loads(raw)
            parts = []
            for idx, cell in enumerate(notebook.get("cells", [])):
                if cell.get("cell_type") != "code":
                    continue
                source = cell.get("source", [])
                if isinstance(source, list):
                    source = "".join(source)
                parts.append((f"code_cell_{idx}", source))
        except Exception:
            continue
    else:
        parts = [("file", raw)]

    for section, content in parts:
        lines = content.splitlines()

        for line_number, line in enumerate(lines, 1):
            matched = [
                name for name, pattern in PATTERNS.items()
                if pattern.search(line)
            ]

            if not matched:
                continue

            # Restrict to relevant snippets without losing
            # source file and line references.
            records.append({
                "file": str(path),
                "section": section,
                "line": line_number,
                "categories": ",".join(matched),
                "snippet": line.strip()[:350],
                "file_sha256": sha256(path)
            })

inventory = pd.DataFrame(
    records,
    columns=[
        "file", "section", "line",
        "categories", "snippet", "file_sha256"
    ]
)

inventory.to_csv(INVENTORY, index=False, encoding="utf-8")

# Prioritize configurations and code rather than
# interpreting keyword occurrences as proof.
priority = inventory[
    inventory["snippet"].str.contains(
        r"num_beams|do_sample|temperature|"
        r"random_state|default_rng|"
        r"from_pretrained|revision=|"
        r"speaker.*sample|sample.*speaker",
        case=False,
        regex=True,
        na=False
    )
].copy()

priority = priority.head(60)

report = {
    "status": "INVENTORY_COMPLETED",
    "purpose": "Locate D07 and D08 documentary evidence",
    "files_searched": searched_count,
    "matching_lines": len(inventory),
    "priority_lines": len(priority),
    "sampling_verified": False,
    "asr_configuration_verified": False,
    "no_models_retrained": True,
    "no_asr_inference": True,
    "inventory_sha256": sha256(INVENTORY),
    "caution": (
        "Keyword matches are leads, not validated provenance. "
        "Source code and configurations must be inspected "
        "before D07 or D08 can be marked resolved."
    )
}

REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 8.1 PROVENANCE INVENTORY ===")
print("Status:", report["status"])
print("Files searched:", searched_count)
print("Matching lines:", len(inventory))
print("Sampling verified: NO")
print("ASR configuration verified: NO")

print("\n=== MATCHES BY CATEGORY ===")
for category in PATTERNS:
    count = inventory["categories"].str.contains(
        category, regex=False
    ).sum()
    print(category, ":", int(count))

print("\n=== PRIORITY SOURCE EVIDENCE ===")
if len(priority):
    print(
        priority[
            ["file", "section", "line", "snippet"]
        ].to_string(index=False, max_colwidth=110)
    )
else:
    print("No priority source lines found.")

print("\n=== SOURCE FILES WITH MOST MATCHES ===")
if len(inventory):
    print(
        inventory["file"]
        .value_counts()
        .head(15)
        .to_string()
    )

print("\nInventory:", INVENTORY)
print("Inventory SHA256:", sha256(INVENTORY))
print("Report:", REPORT)
print("Next: inspect actual source evidence for D07 and D08")


In [ ]:

from pathlib import Path
import hashlib
import json

P = Path("/kaggle/working/acoustic_linguistic_research")

SOURCES = {
    "D07_step01_report": P / "reports/step01_report.md",
    "D07_split_candidate_qc": P / "reports/step05_speaker_split_candidate_qc.json",
    "D07_speaker_independence": P / "reports/speaker_independence_report.json",
    "D07_frozen_cv": P / "configs/speaker_disjoint_cv_v1.json",
    "D08_asr_config": P / "configs/asr_hypothesis_pilot_v1.json",
    "D08_asr_freeze": P / "reports/step03_asr_configuration_freeze.json",
    "D08_whisper_revision": P / "reports/step03_whisper_model_revision.json",
    "D08_beam_test": P / "reports/step03_whisper_beam_test.json"
}

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

evidence = {}

for label, path in SOURCES.items():
    print("\n" + "=" * 65)
    print("SOURCE:", label)
    print("FILE:", path)

    if not path.is_file():
        print("STATUS: MISSING")
        evidence[label] = {"status": "MISSING"}
        continue

    digest = sha256(path)
    print("SHA256:", digest)

    text = path.read_text(encoding="utf-8")
    print("SIZE:", len(text), "characters")

    if path.suffix.lower() == ".json":
        obj = json.loads(text)
        formatted = json.dumps(
            obj,
            indent=2,
            ensure_ascii=False
        )
    else:
        formatted = text

    # Avoid losing important data to notebook output truncation.
    if len(formatted) > 10000:
        print(formatted[:10000])
        print(
            "\n[DISPLAY TRUNCATED — original source unchanged]"
        )
    else:
        print(formatted)

    evidence[label] = {
        "status": "FOUND",
        "file": str(path),
        "sha256": digest,
        "size_bytes": path.stat().st_size
    }

print("\n" + "=" * 65)
print("=== STEP 8.2 ORIGINAL PROVENANCE EVIDENCE ===")
print("Sources inspected:", len(SOURCES))
print(
    "Found:",
    sum(v["status"] == "FOUND" for v in evidence.values())
)
print(
    "Missing:",
    sum(v["status"] == "MISSING" for v in evidence.values())
)

print("\n=== SOURCE HASH SUMMARY ===")
for label, item in evidence.items():
    print(
        label,
        "|",
        item["status"],
        "|",
        item.get("sha256", "-")
    )

print("\nD07 status: EVIDENCE UNDER REVIEW")
print("D08 status: EVIDENCE UNDER REVIEW")
print("No files modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import json, hashlib, re

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

CFG = P / "configs/asr_hypothesis_pilot_v1.json"
FREEZE = P / "reports/step03_asr_configuration_freeze.json"
MODEL = P / "reports/step03_whisper_model_revision.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

cfg = json.loads(CFG.read_text(encoding="utf-8"))
freeze = json.loads(FREEZE.read_text(encoding="utf-8"))
model = json.loads(MODEL.read_text(encoding="utf-8"))

assert cfg["hypotheses"] == freeze["hypotheses"]
assert cfg["model_id"] == freeze["model_id"] == model["model_id"]
assert (
    cfg["model_revision"]
    == freeze["model_revision"]
    == model["model_revision"]
)
assert freeze["configuration_sha256"] == sha(CFG)

print("=== STEP 8.3 FROZEN ASR CROSSCHECK ===")
print("Config and freeze consistent: PASS")
print("Model revision consistent: PASS")
print("Frozen config SHA256 consistent: PASS")
print("Full 400-audio inference code audited: NOT YET")

# Look for potential original sample selection documents.
keywords = re.compile(
    r"sample|select|pilot|speaker|asr|hypothes|"
    r"manifest|progress|inference|batch",
    re.I
)

candidates = sorted(
    path for path in P.rglob("*")
    if path.is_file()
    and keywords.search(path.name)
    and path.suffix.lower() in {
        ".json", ".csv", ".py", ".ipynb", ".md", ".txt"
    }
)

records = []
for path in candidates:
    records.append({
        "path": str(path.relative_to(P)),
        "type": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
        "sha256": sha(path)
    })

inventory = pd.DataFrame(records)
inventory_path = OUT / "step08_03_candidate_files.csv"
inventory.to_csv(inventory_path, index=False)

print("\n=== POTENTIAL PROVENANCE FILES ===")
print(inventory[["path", "type", "size_bytes"]].to_string(
    index=False, max_rows=90, max_colwidth=95
))

# Inspect JSON keys and relevant nested settings; do not
# assume that keyword matches prove a sampling procedure.
relevant_keys = re.compile(
    r"sample|select|seed|random|speaker|revision|"
    r"model|decode|hypothes|inference|"
    r"generation|batch|pilot|source",
    re.I
)

evidence = []

def inspect(value, prefix="", depth=0):
    if depth > 5:
        return
    if isinstance(value, dict):
        for key, item in value.items():
            full = f"{prefix}.{key}" if prefix else str(key)
            if relevant_keys.search(str(key)):
                if isinstance(item, (str, int, float, bool)):
                    evidence.append((source_name, full, str(item)[:180]))
                elif isinstance(item, list):
                    evidence.append((
                        source_name, full,
                        f"list length={len(item)}"
                    ))
            inspect(item, full, depth + 1)
    elif isinstance(value, list):
        for i, item in enumerate(value[:12]):
            inspect(item, f"{prefix}[{i}]", depth + 1)

for path in candidates:
    if path.suffix.lower() != ".json":
        continue
    if path.stat().st_size > 1_000_000:
        continue
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    source_name = str(path.relative_to(P))
    inspect(obj)

evidence_df = pd.DataFrame(
    evidence, columns=["file", "key", "value"]
)
evidence_path = OUT / "step08_03_metadata_evidence.csv"
evidence_df.to_csv(evidence_path, index=False)

# Focus on sample selection and full-run metadata.
focus = evidence_df[
    evidence_df["key"].str.contains(
        r"seed|random|select|sampling|sample_method|"
        r"source.*speaker|model_revision|generation|"
        r"inference|batch",
        case=False, regex=True, na=False
    )
]

print("\n=== TARGETED METADATA ===")
if focus.empty:
    print("No targeted metadata keys found.")
else:
    print(focus.head(90).to_string(
        index=False, max_colwidth=85
    ))

# Inspect existing CSV filenames and schemas only.
print("\n=== CANDIDATE CSV SCHEMAS ===")
csv_candidates = [
    path for path in candidates
    if path.suffix.lower() == ".csv"
    and any(term in path.name.lower() for term in [
        "speaker", "pilot", "sample", "asr", "hypothes"
    ])
]

for path in csv_candidates[:20]:
    try:
        head = pd.read_csv(path, nrows=3, dtype=str)
        print(str(path.relative_to(P)))
        print("  columns:", list(head.columns)[:20])
        print("  example audio IDs:",
              head["audio_id"].tolist()
              if "audio_id" in head.columns else "N/A")
    except Exception as exc:
        print(path.name, "READ ERROR:", str(exc)[:100])

report = {
    "status": "EVIDENCE_INVENTORY_COMPLETE",
    "step": "8.3",
    "frozen_config_consistency": True,
    "model_revision_consistency": True,
    "full_run_code_verified": False,
    "original_sampling_method_verified": False,
    "candidate_files": len(candidates),
    "inventory_sha256": sha(inventory_path),
    "evidence_sha256": sha(evidence_path),
    "source_config_sha256": sha(CFG),
    "no_model_retraining": True,
    "no_asr_inference": True
}

report_path = OUT / "step08_03_targeted_provenance.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== STEP 8.3 SUMMARY ===")
print("Status:", report["status"])
print("Candidate files:", len(candidates))
print("Extracted metadata records:", len(evidence_df))
print("Original sampling method verified: NO")
print("Full-run ASR code verified: NO")
print("Evidence CSV:", evidence_path)
print("Report:", report_path)


In [ ]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

SPEAKER_FILE = P / "checkpoints/pilot_speaker_ids.json"
SELECTION = P / "data/metadata/pilot_selection_400.csv"
ASR = P / "features/asr/full/asr_hypotheses_0001-0400.csv"
CFG = P / "configs/asr_hypothesis_pilot_v1.json"
FREEZE = P / "reports/step03_asr_configuration_freeze.json"

REPORT = OUT / "step08_04_sampling_asr_consistency.json"
ISSUES = OUT / "step08_04_asr_metadata_mismatches.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path in [SPEAKER_FILE, SELECTION, ASR, CFG, FREEZE]:
    assert path.is_file(), f"Missing: {path}"

# Verify the previously frozen ASR output.
assert sha(ASR) == (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

speaker_config = json.loads(
    SPEAKER_FILE.read_text(encoding="utf-8")
)
cfg = json.loads(CFG.read_text(encoding="utf-8"))
freeze = json.loads(FREEZE.read_text(encoding="utf-8"))

assert cfg["hypotheses"] == freeze["hypotheses"]
assert freeze["configuration_sha256"] == sha(CFG)

sel = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)
asr = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(sel) == 400
assert len(asr) == 2000
assert sel["audio_id"].nunique() == 400
assert sel["speaker"].nunique() == 20

assert not sel.duplicated("audio_id").any()
assert not asr.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert asr.groupby("audio_id")[
    "hypothesis_id"
].nunique().eq(5).all()

assert set(sel["audio_id"]) == set(asr["audio_id"])

mapping = asr[
    ["audio_id", "speaker"]
].drop_duplicates()

assert len(mapping) == 400

mapping = mapping.merge(
    sel[["audio_id", "speaker"]],
    on="audio_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_asr", "_selection")
)

assert len(mapping) == 400

speaker_mapping_ok = bool(
    (mapping["speaker_asr"]
     == mapping["speaker_selection"]).all()
)

print("=== D07 SAVED SAMPLING CONFIGURATION ===")
print(json.dumps(
    speaker_config,
    indent=2,
    ensure_ascii=False
))

print("\n=== D07 SELECTION INTEGRITY ===")
print("Selected audios:", len(sel))
print("Selected speakers:", sel["speaker"].nunique())
print("Audio IDs match ASR:", True)
print("Speaker mapping match:", speaker_mapping_ok)

speaker_counts = (
    sel.groupby("speaker")["audio_id"]
    .nunique()
    .sort_index()
)
print("\n=== SELECTED SPEAKERS ===")
print(speaker_counts.to_string())
print("All speakers have 20 audios:",
      bool(speaker_counts.eq(20).all()))

print("\n=== SELECTION ORDER ===")
print(
    sel[
        ["pilot_index", "audio_id", "speaker"]
    ].head(12).to_string(index=False)
)

# Check every full-run ASR row against the
# expected hypothesis-specific decoding metadata.
expected = {
    item["id"]: item
    for item in cfg["hypotheses"]
}

assert set(expected) == {"H1", "H2", "H3", "H4", "H5"}
assert set(asr["hypothesis_id"]) == set(expected)

required_columns = [
    "hypothesis_id", "method", "model_id",
    "model_revision", "do_sample",
    "num_beams", "temperature", "seed"
]

assert set(required_columns).issubset(asr.columns)

def normalize_bool(value):
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    if s in {"true", "1", "1.0"}:
        return True
    if s in {"false", "0", "0.0"}:
        return False
    return "UNRECOGNIZED:" + s

def normalize_number(value):
    if pd.isna(value):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return "UNRECOGNIZED:" + str(value)

def same_value(actual, expected_value):
    if expected_value is None:
        return actual is None
    if isinstance(expected_value, (int, float)):
        return (
            isinstance(actual, (int, float))
            and not isinstance(actual, bool)
            and np.isclose(
                actual, float(expected_value),
                rtol=0, atol=1e-10
            )
        )
    return actual == expected_value

problems = []
checks = []

for hyp, group in asr.groupby("hypothesis_id"):
    spec = expected[hyp]
    mismatch_count = 0

    for _, row in group.iterrows():

        checks_for_row = {
            "method": (
                str(row["method"]).strip(),
                spec["method"]
            ),
            "model_id": (
                str(row["model_id"]).strip(),
                cfg["model_id"]
            ),
            "model_revision": (
                str(row["model_revision"]).strip(),
                cfg["model_revision"]
            ),
            "do_sample": (
                normalize_bool(row["do_sample"]),
                spec["do_sample"]
            )
        }

        # Inspect explicit or conventionally implied fields.
        # Missing values are not silently converted to
        # configured values.
        for field in ["num_beams", "temperature", "seed"]:
            actual = normalize_number(row[field])
            expected_value = spec.get(field)

            if field == "num_beams" and hyp in {"H3", "H4", "H5"}:
                expected_value = cfg[
                    "runtime_generation_rules"
                ]["num_beams_sampling"]

            checks_for_row[field] = (
                actual,
                expected_value
            )

        for field, (actual, wanted) in checks_for_row.items():
            if not same_value(actual, wanted):
                mismatch_count += 1
                problems.append({
                    "audio_id": row["audio_id"],
                    "hypothesis_id": hyp,
                    "field": field,
                    "observed": str(actual),
                    "expected": str(wanted)
                })

    checks.append({
        "hypothesis_id": hyp,
        "rows": len(group),
        "metadata_mismatches": mismatch_count
    })

checks_df = pd.DataFrame(checks)
issues_df = pd.DataFrame(
    problems,
    columns=[
        "audio_id", "hypothesis_id", "field",
        "observed", "expected"
    ]
)

issues_df.to_csv(
    ISSUES, index=False, encoding="utf-8"
)

print("\n=== D08 FULL-RUN ASR METADATA ===")
print(checks_df.to_string(index=False))

print("\n=== D08 ACTUAL FIELD VALUES ===")
for hyp, group in asr.groupby("hypothesis_id"):
    print("\n", hyp)
    for field in [
        "method", "model_id", "model_revision",
        "do_sample", "num_beams", "temperature", "seed"
    ]:
        values = (
            group[field].astype("string")
            .fillna("<missing>")
            .value_counts()
            .head(6)
            .to_dict()
        )
        print(" ", field, ":", values)

print("\n=== ASR METADATA MISMATCH EXAMPLES ===")
if len(issues_df):
    print(issues_df.head(25).to_string(index=False))
else:
    print("None")

report = {
    "step": "8.4",
    "status": (
        "METADATA_CONSISTENT"
        if len(issues_df) == 0 and speaker_mapping_ok
        else "NEEDS_REVIEW"
    ),
    "selected_audios": 400,
    "selected_speakers": 20,
    "speaker_mapping_consistent": speaker_mapping_ok,
    "all_speakers_have_20_audios": bool(
        speaker_counts.eq(20).all()
    ),
    "asr_hypothesis_rows": len(asr),
    "asr_metadata_mismatch_count": len(issues_df),
    "sampling_algorithm_verified": False,
    "full_inference_source_code_verified": False,
    "models_retrained": False,
    "source_hashes": {
        str(p.relative_to(P)): sha(p)
        for p in [SPEAKER_FILE, SELECTION, ASR, CFG, FREEZE]
    },
    "mismatch_csv_sha256": sha(ISSUES),
    "methodology_note": (
        "The audit compares saved generation metadata "
        "against the frozen configuration. It does not "
        "independently execute the original ASR code "
        "or reconstruct the original sampling algorithm."
    )
}

REPORT.write_text(
    json.dumps(
        report, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== STEP 8.4 SUMMARY ===")
print("Status:", report["status"])
print("D07 sampling algorithm independently verified: NO")
print("D08 full inference code independently verified: NO")
print("ASR metadata mismatch count:", len(issues_df))
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Mismatch CSV:", ISSUES)
print("No model retraining or ASR inference performed.")


In [ ]:

from pathlib import Path
import json
import hashlib
import math
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

ASR = P / "features/asr/full/asr_hypotheses_0001-0400.csv"
CFG = P / "configs/asr_hypothesis_pilot_v1.json"
FREEZE = P / "reports/step03_asr_configuration_freeze.json"
SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"
SELECTION = P / "data/metadata/pilot_selection_400.csv"

REPORT = OUT / "step08_04_corrected_metadata_audit.json"
MISMATCH = OUT / "step08_04_corrected_metadata_mismatches.csv"
SUMMARY = OUT / "step08_04_corrected_hypothesis_summary.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert sha(ASR) == (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

cfg = json.loads(CFG.read_text(encoding="utf-8"))
freeze = json.loads(FREEZE.read_text(encoding="utf-8"))
speaker_cfg = json.loads(SPEAKERS.read_text(encoding="utf-8"))

assert cfg["hypotheses"] == freeze["hypotheses"]
assert freeze["configuration_sha256"] == sha(CFG)

asr = pd.read_csv(
    ASR, dtype={"audio_id": "string", "speaker": "string"}
)
sel = pd.read_csv(
    SELECTION, dtype={"audio_id": "string", "speaker": "string"}
)

assert len(asr) == 2000
assert len(sel) == 400
assert not asr.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert set(asr["audio_id"]) == set(sel["audio_id"])
assert set(sel["speaker"]) == set(speaker_cfg["speaker_ids"])
assert speaker_cfg["seed"] == 42

counts = sel.groupby("speaker")["audio_id"].nunique()
assert len(counts) == 20 and counts.eq(20).all()

mapping = (
    asr[["audio_id", "speaker"]]
    .drop_duplicates()
    .merge(
        sel[["audio_id", "speaker"]],
        on="audio_id",
        validate="one_to_one",
        suffixes=("_asr", "_selected")
    )
)
assert len(mapping) == 400
assert mapping["speaker_asr"].eq(
    mapping["speaker_selected"]
).all()

def parse_bool(value):
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    if s in ("true", "1", "1.0"):
        return True
    if s in ("false", "0", "0.0"):
        return False
    return "INVALID_BOOL:" + s

def parse_number(value):
    if pd.isna(value):
        return None
    try:
        x = float(value)
        return x if math.isfinite(x) else "INVALID_NUMBER"
    except (TypeError, ValueError):
        return "INVALID_NUMBER:" + str(value)

def compare(actual, expected):
    # Boolean must be checked BEFORE numeric types.
    if isinstance(expected, bool):
        return type(actual) is bool and actual == expected

    if expected is None:
        return actual is None

    if isinstance(expected, (int, float)):
        return (
            isinstance(actual, (int, float))
            and not isinstance(actual, bool)
            and math.isclose(
                float(actual), float(expected),
                rel_tol=0, abs_tol=1e-10
            )
        )

    return actual == expected

specs = {x["id"]: x for x in cfg["hypotheses"]}
assert set(specs) == {"H1", "H2", "H3", "H4", "H5"}

mismatches = []
summary_rows = []

for hyp in ["H1", "H2", "H3", "H4", "H5"]:
    group = asr[asr["hypothesis_id"] == hyp]
    spec = specs[hyp]

    assert len(group) == 400
    assert group["audio_id"].nunique() == 400

    before = len(mismatches)

    for row in group.itertuples(index=False):
        expected_beams = spec.get("num_beams")

        if expected_beams is None:
            expected_beams = cfg[
                "runtime_generation_rules"
            ]["num_beams_sampling"]

        observed = {
            "method": str(row.method).strip(),
            "model_id": str(row.model_id).strip(),
            "model_revision": str(row.model_revision).strip(),
            "do_sample": parse_bool(row.do_sample),
            "num_beams": parse_number(row.num_beams),
            "temperature": parse_number(row.temperature),
            "seed": parse_number(row.seed)
        }

        wanted = {
            "method": spec["method"],
            "model_id": cfg["model_id"],
            "model_revision": cfg["model_revision"],
            "do_sample": spec["do_sample"],
            "num_beams": expected_beams,
            "temperature": spec.get("temperature"),
            "seed": spec.get("seed")
        }

        for field in wanted:
            if not compare(observed[field], wanted[field]):
                mismatches.append({
                    "audio_id": row.audio_id,
                    "hypothesis_id": hyp,
                    "field": field,
                    "observed": repr(observed[field]),
                    "expected": repr(wanted[field])
                })

    summary_rows.append({
        "hypothesis_id": hyp,
        "rows": len(group),
        "metadata_mismatches": len(mismatches) - before
    })

mismatch_df = pd.DataFrame(
    mismatches,
    columns=[
        "audio_id", "hypothesis_id", "field",
        "observed", "expected"
    ]
)

summary_df = pd.DataFrame(summary_rows)

# Preserve earlier failed audit files; save new filenames.
def save_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing corrected file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_csv(mismatch_df, MISMATCH)
save_csv(summary_df, SUMMARY)

status = (
    "PASS"
    if len(mismatches) == 0
    else "NEEDS_REVIEW"
)

report = {
    "step": "8.4_corrected",
    "status": status,
    "correction_reason": (
        "Prior comparison incorrectly treated Python bool "
        "as a numeric type because bool subclasses int."
    ),
    "previous_report_preserved": True,
    "selected_speakers": 20,
    "selected_audios": 400,
    "asr_hypotheses": 2000,
    "checked_fields_per_hypothesis": 7,
    "metadata_mismatch_count": len(mismatches),
    "sampling_seed_recorded": 42,
    "speaker_selection_algorithm_verified": False,
    "full_inference_source_code_verified": False,
    "max_new_tokens_execution_independently_verified": False,
    "ASR_inference_executed": False,
    "model_training_executed": False,
    "source_sha256": {
        str(path.relative_to(P)): sha(path)
        for path in [ASR, CFG, FREEZE, SPEAKERS, SELECTION]
    },
    "summary_sha256": sha(SUMMARY),
    "mismatch_sha256": sha(MISMATCH)
}

temp = REPORT.with_suffix(".tmp")
temp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
    temp.unlink()
else:
    temp.replace(REPORT)

print("=== STEP 8.4 CORRECTED METADATA AUDIT ===")
print("Status:", status)
print("Total ASR hypotheses:", len(asr))
print("Metadata mismatches:", len(mismatches))
print("Sampling speaker/audio mapping: PASS")
print("Sampling seed recorded:", speaker_cfg["seed"])

print("\n=== PER-HYPOTHESIS RESULT ===")
print(summary_df.to_string(index=False))

print("\n=== MISMATCHES BY FIELD ===")
if mismatch_df.empty:
    print("None")
else:
    print(
        mismatch_df.groupby("field")
        .size()
        .to_string()
    )
    print(mismatch_df.head(20).to_string(index=False))

print("\n=== PROVENANCE STATUS ===")
print("D07 sampling algorithm: NOT YET VERIFIED")
print("D08 saved decoding metadata:",
      "PASS" if status == "PASS" else "NEEDS REVIEW")
print("D08 original inference implementation: NOT YET VERIFIED")

print("\nCorrected report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Original failed report preserved: YES")


In [ ]:

from pathlib import Path
import json
import re
import hashlib
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
W = Path("/kaggle/working")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"
PROGRESS = P / "checkpoints/research_progress.json"
PILOT_CONFIG = P / "configs/pilot_acoustic_v1.json"
SELECTION = P / "data/metadata/pilot_selection_400.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path in [SPEAKERS, PROGRESS, PILOT_CONFIG, SELECTION]:
    assert path.is_file(), f"Missing: {path}"

speaker_info = json.loads(SPEAKERS.read_text(encoding="utf-8"))
progress = json.loads(PROGRESS.read_text(encoding="utf-8"))
pilot_config = json.loads(PILOT_CONFIG.read_text(encoding="utf-8"))

sel = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(sel) == 400
assert sel["speaker"].nunique() == 20
assert sel["audio_id"].nunique() == 400
assert speaker_info["seed"] == 42
assert set(speaker_info["speaker_ids"]) == set(sel["speaker"])
assert sel.groupby("speaker")["audio_id"].nunique().eq(20).all()

print("=== STEP 8.5 SAMPLING PROVENANCE AUDIT ===")

print("\n=== PILOT ACOUSTIC CONFIG ===")
print(json.dumps(pilot_config, indent=2, ensure_ascii=False))

print("\n=== SAMPLING-RELATED PROGRESS KEYS ===")
pattern = re.compile(
    r"seed|random|sample|select|speaker|pilot|row|index",
    re.IGNORECASE
)

for key, value in progress.items():
    if pattern.search(key):
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False)[:500]
        print(f"{key}: {value}")

print("\n=== SELECTION ROW INDEX AUDIT ===")

assert "row" in sel.columns
assert "pilot_index" in sel.columns

sel["row_numeric"] = pd.to_numeric(
    sel["row"], errors="raise"
)
sel["pilot_index_numeric"] = pd.to_numeric(
    sel["pilot_index"], errors="raise"
)

print("Original row min:", sel["row_numeric"].min())
print("Original row max:", sel["row_numeric"].max())
print("Unique original rows:", sel["row_numeric"].nunique())

print(
    "Pilot index consecutive 1-400:",
    sorted(sel["pilot_index_numeric"].tolist())
    == list(range(1, 401))
)

print(
    "Original dataset rows strictly increasing:",
    bool(sel["row_numeric"].is_monotonic_increasing)
)

print(
    "Speaker IDs sorted in selection:",
    sel["speaker"].drop_duplicates().tolist()
    == sorted(sel["speaker"].unique().tolist())
)

print("\n=== SPEAKER-LEVEL ORIGINAL ROW SUMMARY ===")
speaker_rows = (
    sel.groupby("speaker")
    .agg(
        selected_audios=("audio_id", "count"),
        original_row_min=("row_numeric", "min"),
        original_row_max=("row_numeric", "max"),
        selected_index_min=("pilot_index_numeric", "min"),
        selected_index_max=("pilot_index_numeric", "max")
    )
    .reset_index()
)

print(speaker_rows.to_string(index=False))

print("\n=== FIRST AND LAST 5 SELECTIONS ===")
print(
    sel[
        ["pilot_index", "row", "speaker", "audio_id"]
    ].head(5).to_string(index=False)
)
print("...")
print(
    sel[
        ["pilot_index", "row", "speaker", "audio_id"]
    ].tail(5).to_string(index=False)
)

# Inspect only source code, not notebook outputs or unrelated files.
source_files = [
    path for path in P.rglob("*.py")
    if path.is_file()
]

source_files += [
    path for path in W.glob("*.py")
    if path.is_file()
]

source_files += [
    path for path in W.glob("*.ipynb")
    if path.is_file()
]

source_files = sorted(set(source_files))

code_pattern = re.compile(
    r"default_rng|RandomState|random\.sample|"
    r"random\.seed|np\.random|rng\.choice|"
    r"random\.choice|sample\(n=|"
    r"sample\(frac=|speaker_ids|pilot_selection_400",
    re.IGNORECASE
)

matches = []

for path in source_files:
    if path.stat().st_size > 20_000_000:
        continue

    try:
        if path.suffix == ".ipynb":
            obj = json.loads(path.read_text(encoding="utf-8"))
            sections = [
                (
                    f"cell_{i}",
                    "".join(c.get("source", []))
                    if isinstance(c.get("source", []), list)
                    else str(c.get("source", ""))
                )
                for i, c in enumerate(obj.get("cells", []))
                if c.get("cell_type") == "code"
            ]
        else:
            sections = [
                ("source", path.read_text(encoding="utf-8"))
            ]
    except Exception:
        continue

    for section, source in sections:
        for line_no, line in enumerate(source.splitlines(), 1):
            if code_pattern.search(line):
                matches.append({
                    "file": str(path),
                    "section": section,
                    "line": line_no,
                    "snippet": line.strip()[:400],
                    "source_sha256": sha(path)
                })

code_evidence = pd.DataFrame(
    matches,
    columns=[
        "file", "section", "line",
        "snippet", "source_sha256"
    ]
)

CODE_OUT = OUT / "step08_05_sampling_code_evidence.csv"
SPEAKER_OUT = OUT / "step08_05_sampling_speaker_rows.csv"
REPORT = OUT / "step08_05_sampling_provenance.json"

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(code_evidence, CODE_OUT)
safe_csv(speaker_rows, SPEAKER_OUT)

print("\n=== ORIGINAL SAMPLING CODE EVIDENCE ===")
print("Source files searched:", len(source_files))
print("Matching code lines:", len(code_evidence))

if code_evidence.empty:
    print("No matching source code found.")
else:
    print(
        code_evidence[
            ["file", "section", "line", "snippet"]
        ].head(50).to_string(
            index=False, max_colwidth=110
        )
    )

report = {
    "step": "8.5",
    "status": "EVIDENCE_REVIEW",
    "sampling_seed_documented": 42,
    "speaker_ids_documented": 20,
    "selected_audios_verified": 400,
    "selected_audios_per_speaker": 20,
    "saved_selection_consistent": True,
    "original_sampling_algorithm_verified": False,
    "sampling_source_files_checked": len(source_files),
    "sampling_code_matching_lines": len(code_evidence),
    "source_hashes": {
        str(path.relative_to(P)): sha(path)
        for path in [SPEAKERS, PROGRESS, PILOT_CONFIG, SELECTION]
    },
    "output_hashes": {
        CODE_OUT.name: sha(CODE_OUT),
        SPEAKER_OUT.name: sha(SPEAKER_OUT)
    },
    "note": (
        "Recorded seed and selected rows do not independently "
        "establish the original random sampling algorithm. "
        "Potential code matches require human review."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("\n=== STEP 8.5 SUMMARY ===")
print("Status:", report["status"])
print("Sampling seed:", report["sampling_seed_documented"])
print("Selected speaker/audio mapping: PASS")
print("Original algorithm independently verified: NO")
print("Code evidence CSV:", CODE_OUT)
print("Report:", REPORT)
print("Report SHA256:", sha(REPORT))
print("No ASR inference or model training performed.")


In [ ]:

from pathlib import Path
import json
import hashlib
import random

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

SELECTION = P / "data/metadata/pilot_selection_400.csv"
SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"

REPORT = OUT / "step08_06_sampling_reconstruction.json"
RESULTS = OUT / "step08_06_sampling_candidate_comparison.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

sel = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)
saved = json.loads(SPEAKERS.read_text(encoding="utf-8"))

assert saved["seed"] == 42
assert len(sel) == 400
assert len(saved["speaker_ids"]) == 20

sel["row"] = pd.to_numeric(sel["row"], errors="raise")

speaker_rows = (
    sel.groupby("speaker")
    .agg(
        row_min=("row", "min"),
        row_max=("row", "max"),
        n=("row", "nunique")
    )
    .reset_index()
    .sort_values("row_min")
)

assert len(speaker_rows) == 20
assert speaker_rows["n"].eq(20).all()

# Check all 20 consecutive original rows are present.
contiguous = True

for speaker, group in sel.groupby("speaker"):
    rows = sorted(group["row"].astype(int).tolist())
    if rows != list(range(rows[0], rows[0] + 20)):
        contiguous = False

assert contiguous

# Hypothesis: 125 speakers occupy contiguous 20-row
# blocks in the full train dataset, with 1-based row IDs.
block_aligned = bool(
    ((speaker_rows["row_min"] - 1) % 20 == 0).all()
)

assert block_aligned

observed_zero_based = sorted(
    ((speaker_rows["row_min"] - 1) // 20)
    .astype(int)
    .tolist()
)

assert len(set(observed_zero_based)) == 20
assert min(observed_zero_based) >= 0
assert max(observed_zero_based) < 125

candidate_methods = {
    "numpy_default_rng_choice":
        np.random.default_rng(42).choice(
            125, size=20, replace=False
        ).tolist(),

    "numpy_RandomState_choice":
        np.random.RandomState(42).choice(
            125, size=20, replace=False
        ).tolist(),

    "python_random_sample":
        random.Random(42).sample(
            range(125), 20
        ),

    "numpy_default_rng_permutation":
        np.random.default_rng(42).permutation(125)[:20].tolist(),

    "numpy_RandomState_permutation":
        np.random.RandomState(42).permutation(125)[:20].tolist()
}

records = []

for method, selected in candidate_methods.items():
    selected_sorted = sorted(int(x) for x in selected)

    overlap = len(
        set(selected_sorted) & set(observed_zero_based)
    )

    exact_set_match = (
        selected_sorted == observed_zero_based
    )

    records.append({
        "method": method,
        "seed": 42,
        "speaker_population_assumed": 125,
        "sample_size": 20,
        "matching_positions": overlap,
        "exact_set_match": exact_set_match,
        "candidate_positions_zero_based": ",".join(
            map(str, selected_sorted)
        )
    })

results = pd.DataFrame(records)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing result differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(results, RESULTS)

matching = results.loc[
    results["exact_set_match"],
    "method"
].tolist()

report = {
    "step": "8.6",
    "status": "RECONSTRUCTION_COMPLETED",
    "recorded_seed": 42,
    "selected_speakers": 20,
    "selected_audios": 400,
    "audio_rows_consecutive_within_speaker": contiguous,
    "speaker_row_blocks_aligned_to_20": block_aligned,
    "observed_speaker_positions_zero_based": observed_zero_based,
    "candidate_algorithms_tested": len(candidate_methods),
    "exact_matching_candidates": matching,
    "original_algorithm_verified": False,
    "critical_assumption": (
        "All 125 speakers in the original train partition "
        "occupy 20-row blocks in the same order, and "
        "the original selection sampled zero-based positions."
    ),
    "limitations": [
        "Full 125-speaker source ordering was not independently loaded.",
        "An exact random sample match does not prove original code identity.",
        "A mismatch does not exclude other legitimate selection algorithms.",
        "This is a reconstruction audit, not an original code audit."
    ],
    "source_sha256": {
        str(SELECTION.relative_to(P)): sha(SELECTION),
        str(SPEAKERS.relative_to(P)): sha(SPEAKERS)
    },
    "results_sha256": sha(RESULTS),
    "model_retraining": False,
    "asr_inference": False
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("=== STEP 8.6 SAMPLING RECONSTRUCTION ===")
print("Status: RECONSTRUCTION_COMPLETED")
print("Recorded seed:", saved["seed"])
print("Speakers:", len(speaker_rows))
print("Audios:", len(sel))
print("Contiguous audio blocks:", contiguous)
print("20-row block alignment:", block_aligned)

print("\n=== OBSERVED SPEAKER POSITIONS ===")
print("Zero-based:", observed_zero_based)

print("\n=== CANDIDATE ALGORITHM COMPARISON ===")
print(
    results[
        [
            "method",
            "matching_positions",
            "exact_set_match"
        ]
    ].to_string(index=False)
)

print("\n=== EXACT MATCHING CANDIDATES ===")
print(matching if matching else "None")

print("\n=== INTERPRETATION BOUNDARY ===")
print("Original source algorithm proven: NO")
print("125-speaker original order verified: NO")
print("Sampling reconstruction is exploratory.")

print("\nReport:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Comparison CSV:", RESULTS)
print("No model training or ASR inference performed.")


In [ ]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

RAW = P / "data/raw/speechocean762_train.parquet"
SELECTION = P / "data/metadata/pilot_selection_400.csv"
SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"

REPORT = OUT / "step08_07_source_sampling_reconstruction.json"
COMPARE = OUT / "step08_07_reconstructed_selection.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert RAW.is_file(), (
    "Original train parquet is unavailable. "
    "Reattach the pinned dataset before this audit."
)

speaker_info = json.loads(SPEAKERS.read_text(encoding="utf-8"))
selection = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

# Read metadata columns only; no audio bytes decoded.
import pyarrow.parquet as pq

schema = pq.read_schema(RAW)
available = set(schema.names)

assert "audio_id" in available
assert "speaker" in available

raw = pq.read_table(
    RAW,
    columns=["audio_id", "speaker"]
).to_pandas()

raw["audio_id"] = raw["audio_id"].astype("string")
raw["speaker"] = raw["speaker"].astype("string")

assert len(raw) == 2500
assert raw["speaker"].nunique() == 125
assert raw["audio_id"].nunique() == 2500

raw["original_row_1based"] = np.arange(1, 2501)

# Speaker order in the actual stored train data.
original_speaker_order = (
    raw["speaker"].drop_duplicates().tolist()
)

speaker_counts = raw["speaker"].value_counts()
all_have_20 = bool(speaker_counts.eq(20).all())

contiguous_blocks = True

for speaker, g in raw.groupby("speaker", sort=False):
    positions = g["original_row_1based"].to_numpy()
    if not np.array_equal(
        positions,
        np.arange(positions[0], positions[0] + len(positions))
    ):
        contiguous_blocks = False
        break

seed = int(speaker_info["seed"])
rng = np.random.default_rng(seed)

selected_positions = rng.choice(
    len(original_speaker_order),
    size=20,
    replace=False
)

reconstructed_speakers = sorted(
    original_speaker_order[int(i)]
    for i in selected_positions
)

saved_speakers = sorted(speaker_info["speaker_ids"])

speaker_match = reconstructed_speakers == saved_speakers

# Reconstruct: selected speakers, all their 20 source
# rows in original source order, speakers sorted.
reconstructed = (
    raw[raw["speaker"].isin(reconstructed_speakers)]
    .sort_values(
        ["speaker", "original_row_1based"]
    )
    .reset_index(drop=True)
)

reconstructed.insert(
    0, "pilot_index", np.arange(1, len(reconstructed) + 1)
)

assert len(reconstructed) == 400

selected = selection[
    ["pilot_index", "row", "audio_id", "speaker"]
].copy()

selected["pilot_index"] = pd.to_numeric(
    selected["pilot_index"], errors="raise"
).astype(int)

selected["row"] = pd.to_numeric(
    selected["row"], errors="raise"
).astype(int)

selected = selected.sort_values(
    "pilot_index"
).reset_index(drop=True)

audio_match = (
    reconstructed["audio_id"].tolist()
    == selected["audio_id"].tolist()
)

row_match = (
    reconstructed["original_row_1based"].tolist()
    == selected["row"].tolist()
)

order_match = (
    reconstructed["pilot_index"].tolist()
    == selected["pilot_index"].tolist()
)

reconstructed["saved_audio_id"] = selected["audio_id"]
reconstructed["saved_row"] = selected["row"]
reconstructed["audio_id_match"] = (
    reconstructed["audio_id"] == reconstructed["saved_audio_id"]
)
reconstructed["row_match"] = (
    reconstructed["original_row_1based"]
    == reconstructed["saved_row"]
)

# Preserve output without silently replacing previous evidence.
tmp = COMPARE.with_suffix(".tmp")
reconstructed.to_csv(
    tmp, index=False, encoding="utf-8"
)
if COMPARE.exists():
    assert sha(COMPARE) == sha(tmp)
    tmp.unlink()
else:
    tmp.replace(COMPARE)

full_match = (
    all_have_20
    and contiguous_blocks
    and speaker_match
    and audio_match
    and row_match
    and order_match
)

report = {
    "step": "8.7",
    "status": (
        "RECONSTRUCTION_MATCH"
        if full_match else "NEEDS_REVIEW"
    ),
    "source_train_rows": len(raw),
    "source_speakers": len(original_speaker_order),
    "each_source_speaker_has_20_rows": all_have_20,
    "source_speaker_blocks_contiguous": contiguous_blocks,
    "recorded_seed": seed,
    "selected_speaker_ids_match": speaker_match,
    "selected_audio_ids_match": audio_match,
    "selected_original_rows_match": row_match,
    "pilot_index_order_match": order_match,
    "original_source_code_verified": False,
    "selection_rule_reconstructed": (
        "NumPy default_rng(seed).choice(125,20,replace=False), "
        "followed by original source row selection "
        "for selected speakers and speaker-sorted output."
    ),
    "parquet_schema_columns_used": ["audio_id", "speaker"],
    "raw_audio_decoded": False,
    "model_training": False,
    "ASR_inference": False,
    "source_hashes": {
        "original_train_parquet": sha(RAW),
        "pilot_selection_400.csv": sha(SELECTION),
        "pilot_speaker_ids.json": sha(SPEAKERS)
    },
    "reconstructed_selection_sha256": sha(COMPARE),
    "limitation": (
        "Exact reconstruction provides strong evidence for "
        "the sampling procedure but does not prove the "
        "historical source code used this exact implementation."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

print("=== STEP 8.7 ORIGINAL DATASET RECONSTRUCTION ===")
print("Status:", report["status"])
print("Original training rows:", len(raw))
print("Original speakers:", len(original_speaker_order))
print("Every speaker has 20 rows:", all_have_20)
print("Speaker rows contiguous:", contiguous_blocks)

print("\n=== RECONSTRUCTION RESULTS ===")
print("Speaker IDs match:", speaker_match)
print("Audio IDs match:", audio_match)
print("Original row indices match:", row_match)
print("Pilot index order match:", order_match)
print("Full reconstruction match:", full_match)

print("\n=== RECONSTRUCTED SPEAKERS ===")
print(reconstructed_speakers)

print("\n=== RECONSTRUCTED SAMPLE PREVIEW ===")
print(
    reconstructed[
        ["pilot_index", "original_row_1based",
         "audio_id", "speaker", "audio_id_match", "row_match"]
    ].head(12).to_string(index=False)
)

print("\n=== METHODOLOGICAL BOUNDARY ===")
print("Original source code recovered: NO")
print("Reconstruction independently matched:", full_match)
print("Source audio decoded: NO")

print("\nReport:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Comparison CSV:", COMPARE)


In [ ]:

from pathlib import Path
import json
import pyarrow.parquet as pq
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")

RAW = P / "data/raw/speechocean762_train.parquet"
SEL = P / "data/metadata/pilot_selection_400.csv"

assert RAW.is_file(), f"Missing: {RAW}"
assert SEL.is_file(), f"Missing: {SEL}"

pf = pq.ParquetFile(RAW)
schema = pf.schema_arrow

print("=== STEP 8.7A ORIGINAL PARQUET SCHEMA ===")
print("Parquet rows:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)

print("\n=== ALL TOP-LEVEL COLUMNS ===")
for field in schema:
    print(f"{field.name}: {field.type}")

print("\n=== SELECTION METADATA COLUMNS ===")
sel = pd.read_csv(
    SEL,
    nrows=5,
    dtype={"audio_id": "string", "speaker": "string"}
)

print(sel.columns.tolist())

print("\n=== FIRST 5 SAVED SELECTION ROWS ===")
available_selection = [
    c for c in [
        "pilot_index", "row", "speaker",
        "audio_id", "audio_path", "split"
    ]
    if c in sel.columns
]

print(
    sel[available_selection].to_string(index=False)
)

print("\n=== POSSIBLE SOURCE METADATA COLUMNS ===")
metadata_cols = [
    name for name in schema.names
    if any(token in name.lower() for token in [
        "audio", "path", "file", "speaker",
        "id", "row", "text", "fluency"
    ])
    and name.lower() != "audio"
]

print(metadata_cols)

# Read a tiny number of rows from metadata columns only.
# Do not access the potentially large audio column.
safe_cols = [
    name for name in metadata_cols
    if not (
        name.lower() == "audio"
        or "binary" in str(schema.field(name).type).lower()
    )
]

safe_cols = safe_cols[:10]

if safe_cols:
    batch = next(
        pf.iter_batches(
            batch_size=5,
            columns=safe_cols
        ),
        None
    )

    if batch is not None:
        example = batch.to_pandas()

        print("\n=== FIRST 5 SOURCE METADATA ROWS ===")
        print(
            example.to_string(
                index=False,
                max_colwidth=100
            )
        )
else:
    print("\nNo suitable metadata columns found.")

print("\n=== DIAGNOSTIC SUMMARY ===")
print("Schema inspection: COMPLETE")
print("Audio decoded: NO")
print("ASR inference: NO")
print("Model training: NO")
print("Original files modified: NO")
print("Next: adapt STEP 8.7 to actual source schema")


In [ ]:

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

RAW = P / "data/raw/speechocean762_train.parquet"
SEL = P / "data/metadata/pilot_selection_400.csv"
SPEAKERS = P / "checkpoints/pilot_speaker_ids.json"

REPORT = OUT / "step08_07b_source_reconstruction.json"
COMPARE = OUT / "step08_07b_reconstructed_selection.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path in [RAW, SEL, SPEAKERS]:
    assert path.is_file(), f"Missing: {path}"

saved = json.loads(SPEAKERS.read_text(encoding="utf-8"))

selection = pd.read_csv(
    SEL,
    dtype={"speaker": "string", "audio_id": "string"}
)

selection["row"] = pd.to_numeric(
    selection["row"], errors="raise"
).astype(int)

selection["pilot_index"] = pd.to_numeric(
    selection["pilot_index"], errors="raise"
).astype(int)

selection = selection.sort_values(
    "pilot_index"
).reset_index(drop=True)

# Read metadata only, excluding audio.bytes.
pf = pq.ParquetFile(RAW)

leaf_paths = [
    pf.schema.column(i).path
    for i in range(len(pf.schema.names))
]

assert "audio.path" in leaf_paths, leaf_paths

table = pf.read(
    columns=["speaker", "audio.path", "fluency", "text"]
)

raw = table.to_pandas()

# Arrow may expose the nested field as a struct
# rather than a flattened column.
if "audio.path" in raw.columns:
    paths = raw["audio.path"]
elif "audio" in raw.columns:
    paths = raw["audio"].map(
        lambda x: x.get("path")
        if isinstance(x, dict) else None
    )
else:
    raise AssertionError(
        f"Audio path missing from projection: {list(raw.columns)}"
    )

raw["speaker"] = raw["speaker"].astype("string")
raw["audio_path_source"] = paths.astype("string")

assert raw["audio_path_source"].notna().all()
assert len(raw) == 2500
assert raw["speaker"].nunique() == 125

raw["audio_id"] = raw["audio_path_source"].map(
    lambda x: Path(str(x)).stem
).astype("string")

raw["original_row_1based"] = np.arange(1, 2501)

assert raw["audio_id"].nunique() == 2500

original_speaker_order = (
    raw["speaker"].drop_duplicates().tolist()
)

counts = raw.groupby("speaker")["audio_id"].nunique()
all_have_20 = bool(counts.eq(20).all())

contiguous = all(
    np.array_equal(
        group["original_row_1based"].to_numpy(),
        np.arange(
            group["original_row_1based"].iloc[0],
            group["original_row_1based"].iloc[0] + len(group)
        )
    )
    for _, group in raw.groupby("speaker", sort=False)
)

seed = int(saved["seed"])
positions = np.random.default_rng(seed).choice(
    len(original_speaker_order),
    size=20,
    replace=False
)

chosen_speakers = sorted(
    original_speaker_order[int(i)]
    for i in positions
)

speaker_match = (
    chosen_speakers == sorted(saved["speaker_ids"])
)

reconstructed = (
    raw[raw["speaker"].isin(chosen_speakers)]
    .sort_values(["speaker", "original_row_1based"])
    .reset_index(drop=True)
)

reconstructed.insert(
    0, "pilot_index", np.arange(1, len(reconstructed) + 1)
)

assert len(reconstructed) == 400

reconstructed["saved_audio_id"] = selection["audio_id"]
reconstructed["saved_speaker"] = selection["speaker"]
reconstructed["saved_row"] = selection["row"]

reconstructed["audio_id_match"] = (
    reconstructed["audio_id"]
    == reconstructed["saved_audio_id"]
)

reconstructed["speaker_match"] = (
    reconstructed["speaker"]
    == reconstructed["saved_speaker"]
)

reconstructed["row_match"] = (
    reconstructed["original_row_1based"]
    == reconstructed["saved_row"]
)

index_match = (
    reconstructed["pilot_index"].tolist()
    == selection["pilot_index"].tolist()
)

audio_match = bool(
    reconstructed["audio_id_match"].all()
)
row_match = bool(
    reconstructed["row_match"].all()
)
speaker_row_match = bool(
    reconstructed["speaker_match"].all()
)

# Also compare labels, prompt text and source path.
fluency_match = bool(np.allclose(
    reconstructed["fluency"].astype(float),
    selection["fluency"].astype(float)
))

text_match = bool(
    reconstructed["text"].astype("string")
    .fillna("<missing>")
    .eq(selection["text"].astype("string").fillna("<missing>"))
    .all()
)

path_match = bool(
    reconstructed["audio_path_source"]
    .map(lambda x: Path(str(x)).name)
    .eq(
        selection["audio_path"].astype("string")
        .map(lambda x: Path(str(x)).name)
    )
    .all()
)

full_match = all([
    all_have_20,
    contiguous,
    speaker_match,
    speaker_row_match,
    audio_match,
    row_match,
    index_match,
    fluency_match,
    text_match,
    path_match
])

def safe_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp, index=False, encoding="utf-8"
    )
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing output differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

# No raw audio bytes are included in this CSV.
save_columns = [
    "pilot_index", "original_row_1based",
    "speaker", "audio_id", "audio_path_source",
    "saved_speaker", "saved_audio_id", "saved_row",
    "speaker_match", "audio_id_match", "row_match"
]

safe_csv(reconstructed[save_columns], COMPARE)

report = {
    "step": "8.7b",
    "status": (
        "RECONSTRUCTION_MATCH"
        if full_match else "NEEDS_REVIEW"
    ),
    "source_rows": len(raw),
    "source_speakers": len(original_speaker_order),
    "sampling_seed": seed,
    "candidate_algorithm":
        "numpy.default_rng(42).choice(125,20,replace=False)",
    "all_source_speakers_have_20_rows": all_have_20,
    "speaker_blocks_contiguous": contiguous,
    "selected_speakers_match": speaker_match,
    "speaker_mapping_match": speaker_row_match,
    "selected_audio_ids_match": audio_match,
    "source_row_indices_match": row_match,
    "pilot_index_match": index_match,
    "fluency_labels_match": fluency_match,
    "reading_prompts_match": text_match,
    "audio_path_basename_match": path_match,
    "original_selection_source_code_found": False,
    "audio_bytes_read": False,
    "asr_inference": False,
    "model_training": False,
    "source_sha256": {
        str(path.relative_to(P)): sha(path)
        for path in [RAW, SEL, SPEAKERS]
    },
    "comparison_sha256": sha(COMPARE),
    "interpretation": (
        "An exact match supports reconstruction of the "
        "original sample from the pinned training data. "
        "It does not prove the historical code identity."
    )
}

tmp = REPORT.with_suffix(".tmp")
tmp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing report differs"
    tmp.unlink()
else:
    tmp.replace(REPORT)

print("=== STEP 8.7B SOURCE SAMPLING RECONSTRUCTION ===")
print("Status:", report["status"])
print("Original train rows:", len(raw))
print("Original train speakers:", len(original_speaker_order))
print("All speakers have 20 rows:", all_have_20)
print("Speaker blocks contiguous:", contiguous)

print("\n=== RECONSTRUCTION RESULTS ===")
for key in [
    "selected_speakers_match",
    "speaker_mapping_match",
    "selected_audio_ids_match",
    "source_row_indices_match",
    "pilot_index_match",
    "fluency_labels_match",
    "reading_prompts_match",
    "audio_path_basename_match"
]:
    print(key + ":", report[key])

print("FULL MATCH:", full_match)

print("\n=== RECONSTRUCTED SPEAKERS ===")
print(chosen_speakers)

print("\n=== FIRST 5 RECONSTRUCTED ROWS ===")
print(
    reconstructed[save_columns]
    .head(5)
    .to_string(index=False)
)

print("\n=== LIMITATION ===")
print("Historical sampling source code recovered: NO")
print("Exact source-based reconstruction:", full_match)

print("\nReport:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Comparison CSV:", COMPARE)


In [ ]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

BASE = P / "features/asr/full"
FINAL = BASE / "asr_hypotheses_0001-0400.csv"
CONFIG = P / "configs/asr_hypothesis_pilot_v1.json"
PROGRESS = P / "checkpoints/research_progress.json"

BATCHES = {
    "0001-0100": BASE / "asr_hypotheses_0001-0100.csv",
    "0101-0200": BASE / "asr_hypotheses_0101-0200.csv",
    "0201-0300": BASE / "asr_hypotheses_0201-0300.csv",
    "0301-0400": BASE / "asr_hypotheses_0301-0400.csv",
}

REPORT = OUT / "step08_08_asr_batch_integrity.json"
BATCH_REPORT = OUT / "step08_08_batch_comparison.csv"
DIFFERENCES = OUT / "step08_08_row_differences.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read(path):
    return pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "hypothesis_id": "string"
        }
    )

assert sha(FINAL) == (
    "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"
)

final = read(FINAL)
cfg = json.loads(CONFIG.read_text(encoding="utf-8"))
progress = json.loads(PROGRESS.read_text(encoding="utf-8"))

KEYS = [
    "pilot_index", "audio_id",
    "speaker", "hypothesis_id"
]

FIELDS = [
    "method", "model_id", "model_revision",
    "do_sample", "num_beams",
    "temperature", "seed", "transcription",
    "duration_sec", "inference_sec"
]

assert len(final) == 2000
assert not final.duplicated(KEYS).any()
assert set(KEYS + FIELDS).issubset(final.columns)

# Canonical comparison: string columns must agree,
# while floating-point columns tolerate only
# roundtrip serialization differences.
TEXT_FIELDS = [
    "method", "model_id",
    "model_revision", "transcription"
]

NUMBER_FIELDS = [
    "num_beams", "temperature",
    "seed", "duration_sec", "inference_sec"
]

def equal_series(left, right, column):
    if column in TEXT_FIELDS:
        a = left.astype("string")
        b = right.astype("string")
        return (
            a.eq(b).fillna(False)
            | (a.isna() & b.isna())
        ).to_numpy(dtype=bool)

    if column == "do_sample":
        a = left.astype("string").str.lower()
        b = right.astype("string").str.lower()
        return (
            a.eq(b).fillna(False)
            | (a.isna() & b.isna())
        ).to_numpy(dtype=bool)

    a = pd.to_numeric(left, errors="coerce")
    b = pd.to_numeric(right, errors="coerce")

    return np.isclose(
        a.to_numpy(dtype=float),
        b.to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
        equal_nan=True
    )

batch_frames = []
batch_summary = []

for label, path in BATCHES.items():
    assert path.is_file(), f"Missing batch: {path}"

    part = read(path)

    assert len(part) == 500, (
        f"Unexpected rows in {label}: {len(part)}"
    )
    assert part["audio_id"].nunique() == 100
    assert not part.duplicated(KEYS).any()
    assert set(KEYS + FIELDS).issubset(part.columns)

    # The expected batch row range is based on
    # pilot_index rather than audio ID.
    low, high = map(int, label.split("-"))

    assert part["pilot_index"].between(low, high).all()

    batch_frames.append(part)

    batch_summary.append({
        "batch": label,
        "rows": len(part),
        "unique_audios": part["audio_id"].nunique(),
        "unique_hypotheses": part["hypothesis_id"].nunique(),
        "sha256": sha(path)
    })

combined = pd.concat(batch_frames, ignore_index=True)

assert len(combined) == 2000
assert not combined.duplicated(KEYS).any()

joined = final[KEYS + FIELDS].merge(
    combined[KEYS + FIELDS],
    on=KEYS,
    how="outer",
    validate="one_to_one",
    suffixes=("_final", "_batch"),
    indicator=True
)

missing_keys = int(joined["_merge"].ne("both").sum())

differences = []

for field in FIELDS:
    left = joined[f"{field}_final"]
    right = joined[f"{field}_batch"]

    matches = equal_series(left, right, field)

    for idx in np.flatnonzero(~matches):
        row = joined.iloc[int(idx)]
        differences.append({
            "pilot_index": row["pilot_index"],
            "audio_id": row["audio_id"],
            "hypothesis_id": row["hypothesis_id"],
            "field": field,
            "batch_value": str(row[f"{field}_batch"])[:300],
            "final_value": str(row[f"{field}_final"])[:300]
        })

diff = pd.DataFrame(
    differences,
    columns=[
        "pilot_index", "audio_id", "hypothesis_id",
        "field", "batch_value", "final_value"
    ]
)

batch_table = pd.DataFrame(batch_summary)

# Check the recorded batch hashes, when present.
recorded_hash_checks = {}

for label, path in BATCHES.items():
    key = f"step3_full_batch_{label}_sha256"
    expected_hash = progress.get(key)

    recorded_hash_checks[label] = {
        "progress_hash_available": expected_hash is not None,
        "matches": (
            sha(path) == expected_hash
            if expected_hash is not None else None
        )
    }

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp, index=False, encoding="utf-8"
    )
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(batch_table, BATCH_REPORT)
safe_csv(diff, DIFFERENCES)

all_recorded_hashes_match = all(
    item["matches"] is True
    for item in recorded_hash_checks.values()
)

status = (
    "PASS"
    if missing_keys == 0
    and len(diff) == 0
    and all_recorded_hashes_match
    else "NEEDS_REVIEW"
)

report = {
    "step": "8.8",
    "status": status,
    "original_batch_count": 4,
    "combined_hypothesis_rows": len(combined),
    "final_hypothesis_rows": len(final),
    "missing_or_extra_keys": missing_keys,
    "field_level_differences": len(diff),
    "progress_batch_hashes": recorded_hash_checks,
    "frozen_asr_model_id": cfg["model_id"],
    "frozen_asr_model_revision": cfg["model_revision"],
    "saved_metadata_audit":
        "STEP 8.4 CORRECTED PASS",
    "historical_inference_code_recovered": False,
    "runtime_generation_rules_independently_verified": False,
    "raw_audio_decoded": False,
    "asr_inference_performed": False,
    "model_training_performed": False,
    "batch_report_sha256": sha(BATCH_REPORT),
    "differences_sha256": sha(DIFFERENCES),
    "source_hashes": {
        str(path.relative_to(P)): sha(path)
        for path in [FINAL, CONFIG, PROGRESS, *BATCHES.values()]
    }
}

temp = REPORT.with_suffix(".tmp")
temp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing report differs"
    temp.unlink()
else:
    temp.replace(REPORT)

print("=== STEP 8.8 ASR BATCH INTEGRITY AUDIT ===")
print("Status:", status)
print("Final hypothesis rows:", len(final))
print("Combined batch rows:", len(combined))
print("Missing/extra keys:", missing_keys)
print("Field-level differences:", len(diff))

print("\n=== BATCH HASH VERIFICATION ===")
for label, result in recorded_hash_checks.items():
    print(label, result)

print("\n=== BATCH SUMMARY ===")
print(
    batch_table[
        ["batch", "rows", "unique_audios",
         "unique_hypotheses"]
    ].to_string(index=False)
)

print("\n=== DIFFERENCE EXAMPLES ===")
if diff.empty:
    print("None")
else:
    print(diff.head(15).to_string(index=False))

print("\n=== D08 EVIDENCE BOUNDARY ===")
print("Saved generation metadata: VERIFIED")
print("Batch-to-final consistency:",
      "PASS" if status == "PASS" else "NEEDS REVIEW")
print("Historical inference source code: NOT RECOVERED")
print("Full runtime independently replayed: NO")

print("\nReport:", REPORT)
print("Report SHA256:", sha(REPORT))
print("Batch CSV:", BATCH_REPORT)
print("Difference CSV:", DIFFERENCES)


In [ ]:

from pathlib import Path
import json
import hashlib
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_provenance"
OUT.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "D07_reconstruction": (
        OUT / "step08_07b_source_reconstruction.json",
        "64f25a9b45a8720ef2b197630fef16b325b193189b69d6b17396efb18eeeea97"
    ),
    "D08_metadata": (
        OUT / "step08_04_corrected_metadata_audit.json",
        "e8e1f3ac5c5ea143e18114230fc05f012b33d2e19f4eb354794b36b0c37cc45f"
    ),
    "D08_batches": (
        OUT / "step08_08_asr_batch_integrity.json",
        "6fbe46b3093d8cd23b817b33f1ec89e07f8b9e08aaf979277bb5b8b33b608c88"
    )
}

RECORD = OUT / "step08_09_final_provenance_record.json"
TABLE = OUT / "step08_09_provenance_decisions.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

audits = {}

for name, (path, expected) in SOURCES.items():
    assert path.is_file(), f"Missing source: {path}"
    assert sha(path) == expected, f"SHA mismatch: {name}"
    audits[name] = json.loads(path.read_text(encoding="utf-8"))

d07 = audits["D07_reconstruction"]
d08_metadata = audits["D08_metadata"]
d08_batches = audits["D08_batches"]

assert d07["status"] == "RECONSTRUCTION_MATCH"
assert d07["selected_speakers_match"] is True
assert d07["selected_audio_ids_match"] is True
assert d07["source_row_indices_match"] is True
assert d07["fluency_labels_match"] is True
assert d07["reading_prompts_match"] is True

assert d08_metadata["status"] == "PASS"
assert d08_metadata["metadata_mismatch_count"] == 0
assert d08_batches["status"] == "PASS"
assert d08_batches["missing_or_extra_keys"] == 0
assert d08_batches["field_level_differences"] == 0

assert all(
    x["matches"] is True
    for x in d08_batches["progress_batch_hashes"].values()
)

decisions = [
    {
        "decision_id": "D07",
        "status": "EXACT_SOURCE_BASED_RECONSTRUCTION",
        "verified": (
            "20 speaker IDs, all 400 selected audio IDs, "
            "source rows, prompt texts, fluency labels and "
            "audio paths reproduced from source train parquet."
        ),
        "limitation": (
            "Historical sampling source code unavailable; "
            "exact reconstruction does not establish code identity."
        )
    },
    {
        "decision_id": "D08",
        "status": "SAVED_ASR_ARTIFACT_PROVENANCE_VERIFIED",
        "verified": (
            "Frozen model revision and H1-H5 metadata match "
            "all 2000 saved ASR hypotheses. Four original "
            "batches exactly match final dataset."
        ),
        "limitation": (
            "Historical full-run inference code unavailable; "
            "original GPU inference not independently replayed."
        )
    },
    {
        "decision_id": "D09",
        "status": "LIMITATION_RETAINED",
        "verified": "No ASR pretraining-overlap audit performed.",
        "limitation": (
            "Do not claim absence of overlap between "
            "SpeechOcean762 and ASR pretraining data."
        )
    },
    {
        "decision_id": "D10",
        "status": "SCOPE_RESTRICTION",
        "verified": (
            "Prompt-aware scores are evaluated under "
            "a read-aloud task with available prompts."
        ),
        "limitation": (
            "Actual deployment prompt availability "
            "has not been independently established."
        )
    },
    {
        "decision_id": "D11",
        "status": "LIMITATION_RETAINED",
        "verified": (
            "Speaker-disjoint internal cross-validation "
            "was completed on the selected train subset."
        ),
        "limitation": (
            "No external validation or evaluation "
            "on the sealed official test partition."
        )
    }
]

table = pd.DataFrame(decisions)

methods_sampling = (
    "The analysis used 400 recordings from 20 speakers "
    "(20 recordings per speaker) selected from the "
    "SpeechOcean762 training partition. The saved sample "
    "was exactly reconstructed from the original source "
    "metadata using a NumPy default_rng(42) choice of "
    "20 out of 125 speaker positions without replacement, "
    "followed by selection of all 20 recordings for each "
    "selected speaker. Although the original sampling "
    "source code was unavailable, the reconstructed "
    "speaker IDs, recording IDs, source-row positions, "
    "fluency labels, reading prompts, and audio paths "
    "matched the frozen selection manifest."
)

methods_asr = (
    "Five ASR hypotheses per recording were generated "
    "using a frozen configuration documenting "
    "openai/whisper-small.en and its model revision. "
    "H1 used greedy decoding (one beam), H2 used "
    "five-beam search, and H3-H5 used stochastic "
    "decoding at temperature 0.3 with recorded seeds "
    "101, 202, and 303. All 2,000 saved hypothesis "
    "metadata rows agreed with the frozen configuration, "
    "and all four batch outputs matched the final "
    "hypothesis dataset without field-level differences."
)

limitations = (
    "The original historical sampling and full-run ASR "
    "inference source code could not be recovered. "
    "Sampling was reconstructed exactly from source "
    "data, whereas ASR reproducibility was verified "
    "at the configuration and saved-artifact levels "
    "without independently replaying inference. "
    "Potential overlap with ASR pretraining data "
    "was not investigated. External validation "
    "was not performed."
)

report = {
    "step": "8.9",
    "status": "PROVENANCE_DOCUMENTED",
    "D07": decisions[0]["status"],
    "D08": decisions[1]["status"],
    "D09": decisions[2]["status"],
    "D10": decisions[3]["status"],
    "D11": decisions[4]["status"],
    "methods_sampling_draft": methods_sampling,
    "methods_asr_draft": methods_asr,
    "limitations_draft": limitations,
    "historical_sampling_code_available": False,
    "historical_asr_code_available": False,
    "independent_asr_replay_performed": False,
    "source_report_sha256": {
        name: sha(path)
        for name, (path, _) in SOURCES.items()
    },
    "models_retrained": False,
    "original_results_modified": False
}

def save_csv_immutable(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing CSV differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv_immutable(table, TABLE)
report["decision_table_sha256"] = sha(TABLE)

tmp = RECORD.with_suffix(".tmp")
tmp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if RECORD.exists():
    assert json.loads(
        RECORD.read_text(encoding="utf-8")
    ) == report
    tmp.unlink()
else:
    tmp.replace(RECORD)

print("=== STEP 8.9 FINAL PROVENANCE RECORD ===")
print("Status:", report["status"])

print("\n=== PROVENANCE DECISIONS ===")
print(
    table[["decision_id", "status"]]
    .to_string(index=False)
)

print("\n=== METHODS: SAMPLING ===")
print(methods_sampling)

print("\n=== METHODS: ASR ===")
print(methods_asr)

print("\n=== LIMITATIONS ===")
print(limitations)

print("\n=== OUTPUT FILES ===")
print("Decision CSV:", TABLE)
print("Record JSON:", RECORD)
print("Record SHA256:", sha(RECORD))
print("Original predictions modified: NO")
print("Models retrained: NO")


In [ ]:

from pathlib import Path
import hashlib
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
TABLE = P / "results/step06_final_tables/table01_primary_models_with_ci.csv"
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure02_model_performance.png"
PDF = OUT / "figure02_model_performance.pdf"
REPORT = OUT / "figure02_model_performance_manifest.json"

EXPECTED_SHA = (
    "22512d4fdd54feec3f6658f99248653e8c9fb99e110887db64578aadfb43b4c0"
)

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert TABLE.is_file()
assert sha(TABLE) == EXPECTED_SHA

df = pd.read_csv(TABLE)

print("=== SOURCE TABLE COLUMNS ===")
print(df.columns.tolist())

print("\n=== SOURCE TABLE PREVIEW ===")
print(df.head(8).to_string(index=False))

# Column detection is explicit to avoid silently
# plotting the wrong metric.
def find_col(candidates):
    matches = [
        col for col in df.columns
        if col.lower() in candidates
    ]
    assert len(matches) == 1, (
        f"Expected one column from {candidates}; "
        f"found {matches}"
    )
    return matches[0]

MODEL_COL = find_col({"model", "model_name"})
MAE_COL = find_col({"mae"})
RMSE_COL = find_col({"rmse"})

data = df[[MODEL_COL, MAE_COL, RMSE_COL]].copy()
data.columns = ["model", "MAE", "RMSE"]

data["model"] = data["model"].astype(str).str.strip()

ORDER = ["M0", "A", "B1", "B2", "C1", "C2"]

assert set(data["model"]) == set(ORDER)
assert not data["model"].duplicated().any()

data = data.set_index("model").loc[ORDER].reset_index()

expected = {
    "M0": (1.3812, 1.6790),
    "A": (0.9399, 1.2066),
    "B1": (1.2426, 1.9972),
    "B2": (1.1761, 1.9518),
    "C1": (0.8972, 1.3735),
    "C2": (0.7782, 1.2461)
}

for _, row in data.iterrows():
    mae_expected, rmse_expected = expected[row["model"]]
    assert np.isclose(row["MAE"], mae_expected, atol=1e-4)
    assert np.isclose(row["RMSE"], rmse_expected, atol=1e-4)

labels = [
    "M0\nMean baseline",
    "A\nAcoustic",
    "B1\nLinguistic",
    "B2\nLinguistic + prompt",
    "C1\nFusion",
    "C2\nFusion + prompt"
]

# Two aligned panels prevent confusion between
# MAE and RMSE scales.
fig, axes = plt.subplots(
    1, 2,
    figsize=(12.5, 5.2),
    sharex=True
)

x = np.arange(len(ORDER))

for ax, metric in zip(axes, ["MAE", "RMSE"]):
    values = data[metric].to_numpy(dtype=float)

    bars = ax.bar(
        x,
        values,
        width=0.68,
        color="#778899",
        edgecolor="#30343B",
        linewidth=0.6
    )

    best_index = int(np.argmin(values))
    bars[best_index].set_color("#187B78")

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.035,
            f"{value:.3f}",
            ha="center", va="bottom",
            fontsize=9
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, 2.25)
    ax.set_ylabel(metric + " (lower is better)")
    ax.set_title(metric, fontsize=13, weight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.22)
    ax.set_axisbelow(True)

fig.suptitle(
    "Speaker-Disjoint Cross-Validated Fluency Prediction",
    fontsize=14,
    weight="bold"
)

fig.text(
    0.5, 0.015,
    "400 recordings, 20 speakers, five ASR hypotheses per recording. "
    "Fixed Ridge alpha=1.0.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.055, 1, 0.94])

fig.savefig(
    PNG, dpi=300,
    bbox_inches="tight"
)
fig.savefig(
    PDF,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

assert PNG.is_file() and PNG.stat().st_size > 0
assert PDF.is_file() and PDF.stat().st_size > 0

manifest = {
    "figure": "Figure 2",
    "title": "Speaker-disjoint model performance comparison",
    "data_source": str(TABLE.relative_to(P)),
    "source_sha256": sha(TABLE),
    "model_order": ORDER,
    "ridge_alpha": 1.0,
    "original_audio_count": 400,
    "speaker_count": 20,
    "hypotheses_per_audio": 5,
    "best_mae_model": "C2",
    "best_rmse_model": "A",
    "values": data.to_dict(orient="records"),
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "interpretation_note": (
        "Bars show original out-of-fold point estimates. "
        "Uncertainty intervals are not shown in this figure. "
        "Prompt-aware models are secondary ablations."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == manifest, "Existing figure manifest differs"
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("\n=== STEP 8.10 FIGURE 2 ===")
print("Status: PASS")
print("Models:", ", ".join(ORDER))
print("Best MAE:", "C2")
print("Best RMSE:", "A")
print("Original metrics unchanged: YES")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", REPORT)
print("Manifest SHA256:", sha(REPORT))


In [ ]:

from pathlib import Path
import pandas as pd
import hashlib

P = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = (
    P / "results/step06_final_tables/"
    "table01_primary_models_with_ci.csv"
)

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=== STEP 8.10 FIGURE 2 SOURCE DIAGNOSTIC ===")

assert SOURCE.is_file(), f"Missing: {SOURCE}"

df = pd.read_csv(SOURCE)

print("\n=== SOURCE FILE ===")
print("Path:", SOURCE)
print("SHA256:", sha(SOURCE))
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\n=== COLUMN NAMES ===")
for col in df.columns:
    print(repr(col))

print("\n=== COMPLETE SOURCE TABLE ===")
print(df.to_string(index=False, max_colwidth=100))

print("\n=== MODEL IDENTIFIERS ===")
for col in df.columns:
    if "model" in col.lower():
        print(f"\nColumn: {col}")
        print(df[col].astype("string").unique().tolist())

print("\n=== AVAILABLE RELATED TABLES ===")
folder = P / "results/step06_final_tables"

for path in sorted(folder.glob("*.csv")):
    try:
        preview = pd.read_csv(path, nrows=2)
        print(
            path.name,
            "| columns:",
            preview.columns.tolist()[:12]
        )
    except Exception as e:
        print(path.name, "| ERROR:", str(e)[:100])

print("\n=== DIAGNOSTIC SUMMARY ===")
print("Source inspected: YES")
print("Original results modified: NO")
print("Model training: NO")
print("Figure generated: NO")


In [ ]:

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = (
    P / "results/step06_final_tables/"
    "table01_primary_models_with_ci.csv"
)
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure02_model_performance_95ci.png"
PDF = OUT / "figure02_model_performance_95ci.pdf"
MANIFEST = OUT / "figure02_model_performance_95ci.json"

EXPECTED_SHA = (
    "22512d4fdd54feec3f6658f99248653e8c9fb99e110887db64578aadfb43b4c0"
)

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert SOURCE.is_file()
assert sha(SOURCE) == EXPECTED_SHA

df = pd.read_csv(SOURCE)

ORDER = ["A", "B1", "B2", "C1", "C2"]

REQUIRED = [
    "model", "MAE", "MAE_CI_low", "MAE_CI_high",
    "RMSE", "RMSE_CI_low", "RMSE_CI_high"
]

assert set(REQUIRED).issubset(df.columns)
assert set(df["model"]) == set(ORDER)
assert not df["model"].duplicated().any()

data = df.set_index("model").loc[ORDER].reset_index()

expected = {
    "A":  (0.939944, 1.206617),
    "B1": (1.242553, 1.997205),
    "B2": (1.176070, 1.951791),
    "C1": (0.897204, 1.373540),
    "C2": (0.778167, 1.246079)
}

for row in data.itertuples(index=False):
    exp_mae, exp_rmse = expected[row.model]
    assert np.isclose(row.MAE, exp_mae, atol=1e-6)
    assert np.isclose(row.RMSE, exp_rmse, atol=1e-6)

for metric in ["MAE", "RMSE"]:
    assert (
        data[f"{metric}_CI_low"] <= data[metric]
    ).all()
    assert (
        data[metric] <= data[f"{metric}_CI_high"]
    ).all()

labels = [
    "A\nAcoustic",
    "B1\nLinguistic",
    "B2\nLinguistic\n+ prompt",
    "C1\nFusion",
    "C2\nFusion\n+ prompt"
]

x = np.arange(len(ORDER))

# Colors distinguish prompt-independent and
# prompt-aware models, not statistical significance.
colors = [
    "#4D647C",
    "#4D647C",
    "#B5A081",
    "#4D647C",
    "#B5A081"
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, axes = plt.subplots(
    1, 2,
    figsize=(12.2, 5.4),
    sharey=False
)

for ax, metric in zip(axes, ["MAE", "RMSE"]):
    values = data[metric].to_numpy(dtype=float)
    lower = data[f"{metric}_CI_low"].to_numpy(dtype=float)
    upper = data[f"{metric}_CI_high"].to_numpy(dtype=float)

    errors = np.vstack([
        values - lower,
        upper - values
    ])

    ax.bar(
        x,
        values,
        width=0.65,
        color=colors,
        edgecolor="#414141",
        linewidth=0.6,
        zorder=2
    )

    ax.errorbar(
        x,
        values,
        yerr=errors,
        fmt="none",
        ecolor="#242424",
        capsize=4,
        elinewidth=1.2,
        capthick=1.2,
        zorder=3
    )

    for i, val in enumerate(values):
        ax.text(
            i, val + 0.045,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel(metric + " (lower is better)")
    ax.set_title(
        metric + " with 95% bootstrap CI",
        fontsize=12,
        fontweight="bold"
    )

    ymax = max(upper) * 1.18
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.18, zorder=0)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

from matplotlib.patches import Patch

legend_items = [
    Patch(
        facecolor="#4D647C",
        label="Prompt-independent"
    ),
    Patch(
        facecolor="#B5A081",
        label="Prompt-aware (secondary)"
    )
]

fig.legend(
    handles=legend_items,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.905),
    ncol=2,
    frameon=False,
    fontsize=9
)

fig.suptitle(
    "Fluency Prediction Performance Across Model Configurations",
    fontsize=14,
    fontweight="bold",
    y=1.005
)

fig.text(
    0.5, -0.015,
    "Five-fold speaker-disjoint CV; 400 recordings from 20 speakers; "
    "Ridge alpha=1.0. Error bars: speaker-cluster bootstrap 95% CI.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.04, 1, 0.83])

fig.savefig(PNG, dpi=300, bbox_inches="tight")
fig.savefig(PDF, bbox_inches="tight")
plt.show()
plt.close(fig)

assert PNG.stat().st_size > 0
assert PDF.stat().st_size > 0

manifest = {
    "figure": "Figure 2",
    "title": "Model performance with 95% bootstrap confidence intervals",
    "source_file": str(SOURCE.relative_to(P)),
    "source_sha256": sha(SOURCE),
    "model_order": ORDER,
    "model_M0_included": False,
    "ridge_alpha": 1.0,
    "cross_validation": "5-fold speaker-disjoint",
    "confidence_interval": "speaker-cluster bootstrap 95%",
    "bootstrap_draws": 10000,
    "prompt_independent": ["A", "B1", "C1"],
    "prompt_aware_secondary": ["B2", "C2"],
    "data": data[REQUIRED].to_dict(orient="records"),
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "interpretation": (
        "Confidence intervals describe uncertainty in "
        "individual model metrics. Inferential claims "
        "about model differences must use the previously "
        "computed paired speaker-cluster bootstrap results, "
        "not visual overlap of these intervals."
    )
}

tmp = MANIFEST.with_suffix(".tmp")
tmp.write_text(
    json.dumps(
        manifest, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == manifest
    tmp.unlink()
else:
    tmp.replace(MANIFEST)

print("\n=== STEP 8.10 FIGURE 2 FINAL ===")
print("Status: PASS")
print("Models:", ORDER)
print("95% bootstrap CI included: YES")
print("M0 excluded: YES")
print("Best MAE: C2")
print("Best RMSE: A")

print("\n=== OUTPUT ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original results modified: NO")


In [ ]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = P / "results/step06_final_tables/table01_primary_models_with_ci.csv"
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure03_accuracy_stability_tradeoff.png"
PDF = OUT / "figure03_accuracy_stability_tradeoff.pdf"
MANIFEST = OUT / "figure03_accuracy_stability_tradeoff.json"

EXPECTED_SOURCE_SHA = (
    "22512d4fdd54feec3f6658f99248653e8c9fb99e110887db64578aadfb43b4c0"
)

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert SOURCE.is_file()
assert sha(SOURCE) == EXPECTED_SOURCE_SHA

df = pd.read_csv(SOURCE)
ORDER = ["A", "B1", "B2", "C1", "C2"]

required = [
    "model", "MAE", "MAE_CI_low", "MAE_CI_high",
    "mean_prediction_range", "range_CI_low", "range_CI_high"
]

assert set(required).issubset(df.columns)
assert set(df["model"]) == set(ORDER)
assert not df["model"].duplicated().any()

data = df.set_index("model").loc[ORDER].reset_index()

# Cross-check frozen headline values.
expected = {
    "A": (0.939944, 0.000000),
    "B1": (1.242553, 0.330215),
    "B2": (1.176070, 0.478216),
    "C1": (0.897204, 0.176617),
    "C2": (0.778167, 0.228505)
}

for row in data.itertuples(index=False):
    mae, rng = expected[row.model]
    assert np.isclose(row.MAE, mae, atol=1e-6)
    assert np.isclose(
        row.mean_prediction_range, rng, atol=1e-6
    )

for point, lower, upper in [
    ("MAE", "MAE_CI_low", "MAE_CI_high"),
    (
        "mean_prediction_range",
        "range_CI_low",
        "range_CI_high"
    )
]:
    assert data[lower].le(data[point]).all()
    assert data[point].le(data[upper]).all()

# Separate model classes visually.
style = {
    "A":  ("#6B7280", "s"),
    "B1": ("#2563A6", "o"),
    "B2": ("#D18C32", "D"),
    "C1": ("#2563A6", "o"),
    "C2": ("#D18C32", "D")
}

fig, ax = plt.subplots(figsize=(9.2, 6.4))

for row in data.itertuples(index=False):
    name = row.model
    color, marker = style[name]

    x = float(row.mean_prediction_range)
    y = float(row.MAE)

    xerr = np.array([
        [max(0, x - float(row.range_CI_low))],
        [max(0, float(row.range_CI_high) - x)]
    ])
    yerr = np.array([
        [max(0, y - float(row.MAE_CI_low))],
        [max(0, float(row.MAE_CI_high) - y)]
    ])

    ax.errorbar(
        x, y,
        xerr=xerr,
        yerr=yerr,
        fmt=marker,
        markersize=10,
        color=color,
        ecolor=color,
        markeredgecolor="white",
        markeredgewidth=0.9,
        elinewidth=1.2,
        capsize=4,
        alpha=0.9,
        zorder=3
    )

    offsets = {
        "A": (10, -11),
        "B1": (12, 8),
        "B2": (12, -10),
        "C1": (-29, 12),
        "C2": (12, -11)
    }

    ax.annotate(
        name,
        xy=(x, y),
        xytext=offsets[name],
        textcoords="offset points",
        fontsize=11,
        fontweight="bold",
        color=color
    )

ax.set_xlim(-0.07, 0.92)
ax.set_ylim(0.45, 1.85)

ax.set_xlabel(
    "Mean prediction range across H1–H5 "
    "(lower = more stable)",
    fontsize=11
)
ax.set_ylabel(
    "Mean absolute error (lower = more accurate)",
    fontsize=11
)

ax.set_title(
    "Accuracy–Stability Trade-off Across Model Configurations",
    fontsize=13,
    fontweight="bold",
    pad=16
)

ax.grid(alpha=0.18)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

from matplotlib.lines import Line2D

legend_items = [
    Line2D(
        [0], [0], marker="s", color="none",
        markerfacecolor="#6B7280",
        markeredgecolor="#6B7280",
        markersize=9, label="Acoustic-only (A)"
    ),
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor="#2563A6",
        markeredgecolor="#2563A6",
        markersize=9, label="Prompt-independent (B1, C1)"
    ),
    Line2D(
        [0], [0], marker="D", color="none",
        markerfacecolor="#D18C32",
        markeredgecolor="#D18C32",
        markersize=9, label="Prompt-aware (B2, C2)"
    )
]

ax.legend(
    handles=legend_items,
    loc="upper left",
    frameon=True,
    fontsize=9
)

fig.text(
    0.5, 0.025,
    "Error bars: marginal 95% speaker-cluster bootstrap CI "
    "(10,000 draws). Fixed Ridge alpha=1.0.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.06, 1, 1])

fig.savefig(PNG, dpi=300, bbox_inches="tight")
fig.savefig(PDF, bbox_inches="tight")
plt.show()
plt.close(fig)

assert PNG.is_file() and PNG.stat().st_size > 0
assert PDF.is_file() and PDF.stat().st_size > 0

manifest = {
    "figure": "Figure 3",
    "title": "Accuracy-Stability Trade-off",
    "status": "CREATED",
    "source": str(SOURCE.relative_to(P)),
    "source_sha256": sha(SOURCE),
    "models": ORDER,
    "ridge_alpha": 1.0,
    "cross_validation": "5-fold speaker-disjoint",
    "speaker_clusters": 20,
    "bootstrap_draws": 10000,
    "x_axis": "mean_prediction_range",
    "y_axis": "MAE",
    "ci_type": "marginal speaker-cluster bootstrap 95%",
    "values": data[required].to_dict(orient="records"),
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "interpretation_notes": [
        "A has zero ASR-induced prediction range by design.",
        "B1 and C1 are prompt-independent.",
        "B2 and C2 are prompt-aware secondary ablations.",
        "Marginal error bars cannot replace paired model-difference CIs.",
        "The plot does not demonstrate causal effects."
    ]
}

temp = MANIFEST.with_suffix(".tmp")
temp.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == manifest, "Existing manifest differs"
    temp.unlink()
else:
    temp.replace(MANIFEST)

print("\n=== STEP 8.11 FIGURE 3 ===")
print("Status: PASS")
print("Models:", ORDER)
print("Mean prediction range, C1:",
      round(expected["C1"][1], 4))
print("Mean prediction range, C2:",
      round(expected["C2"][1], 4))
print("MAE, C1:", round(expected["C1"][0], 4))
print("MAE, C2:", round(expected["C2"][0], 4))

print("\n=== SAVED FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original data modified: NO")


In [ ]:

from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = P / "results/step06_final_tables/step06_asr_variability_per_audio.csv"
ASSOC = P / "results/step06_final_tables/step06_variability_association.csv"
BOOT = P / "results/step06_final_tables/step06_variable_subset_bootstrap.csv"
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure04_lexical_variation_prediction_range.png"
PDF = OUT / "figure04_lexical_variation_prediction_range.pdf"
MANIFEST = OUT / "figure04_lexical_variation_prediction_range.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

assert all(p.exists() for p in [SOURCE, ASSOC, BOOT])

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 400
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert df["hypothesis_count"].eq(5).all()

MODELS = ["B1", "B2", "C1", "C2"]
assert all(f"{m}_range" in df.columns for m in MODELS)

df["lexical_unique"] = pd.to_numeric(
    df["lexical_unique"], errors="raise"
)
assert df["lexical_unique"].between(1, 5).all()

variable = df.loc[df["lexical_unique"] > 1].copy()

assert len(variable) == 172
assert variable["speaker"].nunique() == 19

EXPECTED_RHO = {
    "B1": 0.5951,
    "B2": 0.4777,
    "C1": 0.4699,
    "C2": 0.4117
}

rhos = {}

for model in MODELS:
    values = pd.to_numeric(
        variable[f"{model}_range"], errors="raise"
    )
    assert np.isfinite(values).all()
    assert values.ge(0).all()

    rho, p_naive = spearmanr(
        variable["lexical_unique"], values
    )

    assert np.isclose(
        rho, EXPECTED_RHO[model], atol=0.002
    ), f"{model} rho mismatch: {rho}"

    rhos[model] = float(rho)

# This is a descriptive visualization only.
# Speaker-cluster CIs remain in frozen STEP 6 tables;
# naive Spearman p-values are not used for inference.
fig, axes = plt.subplots(
    2, 2,
    figsize=(11.5, 8.5),
    sharex=True
)

for ax, model in zip(axes.flat, MODELS):
    ycol = f"{model}_range"

    plot_df = variable[
        ["lexical_unique", ycol]
    ].copy()

    grouped = (
        plot_df.groupby("lexical_unique")[ycol]
        .agg(["mean", "median", "count"])
        .reset_index()
    )

    # Deterministic horizontal jitter only for display.
    rng = np.random.default_rng(100 + MODELS.index(model))

    jitter = rng.uniform(
        -0.13, 0.13, len(plot_df)
    )

    ax.scatter(
        plot_df["lexical_unique"] + jitter,
        plot_df[ycol],
        s=20,
        alpha=0.38,
        color="#3C6E91",
        edgecolors="none",
        rasterized=True
    )

    # Median per lexical diversity level.
    ax.plot(
        grouped["lexical_unique"],
        grouped["median"],
        color="#B65C33",
        marker="D",
        linewidth=1.8,
        markersize=6,
        label="Group median"
    )

    # Use symmetric log display only if extreme outliers
    # otherwise hide the main distribution.
    upper = float(plot_df[ycol].max())

    if upper > 5:
        ax.set_yscale("symlog", linthresh=0.1)
        scale_label = " (symlog scale)"
    else:
        scale_label = ""

    ax.set_title(
        f"{model}  |  Spearman rho = {rhos[model]:.3f}",
        fontsize=11,
        fontweight="bold"
    )
    ax.set_ylabel(
        "Prediction range" + scale_label
    )
    ax.set_xticks([2, 3, 4, 5])
    ax.set_xlim(1.6, 5.4)
    ax.grid(alpha=0.18)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    print(f"\n=== {model} GROUP SUMMARY ===")
    print(grouped.to_string(index=False))

for ax in axes[1, :]:
    ax.set_xlabel(
        "Unique lexical ASR hypotheses per audio (out of 5)"
    )

axes[0, 0].legend(
    loc="upper left",
    fontsize=8,
    frameon=False
)

fig.suptitle(
    "Lexical ASR Diversity and Fluency Prediction Instability",
    fontsize=14,
    fontweight="bold",
    y=0.985
)

fig.text(
    0.5, 0.012,
    "172 lexically variable recordings from 19 speakers. "
    "Spearman associations are descriptive; "
    "each point represents one recording.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.04, 1, 0.95])

fig.savefig(
    PNG, dpi=300, bbox_inches="tight"
)
fig.savefig(
    PDF, bbox_inches="tight"
)
plt.show()
plt.close(fig)

assert PNG.stat().st_size > 0
assert PDF.stat().st_size > 0

manifest = {
    "figure": "Figure 4",
    "status": "CREATED",
    "title": (
        "Lexical ASR Diversity and Fluency "
        "Prediction Instability"
    ),
    "source_file": str(SOURCE.relative_to(P)),
    "source_sha256": sha(SOURCE),
    "association_source_sha256": sha(ASSOC),
    "bootstrap_source_sha256": sha(BOOT),
    "models": MODELS,
    "recording_count_total": 400,
    "lexically_variable_recordings": len(variable),
    "speaker_clusters_in_subset": int(
        variable["speaker"].nunique()
    ),
    "spearman_rho": rhos,
    "x_axis": "lexical_unique",
    "y_axis": "model-specific prediction range",
    "subset": "lexical_unique > 1",
    "interpretation_notes": [
        "Correlations are descriptive associations.",
        "Audio-level observations share speaker clusters.",
        "Naive p-values are not used for inference.",
        "Confidence intervals remain in the STEP 6 "
        "speaker-cluster bootstrap results.",
        "One extreme ASR hallucination was retained.",
        "Symlog axis is used only when maximum range exceeds 5."
    ],
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF)
}

temp = MANIFEST.with_suffix(".tmp")
temp.write_text(
    json.dumps(
        manifest, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == manifest, "Manifest differs from existing version"
    temp.unlink()
else:
    temp.replace(MANIFEST)

print("\n=== STEP 8.12 FIGURE 4 ===")
print("Status: PASS")
print("Total audios:", len(df))
print("Lexically variable audios:", len(variable))
print("Speakers in subset:", variable["speaker"].nunique())

print("\nSpearman rho:")
for model, rho in rhos.items():
    print(f"  {model}: {rho:.4f}")

print("\n=== SAVED FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original data modified: NO")


In [ ]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
ASR = P / "features/asr/full/asr_hypotheses_0001-0400.csv"
DIAG = P / "results/step06_final_tables/step06_ood_diagnostic_per_hypothesis.csv"

OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure05_critical_asr_hallucination.png"
PDF = OUT / "figure05_critical_asr_hallucination.pdf"
CSV = OUT / "figure05_critical_asr_hallucination_data.csv"
MANIFEST = OUT / "figure05_critical_asr_hallucination.json"

TARGET = "030210164"
ORDER = ["H1", "H2", "H3", "H4", "H5"]

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def save_immutable(path, writer):
    tmp = path.with_name(path.stem + ".temporary" + path.suffix)
    writer(tmp)
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing artifact differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

assert ASR.exists() and DIAG.exists()

asr = pd.read_csv(
    ASR, dtype={"audio_id": "string", "speaker": "string"}
)
diag = pd.read_csv(
    DIAG, dtype={"audio_id": "string", "speaker": "string"}
)

required_diag = {
    "audio_id", "speaker", "hypothesis_id",
    "fluency", "c2_prediction",
    "acoustic_ridge_prediction"
}

required_asr = {
    "audio_id", "hypothesis_id", "transcription"
}

assert required_diag.issubset(diag.columns)
assert required_asr.issubset(asr.columns)

d = diag.loc[
    diag["audio_id"] == TARGET
].copy()

a = asr.loc[
    asr["audio_id"] == TARGET
].copy()

assert len(d) == 5
assert len(a) == 5
assert set(d["hypothesis_id"]) == set(ORDER)
assert set(a["hypothesis_id"]) == set(ORDER)

data = d.merge(
    a[["audio_id", "hypothesis_id", "transcription"]],
    on=["audio_id", "hypothesis_id"],
    validate="one_to_one"
)

data["word_count"] = (
    data["transcription"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

data = (
    data.set_index("hypothesis_id")
    .loc[ORDER]
    .reset_index()
)

assert data["fluency"].nunique() == 1
assert np.isclose(float(data["fluency"].iloc[0]), 7.0)

h2 = data.loc[data["hypothesis_id"] == "H2"].iloc[0]

assert int(h2["word_count"]) == 96
assert np.isclose(
    float(h2["c2_prediction"]),
    -26.1267,
    atol=0.002
)

human = float(data["fluency"].iloc[0])
acoustic = float(
    data["acoustic_ridge_prediction"].iloc[0]
)

assert np.allclose(
    data["acoustic_ridge_prediction"].to_numpy(float),
    acoustic,
    atol=1e-8
)

x = np.arange(5)
c2 = data["c2_prediction"].to_numpy(float)
words = data["word_count"].to_numpy(int)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12.5, 5.4),
    gridspec_kw={"width_ratios": [1.3, 1]}
)

# Panel A: downstream scores.
ax = axes[0]

ax.plot(
    x, c2,
    marker="o",
    linewidth=2,
    markersize=8,
    color="#315E83",
    label="C2 prediction"
)

ax.scatter(
    [1], [c2[1]],
    s=155,
    marker="o",
    facecolors="none",
    edgecolors="#B44336",
    linewidths=2.2,
    zorder=5
)

ax.axhline(
    human,
    color="#333333",
    linestyle="--",
    linewidth=1.7,
    label=f"Human fluency = {human:g}"
)

ax.axhline(
    acoustic,
    color="#348369",
    linestyle=":",
    linewidth=1.8,
    label=f"Acoustic-only A = {acoustic:.2f}"
)

ax.annotate(
    f"H2: {c2[1]:.2f}",
    xy=(1, c2[1]),
    xytext=(12, 14),
    textcoords="offset points",
    color="#B44336",
    fontsize=10,
    weight="bold"
)

ax.set_xticks(x, ORDER)
ax.set_ylabel("Predicted fluency score")
ax.set_title("A. Scoring instability", fontweight="bold")
ax.grid(axis="y", alpha=0.2)
ax.legend(fontsize=8.5, loc="lower right")

# Panel B: transcript length.
ax = axes[1]

colors = [
    "#7890A3" if h != "H2" else "#B44336"
    for h in ORDER
]

bars = ax.bar(
    x, words,
    color=colors,
    edgecolor="#333333",
    linewidth=0.5
)

for bar, count in zip(bars, words):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        str(int(count)),
        ha="center", va="bottom",
        fontsize=10
    )

ax.set_xticks(x, ORDER)
ax.set_ylabel("Transcript word count")
ax.set_title("B. ASR transcript length", fontweight="bold")
ax.set_ylim(0, max(words) * 1.16)
ax.grid(axis="y", alpha=0.2)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)

fig.suptitle(
    "Critical ASR Hallucination and Downstream Score Instability",
    fontsize=13.5,
    fontweight="bold"
)

fig.text(
    0.5, 0.015,
    f"Recording {TARGET}; observed human fluency = 7. "
    "An illustrative single-recording case, not a population estimate.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.055, 1, 0.93])

# Write once; never overwrite different existing evidence.
save_immutable(
    PNG,
    lambda p: fig.savefig(
        p, dpi=300, bbox_inches="tight",
        metadata={"Software": "matplotlib"}
    )
)

save_immutable(
    PDF,
    lambda p: fig.savefig(
        p, bbox_inches="tight",
        metadata={
            "Title": "Critical ASR Hallucination Case",
            "Creator": "matplotlib",
            "CreationDate": None
        }
    )
)

plt.show()
plt.close(fig)

save_columns = [
    "audio_id", "speaker", "hypothesis_id",
    "fluency", "c2_prediction",
    "acoustic_ridge_prediction",
    "word_count", "transcription"
]

save_immutable(
    CSV,
    lambda p: data[save_columns].to_csv(
        p, index=False, encoding="utf-8"
    )
)

manifest = {
    "figure": "Figure 5",
    "status": "CREATED",
    "audio_id": TARGET,
    "speaker": str(data["speaker"].iloc[0]),
    "human_fluency": human,
    "acoustic_prediction": acoustic,
    "h2_c2_prediction": float(c2[1]),
    "h2_word_count": int(words[1]),
    "h2_absolute_error": abs(float(c2[1]) - human),
    "source_sha256": {
        str(ASR.relative_to(P)): sha(ASR),
        str(DIAG.relative_to(P)): sha(DIAG)
    },
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "data_csv_sha256": sha(CSV),
    "interpretation": (
        "Descriptive case study of an extreme ASR output "
        "and its association with a downstream C2 score. "
        "No score clipping or case exclusion applied."
    )
}

save_immutable(
    MANIFEST,
    lambda p: p.write_text(
        json.dumps(
            manifest, indent=2, ensure_ascii=False
        ),
        encoding="utf-8"
    )
)

print("\n=== STEP 8.13 FIGURE 5 ===")
print("Status: PASS")
print("Audio ID:", TARGET)
print("Human fluency:", human)
print("Acoustic-only prediction:", round(acoustic, 4))
print("H2 C2 prediction:", round(float(c2[1]), 4))
print("H2 absolute error:", round(abs(float(c2[1])-human), 4))

print("\n=== HYPOTHESIS DATA ===")
print(
    data[
        ["hypothesis_id", "word_count",
         "c2_prediction", "acoustic_ridge_prediction"]
    ].to_string(index=False)
)

print("\n=== OUTPUT FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Data CSV:", CSV)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original predictions modified: NO")


In [ ]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = (
    P / "results/step06_final_tables/"
    "step06_c2_critical_h2_vs_h1_contributions.csv"
)
CASE = (
    P / "results/step08_figures/"
    "figure05_critical_asr_hallucination_data.csv"
)
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure05b_h2_h1_feature_contributions.png"
PDF = OUT / "figure05b_h2_h1_feature_contributions.pdf"
CSV = OUT / "figure05b_feature_contribution_summary.csv"
REPORT = OUT / "figure05b_feature_contribution_manifest.json"

TARGET = "030210164"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

assert SOURCE.is_file()
assert CASE.is_file()

contrib = pd.read_csv(
    SOURCE, dtype={"audio_id": "string"}
)
case = pd.read_csv(
    CASE, dtype={"audio_id": "string"}
)

required = {
    "audio_id", "feature", "feature_group",
    "prediction_delta_contribution"
}
assert required.issubset(contrib.columns)

contrib = contrib.loc[
    contrib["audio_id"] == TARGET
].copy()

case = case.loc[
    case["audio_id"] == TARGET
].copy()

assert len(contrib) > 0
assert len(case) == 5

assert set(case["hypothesis_id"]) == {
    "H1", "H2", "H3", "H4", "H5"
}

h1 = float(
    case.loc[
        case["hypothesis_id"] == "H1",
        "c2_prediction"
    ].iloc[0]
)

h2 = float(
    case.loc[
        case["hypothesis_id"] == "H2",
        "c2_prediction"
    ].iloc[0]
)

observed_delta = h2 - h1

contrib["prediction_delta_contribution"] = pd.to_numeric(
    contrib["prediction_delta_contribution"],
    errors="raise"
)

assert np.isfinite(
    contrib["prediction_delta_contribution"]
).all()

group_summary = (
    contrib.groupby("feature_group", as_index=False)
    ["prediction_delta_contribution"]
    .sum()
)

print("=== ORIGINAL FEATURE GROUP LABELS ===")
print(group_summary.to_string(index=False))

# Match documented groups without silently
# substituting unrecognized feature categories.
def categorize(name):
    value = str(name).lower().strip()
    if "acoustic" in value:
        return "Acoustic"
    if "prompt" in value:
        return "Prompt-aware"
    if "linguistic" in value or "text" in value:
        return "Linguistic"
    raise ValueError(
        f"Unrecognized feature group: {name}"
    )

contrib["plot_group"] = (
    contrib["feature_group"].map(categorize)
)

ORDER = ["Acoustic", "Linguistic", "Prompt-aware"]

summary = (
    contrib.groupby("plot_group")
    ["prediction_delta_contribution"]
    .sum()
    .reindex(ORDER, fill_value=0.0)
)

total = float(summary.sum())
residual = observed_delta - total

# Validate against the previously documented
# decomposition, allowing only rounding tolerance.
expected = {
    "Acoustic": 0.0,
    "Linguistic": -15.7110,
    "Prompt-aware": -17.7889
}

for group, value in expected.items():
    assert np.isclose(
        summary[group], value,
        rtol=0, atol=0.01
    ), f"Unexpected {group}: {summary[group]}"

assert np.isclose(
    total, observed_delta,
    rtol=0, atol=0.001
), (
    f"Contributions do not reconstruct prediction: "
    f"{total} vs {observed_delta}"
)

# Plot feature-group contributions on a common
# signed scale; no posthoc clipping.
fig, ax = plt.subplots(figsize=(9.0, 4.8))

colors = ["#738596", "#376A91", "#B36B49"]
values = summary.to_numpy(dtype=float)

bars = ax.barh(
    ORDER,
    values,
    color=colors,
    edgecolor="#363636",
    linewidth=0.6
)

for bar, val in zip(bars, values):
    ax.text(
        val - 0.35 if val < 0 else val + 0.35,
        bar.get_y() + bar.get_height()/2,
        f"{val:+.3f}",
        ha="right" if val < 0 else "left",
        va="center",
        fontsize=10
    )

ax.axvline(0, color="#333333", linewidth=1)
ax.set_xlim(min(-21, float(values.min()) * 1.17), 3)
ax.invert_yaxis()
ax.set_xlabel(
    "Contribution to C2 prediction change "
    "(H2 minus H1)"
)
ax.set_title(
    "Critical ASR Case: Feature-Group Contributions",
    fontsize=13,
    fontweight="bold",
    pad=12
)
ax.grid(axis="x", alpha=0.18)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.text(
    0.5, 0.025,
    f"Audio {TARGET}: H1 = {h1:.3f}, "
    f"H2 = {h2:.3f}; "
    f"total change = {observed_delta:+.3f}.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.07, 1, 1])

fig.savefig(PNG, dpi=300, bbox_inches="tight")
fig.savefig(PDF, bbox_inches="tight")
plt.show()
plt.close(fig)

assert PNG.is_file() and PNG.stat().st_size > 0
assert PDF.is_file() and PDF.stat().st_size > 0

summary_df = pd.DataFrame({
    "feature_group": ORDER,
    "contribution_H2_minus_H1": values
})
summary_df.to_csv(CSV, index=False)

manifest = {
    "figure": "Figure 5B",
    "status": "PASS",
    "audio_id": TARGET,
    "model": "C2",
    "comparison": "H2 minus H1",
    "h1_prediction": h1,
    "h2_prediction": h2,
    "observed_prediction_delta": observed_delta,
    "summed_feature_contributions": total,
    "reconstruction_residual": residual,
    "contributions": {
        key: float(summary[key])
        for key in ORDER
    },
    "source_sha256": {
        str(SOURCE.relative_to(P)): sha(SOURCE),
        str(CASE.relative_to(P)): sha(CASE)
    },
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "csv_sha256": sha(CSV),
    "interpretation_note": (
        "Exact linear-model decomposition of the "
        "prediction difference between H2 and H1 "
        "within the same validation fold. "
        "This is a descriptive model attribution, "
        "not a causal estimate."
    )
}

tmp = REPORT.with_suffix(".tmp")
tmp.write_text(
    json.dumps(
        manifest, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == manifest, "Existing report differs"
    tmp.unlink()
else:
    tmp.replace(REPORT)

print("\n=== STEP 8.14 FIGURE 5B ===")
print("Status: PASS")
print("H1 prediction:", round(h1, 6))
print("H2 prediction:", round(h2, 6))
print("Observed H2-H1:", round(observed_delta, 6))

print("\n=== FEATURE CONTRIBUTIONS ===")
print(summary_df.to_string(index=False))
print("Summed contribution:", round(total, 6))
print("Reconstruction residual:", round(residual, 10))

print("\n=== OUTPUT FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("CSV:", CSV)
print("Manifest:", REPORT)
print("Manifest SHA256:", sha(REPORT))
print("Original predictions modified: NO")


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

P = Path("/kaggle/working/acoustic_linguistic_research")
SRC = P / "results/step06_final_tables"
OUT = P / "results/step08_publication_tables"
OUT.mkdir(parents=True, exist_ok=True)

MODELS = SRC / "table01_primary_models_with_ci.csv"
PAIRED = SRC / "table02_paired_differences_with_ci.csv"

OUT_MODELS = OUT / "table02a_model_performance.csv"
OUT_PAIRED = OUT / "table02b_paired_bootstrap_comparisons.csv"
MANIFEST = OUT / "step08_15_publication_tables_manifest.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert MODELS.exists() and PAIRED.exists()

model_df = pd.read_csv(MODELS)
paired_df = pd.read_csv(PAIRED)

required_models = {
    "model", "MAE", "MAE_CI_low", "MAE_CI_high",
    "RMSE", "RMSE_CI_low", "RMSE_CI_high",
    "mean_prediction_range", "range_CI_low",
    "range_CI_high"
}

required_paired = {
    "comparison", "metric", "difference",
    "CI_low", "CI_high", "CI_excludes_zero"
}

assert required_models.issubset(model_df.columns)
assert required_paired.issubset(paired_df.columns)

ORDER = ["A", "B1", "B2", "C1", "C2"]
assert set(model_df["model"]) == set(ORDER)
assert not model_df["model"].duplicated().any()

model_df = (
    model_df.set_index("model")
    .loc[ORDER]
    .reset_index()
)

def fmt_ci(point, low, high):
    return f"{point:.4f} [{low:.4f}, {high:.4f}]"

publication_models = pd.DataFrame({
    "Model": model_df["model"],
    "MAE (95% CI)": [
        fmt_ci(r.MAE, r.MAE_CI_low, r.MAE_CI_high)
        for r in model_df.itertuples(index=False)
    ],
    "RMSE (95% CI)": [
        fmt_ci(r.RMSE, r.RMSE_CI_low, r.RMSE_CI_high)
        for r in model_df.itertuples(index=False)
    ],
    "Prediction range (95% CI)": [
        fmt_ci(
            r.mean_prediction_range,
            r.range_CI_low,
            r.range_CI_high
        )
        for r in model_df.itertuples(index=False)
    ],
    "Model role": [
        "Primary: acoustic-only",
        "Primary: linguistic",
        "Secondary: linguistic + prompt",
        "Primary: acoustic + linguistic",
        "Secondary: full fusion + prompt"
    ]
})

# Keep source comparison and metric strings unchanged.
# Do not relabel or reconstruct any missing comparisons.
paired_df["difference"] = pd.to_numeric(
    paired_df["difference"], errors="raise"
)
paired_df["CI_low"] = pd.to_numeric(
    paired_df["CI_low"], errors="raise"
)
paired_df["CI_high"] = pd.to_numeric(
    paired_df["CI_high"], errors="raise"
)

assert np.isfinite(
    paired_df[["difference", "CI_low", "CI_high"]]
).all().all()

assert (
    paired_df["CI_low"] <= paired_df["difference"]
).all()

assert (
    paired_df["difference"] <= paired_df["CI_high"]
).all()

calculated_excludes_zero = (
    (paired_df["CI_low"] > 0)
    | (paired_df["CI_high"] < 0)
)

source_flag = (
    paired_df["CI_excludes_zero"]
    .astype(str).str.strip().str.lower()
)

assert source_flag.isin(["true", "false"]).all()
assert (
    source_flag.eq("true").to_numpy()
    == calculated_excludes_zero.to_numpy()
).all()

publication_paired = paired_df.copy()
publication_paired["Difference (95% CI)"] = [
    fmt_ci(r.difference, r.CI_low, r.CI_high)
    for r in paired_df.itertuples(index=False)
]

publication_paired["CI excludes zero"] = (
    np.where(calculated_excludes_zero, "Yes", "No")
)

publication_paired = publication_paired[
    ["comparison", "metric",
     "Difference (95% CI)", "CI excludes zero"]
].rename(columns={
    "comparison": "Comparison",
    "metric": "Metric"
})

# Verify that previously reported key contrasts are present,
# regardless of ordering or cosmetic spaces in the source.
def norm(value):
    return (
        str(value).lower()
        .replace(" ", "")
        .replace("−", "-")
        .replace("–", "-")
        .replace("_", "")
    )

def find_contrast(comparison, metric_tokens):
    rows = paired_df[
        paired_df["comparison"].map(norm).eq(norm(comparison))
        & paired_df["metric"].map(
            lambda s: any(
                token in norm(s) for token in metric_tokens
            )
        )
    ]
    return rows

checks = [
    ("C2-A", ["mae"], -0.1618),
    ("C2-C1", ["mae"], -0.1190),
    ("C2-C1", ["range"], +0.0519)
]

verified = []
for comparison, tokens, expected in checks:
    matches = find_contrast(comparison, tokens)
    if len(matches) == 1:
        value = float(matches.iloc[0]["difference"])
        assert np.isclose(value, expected, atol=0.0002)
        verified.append(comparison + " / " + tokens[0])
    else:
        print(
            "Contrast not uniquely identified by name:",
            comparison, tokens,
            "matches:", len(matches)
        )

def save_immutable(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing CSV differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_immutable(publication_models, OUT_MODELS)
save_immutable(publication_paired, OUT_PAIRED)

manifest = {
    "step": "8.15",
    "status": "PASS",
    "models": ORDER,
    "model_count": len(publication_models),
    "paired_comparison_rows": len(publication_paired),
    "key_contrasts_identified": verified,
    "source_hashes": {
        str(MODELS.relative_to(P)): sha(MODELS),
        str(PAIRED.relative_to(P)): sha(PAIRED)
    },
    "output_hashes": {
        OUT_MODELS.name: sha(OUT_MODELS),
        OUT_PAIRED.name: sha(OUT_PAIRED)
    },
    "statistical_notes": [
        "95% CIs use speaker-cluster bootstrap.",
        "Paired contrasts use speaker-level paired resampling.",
        "Multiple-comparison adjustments were not applied.",
        "Prompt-aware B2 and C2 are secondary ablations.",
        "A is ASR-invariant by design.",
        "No model retraining or new statistical tests."
    ]
}

temp = MANIFEST.with_suffix(".tmp")
temp.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == manifest
    temp.unlink()
else:
    temp.replace(MANIFEST)

print("=== STEP 8.15 PUBLICATION TABLES ===")
print("Status: PASS")
print("Model rows:", len(publication_models))
print("Paired comparison rows:", len(publication_paired))
print("Verified key contrasts:", verified)

print("\n=== TABLE 2A — MODEL PERFORMANCE ===")
print(publication_models.to_string(index=False))

print("\n=== TABLE 2B — PAIRED COMPARISONS ===")
print(publication_paired.to_string(index=False))

print("\n=== SAVED FILES ===")
print(OUT_MODELS)
print(OUT_PAIRED)
print(MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original frozen results modified: NO")


In [ ]:

from pathlib import Path
import json
import hashlib
import re

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
SRC = P / "results/step06_final_tables"
OUT = P / "results/step08_publication_tables"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE = SRC / "table02_paired_differences_with_ci.csv"
EXISTING = OUT / "table02b_paired_bootstrap_comparisons.csv"

CORRECTED = OUT / "step08_15a_verified_key_contrasts.csv"
REPORT = OUT / "step08_15a_contrast_verification.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert SOURCE.is_file()
assert EXISTING.is_file()

df = pd.read_csv(SOURCE)
pub = pd.read_csv(EXISTING)

assert len(df) == 18
assert len(pub) == 18

required = {
    "comparison", "metric", "difference",
    "CI_low", "CI_high", "CI_excludes_zero"
}
assert required.issubset(df.columns)

# Normalize historical source labels such as
# C2_minus_C1, C2-C1 and C2 minus C1 consistently.
def normalize_contrast(value):
    s = str(value).lower().strip()
    s = s.replace("−", "-").replace("–", "-")
    s = re.sub(r"\s*[_ -]*minus[_ -]*\s*", "-", s)
    s = re.sub(r"[\s_]+", "", s)
    return s

def normalize_metric(value):
    s = str(value).lower().strip()
    s = re.sub(r"[\s_-]+", "", s)
    if s in ("meanrange", "meanpredictionrange", "range"):
        return "range"
    return s

df["canonical_comparison"] = (
    df["comparison"].map(normalize_contrast)
)
df["canonical_metric"] = df["metric"].map(normalize_metric)

assert not df.duplicated(
    ["canonical_comparison", "canonical_metric"]
).any()

targets = [
    ("C2-A", "mae", -0.1618, -0.2367, -0.0858),
    ("C2-C1", "mae", -0.1190, -0.1841, -0.0598),
    ("C2-C1", "rmse", -0.1275, -0.1885, -0.0710),
    ("C2-C1", "range", +0.0519, +0.0095, +0.1013),
]

verified = []

for comp, metric, value, lo, hi in targets:
    match = df.loc[
        df["canonical_comparison"].eq(
            normalize_contrast(comp)
        )
        & df["canonical_metric"].eq(metric)
    ]

    assert len(match) == 1, (
        f"Contrast not unique: {comp}/{metric}; "
        f"matches={len(match)}"
    )

    row = match.iloc[0]

    assert np.isclose(
        row["difference"], value,
        rtol=0, atol=0.00015
    )
    assert np.isclose(
        row["CI_low"], lo,
        rtol=0, atol=0.00015
    )
    assert np.isclose(
        row["CI_high"], hi,
        rtol=0, atol=0.00015
    )

    excludes_zero = bool(
        (row["CI_low"] > 0)
        or (row["CI_high"] < 0)
    )
    assert excludes_zero

    verified.append({
        "comparison": comp,
        "metric": metric,
        "source_comparison": row["comparison"],
        "source_metric": row["metric"],
        "difference": float(row["difference"]),
        "ci_low": float(row["CI_low"]),
        "ci_high": float(row["CI_high"]),
        "nominal_ci_excludes_zero": excludes_zero
    })

result = pd.DataFrame(verified)

def write_csv_immutable(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False)
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing output differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

write_csv_immutable(result, CORRECTED)

report = {
    "step": "8.15a",
    "status": "PASS",
    "correction": (
        "Contrast names in source use _minus_ instead "
        "of hyphen notation; original 8.15 matching "
        "logic failed to identify them."
    ),
    "source_comparison_rows": len(df),
    "publication_comparison_rows": len(pub),
    "key_contrasts_expected": len(targets),
    "key_contrasts_verified": len(verified),
    "multiple_comparison_adjustment": False,
    "model_retraining": False,
    "bootstrap_rerun": False,
    "original_tables_modified": False,
    "source_sha256": sha(SOURCE),
    "existing_publication_table_sha256": sha(EXISTING),
    "verified_contrasts_sha256": sha(CORRECTED)
}

temp = REPORT.with_suffix(".tmp")
temp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
    temp.unlink()
else:
    temp.replace(REPORT)

print("=== STEP 8.15A CONTRAST VERIFICATION ===")
print("Status:", report["status"])
print("Total source contrasts:", len(df))
print("Key contrasts verified:", len(verified), "/", len(targets))

print("\n=== VERIFIED CONTRASTS ===")
print(result.to_string(index=False))

print("\n=== SOURCE PRESERVATION ===")
print("Original STEP 6 table modified: NO")
print("Original STEP 8.15 publication table modified: NO")
print("Bootstrap rerun: NO")

print("\n=== SAVED FILES ===")
print("Corrected contrast CSV:", CORRECTED)
print("Verification report:", REPORT)
print("Report SHA256:", sha(REPORT))


In [ ]:

from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np

P = Path("/kaggle/working/acoustic_linguistic_research")
SRC = P / "results/step06_final_tables"
OUT = P / "results/step08_publication_tables"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE = SRC / "table03_anomaly_sensitivity.csv"
PAIRED = SRC / "table04_sensitivity_paired_differences.csv"
MAIN = SRC / "table01_primary_models_with_ci.csv"

TABLE = OUT / "table03_anomaly_sensitivity_publication.csv"
REPORT = OUT / "step08_16_anomaly_sensitivity_manifest.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path in [SOURCE, PAIRED, MAIN]:
    assert path.is_file(), f"Missing source: {path}"

data = pd.read_csv(SOURCE)
paired = pd.read_csv(PAIRED)
main = pd.read_csv(MAIN)

required = [
    "condition", "model", "audio_count",
    "hypothesis_count", "speaker_count",
    "MAE", "RMSE", "mean_prediction_range",
    "median_prediction_range", "p95_prediction_range"
]
assert set(required).issubset(data.columns)

assert not data.duplicated(["condition", "model"]).any()

for col in ["audio_count", "hypothesis_count", "speaker_count"]:
    data[col] = pd.to_numeric(data[col], errors="raise")
    assert data[col].notna().all()
    assert (data[col] > 0).all()
    assert (data[col] % 1 == 0).all()

for col in [
    "MAE", "RMSE", "mean_prediction_range",
    "median_prediction_range", "p95_prediction_range"
]:
    data[col] = pd.to_numeric(data[col], errors="raise")
    assert np.isfinite(data[col]).all()
    assert data[col].ge(0).all()

assert data["audio_count"].le(400).all()
assert data["speaker_count"].le(20).all()

# Five ASR hypotheses per recording are retained
# within each reported sensitivity condition.
assert (
    data["hypothesis_count"]
    == 5 * data["audio_count"]
).all()

# Verify that the original full sample can be
# identified by counts rather than guessed labels.
full = data.loc[
    (data["audio_count"] == 400)
    & (data["hypothesis_count"] == 2000)
    & (data["speaker_count"] == 20)
]

assert not full.empty, "No full-sample condition identified"

full_conditions = full["condition"].unique().tolist()
assert len(full_conditions) == 1, (
    f"Multiple full-sample conditions: {full_conditions}"
)

full_label = full_conditions[0]
assert set(["A", "B1", "B2", "C1", "C2"]).issubset(
    set(main["model"])
)

# Match full-sample sensitivity rows against
# frozen original metrics wherever present.
checked_models = []

for row in full.itertuples(index=False):
    matching = main.loc[main["model"] == row.model]

    if len(matching) != 1:
        continue

    frozen = matching.iloc[0]

    assert np.isclose(
        row.MAE, float(frozen["MAE"]),
        rtol=0, atol=1e-6
    ), f"MAE mismatch for {row.model}"

    assert np.isclose(
        row.RMSE, float(frozen["RMSE"]),
        rtol=0, atol=1e-6
    ), f"RMSE mismatch for {row.model}"

    checked_models.append(row.model)

# Keep original condition names and exact source
# point estimates, only format publication numbers.
publication = data.copy()

publication["MAE"] = publication["MAE"].round(4)
publication["RMSE"] = publication["RMSE"].round(4)
publication["mean_prediction_range"] = (
    publication["mean_prediction_range"].round(4)
)
publication["median_prediction_range"] = (
    publication["median_prediction_range"].round(4)
)
publication["p95_prediction_range"] = (
    publication["p95_prediction_range"].round(4)
)

condition_order = data["condition"].drop_duplicates().tolist()
model_order = ["A", "B1", "B2", "C1", "C2"]

publication["condition"] = pd.Categorical(
    publication["condition"],
    categories=condition_order,
    ordered=True
)
publication["model"] = pd.Categorical(
    publication["model"],
    categories=model_order,
    ordered=True
)

# If other model IDs exist, avoid dropping them.
extra_models = sorted(
    set(data["model"]) - set(model_order)
)
if extra_models:
    publication["model"] = pd.Categorical(
        data["model"],
        categories=model_order + extra_models,
        ordered=True
    )

publication = publication.sort_values(
    ["condition", "model"]
).reset_index(drop=True)

publication["condition"] = publication["condition"].astype(str)
publication["model"] = publication["model"].astype(str)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing publication table differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(publication, TABLE)

# Validate paired-sensitivity source structure,
# without generating new inferential intervals.
assert set([
    "condition", "comparison", "MAE_difference",
    "mean_range_difference"
]).issubset(paired.columns)

report = {
    "step": "8.16",
    "status": "PASS",
    "publication_rows": len(publication),
    "conditions": condition_order,
    "models": sorted(data["model"].unique().tolist()),
    "full_sample_condition": full_label,
    "full_sample_models_checked": checked_models,
    "source_hashes": {
        str(path.relative_to(P)): sha(path)
        for path in [SOURCE, PAIRED, MAIN]
    },
    "table_sha256": sha(TABLE),
    "original_results_modified": False,
    "models_retrained": False,
    "exclusions_posthoc": True,
    "interpretation_note": (
        "The full 400-recording dataset remains the "
        "primary analysis. Exclusions are post-hoc "
        "sensitivity checks and are not independent "
        "confirmatory analyses."
    )
}

temp = REPORT.with_suffix(".tmp")
temp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing manifest differs"
    temp.unlink()
else:
    temp.replace(REPORT)

print("=== STEP 8.16 ANOMALY SENSITIVITY TABLE ===")
print("Status:", report["status"])
print("Conditions:", condition_order)
print("Publication rows:", len(publication))
print("Full sample condition:", full_label)
print("Frozen models cross-checked:", checked_models)

print("\n=== TABLE 3 — ANOMALY SENSITIVITY ===")
print(publication.to_string(index=False))

print("\n=== EXISTING PAIRED SENSITIVITY RESULTS ===")
print(paired.to_string(index=False))

print("\n=== SAVED FILES ===")
print("Publication CSV:", TABLE)
print("Manifest:", REPORT)
print("Manifest SHA256:", sha(REPORT))
print("Original frozen data modified: NO")
print("Bootstrap rerun: NO")


In [ ]:

from pathlib import Path
import json
import hashlib
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_publication_audit"
OUT.mkdir(parents=True, exist_ok=True)

FIG = P / "results/step08_figures"
TAB = P / "results/step08_publication_tables"
PROV = P / "results/step08_provenance"
SRC = P / "results/step06_final_tables"

MANIFEST = OUT / "step08_17_publication_artifact_audit.json"
INVENTORY = OUT / "step08_17_artifact_inventory.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

groups = {
    "figure02": [
        FIG / "figure02_model_performance_95ci.png",
        FIG / "figure02_model_performance_95ci.pdf",
        FIG / "figure02_model_performance_95ci.json",
    ],
    "figure03": [
        FIG / "figure03_accuracy_stability_tradeoff.png",
        FIG / "figure03_accuracy_stability_tradeoff.pdf",
        FIG / "figure03_accuracy_stability_tradeoff.json",
    ],
    "figure04": [
        FIG / "figure04_lexical_variation_prediction_range.png",
        FIG / "figure04_lexical_variation_prediction_range.pdf",
        FIG / "figure04_lexical_variation_prediction_range.json",
    ],
    "figure05": [
        FIG / "figure05_critical_asr_hallucination.png",
        FIG / "figure05_critical_asr_hallucination.pdf",
        FIG / "figure05_critical_asr_hallucination_data.csv",
        FIG / "figure05_critical_asr_hallucination.json",
    ],
    "figure05b": [
        FIG / "figure05b_h2_h1_feature_contributions.png",
        FIG / "figure05b_h2_h1_feature_contributions.pdf",
        FIG / "figure05b_feature_contribution_summary.csv",
        FIG / "figure05b_feature_contribution_manifest.json",
    ],
    "publication_tables": [
        TAB / "table02a_model_performance.csv",
        TAB / "table02b_paired_bootstrap_comparisons.csv",
        TAB / "step08_15a_verified_key_contrasts.csv",
        TAB / "table03_anomaly_sensitivity_publication.csv",
        TAB / "step08_16_anomaly_sensitivity_manifest.json",
    ],
    "provenance": [
        PROV / "step08_07b_source_reconstruction.json",
        PROV / "step08_04_corrected_metadata_audit.json",
        PROV / "step08_08_asr_batch_integrity.json",
        PROV / "step08_09_final_provenance_record.json",
    ],
    "frozen_sources": [
        SRC / "table01_primary_models_with_ci.csv",
        SRC / "table02_paired_differences_with_ci.csv",
        SRC / "table03_anomaly_sensitivity.csv",
        SRC / "step06_asr_variability_per_audio.csv",
        SRC / "step06_c2_critical_h2_vs_h1_contributions.csv",
    ],
}

rows = []
missing = []

for group, paths in groups.items():
    for path in paths:
        exists = path.is_file()
        if not exists:
            missing.append(str(path.relative_to(P)))

        rows.append({
            "group": group,
            "path": str(path.relative_to(P)),
            "exists": exists,
            "size_bytes": path.stat().st_size if exists else 0,
            "sha256": sha(path) if exists else "",
        })

inventory = pd.DataFrame(rows)

# Independently check saved figure manifests against
# their actual PNG, PDF and CSV artifacts.
figure_manifests = {
    "figure02": (
        FIG / "figure02_model_performance_95ci.json",
        FIG / "figure02_model_performance_95ci.png",
        FIG / "figure02_model_performance_95ci.pdf",
        None
    ),
    "figure03": (
        FIG / "figure03_accuracy_stability_tradeoff.json",
        FIG / "figure03_accuracy_stability_tradeoff.png",
        FIG / "figure03_accuracy_stability_tradeoff.pdf",
        None
    ),
    "figure04": (
        FIG / "figure04_lexical_variation_prediction_range.json",
        FIG / "figure04_lexical_variation_prediction_range.png",
        FIG / "figure04_lexical_variation_prediction_range.pdf",
        None
    ),
    "figure05": (
        FIG / "figure05_critical_asr_hallucination.json",
        FIG / "figure05_critical_asr_hallucination.png",
        FIG / "figure05_critical_asr_hallucination.pdf",
        FIG / "figure05_critical_asr_hallucination_data.csv"
    ),
    "figure05b": (
        FIG / "figure05b_feature_contribution_manifest.json",
        FIG / "figure05b_h2_h1_feature_contributions.png",
        FIG / "figure05b_h2_h1_feature_contributions.pdf",
        FIG / "figure05b_feature_contribution_summary.csv"
    )
}

checks = []

for name, (manifest, png, pdf, csv) in figure_manifests.items():
    if not all(
        x.is_file()
        for x in [manifest, png, pdf]
        + ([csv] if csv is not None else [])
    ):
        checks.append({"figure": name, "status": "MISSING"})
        continue

    meta = json.loads(manifest.read_text(encoding="utf-8"))
    ok = (
        meta["png_sha256"] == sha(png)
        and meta["pdf_sha256"] == sha(pdf)
    )

    if csv is not None:
        key = (
            "data_csv_sha256"
            if name == "figure05"
            else "csv_sha256"
        )
        ok = ok and meta[key] == sha(csv)

    checks.append({
        "figure": name,
        "status": "PASS" if ok else "HASH_MISMATCH"
    })

checks_df = pd.DataFrame(checks)

# Verify the most important frozen statistical values.
models = pd.read_csv(
    SRC / "table01_primary_models_with_ci.csv"
)
paired = pd.read_csv(
    SRC / "table02_paired_differences_with_ci.csv"
)
sensitivity = pd.read_csv(
    SRC / "table03_anomaly_sensitivity.csv"
)

assert len(models) == 5
assert len(paired) == 18
assert len(sensitivity) == 15

assert set(models["model"]) == {"A","B1","B2","C1","C2"}
assert set(sensitivity["audio_count"]) == {400, 399, 396}

# Check the already verified provenance reports.
provenance = json.loads(
    (PROV / "step08_09_final_provenance_record.json")
    .read_text(encoding="utf-8")
)

assert provenance["status"] == "PROVENANCE_DOCUMENTED"

def safe_csv(df, path):
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing inventory changed: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

safe_csv(inventory, INVENTORY)

status = (
    "PASS"
    if len(missing) == 0
    and checks_df["status"].eq("PASS").all()
    else "NEEDS_REVIEW"
)

report = {
    "step": "8.17",
    "status": status,
    "files_expected": len(inventory),
    "files_present": int(inventory["exists"].sum()),
    "missing": missing,
    "figure_integrity_checks": checks,
    "model_rows": len(models),
    "paired_contrast_rows": len(paired),
    "sensitivity_rows": len(sensitivity),
    "provenance_status": provenance["status"],
    "figure01_workflow_created": False,
    "original_results_modified": False,
    "models_retrained": False,
    "inventory_sha256": sha(INVENTORY)
}

tmp = MANIFEST.with_suffix(".tmp")
tmp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == report, "Existing audit manifest differs"
    tmp.unlink()
else:
    tmp.replace(MANIFEST)

print("=== STEP 8.17 PUBLICATION ARTIFACT AUDIT ===")
print("Status:", status)
print("Files expected:", len(inventory))
print("Files present:", int(inventory["exists"].sum()))
print("Missing files:", len(missing))

print("\n=== FIGURE INTEGRITY ===")
print(checks_df.to_string(index=False))

print("\n=== FROZEN TABLE INTEGRITY ===")
print("Model rows:", len(models))
print("Paired contrast rows:", len(paired))
print("Sensitivity rows:", len(sensitivity))
print("Provenance:", provenance["status"])

print("\n=== REMAINING PUBLICATION WORK ===")
print("Figure 1 research workflow: NOT YET CREATED")
print("Captions and manuscript integration: NOT YET FINALIZED")

print("\n=== SAVED FILES ===")
print("Inventory:", INVENTORY)
print("Audit:", MANIFEST)
print("Audit SHA256:", sha(MANIFEST))
print("Original predictions modified: NO")


In [ ]:

from pathlib import Path
import json
import hashlib

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure01_research_workflow.png"
PDF = OUT / "figure01_research_workflow.pdf"
MANIFEST = OUT / "figure01_research_workflow.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def immutable_save(path, writer):
    temp = path.with_name(path.stem + ".temporary" + path.suffix)
    writer(temp)
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing artifact differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

fig, ax = plt.subplots(figsize=(13.4, 10.0))
ax.set_xlim(0, 13.4)
ax.set_ylim(0, 10)
ax.axis("off")

def box(x, y, w, h, heading, details,
        face="#EDF3F7", edge="#61788A",
        fontsize=10):
    rect = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.12,rounding_size=0.16",
        linewidth=1.25,
        edgecolor=edge,
        facecolor=face
    )
    ax.add_patch(rect)
    ax.text(
        x + w/2, y + h*0.68, heading,
        ha="center", va="center",
        fontsize=fontsize + 1,
        fontweight="bold",
        color="#213343"
    )
    ax.text(
        x + w/2, y + h*0.32, details,
        ha="center", va="center",
        fontsize=fontsize - 0.6,
        color="#35485A",
        linespacing=1.28
    )

def arrow(x1, y1, x2, y2, color="#526A7B"):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1), (x2, y2),
            arrowstyle="-|>",
            mutation_scale=14,
            linewidth=1.5,
            color=color,
            connectionstyle="arc3,rad=0"
        )
    )

# 1. Source dataset.
box(
    3.8, 8.65, 5.8, 1.0,
    "SpeechOcean762 — Training Partition",
    "2,500 recordings / 125 speakers\n"
    "Official test partition reserved"
)
arrow(6.7, 8.55, 6.7, 8.13)

# 2. Pilot selection.
box(
    3.8, 7.1, 5.8, 0.95,
    "Speaker-Level Pilot Selection",
    "20 speakers × 20 recordings = 400 audios\n"
    "Recorded seed = 42"
)
arrow(6.7, 7.0, 6.7, 6.58)

# 3. Parallel extraction streams.
box(
    0.55, 5.3, 3.65, 1.12,
    "Acoustic Features",
    "14 audio-derived variables\n"
    "VAD, pauses, pitch, RMS",
    face="#E8F2ED",
    edge="#5B8876"
)

box(
    4.9, 5.3, 3.65, 1.12,
    "ASR Transcriptions",
    "Whisper small.en\n"
    "H1–H5 per recording",
    face="#E8EFF9",
    edge="#56799E"
)

box(
    9.2, 5.3, 3.65, 1.12,
    "Reading Prompt",
    "Reference text available\n"
    "in read-aloud task",
    face="#F8F0E3",
    edge="#A68954"
)

arrow(5.0, 7.0, 2.4, 6.52)
arrow(6.7, 7.0, 6.7, 6.52)
arrow(8.4, 7.0, 11.0, 6.52)

# 4. Feature group preparation.
box(
    0.55, 3.65, 3.65, 1.0,
    "A: Acoustic Features",
    "14 features / recording",
    face="#E8F2ED",
    edge="#5B8876"
)

box(
    4.9, 3.65, 3.65, 1.0,
    "B: Linguistic Features",
    "7 ASR-derived features\n"
    "per hypothesis",
    face="#E8EFF9",
    edge="#56799E"
)

box(
    9.2, 3.65, 3.65, 1.0,
    "Prompt-Aware Features",
    "2 prompt–ASR disagreement\n"
    "measures (not WER/CER)",
    face="#F8F0E3",
    edge="#A68954"
)

arrow(2.4, 5.2, 2.4, 4.78)
arrow(6.7, 5.2, 6.7, 4.78)
arrow(11.0, 5.2, 11.0, 4.78)
arrow(8.7, 5.85, 9.05, 4.23)

# 5. Model configurations.
box(
    0.6, 2.05, 12.2, 1.0,
    "Model Configurations",
    "M0: mean baseline   |   A: acoustic   |   B1: linguistic   |   "
    "C1: acoustic + linguistic\n"
    "Secondary: B2 = linguistic + prompt; C2 = full fusion + prompt",
    face="#F0F1F3",
    edge="#777F88",
    fontsize=9.8
)

arrow(2.4, 3.55, 2.4, 3.17)
arrow(6.7, 3.55, 6.7, 3.17)
arrow(11.0, 3.55, 11.0, 3.17)

# 6. Evaluation.
box(
    0.6, 0.45, 5.7, 0.95,
    "Five-Fold Speaker-Disjoint CV",
    "Ridge alpha=1.0; fold-only preprocessing\n"
    "All H1–H5 of each audio remain together",
    face="#EEF1F5",
    edge="#67798B",
    fontsize=9.5
)

box(
    7.05, 0.45, 5.75, 0.95,
    "Performance & Robustness Analysis",
    "MAE, RMSE, H1–H5 prediction range\n"
    "Speaker-cluster bootstrap + sensitivity",
    face="#EEF1F5",
    edge="#67798B",
    fontsize=9.5
)

arrow(6.7, 1.95, 3.45, 1.52)
arrow(6.3, 0.92, 6.92, 0.92)

fig.suptitle(
    "Research Workflow: ASR Variability and Human-Aligned Fluency Scoring",
    fontsize=16,
    fontweight="bold",
    y=0.995
)

fig.tight_layout(rect=[0.015, 0.015, 0.985, 0.965])

immutable_save(
    PNG,
    lambda path: fig.savefig(
        path, dpi=300, bbox_inches="tight"
    )
)
immutable_save(
    PDF,
    lambda path: fig.savefig(
        path,
        bbox_inches="tight",
        metadata={
            "Title": "Research Workflow",
            "Creator": "matplotlib",
            "CreationDate": None
        }
    )
)

plt.show()
plt.close(fig)

manifest = {
    "figure": "Figure 1",
    "status": "CREATED",
    "title": (
        "Research Workflow: ASR Variability and "
        "Human-Aligned Fluency Scoring"
    ),
    "dataset": "mispeech/speechocean762",
    "dataset_split": "train",
    "source_recordings": 2500,
    "source_speakers": 125,
    "selected_recordings": 400,
    "selected_speakers": 20,
    "asr_hypotheses_per_audio": 5,
    "feature_counts": {
        "acoustic": 14,
        "linguistic": 7,
        "prompt_aware": 2
    },
    "models": ["M0", "A", "B1", "B2", "C1", "C2"],
    "validation": "five-fold speaker-disjoint CV",
    "primary_alpha": 1.0,
    "primary_metrics": ["MAE", "RMSE"],
    "stability_metric": "prediction range across H1-H5",
    "bootstrap_clusters": "speaker",
    "interpretation_notes": [
        "Prompt-aware models require reading prompts.",
        "Prompt–ASR disagreement is not measured WER or CER.",
        "The official test partition was not used.",
        "Acoustic-only A is ASR-invariant by design.",
        "Figure illustrates the workflow, not causal inference."
    ],
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF)
}

immutable_save(
    MANIFEST,
    lambda path: path.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
)

print("\n=== STEP 8.18 FIGURE 1 ===")
print("Status: PASS")
print("Dataset: SpeechOcean762 train")
print("Recordings: 400")
print("Speakers: 20")
print("ASR hypotheses: 2000")
print("Models: M0, A, B1, B2, C1, C2")

print("\n=== OUTPUT FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original predictions modified: NO")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile
import io

P = Path("/kaggle/working/acoustic_linguistic_research")
ROOT = Path("/kaggle/working")

ARCHIVE = ROOT / "asr_fluency_step08_publication_complete.zip"
MANIFEST_NAME = "step08_backup_manifest.json"

FOLDERS = [
    P / "results/step08_provenance",
    P / "results/step08_figures",
    P / "results/step08_publication_tables",
    P / "results/step08_publication_audit",
    P / "results/step06_final_tables"
]

EXTRA = [
    P / "configs/asr_hypothesis_pilot_v1.json",
    P / "configs/speaker_disjoint_cv_v1.json",
    P / "checkpoints/pilot_speaker_ids.json",
    P / "data/metadata/pilot_selection_400.csv"
]

def sha_bytes(content):
    return hashlib.sha256(content).hexdigest()

def sha_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

files = {}

for folder in FOLDERS:
    assert folder.is_dir(), f"Missing folder: {folder}"
    for path in folder.rglob("*"):
        if path.is_file():
            files[str(path.relative_to(P))] = path

for path in EXTRA:
    assert path.is_file(), f"Missing file: {path}"
    files[str(path.relative_to(P))] = path

assert len(files) > 30

# Mandatory completed artifacts.
required = [
    "results/step08_figures/figure01_research_workflow.png",
    "results/step08_figures/figure02_model_performance_95ci.png",
    "results/step08_figures/figure03_accuracy_stability_tradeoff.png",
    "results/step08_figures/figure04_lexical_variation_prediction_range.png",
    "results/step08_figures/figure05_critical_asr_hallucination.png",
    "results/step08_figures/figure05b_h2_h1_feature_contributions.png",
    "results/step08_publication_tables/table02a_model_performance.csv",
    "results/step08_publication_tables/table02b_paired_bootstrap_comparisons.csv",
    "results/step08_publication_tables/table03_anomaly_sensitivity_publication.csv",
    "results/step08_provenance/step08_09_final_provenance_record.json",
    "results/step08_publication_audit/step08_17_publication_artifact_audit.json"
]

missing = [name for name in required if name not in files]
assert not missing, f"Missing required artifacts: {missing}"

entries = []

for name, path in sorted(files.items()):
    entries.append({
        "path": name,
        "size_bytes": path.stat().st_size,
        "sha256": sha_file(path)
    })

manifest = {
    "archive_purpose": "STEP 8 publication outputs and provenance",
    "project": "Within-Utterance ASR Transcription Variability",
    "status": "READY_FOR_ARCHIVE",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "file_count": len(entries),
    "files": entries,
    "exclusions": [
        "Raw audio and large source parquet files",
        "Full STEP 1-7 research backup (stored separately)",
        "Kaggle notebook source code (export separately)"
    ],
    "important_notes": [
        "Primary modeling results are frozen.",
        "STEP 8 only generated publication outputs and audits.",
        "Prompt-aware models are secondary ablations.",
        "No external test-split validation was performed.",
        "Original ASR runtime was not independently replayed."
    ]
}

manifest_bytes = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
).encode("utf-8")

# Write to temporary archive before replacing final path.
TEMP = ROOT / "asr_fluency_step08_publication_complete.tmp.zip"

with zipfile.ZipFile(
    TEMP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as z:
    for name, path in sorted(files.items()):
        z.write(path, arcname=name)

    z.writestr(MANIFEST_NAME, manifest_bytes)

# Verify archive against the independent file manifest.
with zipfile.ZipFile(TEMP, "r") as z:
    assert z.testzip() is None

    names = set(z.namelist())
    assert len(names) == len(entries) + 1
    assert MANIFEST_NAME in names

    restored_manifest = json.loads(
        z.read(MANIFEST_NAME).decode("utf-8")
    )

    assert restored_manifest["file_count"] == len(entries)

    for entry in restored_manifest["files"]:
        archived = z.read(entry["path"])
        assert len(archived) == entry["size_bytes"]
        assert sha_bytes(archived) == entry["sha256"]

TEMP.replace(ARCHIVE)

assert ARCHIVE.is_file()

# Verify final archive is readable.
with zipfile.ZipFile(ARCHIVE, "r") as z:
    assert z.testzip() is None

print("=== STEP 8.19 PUBLICATION BACKUP ===")
print("Status: VERIFIED")
print("Files archived:", len(entries))
print("ZIP entries including manifest:", len(entries) + 1)
print("Required publication artifacts:", len(required))
print("All archive file hashes: PASS")

print("\n=== ARCHIVE ===")
print("Path:", ARCHIVE)
print("Size MB:", round(ARCHIVE.stat().st_size / (1024**2), 2))
print("SHA256:", sha_file(ARCHIVE))

print("\n=== PRESERVATION ===")
print("STEP 1-7 archive modified: NO")
print("Original modeling datasets modified: NO")
print("Raw audio included: NO")
print("Kaggle notebook included: NO")
print("Next: download ZIP and export notebook separately.")
